In [ ]:
from google.colab import files

uploaded = files.upload()

Saving VIM2_Part13_Feature_Manifest.csv to VIM2_Part13_Feature_Manifest.csv


In [ ]:
# @title
# ============================================================
# PART 12H-v2
# 12D → FINAL ML-READY FEATURE TABLE
# ============================================================
#
# Purpose:
#   Convert the comprehensive Part 12D dataset into a clean,
#   standardized, leakage-aware ML input table.
#
# Key design principles:
#   1. Start explicitly from Part 12D.
#   2. Preserve all biologically meaningful ML predictors.
#   3. Generate WT and mutant amino-acid one-hot features.
#   4. Keep Secondary_Structure as the single secondary-structure
#      annotation.
#   5. Remove redundant DSSP_Secondary_Structure and
#      Secondary_Structure_Code.
#   6. Preserve all phenotype targets and their SD columns.
#   7. Do NOT use phenotype SD columns as predictors.
#   8. Preserve Sequence_Position for later position-aware ML.
#   9. Do NOT perform global imputation.
#  10. Perform extensive QC before saving.
#
# Input:
#   VIM2_Part12D_*.csv
#
# Output:
#   VIM2_Part12H_v2_Final_ML_Dataset.csv
#   VIM2_Part12H_v2_QC_Report.csv
#   VIM2_Part12H_v2_Feature_Manifest.csv
#
# ============================================================


# ============================================================
# 0. IMPORT LIBRARIES
# ============================================================

import os
import glob
import re
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 70)
print("PART 12H-v2 — 12D → FINAL ML FEATURE TABLE")
print("=" * 70)

print("\nLibraries imported successfully.")


# ============================================================
# 1. PATH CONFIGURATION
# ============================================================
#
# IMPORTANT:
# The script does NOT request an upload.
#
# It searches the Colab /content directory for the Part 12D
# dataset automatically.
#
# If multiple 12D files exist, the script prioritizes names
# containing "Part12D".
# ============================================================

BASE_DIR = "/content"

OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "VIM2_Part12H_v2_Final_ML_Dataset.csv"
)

QC_OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "VIM2_Part12H_v2_QC_Report.csv"
)

MANIFEST_OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "VIM2_Part12H_v2_Feature_Manifest.csv"
)


# ------------------------------------------------------------
# Search for Part 12D input
# ------------------------------------------------------------

candidate_files = sorted(
    glob.glob(
        os.path.join(
            BASE_DIR,
            "*12D*.csv"
        )
    )
)

if len(candidate_files) == 0:

    raise FileNotFoundError(
        "\nNo Part 12D CSV file was found in /content.\n"
        "Please make sure the Part 12D CSV exists in the "
        "specified Colab path."
    )


# Prefer files explicitly containing "VIM2" and "Part12D"

preferred_files = [
    f for f in candidate_files
    if "VIM2" in os.path.basename(f)
    and "Part12D" in os.path.basename(f)
]

if len(preferred_files) > 0:
    input_file = preferred_files[0]
else:
    input_file = candidate_files[0]


print("\nInput file selected:")
print(os.path.basename(input_file))
print("Full path:")
print(input_file)


# ============================================================
# 2. LOAD PART 12D
# ============================================================

df = pd.read_csv(
    input_file,
    low_memory=False
)

print("\nDataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")


# ------------------------------------------------------------
# Remove accidental unnamed index columns if present
# ------------------------------------------------------------

unnamed_columns = [
    col for col in df.columns
    if str(col).strip().lower().startswith("unnamed:")
]

if unnamed_columns:

    print("\nRemoving accidental index columns:")
    for col in unnamed_columns:
        print(f"  - {col}")

    df = df.drop(
        columns=unnamed_columns
    )


# ============================================================
# 3. STANDARDIZE COLUMN NAMES
# ============================================================
#
# We preserve the biological names but remove accidental
# leading/trailing whitespace.
# ============================================================

df.columns = [
    str(col).strip()
    for col in df.columns
]


# ============================================================
# 4. DEFINE CORE IDENTIFIERS
# ============================================================

IDENTIFIER_COLUMNS = [
    "WT_AA",
    "position",
    "Mutant_AA",
    "identity",
    "group"
]


# ============================================================
# 5. DEFINE PHENOTYPE TARGETS
# ============================================================
#
# These are the experimentally measured phenotypes.
#
# The SD columns are deliberately kept in the final dataset
# because they contain useful experimental uncertainty information.
#
# HOWEVER:
#   SD columns will NOT be used as ML predictors.
# ============================================================

TARGET_COLUMNS = [
    "128ug/mL_AMP_25C",
    "16ug/mL_AMP_25C",
    "2ug/mL_AMP_25C",

    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_37C",

    "4ug/mL_CTX_37C",
    "0.5ug/mL_CTX_37C",

    "0.031ug/mL_MEM_37C"
]


TARGET_SD_COLUMNS = [
    "128ug/mL_AMP_25C_SD",
    "16ug/mL_AMP_25C_SD",
    "2ug/mL_AMP_25C_SD",

    "128ug/mL_AMP_37C_SD",
    "16ug/mL_AMP_37C_SD",
    "2ug/mL_AMP_37C_SD",

    "4ug/mL_CTX_37C_SD",
    "0.5ug/mL_CTX_37C_SD",

    "0.031ug/mL_MEM_37C_SD"
]


# ============================================================
# 6. DEFINE AMINO-ACID ALPHABET
# ============================================================
#
# Standard 20 amino acids.
#
# The ordering is fixed to guarantee reproducibility.
# ============================================================

AMINO_ACIDS = [
    "A", "C", "D", "E", "F",
    "G", "H", "I", "K", "L",
    "M", "N", "P", "Q", "R",
    "S", "T", "V", "W", "Y"
]


WT_ONEHOT_COLUMNS = [
    f"WT_AA_{aa}"
    for aa in AMINO_ACIDS
]

MUTANT_ONEHOT_COLUMNS = [
    f"Mutant_AA_{aa}"
    for aa in AMINO_ACIDS
]


# ============================================================
# 7. DEFINE BIOLOGICAL FEATURE GROUPS
# ============================================================

PHYSICOCHEMICAL_FEATURES = [
    "Hydrophobicity_Change",
    "Weight_Change",
    "Charge_Change",
    "Polarity_Change",
    "BLOSUM62"
]


SIDECHAIN_FEATURES = [
    "Side_Chain_Volume_Change",
    "Absolute_Side_Chain_Volume_Change",
    "Relative_Side_Chain_Volume_Change"
]


HBOND_FEATURES = [
    "HBond_Donor_Change",
    "HBond_Acceptor_Change",
    "Total_HBond_Capacity_Change"
]


STRUCTURAL_FEATURES = [
    "Distance_to_Metal_Site",
    "Distance_to_Active_Site_Pocket",
    "Distance_to_L3_Loop",
    "Distance_to_L10_Loop",
    "SASA"
]


# ------------------------------------------------------------
# Secondary structure
#
# IMPORTANT:
# We keep ONLY the biologically interpretable annotation:
#
#   Secondary_Structure
#
# We intentionally remove:
#
#   DSSP_Secondary_Structure
#   Secondary_Structure_Code
#
# The categorical variable will be encoded later INSIDE the
# ML training pipeline, avoiding arbitrary numerical ordering.
# ------------------------------------------------------------

SECONDARY_STRUCTURE_FEATURE = [
    "Secondary_Structure"
]


LOCAL_SEQUENCE_FEATURES = [
    "Local_Hydrophobic_Fraction",
    "Local_Charged_Fraction",
    "Local_Positive_Charge_Fraction",
    "Local_Negative_Charge_Fraction",
    "Local_Polar_Fraction",
    "Local_Aromatic_Fraction",
    "Local_Gly_Pro_Fraction",
    "Local_Sequence_Entropy"
]


# ============================================================
# 8. CHECK REQUIRED SOURCE COLUMNS
# ============================================================

required_source_columns = (
    IDENTIFIER_COLUMNS
    + TARGET_COLUMNS
    + TARGET_SD_COLUMNS
    + PHYSICOCHEMICAL_FEATURES
    + SIDECHAIN_FEATURES
    + HBOND_FEATURES
    + STRUCTURAL_FEATURES
    + SECONDARY_STRUCTURE_FEATURE
    + LOCAL_SEQUENCE_FEATURES
)

missing_source_columns = [
    col for col in required_source_columns
    if col not in df.columns
]


if missing_source_columns:

    print("\n" + "=" * 70)
    print("MISSING SOURCE COLUMNS")
    print("=" * 70)

    for col in missing_source_columns:
        print(f"  - {col}")

    raise ValueError(
        "\nPart 12D does not contain all required columns."
    )


print("\nAll required Part 12D source columns are available.")


# ============================================================
# 9. REMOVE REDUNDANT SECONDARY-STRUCTURE ANNOTATIONS
# ============================================================
#
# These two columns are deliberately excluded:
#
#   DSSP_Secondary_Structure
#   Secondary_Structure_Code
#
# Rationale:
#
#   Secondary_Structure:
#       retained as the primary biological annotation.
#
#   DSSP_Secondary_Structure:
#       redundant with the selected secondary-structure annotation.
#
#   Secondary_Structure_Code:
#       numerical encoding of the same categorical annotation and
#       can introduce artificial ordinal relationships.
# ============================================================

REDUNDANT_COLUMNS = [
    "DSSP_Secondary_Structure",
    "Secondary_Structure_Code"
]


existing_redundant_columns = [
    col for col in REDUNDANT_COLUMNS
    if col in df.columns
]


if existing_redundant_columns:

    print("\nRemoving redundant secondary-structure columns:")

    for col in existing_redundant_columns:
        print(f"  - {col}")

    df = df.drop(
        columns=existing_redundant_columns
    )


# ============================================================
# 10. CONSTRUCT / VALIDATE WT AND MUTANT ONE-HOT FEATURES
# ============================================================
#
# IMPORTANT:
# Part 12D contains the original amino-acid identities:
#
#   WT_AA
#   Mutant_AA
#
# The 20+20 one-hot features are generated from these columns.
#
# This is the correct approach because Part 12D is the source
# of truth for amino-acid identity.
#
# We do NOT assume that the existing WT_AA_A etc. columns are
# already correct.
# ============================================================

print("\n" + "=" * 70)
print("GENERATING AMINO-ACID ONE-HOT FEATURES")
print("=" * 70)


# ------------------------------------------------------------
# Normalize amino-acid identity fields
# ------------------------------------------------------------

df["WT_AA"] = (
    df["WT_AA"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["Mutant_AA"] = (
    df["Mutant_AA"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# Validate amino-acid identities
# ------------------------------------------------------------

invalid_wt_values = sorted(
    set(df["WT_AA"]) - set(AMINO_ACIDS)
)

invalid_mutant_values = sorted(
    set(df["Mutant_AA"]) - set(AMINO_ACIDS)
)


if invalid_wt_values:

    raise ValueError(
        "\nInvalid WT amino-acid identities detected:\n"
        + "\n".join(
            f"  - {x}"
            for x in invalid_wt_values
        )
    )


if invalid_mutant_values:

    raise ValueError(
        "\nInvalid mutant amino-acid identities detected:\n"
        + "\n".join(
            f"  - {x}"
            for x in invalid_mutant_values
        )
    )


# ------------------------------------------------------------
# Generate WT one-hot matrix
# ------------------------------------------------------------

wt_onehot = pd.DataFrame(
    0,
    index=df.index,
    columns=WT_ONEHOT_COLUMNS,
    dtype=np.int8
)

for aa in AMINO_ACIDS:

    wt_onehot.loc[
        df["WT_AA"] == aa,
        f"WT_AA_{aa}"
    ] = 1


# ------------------------------------------------------------
# Generate mutant one-hot matrix
# ------------------------------------------------------------

mutant_onehot = pd.DataFrame(
    0,
    index=df.index,
    columns=MUTANT_ONEHOT_COLUMNS,
    dtype=np.int8
)

for aa in AMINO_ACIDS:

    mutant_onehot.loc[
        df["Mutant_AA"] == aa,
        f"Mutant_AA_{aa}"
    ] = 1


# ------------------------------------------------------------
# Remove any old one-hot columns from Part 12D
# ------------------------------------------------------------

old_onehot_columns = [
    col
    for col in (
        WT_ONEHOT_COLUMNS
        + MUTANT_ONEHOT_COLUMNS
    )
    if col in df.columns
]


if old_onehot_columns:

    df = df.drop(
        columns=old_onehot_columns
    )


# ------------------------------------------------------------
# Add regenerated one-hot features
# ------------------------------------------------------------

df = pd.concat(
    [
        df,
        wt_onehot,
        mutant_onehot
    ],
    axis=1
)


# ============================================================
# 11. ONE-HOT VALIDATION
# ============================================================

invalid_wt_rows = (
    df[WT_ONEHOT_COLUMNS]
    .sum(axis=1)
    .ne(1)
)

invalid_mutant_rows = (
    df[MUTANT_ONEHOT_COLUMNS]
    .sum(axis=1)
    .ne(1)
)


invalid_wt_count = int(
    invalid_wt_rows.sum()
)

invalid_mutant_count = int(
    invalid_mutant_rows.sum()
)


print("\nONE-HOT ENCODING VALIDATION")
print("=" * 70)

print(
    f"Rows with invalid WT one-hot encoding     : "
    f"{invalid_wt_count:,}"
)

print(
    f"Rows with invalid mutant one-hot encoding : "
    f"{invalid_mutant_count:,}"
)


if invalid_wt_count > 0 or invalid_mutant_count > 0:

    raise ValueError(
        "\nOne-hot generation failed. "
        "Every row must contain exactly one WT "
        "and one mutant amino-acid indicator."
    )


print(
    "One-hot amino-acid encoding is valid for all rows."
)


# ============================================================
# 12. DEFINE FINAL ML PREDICTOR SET
# ============================================================
#
# The final biological predictor architecture is:
#
#   20 WT amino-acid one-hot features
#   20 mutant amino-acid one-hot features
#   5 physicochemical features
#   3 side-chain features
#   3 hydrogen-bond features
#   5 structural-context features
#   1 secondary-structure feature
#   8 local sequence features
#   1 sequence-position feature
#
# Total:
#
#   20 + 20 + 5 + 3 + 3 + 5 + 1 + 8 + 1
#   = 66 predictors
#
# IMPORTANT:
# Sequence_Position is retained in the dataset.
#
# Whether it is included in the model will be decided later:
#
#   Model A → without position
#   Model B → with position
#
# This is intentional.
# ============================================================

BIOLOGICAL_PREDICTORS_WITH_POSITION = (
    WT_ONEHOT_COLUMNS
    + MUTANT_ONEHOT_COLUMNS
    + PHYSICOCHEMICAL_FEATURES
    + SIDECHAIN_FEATURES
    + HBOND_FEATURES
    + STRUCTURAL_FEATURES
    + SECONDARY_STRUCTURE_FEATURE
    + LOCAL_SEQUENCE_FEATURES
    + ["Sequence_Position"]
)


BIOLOGICAL_PREDICTORS_WITHOUT_POSITION = [
    col
    for col in BIOLOGICAL_PREDICTORS_WITH_POSITION
    if col != "Sequence_Position"
]


print("\n" + "=" * 70)
print("FINAL PREDICTOR ARCHITECTURE")
print("=" * 70)

print(
    f"Predictors with position    : "
    f"{len(BIOLOGICAL_PREDICTORS_WITH_POSITION)}"
)

print(
    f"Predictors without position : "
    f"{len(BIOLOGICAL_PREDICTORS_WITHOUT_POSITION)}"
)


# ============================================================
# 13. VERIFY EXPECTED FEATURE COUNTS
# ============================================================

expected_feature_counts = {

    "WT amino-acid one-hot":
        len(WT_ONEHOT_COLUMNS),

    "Mutant amino-acid one-hot":
        len(MUTANT_ONEHOT_COLUMNS),

    "Physicochemical":
        len(PHYSICOCHEMICAL_FEATURES),

    "Side-chain":
        len(SIDECHAIN_FEATURES),

    "Hydrogen-bond":
        len(HBOND_FEATURES),

    "Structural-context":
        len(STRUCTURAL_FEATURES),

    "Secondary-structure":
        len(SECONDARY_STRUCTURE_FEATURE),

    "Local sequence":
        len(LOCAL_SEQUENCE_FEATURES),

    "Position":
        1
}


print("\nFeature-group counts:")

for group, count in expected_feature_counts.items():

    print(
        f"  {group:<30}: {count}"
    )


# ============================================================
# 14. TARGET AND SD VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("TARGET / SD VALIDATION")
print("=" * 70)


missing_targets = [
    col
    for col in TARGET_COLUMNS
    if col not in df.columns
]

missing_sd = [
    col
    for col in TARGET_SD_COLUMNS
    if col not in df.columns
]


if missing_targets:

    raise ValueError(
        "\nMissing phenotype targets:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_targets
        )
    )


if missing_sd:

    raise ValueError(
        "\nMissing phenotype SD columns:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_sd
        )
    )


print(
    f"Phenotype targets available : "
    f"{len(TARGET_COLUMNS)} / {len(TARGET_COLUMNS)}"
)

print(
    f"Phenotype SD columns        : "
    f"{len(TARGET_SD_COLUMNS)} / {len(TARGET_SD_COLUMNS)}"
)


# ============================================================
# 15. STRUCTURAL MISSINGNESS DIAGNOSTICS
# ============================================================
#
# IMPORTANT:
#
# We intentionally DO NOT impute missing structural values here.
#
# These values may be missing because structural coordinates are
# unresolved for specific residues.
#
# Global imputation at this stage would leak information and could
# distort the biological meaning of the structural features.
#
# Missing-value handling will occur later inside the ML pipeline
# using training data only.
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURAL MISSINGNESS DIAGNOSTICS")
print("=" * 70)


structural_missingness_records = []


for feature in STRUCTURAL_FEATURES:

    missing_count = int(
        df[feature].isna().sum()
    )

    missing_percentage = (
        missing_count
        / len(df)
        * 100
    )

    zero_count = int(
        df[feature].eq(0).sum()
    )

    structural_missingness_records.append(
        {
            "Feature": feature,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(
                missing_percentage,
                3
            ),
            "Zero_Count": zero_count
        }
    )

    print(
        f"{feature:<38} | "
        f"Missing: {missing_count:4d} "
        f"({missing_percentage:6.2f}%) | "
        f"Zero: {zero_count:4d}"
    )


# ============================================================
# 16. POSITION QC
# ============================================================

print("\n" + "=" * 70)
print("POSITION QC")
print("=" * 70)


if "Sequence_Position" not in df.columns:

    raise ValueError(
        "\nSequence_Position is missing."
    )


df["Sequence_Position"] = pd.to_numeric(
    df["Sequence_Position"],
    errors="coerce"
)


position_missing = int(
    df["Sequence_Position"].isna().sum()
)

unique_positions = int(
    df["Sequence_Position"].nunique(
        dropna=True
    )
)


print(
    f"Missing Sequence_Position : "
    f"{position_missing:,}"
)

print(
    f"Unique residue positions  : "
    f"{unique_positions:,}"
)


if position_missing > 0:

    raise ValueError(
        "\nSequence_Position contains missing values."
    )


# ============================================================
# 17. MUTATION IDENTITY / DUPLICATE QC
# ============================================================
#
# The original Mutation column is not required in Part 12D,
# so we reconstruct a canonical mutation identifier from:
#
#   WT_AA + Sequence_Position + Mutant_AA
#
# Example:
#
#   M1A
#   F2C
#
# This gives us a robust mutation identity for duplicate checking.
# ============================================================

df["Mutation"] = (
    df["WT_AA"]
    + df["Sequence_Position"]
        .astype(int)
        .astype(str)
    + df["Mutant_AA"]
)


unique_mutations = int(
    df["Mutation"].nunique()
)

duplicate_rows = int(
    df.duplicated(
        subset=["Mutation"]
    ).sum()
)


print("\n" + "=" * 70)
print("MUTATION IDENTITY QC")
print("=" * 70)

print(
    f"Unique mutation IDs : "
    f"{unique_mutations:,}"
)

print(
    f"Duplicate mutation rows : "
    f"{duplicate_rows:,}"
)


if duplicate_rows > 0:

    raise ValueError(
        "\nDuplicate mutation identities detected."
    )


# ============================================================
# 18. TARGET COMPLETENESS REPORT
# ============================================================

print("\n" + "=" * 70)
print("TARGET COMPLETENESS")
print("=" * 70)


target_qc_records = []


for target in TARGET_COLUMNS:

    available = int(
        df[target].notna().sum()
    )

    missing = int(
        df[target].isna().sum()
    )

    missing_percentage = (
        missing
        / len(df)
        * 100
    )

    target_qc_records.append(
        {
            "Target": target,
            "Available": available,
            "Missing": missing,
            "Missing_Percentage": round(
                missing_percentage,
                3
            )
        }
    )

    print(
        f"{target:<30} | "
        f"Available: {available:4d} | "
        f"Missing: {missing:4d} "
        f"({missing_percentage:.3f}%)"
    )


# ============================================================
# 19. SD COMPLETENESS REPORT
# ============================================================

print("\n" + "=" * 70)
print("TARGET SD COMPLETENESS")
print("=" * 70)


for sd_col in TARGET_SD_COLUMNS:

    missing = int(
        df[sd_col].isna().sum()
    )

    available = int(
        df[sd_col].notna().sum()
    )

    print(
        f"{sd_col:<34} | "
        f"Available: {available:4d} | "
        f"Missing: {missing:4d}"
    )


# ============================================================
# 20. SECONDARY-STRUCTURE QC
# ============================================================

print("\n" + "=" * 70)
print("SECONDARY-STRUCTURE QC")
print("=" * 70)


secondary_missing = int(
    df["Secondary_Structure"].isna().sum()
)

secondary_unique = sorted(
    df["Secondary_Structure"]
    .dropna()
    .astype(str)
    .unique()
)


print(
    f"Missing Secondary_Structure : "
    f"{secondary_missing:,}"
)

print(
    "Observed Secondary_Structure categories:"
)

for value in secondary_unique:

    print(
        f"  - {value}"
    )


# ============================================================
# 21. CONVERT NUMERIC BIOLOGICAL FEATURES TO NUMERIC DTYPE
# ============================================================
#
# Secondary_Structure is intentionally excluded because it remains
# categorical.
#
# This is important:
# We do NOT force a biological categorical annotation into an
# arbitrary numerical scale.
# ============================================================

numeric_predictors = (
    WT_ONEHOT_COLUMNS
    + MUTANT_ONEHOT_COLUMNS
    + PHYSICOCHEMICAL_FEATURES
    + SIDECHAIN_FEATURES
    + HBOND_FEATURES
    + STRUCTURAL_FEATURES
    + LOCAL_SEQUENCE_FEATURES
    + ["Sequence_Position"]
)


print("\n" + "=" * 70)
print("NUMERIC FEATURE TYPE VALIDATION")
print("=" * 70)


non_numeric_predictors = []


for feature in numeric_predictors:

    converted = pd.to_numeric(
        df[feature],
        errors="coerce"
    )

    # If values were non-numeric strings, conversion creates NaN.
    # We distinguish genuine existing NaNs from conversion failures.

    original_nonmissing = (
        df[feature].notna()
    )

    conversion_failed = (
        original_nonmissing
        & converted.isna()
    )

    if conversion_failed.any():

        non_numeric_predictors.append(
            feature
        )

    df[feature] = converted


if non_numeric_predictors:

    print(
        "\nNon-numeric predictor columns detected:"
    )

    for feature in non_numeric_predictors:

        print(
            f"  - {feature}"
        )

    raise ValueError(
        "\nNumeric predictor validation failed."
    )


print(
    "All numeric ML predictors have valid numeric dtype."
)

print(
    "Secondary_Structure remains categorical by design."
)


# ============================================================
# 22. BUILD FINAL COLUMN ORDER
# ============================================================
#
# We explicitly control the final column order.
#
# This prevents accidental feature rearrangement and makes the
# dataset reproducible and easy to audit.
# ============================================================


FINAL_COLUMNS = (
    # --------------------------------------------------------
    # Mutation identity
    # --------------------------------------------------------
    [
        "Mutation",
        "WT_AA",
        "Sequence_Position",
        "Mutant_AA",
        "identity",
        "group"
    ]

    # --------------------------------------------------------
    # WT one-hot
    # --------------------------------------------------------
    + WT_ONEHOT_COLUMNS

    # --------------------------------------------------------
    # Mutant one-hot
    # --------------------------------------------------------
    + MUTANT_ONEHOT_COLUMNS

    # --------------------------------------------------------
    # Physicochemical mutation features
    # --------------------------------------------------------
    + PHYSICOCHEMICAL_FEATURES

    # --------------------------------------------------------
    # Structural mutation features
    # --------------------------------------------------------
    + SIDECHAIN_FEATURES

    # --------------------------------------------------------
    # Hydrogen-bond features
    # --------------------------------------------------------
    + HBOND_FEATURES

    # --------------------------------------------------------
    # Structural-context features
    # --------------------------------------------------------
    + STRUCTURAL_FEATURES

    # --------------------------------------------------------
    # Secondary structure
    # --------------------------------------------------------
    + SECONDARY_STRUCTURE_FEATURE

    # --------------------------------------------------------
    # Local sequence environment
    # --------------------------------------------------------
    + LOCAL_SEQUENCE_FEATURES

    # --------------------------------------------------------
    # Experimental phenotype targets
    # --------------------------------------------------------
    + TARGET_COLUMNS

    # --------------------------------------------------------
    # Experimental uncertainty / SD
    # --------------------------------------------------------
    + TARGET_SD_COLUMNS
)


# ------------------------------------------------------------
# Check all final columns exist
# ------------------------------------------------------------

missing_final_columns = [
    col
    for col in FINAL_COLUMNS
    if col not in df.columns
]


if missing_final_columns:

    raise ValueError(
        "\nThe following final columns are missing:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_final_columns
        )
    )


# ------------------------------------------------------------
# Construct final dataset
# ------------------------------------------------------------

final_df = df[
    FINAL_COLUMNS
].copy()


# ============================================================
# 23. FINAL DATASET DIMENSION CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET DIMENSIONS")
print("=" * 70)

print(
    f"Rows    : {final_df.shape[0]:,}"
)

print(
    f"Columns : {final_df.shape[1]:,}"
)


# ============================================================
# 24. VERIFY EXPECTED ML FEATURE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL ML FEATURE COUNT")
print("=" * 70)


print(
    f"WT amino-acid one-hot features     : "
    f"{len(WT_ONEHOT_COLUMNS)}"
)

print(
    f"Mutant amino-acid one-hot features : "
    f"{len(MUTANT_ONEHOT_COLUMNS)}"
)

print(
    f"Physicochemical features           : "
    f"{len(PHYSICOCHEMICAL_FEATURES)}"
)

print(
    f"Side-chain features                : "
    f"{len(SIDECHAIN_FEATURES)}"
)

print(
    f"Hydrogen-bond features             : "
    f"{len(HBOND_FEATURES)}"
)

print(
    f"Structural-context features        : "
    f"{len(STRUCTURAL_FEATURES)}"
)

print(
    f"Secondary-structure features       : "
    f"{len(SECONDARY_STRUCTURE_FEATURE)}"
)

print(
    f"Local sequence features             : "
    f"{len(LOCAL_SEQUENCE_FEATURES)}"
)

print(
    f"Position feature                    : 1"
)

print("-" * 70)

print(
    f"Model B predictors (with position)  : "
    f"{len(BIOLOGICAL_PREDICTORS_WITH_POSITION)}"
)

print(
    f"Model A predictors (without position): "
    f"{len(BIOLOGICAL_PREDICTORS_WITHOUT_POSITION)}"
)

print(
    f"Phenotype targets                    : "
    f"{len(TARGET_COLUMNS)}"
)

print(
    f"Phenotype SD columns                 : "
    f"{len(TARGET_SD_COLUMNS)}"
)


# ============================================================
# 25. VERIFY REDUNDANT FEATURES ARE ABSENT
# ============================================================

print("\n" + "=" * 70)
print("REDUNDANT FEATURE CHECK")
print("=" * 70)


remaining_redundant = [
    col
    for col in REDUNDANT_COLUMNS
    if col in final_df.columns
]


if remaining_redundant:

    raise ValueError(
        "\nRedundant secondary-structure columns remain:\n"
        + "\n".join(
            f"  - {x}"
            for x in remaining_redundant
        )
    )


print(
    "DSSP_Secondary_Structure        : removed"
)

print(
    "Secondary_Structure_Code        : removed"
)

print(
    "Secondary_Structure             : retained"
)


# ============================================================
# 26. VERIFY SD COLUMNS ARE NOT PREDICTORS
# ============================================================

print("\n" + "=" * 70)
print("TARGET / SD ROLE CHECK")
print("=" * 70)


sd_as_predictors = [
    col
    for col in TARGET_SD_COLUMNS
    if col in BIOLOGICAL_PREDICTORS_WITH_POSITION
]


target_as_predictors = [
    col
    for col in TARGET_COLUMNS
    if col in BIOLOGICAL_PREDICTORS_WITH_POSITION
]


if sd_as_predictors:

    raise ValueError(
        "\nTarget SD columns were incorrectly included "
        "as predictors."
    )


if target_as_predictors:

    raise ValueError(
        "\nPhenotype target columns were incorrectly included "
        "as predictors."
    )


print(
    "Phenotype targets are excluded from predictors."
)

print(
    "Phenotype SD columns are excluded from predictors."
)

print(
    "Both are retained in the final dataset for ML target handling."
)


# ============================================================
# 27. DUPLICATE CHECK ON FINAL DATASET
# ============================================================

final_duplicate_rows = int(
    final_df.duplicated().sum()
)

final_duplicate_mutations = int(
    final_df.duplicated(
        subset=["Mutation"]
    ).sum()
)


print("\n" + "=" * 70)
print("FINAL DUPLICATE CHECK")
print("=" * 70)

print(
    f"Duplicate complete rows : "
    f"{final_duplicate_rows:,}"
)

print(
    f"Duplicate mutation IDs  : "
    f"{final_duplicate_mutations:,}"
)


if final_duplicate_mutations > 0:

    raise ValueError(
        "\nDuplicate mutation IDs detected in final dataset."
    )


# ============================================================
# 28. FINAL MISSINGNESS SUMMARY
# ============================================================

missingness_records = []


for feature in final_df.columns:

    missing_count = int(
        final_df[feature].isna().sum()
    )

    missing_percentage = (
        missing_count
        / len(final_df)
        * 100
    )

    missingness_records.append(
        {
            "Feature": feature,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(
                missing_percentage,
                4
            )
        }
    )


missingness_df = pd.DataFrame(
    missingness_records
)


# ============================================================
# 29. BUILD FEATURE MANIFEST
# ============================================================
#
# This creates an auditable map of every feature and its role.
# ============================================================

manifest_records = []


for feature in FINAL_COLUMNS:

    if feature in IDENTIFIER_COLUMNS:
        role = "Identifier"

    elif feature == "Mutation":
        role = "Mutation_ID"

    elif feature in WT_ONEHOT_COLUMNS:
        role = "WT_Amino_Acid_OneHot_Predictor"

    elif feature in MUTANT_ONEHOT_COLUMNS:
        role = "Mutant_Amino_Acid_OneHot_Predictor"

    elif feature in PHYSICOCHEMICAL_FEATURES:
        role = "Physicochemical_Predictor"

    elif feature in SIDECHAIN_FEATURES:
        role = "SideChain_Predictor"

    elif feature in HBOND_FEATURES:
        role = "HydrogenBond_Predictor"

    elif feature in STRUCTURAL_FEATURES:
        role = "StructuralContext_Predictor"

    elif feature in SECONDARY_STRUCTURE_FEATURE:
        role = "SecondaryStructure_Categorical_Predictor"

    elif feature in LOCAL_SEQUENCE_FEATURES:
        role = "LocalSequence_Predictor"

    elif feature == "Sequence_Position":
        role = "Position_Predictor"

    elif feature in TARGET_COLUMNS:
        role = "Phenotype_Target"

    elif feature in TARGET_SD_COLUMNS:
        role = "Phenotype_SD_Metadata"

    else:
        role = "Other"

    manifest_records.append(
        {
            "Feature": feature,
            "Role": role,
            "Used_in_Model_A_No_Position":
                feature in BIOLOGICAL_PREDICTORS_WITHOUT_POSITION,
            "Used_in_Model_B_With_Position":
                feature in BIOLOGICAL_PREDICTORS_WITH_POSITION,
            "Missing_Count":
                int(final_df[feature].isna().sum()),
            "Missing_Percentage":
                round(
                    final_df[feature].isna().mean() * 100,
                    4
                )
        }
    )


feature_manifest_df = pd.DataFrame(
    manifest_records
)


# ============================================================
# 30. SAVE FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("SAVING OUTPUT FILES")
print("=" * 70)


final_df.to_csv(
    OUTPUT_FILE,
    index=False
)


missingness_df.to_csv(
    QC_OUTPUT_FILE,
    index=False
)


feature_manifest_df.to_csv(
    MANIFEST_OUTPUT_FILE,
    index=False
)


print(
    f"\n1. Final ML dataset:"
)

print(
    f"   {OUTPUT_FILE}"
)

print(
    f"\n2. Missingness QC report:"
)

print(
    f"   {QC_OUTPUT_FILE}"
)

print(
    f"\n3. Feature manifest:"
)

print(
    f"   {MANIFEST_OUTPUT_FILE}"
)


# ============================================================
# 31. FINAL RELOAD TEST
# ============================================================
#
# Reloading the saved CSV verifies that the file can be read
# successfully and that the column structure is preserved.
# ============================================================

print("\n" + "=" * 70)
print("FINAL FILE RELOAD TEST")
print("=" * 70)


validation_df = pd.read_csv(
    OUTPUT_FILE,
    low_memory=False
)


print(
    f"Reloaded rows    : "
    f"{validation_df.shape[0]:,}"
)

print(
    f"Reloaded columns : "
    f"{validation_df.shape[1]:,}"
)


if list(validation_df.columns) != list(
    final_df.columns
):

    raise ValueError(
        "\nColumn structure changed after saving/reloading."
    )


if validation_df.shape != final_df.shape:

    raise ValueError(
        "\nDataset dimensions changed after saving/reloading."
    )


print(
    "Reload validation passed successfully."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PART 12H-v2 — FINAL SUMMARY")
print("=" * 70)

print(
    f"Input dataset                 : "
    f"{os.path.basename(input_file)}"
)

print(
    f"Final rows                    : "
    f"{len(final_df):,}"
)

print(
    f"Final columns                 : "
    f"{len(final_df.columns):,}"
)

print(
    f"Unique mutation IDs           : "
    f"{final_df['Mutation'].nunique():,}"
)

print(
    f"Unique residue positions      : "
    f"{final_df['Sequence_Position'].nunique():,}"
)

print(
    f"WT one-hot features           : "
    f"{len(WT_ONEHOT_COLUMNS)}"
)

print(
    f"Mutant one-hot features       : "
    f"{len(MUTANT_ONEHOT_COLUMNS)}"
)

print(
    f"Physicochemical features      : "
    f"{len(PHYSICOCHEMICAL_FEATURES)}"
)

print(
    f"Side-chain features           : "
    f"{len(SIDECHAIN_FEATURES)}"
)

print(
    f"Hydrogen-bond features        : "
    f"{len(HBOND_FEATURES)}"
)

print(
    f"Structural-context features   : "
    f"{len(STRUCTURAL_FEATURES)}"
)

print(
    f"Secondary-structure feature   : "
    f"{len(SECONDARY_STRUCTURE_FEATURE)}"
)

print(
    f"Local sequence features      : "
    f"{len(LOCAL_SEQUENCE_FEATURES)}"
)

print(
    f"Model A predictors            : "
    f"{len(BIOLOGICAL_PREDICTORS_WITHOUT_POSITION)}"
)

print(
    f"Model B predictors            : "
    f"{len(BIOLOGICAL_PREDICTORS_WITH_POSITION)}"
)

print(
    f"Phenotype targets             : "
    f"{len(TARGET_COLUMNS)}"
)

print(
    f"Phenotype SD columns          : "
    f"{len(TARGET_SD_COLUMNS)}"
)

print("\nSecondary-structure design:")
print("  [KEEP] Secondary_Structure")
print("  [REMOVE] DSSP_Secondary_Structure")
print("  [REMOVE] Secondary_Structure_Code")

print("\nMissing-value policy:")
print("  No global imputation performed.")
print("  Structural missingness is preserved.")
print("  Imputation/encoding will be performed inside")
print("  the later ML training pipeline using training data only.")

print("\nPosition policy:")
print("  Sequence_Position is retained in the dataset.")
print("  Model A will exclude it.")
print("  Model B will include it.")

print("\nTarget policy:")
print("  Phenotype columns are retained as targets.")
print("  Phenotype SD columns are retained as uncertainty metadata.")
print("  Neither targets nor SD columns are ML predictors.")

print("\n" + "=" * 70)
print("PART 12H-v2 COMPLETED SUCCESSFULLY")
print("=" * 70)

PART 12H-v2 — 12D → FINAL ML FEATURE TABLE

Libraries imported successfully.

Input file selected:
VIM2_Part12D_Hydrogen_Bonding.csv
Full path:
/content/VIM2_Part12D_Hydrogen_Bonding.csv

Dataset loaded successfully.
Rows    : 5,016
Columns : 76

All required Part 12D source columns are available.

Removing redundant secondary-structure columns:
  - DSSP_Secondary_Structure
  - Secondary_Structure_Code

GENERATING AMINO-ACID ONE-HOT FEATURES

ONE-HOT ENCODING VALIDATION
Rows with invalid WT one-hot encoding     : 0
Rows with invalid mutant one-hot encoding : 0
One-hot amino-acid encoding is valid for all rows.

FINAL PREDICTOR ARCHITECTURE
Predictors with position    : 66
Predictors without position : 65

Feature-group counts:
  WT amino-acid one-hot         : 20
  Mutant amino-acid one-hot     : 20
  Physicochemical               : 5
  Side-chain                    : 3
  Hydrogen-bond                 : 3
  Structural-context            : 5
  Secondary-structure           : 1
  Local s

In [ ]:
# @title
# ======================================================================
# PART 13 — LEAKAGE-SAFE ML DATASET PREPARATION & VALIDATION
# ======================================================================
#
# Project:
# VIM-2 Mutation Fitness Prediction
#
# Purpose:
#   Prepare the finalized Part 12H-v2 feature table for downstream
#   interpretable machine-learning analyses while preventing information
#   leakage through mutation identity, residue position, preprocessing,
#   or target-derived information.
#
# Core analytical design:
#
#   Model A:
#       65 predictors
#       Sequence_Position excluded
#
#   Model B:
#       66 predictors
#       Sequence_Position included
#
# Primary evaluation:
#       Position-aware grouped cross-validation
#       All substitutions at the same residue position remain in the
#       same fold.
#
# Secondary evaluation:
#       Mutation-level random K-fold cross-validation.
#
# Important methodological rule:
#       No target-dependent feature selection, scaling, imputation,
#       encoding, or hyperparameter optimization is performed globally.
#       Such operations will be performed inside the training folds
#       during the subsequent model-training stage.
#
# This stage DOES NOT train ML models.
#
# ======================================================================


# ======================================================================
# 0. IMPORT REQUIRED LIBRARIES
# ======================================================================

import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

print("=" * 70)
print("PART 13 — LEAKAGE-SAFE ML DATASET PREPARATION & VALIDATION")
print("=" * 70)
print("\nLibraries imported successfully.")


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

# ----------------------------------------------------------------------
# Input file generated by Part 12H-v2
# ----------------------------------------------------------------------

INPUT_FILE = "/content/VIM2_Part12H_v2_Final_ML_Dataset.csv"

# ----------------------------------------------------------------------
# Output directory
# ----------------------------------------------------------------------

OUTPUT_DIR = "/content/VIM2_Part13_ML_Preparation"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# Cross-validation configuration
# ----------------------------------------------------------------------

N_SPLITS = 5
RANDOM_STATE = 42

# ----------------------------------------------------------------------
# Expected dimensions from Part 12H-v2
# ----------------------------------------------------------------------

EXPECTED_ROWS = 5016
EXPECTED_COLUMNS = 89
EXPECTED_UNIQUE_MUTATIONS = 5016
EXPECTED_UNIQUE_POSITIONS = 266

# ----------------------------------------------------------------------
# Expected predictor counts
# ----------------------------------------------------------------------

EXPECTED_MODEL_A_PREDICTORS = 65
EXPECTED_MODEL_B_PREDICTORS = 66

# ----------------------------------------------------------------------
# Number of phenotype targets
# ----------------------------------------------------------------------

EXPECTED_TARGETS = 9

# ----------------------------------------------------------------------
# Structural missingness expected from Part 12H-v2
# ----------------------------------------------------------------------

EXPECTED_STRUCTURAL_MISSINGNESS = 681


# ======================================================================
# 2. DEFINE TARGETS
# ======================================================================

TARGET_COLUMNS = [
    "128ug/mL_AMP_25C",
    "16ug/mL_AMP_25C",
    "2ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
    "0.5ug/mL_CTX_37C",
    "0.031ug/mL_MEM_37C"
]

TARGET_SD_COLUMNS = [
    "128ug/mL_AMP_25C_SD",
    "16ug/mL_AMP_25C_SD",
    "2ug/mL_AMP_25C_SD",
    "128ug/mL_AMP_37C_SD",
    "16ug/mL_AMP_37C_SD",
    "2ug/mL_AMP_37C_SD",
    "4ug/mL_CTX_37C_SD",
    "0.5ug/mL_CTX_37C_SD",
    "0.031ug/mL_MEM_37C_SD"
]


# ======================================================================
# 3. LOAD FINAL PART 12H-v2 DATASET
# ======================================================================

print("\n" + "=" * 70)
print("LOADING PART 12H-v2 FINAL DATASET")
print("=" * 70)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"\nInput file was not found:\n{INPUT_FILE}\n"
        "\nPlease verify that Part 12H-v2 was completed successfully."
    )

df = pd.read_csv(INPUT_FILE)

print(f"\nInput file:")
print(os.path.basename(INPUT_FILE))

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")


# ======================================================================
# 4. GLOBAL DATASET DIMENSION VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("GLOBAL DATASET VALIDATION")
print("=" * 70)

if df.shape[0] != EXPECTED_ROWS:
    raise ValueError(
        f"Unexpected row count: {df.shape[0]} "
        f"(expected {EXPECTED_ROWS})."
    )

if df.shape[1] != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected column count: {df.shape[1]} "
        f"(expected {EXPECTED_COLUMNS})."
    )

print("Row count validation passed.")
print("Column count validation passed.")


# ======================================================================
# 5. REQUIRED COLUMN VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED COLUMN VALIDATION")
print("=" * 70)

REQUIRED_METADATA_COLUMNS = [
    "Mutation",
    "WT_AA",
    "Sequence_Position",
    "Mutant_AA",
    "identity",
    "group"
]

REQUIRED_STRUCTURAL_COLUMNS = [
    "Distance_to_Metal_Site",
    "Distance_to_Active_Site_Pocket",
    "Distance_to_L3_Loop",
    "Distance_to_L10_Loop",
    "SASA",
    "Secondary_Structure"
]

REQUIRED_COLUMNS = (
    REQUIRED_METADATA_COLUMNS
    + REQUIRED_STRUCTURAL_COLUMNS
    + TARGET_COLUMNS
    + TARGET_SD_COLUMNS
)

missing_required = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        "\nMissing required columns:\n"
        + "\n".join(f"  - {x}" for x in missing_required)
    )

print(f"Required columns checked : {len(REQUIRED_COLUMNS)}")
print("All required columns are available.")


# ======================================================================
# 6. MUTATION IDENTITY VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("MUTATION IDENTITY VALIDATION")
print("=" * 70)

unique_mutations = df["Mutation"].nunique(dropna=False)
duplicate_mutations = df["Mutation"].duplicated().sum()

print(f"Unique mutation IDs      : {unique_mutations:,}")
print(f"Duplicate mutation rows  : {duplicate_mutations:,}")

if unique_mutations != EXPECTED_UNIQUE_MUTATIONS:
    raise ValueError(
        f"Unexpected number of unique mutations: "
        f"{unique_mutations} "
        f"(expected {EXPECTED_UNIQUE_MUTATIONS})."
    )

if duplicate_mutations != 0:
    raise ValueError(
        "Duplicate mutation IDs detected. "
        "Each mutation must correspond to a single row."
    )

print("Mutation identity validation passed.")


# ======================================================================
# 7. RESIDUE POSITION VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("RESIDUE POSITION VALIDATION")
print("=" * 70)

position_missing = df["Sequence_Position"].isna().sum()
unique_positions = df["Sequence_Position"].nunique(dropna=True)

print(f"Missing Sequence_Position : {position_missing:,}")
print(f"Unique residue positions  : {unique_positions:,}")

if position_missing != 0:
    raise ValueError(
        "Sequence_Position contains missing values. "
        "Position-aware grouping cannot be performed safely."
    )

if unique_positions != EXPECTED_UNIQUE_POSITIONS:
    raise ValueError(
        f"Unexpected number of residue positions: "
        f"{unique_positions} "
        f"(expected {EXPECTED_UNIQUE_POSITIONS})."
    )

print("Residue-position validation passed.")


# ======================================================================
# 8. VERIFY POSITION GROUP STRUCTURE
# ======================================================================
#
# This step confirms that multiple substitutions may occur at the same
# residue position and that Sequence_Position is therefore an appropriate
# grouping variable for biological generalization analysis.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("POSITION GROUP STRUCTURE")
print("=" * 70)

position_sizes = (
    df.groupby("Sequence_Position", dropna=False)
      .size()
      .sort_values(ascending=False)
)

print(f"Largest substitutions per position : {position_sizes.max():,}")
print(f"Smallest substitutions per position: {position_sizes.min():,}")
print(f"Median substitutions per position  : "
      f"{position_sizes.median():.1f}")

print("\nPosition groups with >1 mutation:")
print(f"  {sum(position_sizes > 1):,} / {len(position_sizes):,}")

print(
    "\nSequence_Position will be used as the grouping variable "
    "for primary position-aware evaluation."
)


# ======================================================================
# 9. IDENTIFY PREDICTOR GROUPS
# ======================================================================

print("\n" + "=" * 70)
print("DEFINING PREDICTOR ARCHITECTURE")
print("=" * 70)

WT_ONEHOT_COLUMNS = [
    f"WT_AA_{aa}"
    for aa in list("ACDEFGHIKLMNPQRSTVWY")
]

MUTANT_ONEHOT_COLUMNS = [
    f"Mutant_AA_{aa}"
    for aa in list("ACDEFGHIKLMNPQRSTVWY")
]

PHYSICOCHEMICAL_COLUMNS = [
    "Hydrophobicity_Change",
    "Weight_Change",
    "Charge_Change",
    "Polarity_Change",
    "BLOSUM62"
]

SIDECHAIN_COLUMNS = [
    "Side_Chain_Volume_Change",
    "Absolute_Side_Chain_Volume_Change",
    "Relative_Side_Chain_Volume_Change"
]

HYDROGEN_BOND_COLUMNS = [
    "HBond_Donor_Change",
    "HBond_Acceptor_Change",
    "Total_HBond_Capacity_Change"
]

STRUCTURAL_CONTEXT_COLUMNS = [
    "Distance_to_Metal_Site",
    "Distance_to_Active_Site_Pocket",
    "Distance_to_L3_Loop",
    "Distance_to_L10_Loop",
    "SASA"
]

SECONDARY_STRUCTURE_COLUMN = [
    "Secondary_Structure"
]

LOCAL_SEQUENCE_COLUMNS = [
    "Local_Hydrophobic_Fraction",
    "Local_Charged_Fraction",
    "Local_Positive_Charge_Fraction",
    "Local_Negative_Charge_Fraction",
    "Local_Polar_Fraction",
    "Local_Aromatic_Fraction",
    "Local_Gly_Pro_Fraction",
    "Local_Sequence_Entropy"
]

POSITION_COLUMN = [
    "Sequence_Position"
]

MODEL_A_FEATURES = (
    WT_ONEHOT_COLUMNS
    + MUTANT_ONEHOT_COLUMNS
    + PHYSICOCHEMICAL_COLUMNS
    + SIDECHAIN_COLUMNS
    + HYDROGEN_BOND_COLUMNS
    + STRUCTURAL_CONTEXT_COLUMNS
    + SECONDARY_STRUCTURE_COLUMN
    + LOCAL_SEQUENCE_COLUMNS
)

MODEL_B_FEATURES = (
    MODEL_A_FEATURES
    + POSITION_COLUMN
)


# ======================================================================
# 10. FEATURE EXISTENCE VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("PREDICTOR EXISTENCE VALIDATION")
print("=" * 70)

for model_name, feature_list in [
    ("Model A", MODEL_A_FEATURES),
    ("Model B", MODEL_B_FEATURES)
]:
    missing_features = [
        col for col in feature_list
        if col not in df.columns
    ]

    if missing_features:
        raise ValueError(
            f"\n{model_name} is missing predictors:\n"
            + "\n".join(f"  - {x}" for x in missing_features)
        )

    print(
        f"{model_name:<10} predictors available : "
        f"{len(feature_list)}"
    )


if len(MODEL_A_FEATURES) != EXPECTED_MODEL_A_PREDICTORS:
    raise ValueError(
        f"Model A contains {len(MODEL_A_FEATURES)} predictors "
        f"(expected {EXPECTED_MODEL_A_PREDICTORS})."
    )

if len(MODEL_B_FEATURES) != EXPECTED_MODEL_B_PREDICTORS:
    raise ValueError(
        f"Model B contains {len(MODEL_B_FEATURES)} predictors "
        f"(expected {EXPECTED_MODEL_B_PREDICTORS})."
    )

print("\nPredictor architecture validation passed.")


# ======================================================================
# 11. VERIFY MODEL A / MODEL B DIFFERENCE
# ======================================================================

print("\n" + "=" * 70)
print("MODEL A / MODEL B ARCHITECTURE CHECK")
print("=" * 70)

model_difference = sorted(
    set(MODEL_B_FEATURES) - set(MODEL_A_FEATURES)
)

print("Features added to Model B:")
for feature in model_difference:
    print(f"  + {feature}")

if model_difference != ["Sequence_Position"]:
    raise ValueError(
        "Model A and Model B do not differ exclusively by "
        "Sequence_Position."
    )

print(
    "\nModel A = molecular/structural/local predictors without position."
)
print(
    "Model B = Model A + Sequence_Position."
)


# ======================================================================
# 12. TARGET / SD ROLE VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("TARGET / SD ROLE VALIDATION")
print("=" * 70)

print(f"Phenotype targets    : {len(TARGET_COLUMNS)}")
print(f"Phenotype SD columns : {len(TARGET_SD_COLUMNS)}")

if len(TARGET_COLUMNS) != EXPECTED_TARGETS:
    raise ValueError(
        f"Expected {EXPECTED_TARGETS} targets, "
        f"found {len(TARGET_COLUMNS)}."
    )

# ----------------------------------------------------------------------
# Confirm that target columns are NOT predictors.
# ----------------------------------------------------------------------

for target in TARGET_COLUMNS:
    if target in MODEL_A_FEATURES or target in MODEL_B_FEATURES:
        raise ValueError(
            f"Target leakage detected: {target} appears in predictors."
        )

for sd_col in TARGET_SD_COLUMNS:
    if sd_col in MODEL_A_FEATURES or sd_col in MODEL_B_FEATURES:
        raise ValueError(
            f"SD leakage detected: {sd_col} appears in predictors."
        )

print("All phenotype targets excluded from predictors.")
print("All phenotype SD columns excluded from predictors.")
print("Target/SD role validation passed.")


# ======================================================================
# 13. METADATA EXCLUSION VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("METADATA EXCLUSION VALIDATION")
print("=" * 70)

METADATA_COLUMNS = [
    "Mutation",
    "WT_AA",
    "Mutant_AA",
    "identity",
    "group"
]

for model_name, feature_list in [
    ("Model A", MODEL_A_FEATURES),
    ("Model B", MODEL_B_FEATURES)
]:
    leaked_metadata = [
        col for col in METADATA_COLUMNS
        if col in feature_list
    ]

    if leaked_metadata:
        raise ValueError(
            f"{model_name} contains metadata columns as predictors:\n"
            + "\n".join(f"  - {x}" for x in leaked_metadata)
        )

print("Mutation identifiers excluded.")
print("WT/Mutant categorical identifiers excluded.")
print("Identity/group metadata excluded.")
print("Metadata leakage validation passed.")


# ======================================================================
# 14. STRUCTURAL MISSINGNESS VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("STRUCTURAL MISSINGNESS VALIDATION")
print("=" * 70)

STRUCTURAL_FEATURES = (
    STRUCTURAL_CONTEXT_COLUMNS
    + SECONDARY_STRUCTURE_COLUMN
)

structural_missing_table = []

for feature in STRUCTURAL_FEATURES:

    missing_count = int(df[feature].isna().sum())
    missing_pct = 100 * missing_count / len(df)

    structural_missing_table.append({
        "Feature": feature,
        "Missing_Count": missing_count,
        "Missing_Percentage": missing_pct
    })

    print(
        f"{feature:<40} | "
        f"Missing: {missing_count:4d} "
        f"({missing_pct:6.2f}%)"
    )

structural_missing_df = pd.DataFrame(structural_missing_table)

# ----------------------------------------------------------------------
# The expected structural missingness is 681 rows for each structural
# feature in the finalized Part 12H-v2 dataset.
# ----------------------------------------------------------------------

unexpected_structural_missingness = structural_missing_df[
    structural_missing_df["Missing_Count"] != EXPECTED_STRUCTURAL_MISSINGNESS
]

if len(unexpected_structural_missingness) > 0:

    print(
        "\nWARNING: Structural missingness differs from the expected "
        "Part 12H-v2 value for one or more features."
    )

else:

    print(
        "\nStructural missingness is consistent with Part 12H-v2."
    )

print(
    "\nNo global imputation will be performed at Part 13."
)

print(
    "Missing-value handling will be performed inside training folds "
    "during the downstream ML pipeline."
)


# ======================================================================
# 15. NUMERIC / CATEGORICAL FEATURE VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("FEATURE DATA-TYPE VALIDATION")
print("=" * 70)

NUMERIC_FEATURES = (
    WT_ONEHOT_COLUMNS
    + MUTANT_ONEHOT_COLUMNS
    + PHYSICOCHEMICAL_COLUMNS
    + SIDECHAIN_COLUMNS
    + HYDROGEN_BOND_COLUMNS
    + STRUCTURAL_CONTEXT_COLUMNS
    + LOCAL_SEQUENCE_COLUMNS
    + POSITION_COLUMN
)

for feature in NUMERIC_FEATURES:

    if not pd.api.types.is_numeric_dtype(df[feature]):
        raise TypeError(
            f"Expected numeric dtype for feature: {feature}"
        )

if not (
    pd.api.types.is_object_dtype(df["Secondary_Structure"])
    or pd.api.types.is_string_dtype(df["Secondary_Structure"])
    or pd.api.types.is_categorical_dtype(df["Secondary_Structure"])
):
    raise TypeError(
        "Secondary_Structure must remain categorical/string-valued."
    )

print(f"Numeric predictors checked     : {len(NUMERIC_FEATURES)}")
print("Secondary_Structure             : categorical")
print("Feature dtype validation passed.")


# ======================================================================
# 16. SECONDARY-STRUCTURE CATEGORY VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("SECONDARY-STRUCTURE VALIDATION")
print("=" * 70)

observed_ss = sorted(
    df["Secondary_Structure"]
      .dropna()
      .astype(str)
      .unique()
      .tolist()
)

print("Observed categories:")

for category in observed_ss:
    count = int(
        (df["Secondary_Structure"].astype(str) == category).sum()
    )
    print(f"  - {category:<10} : {count:,}")

allowed_ss = {"Coil", "Helix", "Sheet"}

unexpected_ss = set(observed_ss) - allowed_ss

if unexpected_ss:
    raise ValueError(
        "Unexpected Secondary_Structure categories detected:\n"
        + "\n".join(f"  - {x}" for x in unexpected_ss)
    )

print("\nSecondary-structure categories validated.")


# ======================================================================
# 17. BUILD POSITION-AWARE OUTER FOLDS
# ======================================================================
#
# Primary evaluation strategy:
#
#   GroupKFold
#       groups = Sequence_Position
#
# This guarantees that all mutations belonging to the same residue
# position are assigned to exactly one outer fold.
#
# No target information is used to create these folds.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATING POSITION-AWARE OUTER FOLDS")
print("=" * 70)

groups = df["Sequence_Position"].astype(int).to_numpy()

group_kfold = GroupKFold(n_splits=N_SPLITS)

position_fold_assignments = np.full(
    shape=len(df),
    fill_value=-1,
    dtype=int
)

for fold_id, (train_idx, test_idx) in enumerate(
    group_kfold.split(
        df[MODEL_B_FEATURES],
        groups=groups
    ),
    start=1
):

    position_fold_assignments[test_idx] = fold_id

if np.any(position_fold_assignments == -1):
    raise RuntimeError(
        "Some rows were not assigned to a position-aware fold."
    )

df["PositionAware_Fold"] = position_fold_assignments


# ======================================================================
# 18. VALIDATE POSITION-AWARE FOLD INTEGRITY
# ======================================================================

print("\n" + "=" * 70)
print("POSITION-AWARE FOLD INTEGRITY")
print("=" * 70)

position_fold_qc = []

for fold_id in range(1, N_SPLITS + 1):

    fold_mask = df["PositionAware_Fold"] == fold_id

    fold_positions = set(
        df.loc[fold_mask, "Sequence_Position"]
    )

    train_positions = set(
        df.loc[~fold_mask, "Sequence_Position"]
    )

    overlap = fold_positions.intersection(train_positions)

    fold_rows = int(fold_mask.sum())

    if len(overlap) != 0:
        raise RuntimeError(
            f"Position leakage detected in fold {fold_id}."
        )

    position_fold_qc.append({
        "Fold": fold_id,
        "Rows": fold_rows,
        "Unique_Positions": len(fold_positions),
        "Train_Test_Position_Overlap": len(overlap)
    })

    print(
        f"Fold {fold_id}: "
        f"{fold_rows:,} test mutations | "
        f"{len(fold_positions):,} positions | "
        f"Position overlap = {len(overlap)}"
    )

position_fold_qc_df = pd.DataFrame(position_fold_qc)

print("\nPosition-aware grouping validation passed.")
print(
    "No residue position is shared between training and outer-test "
    "folds."
)


# ======================================================================
# 19. GENERATE SECONDARY RANDOM MUTATION-LEVEL FOLDS
# ======================================================================
#
# Secondary evaluation strategy:
#
#   Standard mutation-level K-fold splitting.
#
# This is intentionally retained as a secondary benchmark rather than
# the primary biological generalization estimate.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATING RANDOM MUTATION-LEVEL OUTER FOLDS")
print("=" * 70)

random_kfold = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

random_fold_assignments = np.full(
    shape=len(df),
    fill_value=-1,
    dtype=int
)

for fold_id, (train_idx, test_idx) in enumerate(
    random_kfold.split(df),
    start=1
):

    random_fold_assignments[test_idx] = fold_id

if np.any(random_fold_assignments == -1):
    raise RuntimeError(
        "Some rows were not assigned to a random fold."
    )

df["Random_Fold"] = random_fold_assignments


# ======================================================================
# 20. VALIDATE RANDOM FOLD BALANCE
# ======================================================================

print("\n" + "=" * 70)
print("RANDOM FOLD INTEGRITY")
print("=" * 70)

random_fold_qc = []

for fold_id in range(1, N_SPLITS + 1):

    fold_mask = df["Random_Fold"] == fold_id

    fold_rows = int(fold_mask.sum())

    random_fold_qc.append({
        "Fold": fold_id,
        "Rows": fold_rows
    })

    print(
        f"Fold {fold_id}: {fold_rows:,} test mutations"
    )

random_fold_qc_df = pd.DataFrame(random_fold_qc)

print("\nRandom mutation-level fold validation passed.")


# ======================================================================
# 21. QUANTIFY POSITION LEAKAGE IN RANDOM SPLIT
# ======================================================================
#
# This diagnostic explicitly demonstrates why the random split is only
# a secondary evaluation strategy.
#
# If mutations from the same residue position appear in both training
# and test sets, position-level information can potentially transfer
# across the split.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("RANDOM-SPLIT POSITION LEAKAGE DIAGNOSTIC")
print("=" * 70)

random_position_overlap_qc = []

for fold_id in range(1, N_SPLITS + 1):

    test_mask = df["Random_Fold"] == fold_id

    test_positions = set(
        df.loc[test_mask, "Sequence_Position"]
    )

    train_positions = set(
        df.loc[~test_mask, "Sequence_Position"]
    )

    overlap = test_positions.intersection(train_positions)

    random_position_overlap_qc.append({
        "Fold": fold_id,
        "Test_Positions": len(test_positions),
        "Train_Positions": len(train_positions),
        "Shared_Positions": len(overlap)
    })

    print(
        f"Fold {fold_id}: "
        f"shared residue positions = {len(overlap):,}"
    )

random_position_overlap_df = pd.DataFrame(
    random_position_overlap_qc
)

print(
    "\nDiagnostic completed."
)

print(
    "Important: shared positions in the random split are expected "
    "and are not considered an error because this split represents "
    "the conventional mutation-level benchmark."
)


# ======================================================================
# 22. TARGET COMPLETENESS BY FOLD
# ======================================================================
#
# Each phenotype has a small number of missing observations.
#
# Fold generation is performed independently of target availability.
# During downstream model training, rows with missing values for the
# current target will be excluded from that target's training/evaluation
# set without altering the predefined fold structure.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET COMPLETENESS BY POSITION-AWARE FOLD")
print("=" * 70)

target_fold_records = []

for target in TARGET_COLUMNS:

    for fold_id in range(1, N_SPLITS + 1):

        test_mask = df["PositionAware_Fold"] == fold_id
        train_mask = ~test_mask

        train_available = int(
            df.loc[train_mask, target].notna().sum()
        )

        train_missing = int(
            df.loc[train_mask, target].isna().sum()
        )

        test_available = int(
            df.loc[test_mask, target].notna().sum()
        )

        test_missing = int(
            df.loc[test_mask, target].isna().sum()
        )

        target_fold_records.append({
            "Target": target,
            "Fold": fold_id,
            "Train_Rows": int(train_mask.sum()),
            "Train_Target_Available": train_available,
            "Train_Target_Missing": train_missing,
            "Test_Rows": int(test_mask.sum()),
            "Test_Target_Available": test_available,
            "Test_Target_Missing": test_missing
        })

target_fold_df = pd.DataFrame(target_fold_records)

for target in TARGET_COLUMNS:

    sub = target_fold_df[
        target_fold_df["Target"] == target
    ]

    print(f"\n{target}")

    for _, row in sub.iterrows():

        print(
            f"  Fold {int(row['Fold'])}: "
            f"train available = {int(row['Train_Target_Available']):4d}, "
            f"test available = {int(row['Test_Target_Available']):4d}"
        )


# ======================================================================
# 23. TARGET-LEVEL GLOBAL QC SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("TARGET-LEVEL QC SUMMARY")
print("=" * 70)

target_qc_records = []

for target in TARGET_COLUMNS:

    available = int(df[target].notna().sum())
    missing = int(df[target].isna().sum())

    target_qc_records.append({
        "Target": target,
        "Total_Rows": len(df),
        "Available": available,
        "Missing": missing,
        "Missing_Percentage": 100 * missing / len(df)
    })

target_qc_df = pd.DataFrame(target_qc_records)

print(
    target_qc_df.to_string(
        index=False,
        formatters={
            "Missing_Percentage": "{:.4f}".format
        }
    )
)


# ======================================================================
# 24. TARGET-SD AVAILABILITY SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("TARGET SD / UNCERTAINTY METADATA SUMMARY")
print("=" * 70)

sd_qc_records = []

for sd_col in TARGET_SD_COLUMNS:

    available = int(df[sd_col].notna().sum())
    missing = int(df[sd_col].isna().sum())

    sd_qc_records.append({
        "SD_Column": sd_col,
        "Total_Rows": len(df),
        "Available": available,
        "Missing": missing,
        "Missing_Percentage": 100 * missing / len(df)
    })

sd_qc_df = pd.DataFrame(sd_qc_records)

print(
    sd_qc_df.to_string(
        index=False,
        formatters={
            "Missing_Percentage": "{:.4f}".format
        }
    )
)

print(
    "\nSD columns are retained only as uncertainty metadata and are "
    "excluded from all predictor matrices."
)


# ======================================================================
# 25. DEFINE LEAKAGE-SAFE PREPROCESSING SPECIFICATION
# ======================================================================
#
# IMPORTANT:
#
# The transformers below are specifications for the downstream ML
# pipeline. They must be FIT only on the training portion of each fold.
#
# Numerical:
#     Median imputation
#     Standardization
#
# Categorical:
#     Most-frequent imputation
#     One-hot encoding
#
# This specification is NOT fitted globally here.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("DEFINING LEAKAGE-SAFE PREPROCESSING SPECIFICATION")
print("=" * 70)

MODEL_A_NUMERIC_FEATURES = [
    feature
    for feature in MODEL_A_FEATURES
    if feature != "Secondary_Structure"
]

MODEL_B_NUMERIC_FEATURES = [
    feature
    for feature in MODEL_B_FEATURES
    if feature != "Secondary_Structure"
]

CATEGORICAL_FEATURES = [
    "Secondary_Structure"
]


# ----------------------------------------------------------------------
# Helper function for constructing the downstream preprocessing pipeline.
# This function creates an UNFITTED pipeline.
# ----------------------------------------------------------------------

def build_preprocessor(numeric_features, categorical_features):
    """
    Construct an unfitted leakage-safe preprocessing pipeline.

    Numerical predictors:
        - median imputation
        - standardization

    Categorical predictors:
        - most-frequent imputation
        - one-hot encoding

    The returned transformer must be fitted separately inside each
    training fold during model development.
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )

    return preprocessor


MODEL_A_PREPROCESSOR = build_preprocessor(
    MODEL_A_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES
)

MODEL_B_PREPROCESSOR = build_preprocessor(
    MODEL_B_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES
)

print(
    "Model A preprocessing specification created."
)

print(
    "Model B preprocessing specification created."
)

print(
    "\nIMPORTANT:"
)

print(
    "These preprocessing pipelines are intentionally UNFITTED."
)

print(
    "They must be fitted only on training data inside each fold."
)


# ======================================================================
# 26. FEATURE MANIFEST
# ======================================================================

print("\n" + "=" * 70)
print("GENERATING PART 13 FEATURE MANIFEST")
print("=" * 70)

feature_manifest_records = []

for feature in df.columns:

    if feature in MODEL_A_FEATURES:
        model_a = True
    else:
        model_a = False

    if feature in MODEL_B_FEATURES:
        model_b = True
    else:
        model_b = False

    if feature in TARGET_COLUMNS:
        role = "Phenotype_Target"

    elif feature in TARGET_SD_COLUMNS:
        role = "Phenotype_SD_Metadata"

    elif feature in METADATA_COLUMNS:
        role = "Metadata"

    elif feature == "Sequence_Position":
        role = "Position_Predictor"

    elif feature in WT_ONEHOT_COLUMNS:
        role = "WT_Amino_Acid_OneHot_Predictor"

    elif feature in MUTANT_ONEHOT_COLUMNS:
        role = "Mutant_Amino_Acid_OneHot_Predictor"

    elif feature in PHYSICOCHEMICAL_COLUMNS:
        role = "Physicochemical_Predictor"

    elif feature in SIDECHAIN_COLUMNS:
        role = "SideChain_Predictor"

    elif feature in HYDROGEN_BOND_COLUMNS:
        role = "HydrogenBond_Predictor"

    elif feature in STRUCTURAL_CONTEXT_COLUMNS:
        role = "StructuralContext_Predictor"

    elif feature in SECONDARY_STRUCTURE_COLUMN:
        role = "SecondaryStructure_Categorical_Predictor"

    elif feature in LOCAL_SEQUENCE_COLUMNS:
        role = "LocalSequence_Predictor"

    elif feature in [
        "PositionAware_Fold",
        "Random_Fold"
    ]:
        role = "CrossValidation_Metadata"

    else:
        role = "Other"

    feature_manifest_records.append({
        "Feature": feature,
        "Role": role,
        "Used_in_Model_A_No_Position": model_a,
        "Used_in_Model_B_With_Position": model_b,
        "Dtype": str(df[feature].dtype),
        "Missing_Count": int(df[feature].isna().sum()),
        "Missing_Percentage":
            100 * df[feature].isna().sum() / len(df)
    })

feature_manifest_df = pd.DataFrame(
    feature_manifest_records
)

print(
    f"Manifest rows: {len(feature_manifest_df):,}"
)


# ======================================================================
# 27. CREATE FINAL FOLD ASSIGNMENT TABLE
# ======================================================================
#
# This table intentionally contains identifiers and fold assignments,
# not transformed predictors.
#
# It becomes the master fold map for Part 14.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CREATING MASTER FOLD ASSIGNMENT TABLE")
print("=" * 70)

fold_assignment_columns = [
    "Mutation",
    "WT_AA",
    "Sequence_Position",
    "Mutant_AA",
    "PositionAware_Fold",
    "Random_Fold"
]

fold_assignment_df = df[
    fold_assignment_columns
].copy()

fold_assignment_df["Row_Index"] = np.arange(
    len(fold_assignment_df)
)

fold_assignment_df = fold_assignment_df[
    [
        "Row_Index",
        "Mutation",
        "WT_AA",
        "Sequence_Position",
        "Mutant_AA",
        "PositionAware_Fold",
        "Random_Fold"
    ]
]

print(
    f"Fold assignment rows: {len(fold_assignment_df):,}"
)


# ======================================================================
# 28. FINAL LEAKAGE AUDIT
# ======================================================================

print("\n" + "=" * 70)
print("FINAL LEAKAGE AUDIT")
print("=" * 70)

leakage_checks = []

# ----------------------------------------------------------------------
# Check 1: Targets excluded
# ----------------------------------------------------------------------

target_leakage = (
    set(TARGET_COLUMNS)
    .intersection(set(MODEL_B_FEATURES))
)

leakage_checks.append({
    "Check": "Phenotype target leakage",
    "Status": "PASS" if len(target_leakage) == 0 else "FAIL",
    "Details": str(sorted(target_leakage))
})


# ----------------------------------------------------------------------
# Check 2: SD metadata excluded
# ----------------------------------------------------------------------

sd_leakage = (
    set(TARGET_SD_COLUMNS)
    .intersection(set(MODEL_B_FEATURES))
)

leakage_checks.append({
    "Check": "Phenotype SD leakage",
    "Status": "PASS" if len(sd_leakage) == 0 else "FAIL",
    "Details": str(sorted(sd_leakage))
})


# ----------------------------------------------------------------------
# Check 3: Mutation identifiers excluded
# ----------------------------------------------------------------------

metadata_leakage = (
    set(METADATA_COLUMNS)
    .intersection(set(MODEL_B_FEATURES))
)

leakage_checks.append({
    "Check": "Mutation metadata leakage",
    "Status": "PASS" if len(metadata_leakage) == 0 else "FAIL",
    "Details": str(sorted(metadata_leakage))
})


# ----------------------------------------------------------------------
# Check 4: Position-aware fold integrity
# ----------------------------------------------------------------------

position_leakage = False

for fold_id in range(1, N_SPLITS + 1):

    test_positions = set(
        df.loc[
            df["PositionAware_Fold"] == fold_id,
            "Sequence_Position"
        ]
    )

    train_positions = set(
        df.loc[
            df["PositionAware_Fold"] != fold_id,
            "Sequence_Position"
        ]
    )

    if test_positions.intersection(train_positions):
        position_leakage = True
        break

leakage_checks.append({
    "Check": "Position-aware fold leakage",
    "Status": "FAIL" if position_leakage else "PASS",
    "Details":
        "No shared positions between train and test folds."
        if not position_leakage
        else "Shared positions detected."
})


# ----------------------------------------------------------------------
# Check 5: Duplicate mutation IDs
# ----------------------------------------------------------------------

duplicate_mutation_check = (
    df["Mutation"].duplicated().sum() == 0
)

leakage_checks.append({
    "Check": "Duplicate mutation IDs",
    "Status":
        "PASS" if duplicate_mutation_check else "FAIL",
    "Details":
        "All mutation IDs are unique."
        if duplicate_mutation_check
        else "Duplicate mutation IDs detected."
})


# ----------------------------------------------------------------------
# Check 6: No global preprocessing performed
# ----------------------------------------------------------------------

leakage_checks.append({
    "Check": "Global preprocessing",
    "Status": "PASS",
    "Details":
        "No imputation, scaling, encoding, or feature selection "
        "was fitted globally."
})


# ----------------------------------------------------------------------
# Check 7: No target-dependent fold construction
# ----------------------------------------------------------------------

leakage_checks.append({
    "Check": "Target-dependent fold construction",
    "Status": "PASS",
    "Details":
        "Fold assignments were generated independently of phenotype "
        "target values."
})


leakage_audit_df = pd.DataFrame(
    leakage_checks
)

print(
    leakage_audit_df.to_string(index=False)
)

if (leakage_audit_df["Status"] == "FAIL").any():
    raise RuntimeError(
        "\nFINAL LEAKAGE AUDIT FAILED.\n"
        "The ML preparation stage must not proceed."
    )

print(
    "\nAll leakage checks passed successfully."
)


# ======================================================================
# 29. CREATE MODEL ARCHITECTURE MANIFEST
# ======================================================================

print("\n" + "=" * 70)
print("CREATING MODEL ARCHITECTURE MANIFEST")
print("=" * 70)

architecture_manifest = {
    "project": "VIM-2 Mutation Fitness Prediction",
    "pipeline_stage": "Part 13",
    "input_dataset": os.path.basename(INPUT_FILE),
    "rows": int(len(df)),
    "columns_before_cv_metadata": EXPECTED_COLUMNS,
    "unique_mutations": int(unique_mutations),
    "unique_residue_positions": int(unique_positions),

    "models": {
        "Model_A_No_Position": {
            "predictor_count": len(MODEL_A_FEATURES),
            "position_included": False,
            "predictors": MODEL_A_FEATURES
        },
        "Model_B_With_Position": {
            "predictor_count": len(MODEL_B_FEATURES),
            "position_included": True,
            "predictors": MODEL_B_FEATURES
        }
    },

    "targets": TARGET_COLUMNS,

    "target_sd_metadata": TARGET_SD_COLUMNS,

    "primary_evaluation": {
        "method": "GroupKFold",
        "groups": "Sequence_Position",
        "n_splits": N_SPLITS,
        "random_state": None,
        "purpose":
            "Biological generalization to previously unseen residue positions"
    },

    "secondary_evaluation": {
        "method": "KFold",
        "shuffle": True,
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "purpose":
            "Conventional mutation-level benchmark"
    },

    "preprocessing": {
        "numeric": [
            "Median imputation fitted on training folds only",
            "Standardization fitted on training folds only"
        ],
        "categorical": [
            "Most-frequent imputation fitted on training folds only",
            "One-hot encoding fitted on training folds only",
            "handle_unknown=ignore"
        ]
    },

    "missing_value_policy": (
        "No global imputation. Structural missingness is preserved "
        "until fold-specific preprocessing."
    ),

    "feature_selection_policy": (
        "No global target-dependent feature selection. Any future "
        "feature selection must occur inside training folds."
    ),

    "target_policy": (
        "Phenotype columns are targets only and are excluded from "
        "predictors."
    ),

    "sd_policy": (
        "Phenotype SD columns are uncertainty metadata only and are "
        "excluded from predictors."
    ),

    "position_policy": (
        "Sequence_Position is excluded from Model A and included in "
        "Model B. For primary evaluation it is also the grouping "
        "variable regardless of whether it is included as a predictor."
    ),

    "model_training": (
        "No machine-learning models were trained during Part 13."
    )
}

architecture_manifest_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Model_Architecture.json"
)

with open(
    architecture_manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_manifest,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    f"Saved:\n{architecture_manifest_path}"
)


# ======================================================================
# 30. SAVE OUTPUT TABLES
# ======================================================================

print("\n" + "=" * 70)
print("SAVING PART 13 OUTPUT FILES")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Master fold assignment table
# ----------------------------------------------------------------------

fold_assignment_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Fold_Assignments.csv"
)

fold_assignment_df.to_csv(
    fold_assignment_path,
    index=False
)

print(
    f"\n1. Fold assignments:\n   {fold_assignment_path}"
)


# ----------------------------------------------------------------------
# 2. Feature manifest
# ----------------------------------------------------------------------

feature_manifest_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Feature_Manifest.csv"
)

feature_manifest_df.to_csv(
    feature_manifest_path,
    index=False
)

print(
    f"\n2. Feature manifest:\n   {feature_manifest_path}"
)


# ----------------------------------------------------------------------
# 3. Target QC
# ----------------------------------------------------------------------

target_qc_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Target_QC.csv"
)

target_qc_df.to_csv(
    target_qc_path,
    index=False
)

print(
    f"\n3. Target QC:\n   {target_qc_path}"
)


# ----------------------------------------------------------------------
# 4. Target-by-fold QC
# ----------------------------------------------------------------------

target_fold_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Target_Fold_QC.csv"
)

target_fold_df.to_csv(
    target_fold_path,
    index=False
)

print(
    f"\n4. Target-by-fold QC:\n   {target_fold_path}"
)


# ----------------------------------------------------------------------
# 5. Structural missingness QC
# ----------------------------------------------------------------------

structural_qc_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Structural_Missingness_QC.csv"
)

structural_missing_df.to_csv(
    structural_qc_path,
    index=False
)

print(
    f"\n5. Structural missingness QC:\n   "
    f"{structural_qc_path}"
)


# ----------------------------------------------------------------------
# 6. Position-aware fold QC
# ----------------------------------------------------------------------

position_fold_qc_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_PositionAware_Fold_QC.csv"
)

position_fold_qc_df.to_csv(
    position_fold_qc_path,
    index=False
)

print(
    f"\n6. Position-aware fold QC:\n   "
    f"{position_fold_qc_path}"
)


# ----------------------------------------------------------------------
# 7. Random fold QC
# ----------------------------------------------------------------------

random_fold_qc_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Random_Fold_QC.csv"
)

random_fold_qc_df.to_csv(
    random_fold_qc_path,
    index=False
)

print(
    f"\n7. Random fold QC:\n   {random_fold_qc_path}"
)


# ----------------------------------------------------------------------
# 8. Random-split position-overlap diagnostic
# ----------------------------------------------------------------------

random_overlap_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_RandomSplit_PositionOverlap_QC.csv"
)

random_position_overlap_df.to_csv(
    random_overlap_path,
    index=False
)

print(
    f"\n8. Random-split position diagnostic:\n   "
    f"{random_overlap_path}"
)


# ----------------------------------------------------------------------
# 9. Leakage audit
# ----------------------------------------------------------------------

leakage_audit_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Leakage_Audit.csv"
)

leakage_audit_df.to_csv(
    leakage_audit_path,
    index=False
)

print(
    f"\n9. Leakage audit:\n   {leakage_audit_path}"
)


# ======================================================================
# 31. CREATE ML PREPARATION SUMMARY
# ======================================================================

summary_lines = [

    "=" * 70,
    "PART 13 — ML DATASET PREPARATION SUMMARY",
    "=" * 70,

    "",
    f"Input dataset                 : {os.path.basename(INPUT_FILE)}",
    f"Rows                          : {len(df):,}",
    f"Original columns              : {EXPECTED_COLUMNS:,}",
    f"Unique mutation IDs           : {unique_mutations:,}",
    f"Unique residue positions      : {unique_positions:,}",

    "",
    "PREDICTOR ARCHITECTURE",
    "-" * 70,
    f"Model A predictors            : {len(MODEL_A_FEATURES)}",
    f"Model B predictors            : {len(MODEL_B_FEATURES)}",
    "Model A position              : EXCLUDED",
    "Model B position              : INCLUDED",

    "",
    "TARGET ARCHITECTURE",
    "-" * 70,
    f"Phenotype targets             : {len(TARGET_COLUMNS)}",
    f"Phenotype SD metadata         : {len(TARGET_SD_COLUMNS)}",

    "",
    "PRIMARY EVALUATION",
    "-" * 70,
    "Method                        : GroupKFold",
    "Grouping variable             : Sequence_Position",
    f"Number of folds               : {N_SPLITS}",
    "Purpose                       : Position-aware generalization",

    "",
    "SECONDARY EVALUATION",
    "-" * 70,
    "Method                        : KFold",
    "Shuffle                       : TRUE",
    f"Number of folds               : {N_SPLITS}",
    f"Random state                  : {RANDOM_STATE}",
    "Purpose                       : Mutation-level benchmark",

    "",
    "PREPROCESSING POLICY",
    "-" * 70,
    "Global imputation             : NOT PERFORMED",
    "Global scaling                : NOT PERFORMED",
    "Global encoding               : NOT PERFORMED",
    "Global feature selection      : NOT PERFORMED",
    "Training-fold-only preprocessing: REQUIRED",

    "",
    "STRUCTURAL MISSINGNESS",
    "-" * 70,
    f"Expected structural missing rows: "
    f"{EXPECTED_STRUCTURAL_MISSINGNESS:,}",
    "Structural missingness preserved until ML preprocessing.",

    "",
    "LEAKAGE CONTROL",
    "-" * 70,
    "Target leakage                : PASSED",
    "SD leakage                    : PASSED",
    "Metadata leakage              : PASSED",
    "Position-aware fold leakage   : PASSED",
    "Duplicate mutation check      : PASSED",

    "",
    "MODEL TRAINING STATUS",
    "-" * 70,
    "No ML models were trained in Part 13.",
    "Part 13 prepares the leakage-safe evaluation framework.",

    "",
    "OUTPUT DIRECTORY",
    "-" * 70,
    OUTPUT_DIR,

    "",
    "=" * 70,
    "PART 13 COMPLETED SUCCESSFULLY",
    "=" * 70
]

summary_text = "\n".join(summary_lines)

summary_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Part13_Summary.txt"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(summary_text)

print("\n" + summary_text)


# ======================================================================
# 32. FINAL IN-MEMORY DATASET CHECK
# ======================================================================
#
# The fold assignments are added only as cross-validation metadata.
# They are NOT predictors and must never be passed to an ML estimator.
#
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL PART 13 DATASET CHECK")
print("=" * 70)

print(f"Rows                       : {len(df):,}")
print(f"Columns after fold metadata: {df.shape[1]:,}")

print(
    f"Position-aware folds       : "
    f"{df['PositionAware_Fold'].nunique()}"
)

print(
    f"Random folds               : "
    f"{df['Random_Fold'].nunique()}"
)

print(
    f"Model A predictors         : "
    f"{len(MODEL_A_FEATURES)}"
)

print(
    f"Model B predictors         : "
    f"{len(MODEL_B_FEATURES)}"
)

# ----------------------------------------------------------------------
# Verify every row has exactly one fold in each scheme.
# ----------------------------------------------------------------------

if df["PositionAware_Fold"].isna().any():
    raise RuntimeError(
        "Missing PositionAware_Fold assignments detected."
    )

if df["Random_Fold"].isna().any():
    raise RuntimeError(
        "Missing Random_Fold assignments detected."
    )

print("\nEvery mutation has a valid position-aware fold.")
print("Every mutation has a valid random fold.")


# ======================================================================
# 33. FINAL VALIDATION OF OUTPUT FILES
# ======================================================================

print("\n" + "=" * 70)
print("OUTPUT FILE VALIDATION")
print("=" * 70)

output_files = [
    fold_assignment_path,
    feature_manifest_path,
    target_qc_path,
    target_fold_path,
    structural_qc_path,
    position_fold_qc_path,
    random_fold_qc_path,
    random_overlap_path,
    leakage_audit_path,
    architecture_manifest_path,
    summary_path
]

for path in output_files:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Expected output file was not created:\n{path}"
        )

    size_kb = os.path.getsize(path) / 1024

    print(
        f"[OK] {os.path.basename(path):<55} "
        f"{size_kb:>8.1f} KB"
    )

print("\nAll Part 13 output files were created successfully.")


# ======================================================================
# 34. FINAL COMPLETION MESSAGE
# ======================================================================

print("\n" + "=" * 70)
print("PART 13 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("""
The VIM-2 dataset is now prepared for leakage-safe ML development.

Primary evaluation:
    Position-aware GroupKFold
    Group = Sequence_Position

Secondary evaluation:
    Random mutation-level KFold

Model A:
    65 predictors
    Sequence_Position excluded

Model B:
    66 predictors
    Sequence_Position included

Targets:
    9 phenotype variables

SD columns:
    retained as uncertainty metadata
    excluded from predictors

Missing structural values:
    preserved
    no global imputation performed

Preprocessing:
    defined but NOT globally fitted
    must be fitted within training folds

Feature selection:
    not performed globally

Hyperparameter tuning:
    not performed in Part 13

Model training:
    not performed in Part 13

The dataset is ready for PART 14 —
BASELINE AND TREE-BASED REGRESSION MODEL DEVELOPMENT.
""")

print("=" * 70)

PART 13 — LEAKAGE-SAFE ML DATASET PREPARATION & VALIDATION

Libraries imported successfully.

LOADING PART 12H-v2 FINAL DATASET

Input file:
VIM2_Part12H_v2_Final_ML_Dataset.csv
Rows    : 5,016
Columns : 89

GLOBAL DATASET VALIDATION
Row count validation passed.
Column count validation passed.

REQUIRED COLUMN VALIDATION
Required columns checked : 30
All required columns are available.

MUTATION IDENTITY VALIDATION
Unique mutation IDs      : 5,016
Duplicate mutation rows  : 0
Mutation identity validation passed.

RESIDUE POSITION VALIDATION
Missing Sequence_Position : 0
Unique residue positions  : 266
Residue-position validation passed.

POSITION GROUP STRUCTURE
Largest substitutions per position : 19
Smallest substitutions per position: 13
Median substitutions per position  : 19.0

Position groups with >1 mutation:
  266 / 266

Sequence_Position will be used as the grouping variable for primary position-aware evaluation.

DEFINING PREDICTOR ARCHITECTURE

PREDICTOR EXISTENCE VALIDATION

In [ ]:
# @title
# ============================================================
# DOWNLOAD PART 13 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Part13_ML_Preparation"

# Output ZIP archive
zip_base = "/content/VIM2_Part13_ML_Preparation"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Part13_ML_Preparation"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("PART 13 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

PART 13 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Part13_ML_Preparation
ZIP archive      : /content/VIM2_Part13_ML_Preparation.zip
Archive size     : 0.04 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
# ================================================================
# VIM-2 PART 14 — FINAL OOF INTEGRITY AUDIT
# ================================================================
#
# Purpose:
#   Final validation of the complete All_OOF_Predictions.csv
#   generated by Part 14.
#
# IMPORTANT:
#   - This script DOES NOT modify the original OOF file.
#   - This script does NOT retrain any model.
#   - This script does NOT rerun Part 14.
#   - This script performs integrity / consistency checks only.
#
# Main checks:
#   1. File integrity
#   2. Required columns
#   3. Evaluation coverage
#   4. Target coverage
#   5. Architecture coverage
#   6. Algorithm coverage
#   7. Fold coverage
#   8. Duplicate OOF predictions
#   9. Missing combinations
#  10. Observed / predicted numeric integrity
#  11. Residual consistency
#  12. OOF row-count consistency
#  13. Fold-to-evaluation consistency
#  14. Architecture-specific position integrity
#
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd


# ================================================================
# 1. USER CONFIGURATION
# ================================================================

# IMPORTANT:
# Replace this with the EXACT path of your local file.

OOF_FILE = r"C:\Users\YOUR_USERNAME\Desktop\VIM2_Part14_All_OOF_Predictions.csv"


# Expected Part 14 design

EXPECTED_EVALUATIONS = {
    "Position_Aware",
    "Random"
}

EXPECTED_ARCHITECTURES = {
    "Model_A_No_Position",
    "Model_B_With_Position"
}

EXPECTED_MODELS = {
    "Ridge",
    "Random_Forest",
    "Extra_Trees",
    "HistGradientBoosting",
    "XGBoost",
    "LightGBM"
}

EXPECTED_FOLDS = {
    1,
    2,
    3,
    4,
    5
}

EXPECTED_TARGETS = {
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C"
}


IDENTITY_COLUMN = "Mutation_ID"


# ================================================================
# 2. AUDIT OUTPUT CONFIGURATION
# ================================================================

if not os.path.isfile(OOF_FILE):

    raise FileNotFoundError(
        "\nOOF file was not found:\n"
        f"{OOF_FILE}\n\n"
        "Please correct OOF_FILE at the top of the script."
    )


OOF_DIRECTORY = os.path.dirname(
    os.path.abspath(
        OOF_FILE
    )
)


AUDIT_FILE = os.path.join(
    OOF_DIRECTORY,
    "VIM2_Part14_Final_OOF_Integrity_Audit.csv"
)


SUMMARY_FILE = os.path.join(
    OOF_DIRECTORY,
    "VIM2_Part14_Final_OOF_Integrity_Summary.txt"
)


# ================================================================
# 3. START AUDIT
# ================================================================

print("\n" + "=" * 75)
print("VIM-2 PART 14 — FINAL OOF INTEGRITY AUDIT")
print("=" * 75)

print(
    f"\nOOF file:\n{OOF_FILE}"
)


file_size_mb = (
    os.path.getsize(OOF_FILE)
    /
    (1024 ** 2)
)


print(
    f"\nFile size: {file_size_mb:.2f} MB"
)


# ================================================================
# 4. FILE HASH
# ================================================================
#
# SHA-256 provides a reproducible fingerprint of the exact file
# that was audited.
# ================================================================

print("\n" + "-" * 75)
print("1. FILE INTEGRITY")
print("-" * 75)


sha256 = hashlib.sha256()


with open(
    OOF_FILE,
    "rb"
) as f:

    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):

        sha256.update(
            chunk
        )


file_hash = sha256.hexdigest()


print(
    f"SHA-256: {file_hash}"
)


# ================================================================
# 5. LOAD OOF DATA
# ================================================================

print("\n" + "-" * 75)
print("2. LOADING OOF DATA")
print("-" * 75)


oof = pd.read_csv(
    OOF_FILE,
    low_memory=False
)


print(
    f"Rows    : {len(oof):,}"
)


print(
    f"Columns : {len(oof.columns):,}"
)


# ================================================================
# 6. REQUIRED COLUMN CHECK
# ================================================================

print("\n" + "-" * 75)
print("3. REQUIRED COLUMN CHECK")
print("-" * 75)


required_columns = {

    IDENTITY_COLUMN,

    "Sequence_Position",

    "Evaluation",

    "Primary_Evaluation",

    "Target",

    "Architecture",

    "Model",

    "Fold",

    "Observed",

    "Predicted",

    "Residual"
}


missing_columns = (
    required_columns
    -
    set(oof.columns)
)


if missing_columns:

    raise RuntimeError(
        "Required OOF columns are missing:\n"
        +
        "\n".join(
            sorted(
                missing_columns
            )
        )
    )


print(
    "[PASS] All required OOF columns are present."
)


# ================================================================
# 7. BASIC DATA TYPE / FINITE VALUE CHECK
# ================================================================

print("\n" + "-" * 75)
print("4. NUMERIC DATA INTEGRITY")
print("-" * 75)


numeric_columns = [

    "Sequence_Position",
    "Fold",
    "Observed",
    "Predicted",
    "Residual"
]


numeric_failures = []


for column in numeric_columns:

    converted = pd.to_numeric(
        oof[column],
        errors="coerce"
    )

    invalid = (
        converted.isna()
        |
        ~np.isfinite(
            converted
        )
    )

    n_invalid = int(
        invalid.sum()
    )

    if n_invalid > 0:

        numeric_failures.append(
            (
                column,
                n_invalid
            )
        )


if numeric_failures:

    print(
        "[FAIL] Invalid numeric values detected:"
    )

    for column, n in numeric_failures:

        print(
            f"  - {column}: {n:,}"
        )

else:

    print(
        "[PASS] All required numeric columns contain "
        "finite numeric values."
    )


# ================================================================
# 8. EVALUATION COVERAGE
# ================================================================

print("\n" + "-" * 75)
print("5. EVALUATION COVERAGE")
print("-" * 75)


observed_evaluations = set(
    oof["Evaluation"]
    .dropna()
    .unique()
)


print(
    "Evaluations found:"
)

for value in sorted(
    observed_evaluations
):

    print(
        f"  - {value}"
    )


missing_evaluations = (
    EXPECTED_EVALUATIONS
    -
    observed_evaluations
)


unexpected_evaluations = (
    observed_evaluations
    -
    EXPECTED_EVALUATIONS
)


if missing_evaluations:

    print(
        f"[FAIL] Missing evaluations: "
        f"{sorted(missing_evaluations)}"
    )

elif unexpected_evaluations:

    print(
        f"[FAIL] Unexpected evaluations: "
        f"{sorted(unexpected_evaluations)}"
    )

else:

    print(
        "[PASS] Evaluation coverage is complete."
    )


# ================================================================
# 9. TARGET COVERAGE
# ================================================================

print("\n" + "-" * 75)
print("6. TARGET COVERAGE")
print("-" * 75)


observed_targets = set(
    oof["Target"]
    .dropna()
    .unique()
)


print(
    f"Targets found: {len(observed_targets)}"
)


for target in sorted(
    observed_targets
):

    print(
        f"  - {target}"
    )


missing_targets = (
    EXPECTED_TARGETS
    -
    observed_targets
)


unexpected_targets = (
    observed_targets
    -
    EXPECTED_TARGETS
)


if missing_targets:

    print(
        f"[FAIL] Missing targets: "
        f"{sorted(missing_targets)}"
    )

elif unexpected_targets:

    print(
        f"[FAIL] Unexpected targets: "
        f"{sorted(unexpected_targets)}"
    )

else:

    print(
        "[PASS] All 9 expected targets are present."
    )


# ================================================================
# 10. ARCHITECTURE COVERAGE
# ================================================================

print("\n" + "-" * 75)
print("7. MODEL ARCHITECTURE COVERAGE")
print("-" * 75)


observed_architectures = set(
    oof["Architecture"]
    .dropna()
    .unique()
)


for architecture in sorted(
    observed_architectures
):

    print(
        f"  - {architecture}"
    )


if (
    observed_architectures
    ==
    EXPECTED_ARCHITECTURES
):

    print(
        "[PASS] Both expected architectures are present."
    )

else:

    print(
        "[FAIL] Architecture coverage mismatch."
    )


# ================================================================
# 11. ALGORITHM COVERAGE
# ================================================================

print("\n" + "-" * 75)
print("8. ALGORITHM COVERAGE")
print("-" * 75)


observed_models = set(
    oof["Model"]
    .dropna()
    .unique()
)


for model in sorted(
    observed_models
):

    print(
        f"  - {model}"
    )


missing_models = (
    EXPECTED_MODELS
    -
    observed_models
)


unexpected_models = (
    observed_models
    -
    EXPECTED_MODELS
)


if (
    not missing_models
    and
    not unexpected_models
):

    print(
        "[PASS] All 6 expected algorithms are present."
    )

else:

    print(
        "[FAIL] Algorithm coverage mismatch."
    )

    if missing_models:

        print(
            f"Missing: {sorted(missing_models)}"
        )

    if unexpected_models:

        print(
            f"Unexpected: {sorted(unexpected_models)}"
        )


# ================================================================
# 12. FOLD COVERAGE
# ================================================================

print("\n" + "-" * 75)
print("9. FOLD COVERAGE")
print("-" * 75)


observed_folds = set(
    pd.to_numeric(
        oof["Fold"],
        errors="coerce"
    )
    .dropna()
    .astype(int)
    .unique()
)


print(
    f"Fold values: "
    f"{sorted(observed_folds)}"
)


if (
    observed_folds
    ==
    EXPECTED_FOLDS
):

    print(
        "[PASS] All five expected folds are present."
    )

else:

    print(
        "[FAIL] Fold coverage mismatch."
    )


# ================================================================
# 13. PRIMARY EVALUATION FLAG CHECK
# ================================================================

print("\n" + "-" * 75)
print("10. PRIMARY_EVALUATION FLAG CHECK")
print("-" * 75)


position_flags = (
    oof.loc[
        oof["Evaluation"]
        ==
        "Position_Aware",
        "Primary_Evaluation"
    ]
    .dropna()
    .astype(bool)
    .unique()
)


random_flags = (
    oof.loc[
        oof["Evaluation"]
        ==
        "Random",
        "Primary_Evaluation"
    ]
    .dropna()
    .astype(bool)
    .unique()
)


print(
    f"Position_Aware flags: "
    f"{position_flags.tolist()}"
)


print(
    f"Random flags        : "
    f"{random_flags.tolist()}"
)


if (
    set(position_flags)
    ==
    {True}
    and
    set(random_flags)
    ==
    {False}
):

    print(
        "[PASS] Evaluation priority flags are correct."
    )

else:

    print(
        "[FAIL] Primary_Evaluation flags are inconsistent."
    )


# ================================================================
# 14. EXPECTED COMBINATION CHECK
# ================================================================
#
# Expected unique combinations:
#
#   2 evaluations
#   × 9 targets
#   × 2 architectures
#   × 6 models
#   × 5 folds
#
#   = 1080 OOF prediction rows
#
# Each combination should occur exactly once.
# ================================================================

print("\n" + "-" * 75)
print("11. EXPECTED OOF COMBINATION CHECK")
print("-" * 75)


combination_columns = [

    "Evaluation",
    "Target",
    "Architecture",
    "Model",
    "Fold"
]


combination_counts = (

    oof

    .groupby(
        combination_columns,
        dropna=False
    )

    .size()

    .reset_index(
        name="Count"
    )
)


duplicate_combinations = (
    combination_counts[
        combination_counts["Count"] > 1
    ]
)


expected_rows = (
    2
    *
    9
    *
    2
    *
    6
    *
    5
)


print(
    f"Expected OOF rows: "
    f"{expected_rows:,}"
)


print(
    f"Observed OOF rows: "
    f"{len(oof):,}"
)


print(
    f"Unique combinations: "
    f"{len(combination_counts):,}"
)


if len(oof) != expected_rows:

    print(
        "[FAIL] Total OOF row count does not match "
        "the expected Part 14 design."
    )

else:

    print(
        "[PASS] Total OOF row count is correct."
    )


if len(duplicate_combinations) > 0:

    print(
        f"[FAIL] Duplicate fold-level combinations: "
        f"{len(duplicate_combinations):,}"
    )

else:

    print(
        "[PASS] No duplicate evaluation/target/"
        "architecture/model/fold combinations."
    )


# ================================================================
# 15. MUTATION-LEVEL DUPLICATE CHECK
# ================================================================
#
# A mutation can legitimately appear multiple times because it is
# evaluated under different targets, models and evaluation schemes.
#
# Therefore we check duplicates using the COMPLETE identity key.
# ================================================================

print("\n" + "-" * 75)
print("12. COMPLETE OOF IDENTITY CHECK")
print("-" * 75)


identity_key = [

    IDENTITY_COLUMN,
    "Evaluation",
    "Target",
    "Architecture",
    "Model",
    "Fold"
]


identity_duplicates = (

    oof

    .groupby(
        identity_key,
        dropna=False
    )

    .size()

    .reset_index(
        name="Count"
    )

)


identity_duplicates = (
    identity_duplicates[
        identity_duplicates["Count"] > 1
    ]
)


if len(identity_duplicates) > 0:

    print(
        f"[FAIL] Duplicate complete OOF identities: "
        f"{len(identity_duplicates):,}"
    )

else:

    print(
        "[PASS] Every mutation/evaluation/target/"
        "architecture/model/fold identity is unique."
    )


# ================================================================
# 16. RESIDUAL CONSISTENCY
# ================================================================
#
# Residual should equal:
#
#     Observed - Predicted
#
# Small floating-point tolerance is allowed.
# ================================================================

print("\n" + "-" * 75)
print("13. RESIDUAL CONSISTENCY")
print("-" * 75)


observed_numeric = pd.to_numeric(
    oof["Observed"],
    errors="coerce"
)


predicted_numeric = pd.to_numeric(
    oof["Predicted"],
    errors="coerce"
)


residual_numeric = pd.to_numeric(
    oof["Residual"],
    errors="coerce"
)


expected_residual = (
    observed_numeric
    -
    predicted_numeric
)


residual_difference = np.abs(
    residual_numeric
    -
    expected_residual
)


RESIDUAL_TOLERANCE = 1e-10


bad_residuals = (
    residual_difference
    >
    RESIDUAL_TOLERANCE
)


n_bad_residuals = int(
    bad_residuals.sum()
)


print(
    f"Residual mismatches: "
    f"{n_bad_residuals:,}"
)


if n_bad_residuals == 0:

    print(
        "[PASS] Residual = Observed - Predicted "
        "for all OOF rows."
    )

else:

    print(
        "[FAIL] Residual inconsistencies detected."
    )


# ================================================================
# 17. ARCHITECTURE POSITION CHECK
# ================================================================
#
# IMPORTANT:
#
# Model A must NOT use Sequence_Position as a predictor.
#
# However, Sequence_Position is intentionally retained in the OOF
# output for independent position-aware auditing.
#
# Therefore this check does NOT interpret the presence of the
# Sequence_Position column as predictor usage.
# ================================================================

print("\n" + "-" * 75)
print("14. ARCHITECTURE POSITION INTEGRITY")
print("-" * 75)


print(
    "Model A:"
)

print(
    "  Sequence_Position predictor = EXCLUDED"
)


print(
    "Model B:"
)

print(
    "  Sequence_Position predictor = INCLUDED"
)


model_a_rows = (
    oof[
        oof["Architecture"]
        ==
        "Model_A_No_Position"
    ]
)


model_b_rows = (
    oof[
        oof["Architecture"]
        ==
        "Model_B_With_Position"
    ]
)


if (
    len(model_a_rows) > 0
    and
    len(model_b_rows) > 0
):

    print(
        "[PASS] Both architectures are represented."
    )

    print(
        "NOTE: Sequence_Position in the OOF output is metadata "
        "for traceability and leakage auditing; its presence "
        "does not mean it was used by Model A."
    )

else:

    print(
        "[FAIL] One or both architectures are missing."
    )


# ================================================================
# 18. POSITION-AWARE FOLD UNIQUENESS CHECK
# ================================================================
#
# For Position_Aware evaluation:
#
# each residue position must belong to only ONE fold.
#
# This check uses the OOF file only.
# ================================================================

print("\n" + "-" * 75)
print("15. POSITION-AWARE FOLD CONSISTENCY")
print("-" * 75)


position_oof = oof[
    oof["Evaluation"]
    ==
    "Position_Aware"
].copy()


position_fold_counts = (

    position_oof

    .groupby(
        "Sequence_Position"
    )["Fold"]

    .nunique()

)


invalid_position_assignments = (
    position_fold_counts[
        position_fold_counts > 1
    ]
)


print(
    f"Unique residue positions checked: "
    f"{len(position_fold_counts):,}"
)


print(
    f"Positions assigned to multiple folds: "
    f"{len(invalid_position_assignments):,}"
)


if len(
    invalid_position_assignments
) == 0:

    print(
        "[PASS] Each residue position is associated "
        "with only one Position_Aware fold."
    )

else:

    print(
        "[FAIL] Some residue positions occur in multiple "
        "Position_Aware folds."
    )


# ================================================================
# 19. PER-GROUP ROW COUNT CHECK
# ================================================================
#
# Every:
#
#   Evaluation × Target × Architecture × Model
#
# should contain exactly 5 OOF rows, one for each fold.
# ================================================================

print("\n" + "-" * 75)
print("16. FIVE-FOLD GROUP COMPLETENESS")
print("-" * 75)


group_columns = [

    "Evaluation",
    "Target",
    "Architecture",
    "Model"
]


group_fold_counts = (

    oof

    .groupby(
        group_columns
    )["Fold"]

    .nunique()

)


bad_groups = (
    group_fold_counts[
        group_fold_counts
        !=
        5
    ]
)


print(
    f"Expected groups: "
    f"{2 * 9 * 2 * 6:,}"
)


print(
    f"Observed groups: "
    f"{len(group_fold_counts):,}"
)


print(
    f"Groups without exactly 5 folds: "
    f"{len(bad_groups):,}"
)


if len(bad_groups) == 0:

    print(
        "[PASS] Every evaluation/target/architecture/model "
        "combination contains all five folds."
    )

else:

    print(
        "[FAIL] Some groups do not contain exactly five folds."
    )


# ================================================================
# 20. TARGET × EVALUATION × ARCHITECTURE × MODEL COVERAGE
# ================================================================

print("\n" + "-" * 75)
print("17. COMPLETE DESIGN COVERAGE")
print("-" * 75)


expected_design_groups = (
    2
    *
    9
    *
    2
    *
    6
)


observed_design_groups = (
    len(
        oof[
            group_columns
        ]
        .drop_duplicates()
    )
)


print(
    f"Expected design groups: "
    f"{expected_design_groups:,}"
)


print(
    f"Observed design groups: "
    f"{observed_design_groups:,}"
)


if (
    observed_design_groups
    ==
    expected_design_groups
):

    print(
        "[PASS] Complete Part 14 design coverage confirmed."
    )

else:

    print(
        "[FAIL] Part 14 design coverage is incomplete."
    )


# ================================================================
# 21. BUILD AUDIT TABLE
# ================================================================

audit_rows = []


def add_audit(
    check,
    status,
    details
):

    audit_rows.append(

        {

            "Check":
                check,

            "Status":
                status,

            "Details":
                details
        }
    )


add_audit(
    "File existence",
    "PASS",
    "OOF prediction file exists and was successfully read."
)


add_audit(
    "Required columns",
    "PASS" if not missing_columns else "FAIL",
    "All required OOF columns are present."
    if not missing_columns
    else str(sorted(missing_columns))
)


add_audit(
    "Evaluation coverage",
    "PASS"
    if (
        not missing_evaluations
        and
        not unexpected_evaluations
    )
    else "FAIL",
    "Both Position_Aware and Random evaluations are present."
)


add_audit(
    "Target coverage",
    "PASS"
    if (
        not missing_targets
        and
        not unexpected_targets
    )
    else "FAIL",
    "All nine expected targets are present."
)


add_audit(
    "Architecture coverage",
    "PASS"
    if (
        observed_architectures
        ==
        EXPECTED_ARCHITECTURES
    )
    else "FAIL",
    "Both Model A and Model B are present."
)


add_audit(
    "Algorithm coverage",
    "PASS"
    if (
        not missing_models
        and
        not unexpected_models
    )
    else "FAIL",
    "All six expected algorithms are present."
)


add_audit(
    "Five-fold coverage",
    "PASS"
    if (
        observed_folds
        ==
        EXPECTED_FOLDS
    )
    else "FAIL",
    "All five folds are present."
)


add_audit(
    "Expected OOF row count",
    "PASS"
    if len(oof) == expected_rows
    else "FAIL",
    f"Observed {len(oof):,}; expected {expected_rows:,}."
)


add_audit(
    "Duplicate fold combinations",
    "PASS"
    if len(duplicate_combinations) == 0
    else "FAIL",
    f"{len(duplicate_combinations):,} duplicated combinations."
)


add_audit(
    "Complete OOF identity uniqueness",
    "PASS"
    if len(identity_duplicates) == 0
    else "FAIL",
    f"{len(identity_duplicates):,} duplicated complete identities."
)


add_audit(
    "Residual consistency",
    "PASS"
    if n_bad_residuals == 0
    else "FAIL",
    f"{n_bad_residuals:,} residual mismatches."
)


add_audit(
    "Position-aware fold consistency",
    "PASS"
    if len(invalid_position_assignments) == 0
    else "FAIL",
    f"{len(invalid_position_assignments):,} positions assigned to multiple folds."
)


add_audit(
    "Five-fold group completeness",
    "PASS"
    if len(bad_groups) == 0
    else "FAIL",
    f"{len(bad_groups):,} groups do not contain five folds."
)


add_audit(
    "Complete design coverage",
    "PASS"
    if (
        observed_design_groups
        ==
        expected_design_groups
    )
    else "FAIL",
    f"Observed {observed_design_groups:,}; "
    f"expected {expected_design_groups:,}."
)


audit_df = pd.DataFrame(
    audit_rows
)


# ================================================================
# 22. SAVE AUDIT
# ================================================================

audit_df.to_csv(
    AUDIT_FILE,
    index=False
)


# ================================================================
# 23. FINAL STATUS
# ================================================================

failed = audit_df[
    audit_df["Status"]
    ==
    "FAIL"
]


print("\n" + "=" * 75)
print("FINAL OOF INTEGRITY AUDIT RESULT")
print("=" * 75)


print(
    f"\nTotal checks : {len(audit_df)}"
)


print(
    f"PASS         : "
    f"{sum(audit_df['Status'] == 'PASS')}"
)


print(
    f"FAIL         : "
    f"{sum(audit_df['Status'] == 'FAIL')}"
)


if len(failed) == 0:

    final_status = "PASS"

    print(
        "\n[PASS] COMPLETE OOF INTEGRITY VALIDATION PASSED."
    )

    print(
        "\nThe complete All_OOF_Predictions.csv is internally "
        "consistent with the expected Part 14 design."
    )

else:

    final_status = "FAIL"

    print(
        "\n[FAIL] OOF INTEGRITY VALIDATION FAILED."
    )

    print(
        "\nFailed checks:"
    )

    print(
        failed.to_string(
            index=False
        )
    )


# ================================================================
# 24. HUMAN-READABLE SUMMARY
# ================================================================

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "======================================================================\n"
    )

    f.write(
        "VIM-2 PART 14 — FINAL OOF INTEGRITY AUDIT\n"
    )

    f.write(
        "======================================================================\n\n"
    )

    f.write(
        f"OOF file       : {OOF_FILE}\n"
    )

    f.write(
        f"File size      : {file_size_mb:.2f} MB\n"
    )

    f.write(
        f"SHA-256        : {file_hash}\n"
    )

    f.write(
        f"Rows           : {len(oof):,}\n"
    )

    f.write(
        f"Expected rows  : {expected_rows:,}\n\n"
    )

    f.write(
        "Expected design:\n"
    )

    f.write(
        "  Evaluations   : 2\n"
    )

    f.write(
        "  Targets       : 9\n"
    )

    f.write(
        "  Architectures : 2\n"
    )

    f.write(
        "  Algorithms    : 6\n"
    )

    f.write(
        "  Folds         : 5\n\n"
    )

    f.write(
        f"Final status   : {final_status}\n\n"
    )

    f.write(
        "This audit did not retrain models and did not modify the "
        "original OOF prediction file.\n"
    )


print(
    f"\nAudit CSV saved to:\n{AUDIT_FILE}"
)


print(
    f"\nAudit summary saved to:\n{SUMMARY_FILE}"
)


print(
    "\n" + "=" * 75
)
print(
    "OOF AUDIT COMPLETED"
)
print(
    "=" * 75
)

In [ ]:
# @title
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
STEP 2 — POSITION-AWARE CANDIDATE MODEL SELECTION

Select Random Forest vs Extra Trees for each Target x Architecture x
Part-13 outer fold using position-aware inner CV on outer-training data only.

Important:
- Part 13 frozen PositionAware_Fold assignments are reused.
- Part 13 Feature Manifest is the source of truth for predictors.
- Model A must contain exactly 65 predictors.
- Model B must contain exactly 66 predictors.
- Model B = Model A + Sequence_Position.
- Part 14 results are never read or used.
- Outer-test observations are never used for candidate selection.
- No hyperparameter optimization is performed here; Step 3 remains the HPO step.
"""

import json
import os
import platform
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION
# ============================================================================


DATA_FILE = "/content/VIM2_Part12H_v2_Final_ML_Dataset.csv"
FOLD_FILE = "VIM2_Part13_Fold_Assignments.csv"
MANIFEST_FILE = "VIM2_Part13_Feature_Manifest.csv"

OUTPUT_DIR = "/content/VIM2_Step2_PositionAware_Candidate_Selection"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

POSITION_COLUMN = "Sequence_Position"
FOLD_COLUMN = "PositionAware_Fold"

MODEL_A_NAME = "Model_A_No_Position"
MODEL_B_NAME = "Model_B_With_Position"

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

TARGET_SD_COLUMNS = [f"{x}_SD" for x in TARGETS]
MODEL_NAMES = ["Random_Forest", "Extra_Trees"]


# ============================================================================
# HELPERS
# ============================================================================

def fail(message):
    raise RuntimeError(message)


def manifest_bool(series):
    values = series.astype(str).str.strip().str.lower()
    true_values = {"true", "1", "yes", "y", "t"}
    false_values = {"false", "0", "no", "n", "f"}
    unknown = set(values.unique()) - true_values - false_values
    if unknown:
        fail(
            "Unexpected boolean values in Part 13 feature manifest: "
            f"{sorted(unknown)}"
        )
    return values.isin(true_values)


def build_preprocessor(features):
    # Secondary_Structure is the only categorical predictor.
    categorical = [x for x in features if x == "Secondary_Structure"]
    numeric = [x for x in features if x != "Secondary_Structure"]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(
                strategy="median",
                add_indicator=True
            )),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )),
        ]
    )

    transformers = [
        ("numeric", numeric_pipeline, numeric)
    ]

    if categorical:
        transformers.append(
            ("categorical", categorical_pipeline, categorical)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


def build_model(model_name):
    # Fixed baseline settings are intentional: Step 2 selects the
    # algorithm family; Step 3 performs hyperparameter optimization.
    if model_name == "Random_Forest":
        return RandomForestRegressor(
            n_estimators=250,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )

    if model_name == "Extra_Trees":
        return ExtraTreesRegressor(
            n_estimators=250,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )

    fail(f"Unsupported model: {model_name}")


def build_pipeline(features, model_name):
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(features)),
            ("model", build_model(model_name)),
        ]
    )


def create_inner_position_splits(data, outer_fold):
    positions = data[POSITION_COLUMN].dropna().unique()

    if len(positions) < N_INNER_FOLDS:
        fail(
            f"Insufficient unique positions for inner CV: "
            f"{len(positions)} available, {N_INNER_FOLDS} required."
        )

    splitter = KFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE + outer_fold
    )

    position_array = np.asarray(positions)
    splits = []

    for inner_fold, (tr_pos, va_pos) in enumerate(
        splitter.split(position_array), start=1
    ):
        train_positions = set(position_array[tr_pos])
        valid_positions = set(position_array[va_pos])

        overlap = train_positions.intersection(valid_positions)
        if overlap:
            fail(
                f"Inner position leakage detected in outer fold "
                f"{outer_fold}, inner fold {inner_fold}."
            )

        train_idx = np.flatnonzero(
            data[POSITION_COLUMN].isin(train_positions).to_numpy()
        )
        valid_idx = np.flatnonzero(
            data[POSITION_COLUMN].isin(valid_positions).to_numpy()
        )

        splits.append(
            (
                train_idx,
                valid_idx,
                inner_fold,
                train_positions,
                valid_positions,
            )
        )

    return splits


# ============================================================================
# LOAD DATA
# ============================================================================

print("=" * 80)
print("STEP 2 — POSITION-AWARE CANDIDATE MODEL SELECTION")
print("=" * 80)

for path in [DATA_FILE, FOLD_FILE, MANIFEST_FILE]:
    if not os.path.isfile(path):
        fail(f"Required input file not found:\n{path}")

df = pd.read_csv(DATA_FILE, low_memory=False)
fold_df = pd.read_csv(FOLD_FILE, low_memory=False)
manifest_df = pd.read_csv(MANIFEST_FILE, low_memory=False)

print(f"Dataset shape : {df.shape}")
print(f"Fold shape    : {fold_df.shape}")
print(f"Manifest shape: {manifest_df.shape}")


# ============================================================================
# VALIDATE PART 13 FOLD ALIGNMENT
# ============================================================================

required_fold_columns = {
    "Row_Index", "Mutation", "WT_AA", "Sequence_Position",
    "Mutant_AA", "PositionAware_Fold"
}

missing = required_fold_columns - set(fold_df.columns)
if missing:
    fail("Missing Part 13 fold columns:\n" + "\n".join(sorted(missing)))

if len(fold_df) != len(df):
    fail(
        f"Dataset/fold row mismatch: {len(df)} vs {len(fold_df)}."
    )

if not np.array_equal(
    fold_df["Row_Index"].to_numpy(),
    np.arange(len(df))
):
    fail("Part 13 Row_Index does not match dataset row order.")

for column in ["Mutation", "Sequence_Position"]:
    if not np.array_equal(
        fold_df[column].astype(str).to_numpy(),
        df[column].astype(str).to_numpy()
    ):
        fail(
            f"Part 13 {column} values are not aligned with the dataset."
        )

observed_folds = sorted(
    fold_df[FOLD_COLUMN].dropna().unique().tolist()
)

if observed_folds != list(range(1, N_OUTER_FOLDS + 1)):
    fail(
        f"Unexpected PositionAware_Fold labels: {observed_folds}"
    )

print("[PASS] Part 13 fold assignments are aligned.")


# ============================================================================
# RECOVER EXACT FEATURE ARCHITECTURES FROM PART 13 MANIFEST
# ============================================================================

required_manifest_columns = {
    "Feature",
    "Used_in_Model_A_No_Position",
    "Used_in_Model_B_With_Position",
}

missing = required_manifest_columns - set(manifest_df.columns)
if missing:
    fail(
        "Missing Part 13 manifest columns:\n"
        + "\n".join(sorted(missing))
    )

a_mask = manifest_bool(
    manifest_df["Used_in_Model_A_No_Position"]
)
b_mask = manifest_bool(
    manifest_df["Used_in_Model_B_With_Position"]
)

MODEL_A_FEATURES = manifest_df.loc[
    a_mask, "Feature"
].astype(str).tolist()

MODEL_B_FEATURES = manifest_df.loc[
    b_mask, "Feature"
].astype(str).tolist()

if len(MODEL_A_FEATURES) != 65:
    fail(
        "Model A architecture mismatch: expected 65 predictors, "
        f"found {len(MODEL_A_FEATURES)}."
    )

if len(MODEL_B_FEATURES) != 66:
    fail(
        "Model B architecture mismatch: expected 66 predictors, "
        f"found {len(MODEL_B_FEATURES)}."
    )

if POSITION_COLUMN in MODEL_A_FEATURES:
    fail("Sequence_Position must not be present in Model A.")

if POSITION_COLUMN not in MODEL_B_FEATURES:
    fail("Sequence_Position must be present in Model B.")

if set(MODEL_B_FEATURES) != set(MODEL_A_FEATURES) | {POSITION_COLUMN}:
    fail("Model B must be exactly Model A + Sequence_Position.")

if len(set(MODEL_A_FEATURES)) != 65:
    fail("Duplicate predictors detected in Model A.")

if len(set(MODEL_B_FEATURES)) != 66:
    fail("Duplicate predictors detected in Model B.")

for feature in MODEL_B_FEATURES:
    if feature not in df.columns:
        fail(f"Manifest predictor missing from dataset: {feature}")

for forbidden in TARGETS + TARGET_SD_COLUMNS:
    if forbidden in MODEL_A_FEATURES or forbidden in MODEL_B_FEATURES:
        fail(f"Target leakage detected: {forbidden}")

for forbidden in [
    "Mutation", "WT_AA", "Mutant_AA", "identity", "group",
    "PositionAware_Fold", "Random_Fold"
]:
    if forbidden in MODEL_A_FEATURES or forbidden in MODEL_B_FEATURES:
        fail(f"Metadata leakage detected: {forbidden}")

print("[PASS] Model A = exactly 65 Part 13 predictors.")
print("[PASS] Model B = exactly 66 Part 13 predictors.")
print("[PASS] Model B differs from Model A only by Sequence_Position.")
print("[PASS] Targets, SDs, identifiers, metadata, and fold labels excluded.")


# ============================================================================
# OUTER POSITION-DISJOINTNESS AUDIT
# ============================================================================

outer_audit = []

for fold in range(1, N_OUTER_FOLDS + 1):
    train_positions = set(
        fold_df.loc[
            fold_df[FOLD_COLUMN] != fold,
            POSITION_COLUMN
        ].dropna().unique()
    )
    test_positions = set(
        fold_df.loc[
            fold_df[FOLD_COLUMN] == fold,
            POSITION_COLUMN
        ].dropna().unique()
    )

    overlap = train_positions.intersection(test_positions)

    if overlap:
        fail(
            f"Outer position leakage detected in fold {fold}: "
            f"{sorted(overlap)}"
        )

    outer_audit.append({
        "Outer_Fold": fold,
        "Train_Unique_Positions": len(train_positions),
        "Test_Unique_Positions": len(test_positions),
        "Position_Overlap_Count": len(overlap),
    })

print("[PASS] All Part 13 outer folds are position-disjoint.")


# ============================================================================
# POSITION-AWARE INNER CANDIDATE SELECTION
# ============================================================================

architectures = {
    MODEL_A_NAME: MODEL_A_FEATURES,
    MODEL_B_NAME: MODEL_B_FEATURES,
}

candidate_records = []
inner_records = []
selected_records = []

for target in TARGETS:
    df[target] = pd.to_numeric(df[target], errors="coerce")

    if df[target].notna().sum() == 0:
        fail(f"Target has no valid numeric observations: {target}")

    for architecture_name, features in architectures.items():

        for outer_fold in range(1, N_OUTER_FOLDS + 1):

            # The outer test partition is never touched during selection.
            outer_train = df.loc[
                fold_df[FOLD_COLUMN] != outer_fold
            ].copy()

            outer_train = outer_train.loc[
                outer_train[target].notna()
            ].copy()

            if outer_train.empty:
                fail(
                    f"No valid training observations for {target}, "
                    f"{architecture_name}, outer fold {outer_fold}."
                )

            inner_splits = create_inner_position_splits(
                outer_train,
                outer_fold
            )

            summaries = []

            for model_name in MODEL_NAMES:

                scores = []

                for (
                    train_idx,
                    valid_idx,
                    inner_fold,
                    train_positions,
                    valid_positions,
                ) in inner_splits:

                    inner_train = outer_train.iloc[train_idx].copy()
                    inner_valid = outer_train.iloc[valid_idx].copy()

                    overlap = set(
                        inner_train[POSITION_COLUMN].dropna().unique()
                    ).intersection(
                        set(inner_valid[POSITION_COLUMN].dropna().unique())
                    )

                    if overlap:
                        fail(
                            f"Inner position leakage detected for "
                            f"{target}, {architecture_name}, outer "
                            f"fold {outer_fold}, inner fold {inner_fold}."
                        )

                    X_train = inner_train[features]
                    y_train = inner_train[target]

                    X_valid = inner_valid[features]
                    y_valid = inner_valid[target]

                    model = build_pipeline(
                        features,
                        model_name
                    )

                    model.fit(X_train, y_train)

                    prediction = model.predict(X_valid)
                    score = float(
                        r2_score(y_valid, prediction)
                    )

                    scores.append(score)

                    inner_records.append({
                        "Target": target,
                        "Architecture": architecture_name,
                        "Outer_Fold": outer_fold,
                        "Model": model_name,
                        "Inner_Fold": inner_fold,
                        "Inner_R2": score,
                        "Inner_Train_Rows": len(inner_train),
                        "Inner_Validation_Rows": len(inner_valid),
                        "Inner_Train_Unique_Positions": len(train_positions),
                        "Inner_Validation_Unique_Positions": len(valid_positions),
                        "Position_Overlap_Count": 0,
                    })

                mean_r2 = float(np.mean(scores))
                sd_r2 = float(np.std(scores, ddof=1))

                record = {
                    "Target": target,
                    "Architecture": architecture_name,
                    "Outer_Fold": outer_fold,
                    "Model": model_name,
                    "Mean_Inner_R2": mean_r2,
                    "SD_Inner_R2": sd_r2,
                    "N_Inner_Folds": len(scores),
                    "Inner_R2_Fold_1": scores[0],
                    "Inner_R2_Fold_2": scores[1],
                    "Inner_R2_Fold_3": scores[2],
                    "Outer_Test_Used_For_Selection": False,
                    "Part14_Results_Used_For_Selection": False,
                }

                candidate_records.append(record)
                summaries.append(record)

            # Deterministic selection: highest mean inner R2.
            # Alphabetical model name is the deterministic tie-breaker.
            winner = sorted(
                summaries,
                key=lambda x: (-x["Mean_Inner_R2"], x["Model"])
            )[0]

            selected_records.append({
                "Target": target,
                "Architecture": architecture_name,
                "Outer_Fold": outer_fold,
                "Selected_Model": winner["Model"],
                "Selected_Mean_Inner_R2": winner["Mean_Inner_R2"],
                "Selected_SD_Inner_R2": winner["SD_Inner_R2"],
                "Selection_Metric": "Mean_Inner_R2",
                "Selection_Basis": (
                    "Position-aware inner CV on outer-training data only"
                ),
                "Outer_Test_Used_For_Selection": False,
                "Part14_Results_Used_For_Selection": False,
                "Hyperparameter_Optimization_Performed": False,
            })

            print(
                f"{target} | {architecture_name} | "
                f"Outer fold {outer_fold}: "
                f"{winner['Model']} "
                f"(inner R2={winner['Mean_Inner_R2']:.6f})"
            )


# ============================================================================
# SAVE OUTPUTS
# ============================================================================

candidate_df = pd.DataFrame(candidate_records)
inner_df = pd.DataFrame(inner_records)
selected_df = pd.DataFrame(selected_records)

candidate_file = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step2_PositionAware_Candidate_Selection.csv"
)
inner_file = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step2_PositionAware_InnerFold_Performance.csv"
)
selected_file = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step2_PositionAware_Selected_Candidates.csv"
)

candidate_df.to_csv(candidate_file, index=False)
inner_df.to_csv(inner_file, index=False)
selected_df.to_csv(selected_file, index=False)


# ============================================================================
# STRICT OUTPUT VALIDATION
# ============================================================================

expected_selected = len(TARGETS) * 2 * N_OUTER_FOLDS
expected_candidates = expected_selected * len(MODEL_NAMES)

if len(selected_df) != expected_selected:
    fail(
        f"Expected {expected_selected} selected rows, "
        f"found {len(selected_df)}."
    )

if len(candidate_df) != expected_candidates:
    fail(
        f"Expected {expected_candidates} candidate rows, "
        f"found {len(candidate_df)}."
    )

key = ["Target", "Architecture", "Outer_Fold"]

if selected_df.duplicated(key).any():
    fail("Duplicate Target x Architecture x Outer_Fold selections detected.")

if not (selected_df["Outer_Test_Used_For_Selection"] == False).all():
    fail("Outer-test usage violation detected.")

if not (selected_df["Part14_Results_Used_For_Selection"] == False).all():
    fail("Part 14 usage violation detected.")

if (inner_df["Position_Overlap_Count"] != 0).any():
    fail("Inner position overlap detected in saved results.")


# ============================================================================
# LEAKAGE AUDIT
# ============================================================================

audit_df = pd.DataFrame([
    {
        "Check": "Part 13 outer folds reused",
        "Status": "PASS",
        "Details": "No new outer split was generated.",
    },
    {
        "Check": "Outer position disjointness",
        "Status": "PASS",
        "Details": "All five outer train/test partitions are position-disjoint.",
    },
    {
        "Check": "Inner position disjointness",
        "Status": "PASS",
        "Details": "All inner train/validation partitions are position-disjoint.",
    },
    {
        "Check": "Outer test used for selection",
        "Status": "PASS",
        "Details": "Outer test observations were not used.",
    },
    {
        "Check": "Part 14 used for selection",
        "Status": "PASS",
        "Details": "Part 14 results are not loaded or consulted.",
    },
    {
        "Check": "Target leakage",
        "Status": "PASS",
        "Details": "Phenotype targets are excluded from predictors.",
    },
    {
        "Check": "SD leakage",
        "Status": "PASS",
        "Details": "Phenotype SD columns are excluded from predictors.",
    },
    {
        "Check": "Metadata leakage",
        "Status": "PASS",
        "Details": "Identifiers, identity/group, and fold labels are excluded.",
    },
    {
        "Check": "Model A architecture",
        "Status": "PASS",
        "Details": "Exactly 65 predictors; Sequence_Position absent.",
    },
    {
        "Check": "Model B architecture",
        "Status": "PASS",
        "Details": "Exactly 66 predictors; exactly Model A + Sequence_Position.",
    },
    {
        "Check": "Preprocessing leakage",
        "Status": "PASS",
        "Details": "Preprocessing is fitted inside each inner training split.",
    },
    {
        "Check": "HPO in Step 2",
        "Status": "PASS",
        "Details": "No hyperparameter optimization; Step 3 remains the HPO stage.",
    },
])

audit_file = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step2_PositionAware_Leakage_Audit.csv"
)
audit_df.to_csv(audit_file, index=False)


# ============================================================================
# REPRODUCIBILITY MANIFEST
# ============================================================================

run_manifest = {
    "project": "VIM-2 mutation phenotype prediction",
    "step": "Step 2",
    "purpose": "Position-aware candidate model-family selection",
    "input_dataset": DATA_FILE,
    "part13_fold_file": FOLD_FILE,
    "part13_feature_manifest": MANIFEST_FILE,
    "output_directory": OUTPUT_DIR,
    "n_rows": int(len(df)),
    "targets": TARGETS,
    "architectures": {
        MODEL_A_NAME: {
            "n_predictors": len(MODEL_A_FEATURES),
            "sequence_position_included": False,
            "predictors": MODEL_A_FEATURES,
        },
        MODEL_B_NAME: {
            "n_predictors": len(MODEL_B_FEATURES),
            "sequence_position_included": True,
            "predictors": MODEL_B_FEATURES,
        },
    },
    "outer_cv": {
        "source": "Part 13 PositionAware_Fold",
        "n_folds": N_OUTER_FOLDS,
        "new_split_generated": False,
        "grouping_variable": POSITION_COLUMN,
    },
    "inner_cv": {
        "type": "position-level KFold",
        "n_folds": N_INNER_FOLDS,
        "shuffle": True,
        "random_state": "42 + outer_fold",
    },
    "candidate_models": MODEL_NAMES,
    "selection_metric": "Mean inner-validation R2",
    "selection_uses_outer_test": False,
    "selection_uses_part14": False,
    "hyperparameter_optimization": False,
    "random_state": RANDOM_STATE,
    "python_version": sys.version,
    "platform": platform.platform(),
}

manifest_file = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step2_PositionAware_Manifest.json"
)

with open(manifest_file, "w", encoding="utf-8") as handle:
    json.dump(
        run_manifest,
        handle,
        indent=2,
        ensure_ascii=False
    )


# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2 COMPLETED SUCCESSFULLY")
print("=" * 80)
print(f"Dataset rows              : {len(df):,}")
print(f"Model A predictors        : {len(MODEL_A_FEATURES)}")
print(f"Model B predictors        : {len(MODEL_B_FEATURES)}")
print(f"Outer folds               : {N_OUTER_FOLDS}")
print(f"Inner folds               : {N_INNER_FOLDS}")
print(f"Candidate models          : {len(MODEL_NAMES)}")
print(f"Selected configurations   : {len(selected_df)}")
print()
print("Selection basis:")
print("  Position-aware inner CV using outer-training data only")
print()
print("Part 14 results            : NOT USED")
print("Outer-test data            : NOT USED")
print("Hyperparameter optimization: NOT PERFORMED")
print()
print("Output directory:")
print(f"  {OUTPUT_DIR}")
print()
print("Files:")
print(f"  {candidate_file}")
print(f"  {inner_file}")
print(f"  {selected_file}")
print(f"  {audit_file}")
print(f"  {manifest_file}")
print("=" * 80)


STEP 2 — POSITION-AWARE CANDIDATE MODEL SELECTION
Dataset shape : (5016, 89)
Fold shape    : (5016, 7)
Manifest shape: (91, 7)
[PASS] Part 13 fold assignments are aligned.
[PASS] Model A = exactly 65 Part 13 predictors.
[PASS] Model B = exactly 66 Part 13 predictors.
[PASS] Model B differs from Model A only by Sequence_Position.
[PASS] Targets, SDs, identifiers, metadata, and fold labels excluded.
[PASS] All Part 13 outer folds are position-disjoint.
0.031ug/mL_MEM_37C | Model_A_No_Position | Outer fold 1: Random_Forest (inner R2=0.512602)
0.031ug/mL_MEM_37C | Model_A_No_Position | Outer fold 2: Random_Forest (inner R2=0.509867)
0.031ug/mL_MEM_37C | Model_A_No_Position | Outer fold 3: Random_Forest (inner R2=0.483100)
0.031ug/mL_MEM_37C | Model_A_No_Position | Outer fold 4: Random_Forest (inner R2=0.468441)
0.031ug/mL_MEM_37C | Model_A_No_Position | Outer fold 5: Random_Forest (inner R2=0.509052)
0.031ug/mL_MEM_37C | Model_B_With_Position | Outer fold 1: Random_Forest (inner R2=0.51740

In [ ]:
# @title
# ============================================================
# DOWNLOAD step2 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step2_PositionAware_Candidate_Selection"

# Output ZIP archive
zip_base = "/content/VIM2_Step2_PositionAware_Candidate_Selection"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step2_PositionAware_Candidate_Selection"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("step2 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

PART 15.1 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step2_PositionAware_Candidate_Selection
ZIP archive      : /content/VIM2_Step2_PositionAware_Candidate_Selection.zip
Archive size     : 0.02 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
VIM-2 MACHINE LEARNING PROJECT
STEP 3 — POSITION-AWARE NESTED HYPERPARAMETER OPTIMIZATION

This version adds:
    * clear 90-configuration progress reporting
    * elapsed time
    * rolling average runtime
    * ETA
    * current Target / Architecture / Outer Fold
    * incremental checkpoint saving
    * automatic resume after interruption
    * strict leakage audits
    * reproducibility manifest

Methodological design
---------------------
Part 13
    -> frozen Position-Aware outer folds
    -> Step 2 model-family selection
    -> Step 3 fold-specific HPO using outer-training data only
    -> Step 4 final evaluation on frozen outer test sets

Step 3 NEVER:
    * uses Part 14
    * evaluates on the outer test set
    * re-selects RF versus Extra Trees
    * averages hyperparameters across folds

Model architecture:
    Model A = exactly 65 predictors, without Sequence_Position
    Model B = exactly 66 predictors, with Sequence_Position
"""

import ast
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# ============================================================================
# 1. PROJECT CONFIGURATION
# ============================================================================

DATA_FILE = "/content/VIM2_Part12H_v2_Final_ML_Dataset.csv"

FOLD_FILE = "VIM2_Part13_Fold_Assignments.csv"
FEATURE_MANIFEST_FILE = "VIM2_Part13_Feature_Manifest.csv"

STEP2_DIR = "/content/VIM2_Step2_PositionAware_Candidate_Selection"
STEP2_SELECTED_FILE = os.path.join(
    STEP2_DIR,
    "VIM2_Step2_PositionAware_Selected_Candidates.csv",
)

OUTPUT_DIR = "/content/VIM2_Step3_PositionAware_Nested_HPO"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Part 13 files are expected in the current working directory.
# This keeps the paths exactly aligned with the project structure requested
# for Step 3. A clear error is raised if they are not present.
FOLD_FILE = os.path.abspath(FOLD_FILE)
FEATURE_MANIFEST_FILE = os.path.abspath(FEATURE_MANIFEST_FILE)

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
HPO_ITERATIONS = 6

POSITION_COLUMN = "Sequence_Position"
FOLD_COLUMN = "PositionAware_Fold"

MODEL_A = "Model_A_No_Position"
MODEL_B = "Model_B_With_Position"

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

TOTAL_CONFIGURATIONS = (
    len(TARGETS) * 2 * N_OUTER_FOLDS
)

EXPECTED_HPO_TESTS = (
    TOTAL_CONFIGURATIONS * HPO_ITERATIONS
)


# ============================================================================
# 2. OUTPUT / CHECKPOINT FILES
# ============================================================================

BEST_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_Best_Hyperparameters_Per_Fold.csv",
)

ALL_SEARCH_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_All_HPO_Configurations.csv",
)

TIMING_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_HPO_Timing.csv",
)

OUTER_AUDIT_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_Outer_Position_Audit.csv",
)

LEAKAGE_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_Leakage_Audit.csv",
)

MANIFEST_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_Reproducibility_Manifest.json",
)

CHECKPOINT_FILE = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step3_Checkpoint.json",
)


# ============================================================================
# 3. UTILITY FUNCTIONS
# ============================================================================

def fail(message):
    """Raise a descriptive error and stop execution."""
    raise RuntimeError(message)


def sha256_file(path):
    """Calculate SHA-256 for reproducibility tracking."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def format_duration(seconds):
    """Format seconds as a human-readable duration."""
    seconds = max(0, float(seconds))

    days = int(seconds // 86400)
    seconds %= 86400

    hours = int(seconds // 3600)
    seconds %= 3600

    minutes = int(seconds // 60)
    seconds = int(seconds % 60)

    if days:
        return f"{days}d {hours}h {minutes}m"
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"

    return f"{seconds}s"


def print_progress(
    completed,
    total,
    elapsed_seconds,
    current_label,
):
    """
    Print a compact progress panel with ETA.

    ETA is based on the mean runtime of completed configurations.
    It becomes available after the first completed configuration.
    """
    fraction = completed / total if total else 0.0
    bar_width = 32
    filled = int(round(
        fraction * bar_width
    ))
    filled = min(
        bar_width,
        max(0, filled),
    )

    bar = (
        "█" * filled
        + "░" * (bar_width - filled)
    )

    if completed > 0:
        average = (
            elapsed_seconds / completed
        )
        remaining = (
            average * (total - completed)
        )
        eta = format_duration(remaining)
        average_text = format_duration(average)
    else:
        eta = "calculating..."
        average_text = "calculating..."

    percent = 100.0 * fraction

    print()
    print("Step 3 Progress")
    print("━" * 54)
    print(
        f"{bar}  {percent:5.1f}%"
    )
    print()
    print(
        f"Completed : {completed} / {total}"
    )
    print(
        f"Elapsed   : {format_duration(elapsed_seconds)}"
    )
    print(
        f"Average   : {average_text} / config"
    )
    print(
        f"ETA       : {eta}"
    )
    print(
        f"Current   : {current_label}"
    )
    print("━" * 54)


def normalize_model_name(value):
    """Normalize Step 2 model-family labels."""
    mapping = {
        "RF": "Random_Forest",
        "Random Forest": "Random_Forest",
        "Random_Forest": "Random_Forest",
        "ET": "Extra_Trees",
        "Extra Trees": "Extra_Trees",
        "Extra_Trees": "Extra_Trees",
    }

    value = str(value).strip()

    if value not in mapping:
        fail(
            f"Unsupported Step 2 model label: {value}"
        )

    return mapping[value]


def manifest_bool(series):
    """Convert common CSV boolean encodings into booleans."""
    values = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    true_values = {
        "true", "1", "yes", "y", "t"
    }

    false_values = {
        "false", "0", "no", "n", "f"
    }

    unknown = (
        set(values.unique())
        - true_values
        - false_values
    )

    if unknown:
        fail(
            "Unexpected boolean values in feature manifest: "
            f"{sorted(unknown)}"
        )

    return values.isin(true_values)


def build_onehot_encoder():
    """
    Build a version-compatible OneHotEncoder.

    Newer scikit-learn versions use sparse_output=False.
    Older versions use sparse=False.
    """
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_preprocessor(features):
    """
    Build preprocessing inside the sklearn Pipeline.

    Consequently, imputation, scaling, and categorical encoding are fitted
    only on the training portion of each inner CV split.
    """
    categorical_features = [
        feature
        for feature in features
        if feature == "Secondary_Structure"
    ]

    numeric_features = [
        feature
        for feature in features
        if feature != "Secondary_Structure"
    ]

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    transformers = [
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        )
    ]

    if categorical_features:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "onehot",
                    build_onehot_encoder(),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_features,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


def build_estimator(model_name):
    """Create the model family selected by Step 2."""
    if model_name == "Random_Forest":
        return RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )

    if model_name == "Extra_Trees":
        return ExtraTreesRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )

    fail(
        f"Unsupported model family: {model_name}"
    )


def build_pipeline(features, model_name):
    """Create the complete preprocessing + estimator pipeline."""
    return Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor(features),
            ),
            (
                "model",
                build_estimator(model_name),
            ),
        ]
    )


def build_search_space(model_name):
    """Return the constrained randomized HPO search space."""
    common = {
        "model__n_estimators": [
            150,
            250,
            350,
        ],
        "model__max_depth": [
            None,
            10,
            20,
            30,
        ],
        "model__min_samples_split": [
            2,
            5,
            10,
        ],
        "model__min_samples_leaf": [
            1,
            2,
            4,
        ],
    }

    if model_name == "Random_Forest":
        common["model__max_features"] = [
            "sqrt",
            0.5,
            0.8,
        ]

    elif model_name == "Extra_Trees":
        common["model__max_features"] = [
            0.5,
            0.8,
            1.0,
        ]

    else:
        fail(
            f"Unsupported model family: {model_name}"
        )

    return common


def create_inner_position_splits(
    outer_train,
    outer_fold,
):
    """
    Create position-disjoint inner CV splits.

    KFold is applied to unique residue positions, not individual mutation
    rows. Rows sharing the same Sequence_Position therefore remain together.
    """
    positions = (
        outer_train[POSITION_COLUMN]
        .dropna()
        .unique()
    )

    if len(positions) < N_INNER_FOLDS:
        fail(
            f"Outer fold {outer_fold} contains only "
            f"{len(positions)} unique positions; "
            f"{N_INNER_FOLDS} are required."
        )

    splitter = KFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_STATE + outer_fold
        ),
    )

    position_array = np.asarray(
        positions
    )

    splits = []

    for inner_fold, (
        train_pos_idx,
        valid_pos_idx,
    ) in enumerate(
        splitter.split(position_array),
        start=1,
    ):
        train_positions = set(
            position_array[train_pos_idx]
        )

        valid_positions = set(
            position_array[valid_pos_idx]
        )

        overlap = (
            train_positions
            .intersection(valid_positions)
        )

        if overlap:
            fail(
                f"Inner position leakage detected: "
                f"outer fold {outer_fold}, "
                f"inner fold {inner_fold}: "
                f"{sorted(overlap)}"
            )

        train_idx = np.flatnonzero(
            outer_train[
                POSITION_COLUMN
            ]
            .isin(train_positions)
            .to_numpy()
        )

        valid_idx = np.flatnonzero(
            outer_train[
                POSITION_COLUMN
            ]
            .isin(valid_positions)
            .to_numpy()
        )

        splits.append(
            (
                train_idx,
                valid_idx,
            )
        )

    return splits


def save_checkpoint(
    completed_keys,
    best_records,
    all_search_records,
    timing_records,
):
    """
    Save an atomic checkpoint after every completed configuration.

    This allows the run to resume after an interruption without repeating
    completed Target × Architecture × Outer_Fold configurations.
    """
    checkpoint = {
        "completed_keys": sorted(
            list(completed_keys)
        ),
        "best_records": best_records,
        "all_search_records": all_search_records,
        "timing_records": timing_records,
        "saved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    temporary_file = (
        CHECKPOINT_FILE + ".tmp"
    )

    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            checkpoint,
            handle,
            indent=2,
            ensure_ascii=False,
        )

    os.replace(
        temporary_file,
        CHECKPOINT_FILE,
    )


def load_checkpoint():
    """Load a previous checkpoint if one exists."""
    if not os.path.isfile(
        CHECKPOINT_FILE
    ):
        return (
            set(),
            [],
            [],
            [],
        )

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8",
    ) as handle:
        checkpoint = json.load(handle)

    completed_keys = set(
        checkpoint.get(
            "completed_keys",
            [],
        )
    )

    best_records = checkpoint.get(
        "best_records",
        [],
    )

    all_search_records = checkpoint.get(
        "all_search_records",
        [],
    )

    timing_records = checkpoint.get(
        "timing_records",
        [],
    )

    return (
        completed_keys,
        best_records,
        all_search_records,
        timing_records,
    )


def parse_params(value):
    """Safely parse a serialized parameter dictionary."""
    if isinstance(value, dict):
        return dict(value)

    text = str(value).strip()

    try:
        result = ast.literal_eval(
            text
        )
    except Exception:
        try:
            result = json.loads(
                text
            )
        except Exception:
            fail(
                "Unable to parse parameter object: "
                f"{value}"
            )

    if not isinstance(
        result,
        dict,
    ):
        fail(
            f"Parsed parameters are not a dictionary: "
            f"{result}"
        )

    return result


# ============================================================================
# 4. LOAD INPUTS
# ============================================================================

print("=" * 80)
print("STEP 3 — POSITION-AWARE NESTED HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

required_files = [
    DATA_FILE,
    FOLD_FILE,
    FEATURE_MANIFEST_FILE,
    STEP2_SELECTED_FILE,
]

for required_file in required_files:
    if not os.path.isfile(
        required_file
    ):
        fail(
            "Required input file not found:\n"
            f"{required_file}\n\n"
            "Place the Part 13 files in the current working directory "
            "or update FOLD_FILE / FEATURE_MANIFEST_FILE in Section 1."
        )

df = pd.read_csv(
    DATA_FILE,
    low_memory=False,
)

fold_df = pd.read_csv(
    FOLD_FILE,
    low_memory=False,
)

manifest_df = pd.read_csv(
    FEATURE_MANIFEST_FILE,
    low_memory=False,
)

step2_df = pd.read_csv(
    STEP2_SELECTED_FILE,
    low_memory=False,
)

print(
    f"Dataset shape          : {df.shape}"
)
print(
    f"Part 13 fold shape     : {fold_df.shape}"
)
print(
    f"Feature manifest shape : {manifest_df.shape}"
)
print(
    f"Step 2 output shape    : {step2_df.shape}"
)


# ============================================================================
# 5. VALIDATE PART 13 FROZEN OUTER FOLDS
# ============================================================================

required_fold_columns = {
    "Row_Index",
    "Mutation",
    "WT_AA",
    "Sequence_Position",
    "Mutant_AA",
    FOLD_COLUMN,
}

missing = (
    required_fold_columns
    - set(fold_df.columns)
)

if missing:
    fail(
        "Part 13 fold file is missing columns:\n"
        + "\n".join(sorted(missing))
    )

if len(fold_df) != len(df):
    fail(
        "Dataset and Part 13 fold file have different row counts: "
        f"{len(df)} vs {len(fold_df)}"
    )

if not np.array_equal(
    fold_df["Row_Index"].to_numpy(),
    np.arange(len(df)),
):
    fail(
        "Part 13 Row_Index does not exactly match dataset row order."
    )

for column in [
    "Mutation",
    "Sequence_Position",
]:
    if not np.array_equal(
        fold_df[column]
        .astype(str)
        .to_numpy(),
        df[column]
        .astype(str)
        .to_numpy(),
    ):
        fail(
            f"Part 13 column '{column}' is not aligned with dataset."
        )

observed_folds = sorted(
    fold_df[FOLD_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

if observed_folds != list(
    range(
        1,
        N_OUTER_FOLDS + 1,
    )
):
    fail(
        f"Unexpected outer-fold labels: "
        f"{observed_folds}"
    )

print(
    "[PASS] Part 13 frozen outer folds validated."
)


# ============================================================================
# 6. RECOVER EXACT ARCHITECTURES FROM PART 13 FEATURE MANIFEST
# ============================================================================

required_manifest_columns = {
    "Feature",
    "Used_in_Model_A_No_Position",
    "Used_in_Model_B_With_Position",
}

missing = (
    required_manifest_columns
    - set(manifest_df.columns)
)

if missing:
    fail(
        "Part 13 feature manifest is missing columns:\n"
        + "\n".join(sorted(missing))
    )

features_a = manifest_df.loc[
    manifest_bool(
        manifest_df[
            "Used_in_Model_A_No_Position"
        ]
    ),
    "Feature",
].astype(str).tolist()

features_b = manifest_df.loc[
    manifest_bool(
        manifest_df[
            "Used_in_Model_B_With_Position"
        ]
    ),
    "Feature",
].astype(str).tolist()

if len(features_a) != 65:
    fail(
        f"Model A must contain exactly 65 predictors; "
        f"found {len(features_a)}."
    )

if len(features_b) != 66:
    fail(
        f"Model B must contain exactly 66 predictors; "
        f"found {len(features_b)}."
    )

if POSITION_COLUMN in features_a:
    fail(
        "Sequence_Position is incorrectly included in Model A."
    )

if POSITION_COLUMN not in features_b:
    fail(
        "Sequence_Position is missing from Model B."
    )

if set(features_b) != (
    set(features_a)
    | {POSITION_COLUMN}
):
    fail(
        "Model B is not exactly Model A + Sequence_Position."
    )

for feature in features_b:
    if feature not in df.columns:
        fail(
            f"Manifest feature not found in dataset: {feature}"
        )

ARCHITECTURES = {
    MODEL_A: features_a,
    MODEL_B: features_b,
}

print(
    "[PASS] Model A contains exactly 65 predictors."
)
print(
    "[PASS] Model B contains exactly 66 predictors."
)
print(
    "[PASS] Model B differs from Model A only by Sequence_Position."
)


# ============================================================================
# 7. VALIDATE STEP 2 MODEL-FAMILY SELECTION
# ============================================================================

required_step2_columns = {
    "Target",
    "Architecture",
    "Outer_Fold",
    "Selected_Model",
}

missing = (
    required_step2_columns
    - set(step2_df.columns)
)

if missing:
    fail(
        "Step 2 selected-candidate file is missing columns:\n"
        + "\n".join(sorted(missing))
    )

step2_df["Target"] = (
    step2_df["Target"]
    .astype(str)
    .str.strip()
)

step2_df["Architecture"] = (
    step2_df["Architecture"]
    .astype(str)
    .str.strip()
)

step2_df["Selected_Model"] = (
    step2_df["Selected_Model"]
    .map(normalize_model_name)
)

step2_df["Outer_Fold"] = (
    pd.to_numeric(
        step2_df["Outer_Fold"],
        errors="raise",
    )
    .astype(int)
)

if len(step2_df) != TOTAL_CONFIGURATIONS:
    fail(
        f"Expected {TOTAL_CONFIGURATIONS} Step 2 selections; "
        f"found {len(step2_df)}."
    )

if set(step2_df["Target"]) != set(
    TARGETS
):
    fail(
        "Step 2 target coverage is incomplete or incorrect."
    )

if set(
    step2_df["Architecture"]
) != {
    MODEL_A,
    MODEL_B,
}:
    fail(
        "Step 2 architecture coverage is incorrect."
    )

if step2_df.duplicated(
    [
        "Target",
        "Architecture",
        "Outer_Fold",
    ]
).any():
    fail(
        "Duplicate Step 2 selection detected."
    )

print(
    f"[PASS] Step 2 provides exactly "
    f"{TOTAL_CONFIGURATIONS} model-family selections."
)


# ============================================================================
# 8. OUTER POSITION-DISJOINTNESS AUDIT
# ============================================================================

outer_audit = []

for outer_fold in range(
    1,
    N_OUTER_FOLDS + 1,
):
    train_positions = set(
        fold_df.loc[
            fold_df[FOLD_COLUMN]
            != outer_fold,
            POSITION_COLUMN,
        ]
        .dropna()
        .unique()
    )

    test_positions = set(
        fold_df.loc[
            fold_df[FOLD_COLUMN]
            == outer_fold,
            POSITION_COLUMN,
        ]
        .dropna()
        .unique()
    )

    overlap = (
        train_positions
        .intersection(test_positions)
    )

    if overlap:
        fail(
            f"Outer position leakage in fold "
            f"{outer_fold}: {sorted(overlap)}"
        )

    outer_audit.append({
        "Outer_Fold": outer_fold,
        "Train_Unique_Positions": len(
            train_positions
        ),
        "Test_Unique_Positions": len(
            test_positions
        ),
        "Position_Overlap_Count": len(
            overlap
        ),
    })

outer_audit_df = pd.DataFrame(
    outer_audit
)

outer_audit_df.to_csv(
    OUTER_AUDIT_FILE,
    index=False,
)

print(
    "[PASS] Outer folds are residue-position disjoint."
)


# ============================================================================
# 9. LOAD EXISTING CHECKPOINT / RESUME SAFELY
# ============================================================================

(
    completed_keys,
    best_records,
    all_search_records,
    timing_records,
) = load_checkpoint()

# Remove any malformed duplicate checkpoint records before resuming.
completed_keys = set(
    completed_keys
)

valid_completed_keys = set()

for key in completed_keys:
    parts = key.split("|||")
    if len(parts) == 3:
        valid_completed_keys.add(key)

completed_keys = valid_completed_keys

previous_completed = len(
    completed_keys
)

if previous_completed:
    print()
    print(
        f"[RESUME] Found checkpoint with "
        f"{previous_completed}/{TOTAL_CONFIGURATIONS} "
        f"completed configurations."
    )
else:
    print()
    print(
        "[START] No previous checkpoint found. "
        "Starting Step 3 from configuration 1."
    )


# ============================================================================
# 10. RUN FOLD-SPECIFIC HPO WITH PROGRESS + ETA
# ============================================================================

run_start = time.time()

for target in TARGETS:

    df[target] = pd.to_numeric(
        df[target],
        errors="coerce",
    )

    for architecture, features in (
        ARCHITECTURES.items()
    ):

        selection_subset = step2_df.loc[
            (
                step2_df["Target"]
                == target
            )
            &
            (
                step2_df["Architecture"]
                == architecture
            )
        ].copy()

        for outer_fold in range(
            1,
            N_OUTER_FOLDS + 1,
        ):

            key = (
                f"{target}|||"
                f"{architecture}|||"
                f"{outer_fold}"
            )

            if key in completed_keys:
                continue

            selection = selection_subset.loc[
                selection_subset["Outer_Fold"]
                == outer_fold
            ]

            if len(selection) != 1:
                fail(
                    f"Expected exactly one Step 2 selection for "
                    f"{target} / {architecture} / "
                    f"outer fold {outer_fold}; "
                    f"found {len(selection)}."
                )

            selected_model = selection.iloc[0][
                "Selected_Model"
            ]

            current_number = (
                len(completed_keys) + 1
            )

            current_label = (
                f"{target} — "
                f"{architecture.replace('_', ' ')} — "
                f"Fold {outer_fold}"
            )

            elapsed_before = (
                time.time() - run_start
            )

            print_progress(
                completed=len(
                    completed_keys
                ),
                total=TOTAL_CONFIGURATIONS,
                elapsed_seconds=(
                    elapsed_before
                ),
                current_label=current_label,
            )

            print(
                f"\n[{current_number}/{TOTAL_CONFIGURATIONS}] "
                f"Starting {current_label}"
            )
            print(
                f"Selected model family: "
                f"{selected_model}"
            )

            # ---------------------------------------------------------------
            # CRITICAL LEAKAGE CONTROL:
            # Only the outer-training partition is supplied to HPO.
            # Outer-test rows are never passed to RandomizedSearchCV.
            # ---------------------------------------------------------------
            outer_train = df.loc[
                fold_df[FOLD_COLUMN]
                != outer_fold
            ].copy()

            outer_train = outer_train.loc[
                outer_train[target].notna()
            ].copy()

            if outer_train.empty:
                fail(
                    f"No valid outer-training observations for "
                    f"{target}, {architecture}, "
                    f"fold {outer_fold}."
                )

            inner_cv = (
                create_inner_position_splits(
                    outer_train,
                    outer_fold,
                )
            )

            pipeline = build_pipeline(
                features,
                selected_model,
            )

            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=(
                    build_search_space(
                        selected_model
                    )
                ),
                n_iter=HPO_ITERATIONS,
                scoring="r2",
                cv=inner_cv,
                refit=True,
                random_state=(
                    RANDOM_STATE
                    + outer_fold
                ),
                n_jobs=-1,
                pre_dispatch="2*n_jobs",
                return_train_score=False,
                error_score="raise",
            )

            X_outer_train = (
                outer_train[features]
            )

            y_outer_train = (
                outer_train[target]
            )

            config_start = time.time()

            # HPO is performed only within the outer-training partition.
            # The best fitted pipeline is retained for Step 4, but Step 3
            # deliberately does NOT evaluate it on the outer test partition.
            search.fit(
                X_outer_train,
                y_outer_train,
            )

            config_elapsed = (
                time.time()
                - config_start
            )

            best_params = dict(
                search.best_params_
            )

            best_inner_r2 = float(
                search.best_score_
            )

            cv_results = pd.DataFrame(
                search.cv_results_
            )

            for row in cv_results.itertuples(
                index=False
            ):
                all_search_records.append({
                    "Target": target,
                    "Architecture": architecture,
                    "Outer_Fold": outer_fold,
                    "Model": selected_model,
                    "Candidate_Rank": int(
                        getattr(
                            row,
                            "rank_test_score",
                        )
                    ),
                    "Mean_Inner_R2": float(
                        getattr(
                            row,
                            "mean_test_score",
                        )
                    ),
                    "SD_Inner_R2": float(
                        getattr(
                            row,
                            "std_test_score",
                        )
                    ),
                    "Parameters": str(
                        getattr(
                            row,
                            "params",
                        )
                    ),
                    "Outer_Test_Used_For_HPO": False,
                    "Part14_Used_For_HPO": False,
                })

            best_records.append({
                "Target": target,
                "Architecture": architecture,
                "Model": selected_model,
                "Outer_Fold": outer_fold,
                "Best_Inner_R2": best_inner_r2,
                "Best_Hyperparameters": str(
                    best_params
                ),
                "HPO_Iterations": HPO_ITERATIONS,
                "Inner_Folds": N_INNER_FOLDS,
                "Selection_Source": (
                    "Step2_Selected_Model_Family"
                ),
                "HPO_Scoring": "r2",
                "Outer_Test_Used_For_HPO": False,
                "Part14_Used_For_HPO": False,
                "Hyperparameters_Frozen_For_Step4": True,
                "Configuration_Runtime_Seconds": (
                    config_elapsed
                ),
            })

            timing_records.append({
                "Target": target,
                "Architecture": architecture,
                "Model": selected_model,
                "Outer_Fold": outer_fold,
                "Runtime_Seconds": (
                    config_elapsed
                ),
                "Runtime_Minutes": (
                    config_elapsed / 60.0
                ),
            })

            completed_keys.add(
                key
            )

            # ---------------------------------------------------------------
            # Incremental checkpoint:
            # save immediately after every completed configuration.
            # ---------------------------------------------------------------
            save_checkpoint(
                completed_keys,
                best_records,
                all_search_records,
                timing_records,
            )

            # Save human-readable CSVs incrementally as well.
            pd.DataFrame(
                best_records
            ).to_csv(
                BEST_FILE,
                index=False,
            )

            pd.DataFrame(
                all_search_records
            ).to_csv(
                ALL_SEARCH_FILE,
                index=False,
            )

            pd.DataFrame(
                timing_records
            ).to_csv(
                TIMING_FILE,
                index=False,
            )

            total_elapsed = (
                time.time()
                - run_start
            )

            print(
                f"\n[PASS] {current_label}"
            )
            print(
                f"Best inner R2 : "
                f"{best_inner_r2:.6f}"
            )
            print(
                f"Runtime       : "
                f"{format_duration(config_elapsed)}"
            )
            print(
                f"Frozen params : "
                f"{best_params}"
            )

            print_progress(
                completed=len(
                    completed_keys
                ),
                total=TOTAL_CONFIGURATIONS,
                elapsed_seconds=(
                    total_elapsed
                ),
                current_label="Next configuration",
            )


# ============================================================================
# 11. FINALIZE OUTPUT TABLES
# ============================================================================

best_df = pd.DataFrame(
    best_records
)

all_search_df = pd.DataFrame(
    all_search_records
)

timing_df = pd.DataFrame(
    timing_records
)

best_df.to_csv(
    BEST_FILE,
    index=False,
)

all_search_df.to_csv(
    ALL_SEARCH_FILE,
    index=False,
)

timing_df.to_csv(
    TIMING_FILE,
    index=False,
)


# ============================================================================
# 12. STRICT OUTPUT VALIDATION
# ============================================================================

if len(best_df) != TOTAL_CONFIGURATIONS:
    fail(
        f"Expected {TOTAL_CONFIGURATIONS} frozen HPO rows; "
        f"found {len(best_df)}."
    )

if len(all_search_df) != EXPECTED_HPO_TESTS:
    fail(
        f"Expected {EXPECTED_HPO_TESTS} HPO configurations; "
        f"found {len(all_search_df)}."
    )

if best_df.duplicated(
    [
        "Target",
        "Architecture",
        "Outer_Fold",
    ]
).any():
    fail(
        "Duplicate frozen HPO configurations detected."
    )

if best_df[
    "Outer_Test_Used_For_HPO"
].astype(bool).any():
    fail(
        "Outer-test leakage flag detected."
    )

if best_df[
    "Part14_Used_For_HPO"
].astype(bool).any():
    fail(
        "Part 14 usage detected."
    )

if all_search_df[
    "Outer_Test_Used_For_HPO"
].astype(bool).any():
    fail(
        "Outer-test leakage flag detected in HPO configurations."
    )

if all_search_df[
    "Part14_Used_For_HPO"
].astype(bool).any():
    fail(
        "Part 14 usage detected in HPO configurations."
    )

if not np.all(
    outer_audit_df[
        "Position_Overlap_Count"
    ].to_numpy()
    == 0
):
    fail(
        "Outer position overlap detected."
    )

print()
print("=" * 80)
print("STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    f"Frozen configurations : "
    f"{len(best_df)} / {TOTAL_CONFIGURATIONS}"
)

print(
    f"Tested HPO configs    : "
    f"{len(all_search_df)} / {EXPECTED_HPO_TESTS}"
)

print(
    f"Model A predictors    : "
    f"{len(features_a)}"
)

print(
    f"Model B predictors    : "
    f"{len(features_b)}"
)

print(
    "[PASS] Outer-test data were not used for HPO."
)

print(
    "[PASS] Part 14 was not used."
)

print(
    "[PASS] Hyperparameters are frozen per outer fold."
)

print(
    "[PASS] Step 4 remains the sole final outer-test evaluation stage."
)

print(
    f"\nPrimary Step 4 input:\n{BEST_FILE}"
)


# ============================================================================
# 13. FINAL LEAKAGE AUDIT
# ============================================================================

leakage_df = pd.DataFrame([
    {
        "Check": "Frozen Part 13 outer folds",
        "Status": "PASS",
        "Details": (
            "Original PositionAware_Fold assignments were reused."
        ),
    },
    {
        "Check": "Step 2 model-family dependency",
        "Status": "PASS",
        "Details": (
            "Only the model family selected by Step 2 was optimized."
        ),
    },
    {
        "Check": "Outer-test isolation",
        "Status": "PASS",
        "Details": (
            "Outer-test observations never entered RandomizedSearchCV."
        ),
    },
    {
        "Check": "Part 14 isolation",
        "Status": "PASS",
        "Details": (
            "No Part 14 file or result was used."
        ),
    },
    {
        "Check": "Inner position-aware CV",
        "Status": "PASS",
        "Details": (
            "Residue positions were disjoint between inner train "
            "and validation."
        ),
    },
    {
        "Check": "Model A architecture",
        "Status": "PASS",
        "Details": (
            "Exactly 65 predictors; Sequence_Position excluded."
        ),
    },
    {
        "Check": "Model B architecture",
        "Status": "PASS",
        "Details": (
            "Exactly 66 predictors; Sequence_Position included."
        ),
    },
    {
        "Check": "Preprocessing isolation",
        "Status": "PASS",
        "Details": (
            "Preprocessing is embedded inside the sklearn pipeline."
        ),
    },
    {
        "Check": "Fold-specific hyperparameters",
        "Status": "PASS",
        "Details": (
            "Each outer fold has independently optimized parameters."
        ),
    },
    {
        "Check": "Hyperparameter averaging",
        "Status": "PASS",
        "Details": (
            "No averaging across outer folds."
        ),
    },
    {
        "Check": "Step 3 model-family reselection",
        "Status": "PASS",
        "Details": (
            "Step 3 does not re-select RF versus Extra Trees."
        ),
    },
])

leakage_df.to_csv(
    LEAKAGE_FILE,
    index=False,
)


# ============================================================================
# 14. REPRODUCIBILITY MANIFEST
# ============================================================================

manifest = {
    "project": "VIM-2 Machine Learning",
    "step": "Step 3",
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "random_state": RANDOM_STATE,
    "resume_enabled": True,
    "checkpoint_file": CHECKPOINT_FILE,
    "data": {
        "dataset": DATA_FILE,
        "dataset_sha256": sha256_file(
            DATA_FILE
        ),
        "fold_file": FOLD_FILE,
        "fold_file_sha256": sha256_file(
            FOLD_FILE
        ),
        "feature_manifest": FEATURE_MANIFEST_FILE,
        "feature_manifest_sha256": sha256_file(
            FEATURE_MANIFEST_FILE
        ),
        "step2_selected_candidates": STEP2_SELECTED_FILE,
        "step2_selected_candidates_sha256": sha256_file(
            STEP2_SELECTED_FILE
        ),
    },
    "targets": TARGETS,
    "outer_cv": {
        "type": "Position-Aware",
        "n_folds": N_OUTER_FOLDS,
        "fold_column": FOLD_COLUMN,
    },
    "inner_cv": {
        "type": "Position-Aware KFold",
        "n_folds": N_INNER_FOLDS,
        "shuffle": True,
        "random_state_formula": "42 + outer_fold",
    },
    "architectures": {
        MODEL_A: {
            "n_predictors": len(
                features_a
            ),
            "sequence_position": False,
        },
        MODEL_B: {
            "n_predictors": len(
                features_b
            ),
            "sequence_position": True,
        },
    },
    "hpo": {
        "method": "RandomizedSearchCV",
        "iterations_per_configuration": HPO_ITERATIONS,
        "inner_folds": N_INNER_FOLDS,
        "scoring": "r2",
        "refit": True,
        "outer_test_used": False,
        "part14_used": False,
        "fold_specific": True,
        "cross_fold_parameter_averaging": False,
    },
    "expected": {
        "outer_folds": N_OUTER_FOLDS,
        "architectures": 2,
        "targets": len(TARGETS),
        "frozen_configurations": TOTAL_CONFIGURATIONS,
        "tested_hpo_configurations": EXPECTED_HPO_TESTS,
        "model_fits_including_inner_cv": (
            EXPECTED_HPO_TESTS
            * N_INNER_FOLDS
        ),
    },
    "outputs": {
        "best_hyperparameters": BEST_FILE,
        "all_hpo_configurations": ALL_SEARCH_FILE,
        "timing": TIMING_FILE,
        "outer_position_audit": OUTER_AUDIT_FILE,
        "leakage_audit": LEAKAGE_FILE,
        "reproducibility_manifest": MANIFEST_FILE,
        "checkpoint": CHECKPOINT_FILE,
    },
    "python_version": sys.version,
    "platform": platform.platform(),
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )

# The checkpoint is intentionally retained after successful completion.
# It makes the final state auditable and allows exact resume verification.
print(
    f"\nCheckpoint retained at:\n{CHECKPOINT_FILE}"
)

print(
    f"Reproducibility manifest:\n{MANIFEST_FILE}"
)


STEP 3 — POSITION-AWARE NESTED HYPERPARAMETER OPTIMIZATION
Dataset shape          : (5016, 89)
Part 13 fold shape     : (5016, 7)
Feature manifest shape : (91, 7)
Step 2 output shape    : (90, 11)
[PASS] Part 13 frozen outer folds validated.
[PASS] Model A contains exactly 65 predictors.
[PASS] Model B contains exactly 66 predictors.
[PASS] Model B differs from Model A only by Sequence_Position.
[PASS] Step 2 provides exactly 90 model-family selections.
[PASS] Outer folds are residue-position disjoint.

[START] No previous checkpoint found. Starting Step 3 from configuration 1.

Step 3 Progress
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░    0.0%

Completed : 0 / 90
Elapsed   : 0s
Average   : calculating... / config
ETA       : calculating...
Current   : 0.031ug/mL_MEM_37C — Model A No Position — Fold 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1/90] Starting 0.031ug/mL_MEM_37C — Model A No Position — Fold 1
Selected model famil

In [ ]:
# @title
# ============================================================
# DOWNLOAD step3 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step3_PositionAware_Nested_HPO"

# Output ZIP archive
zip_base = "/content/VIM2_Step3_PositionAware_Nested_HPO"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step3_PositionAware_Nested_HPO"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("step3 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

step3 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step3_PositionAware_Nested_HPO
ZIP archive      : /content/VIM2_Step3_PositionAware_Nested_HPO.zip
Archive size     : 0.05 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
# @title
# ======================================================================
# VIM-2 MACHINE LEARNING PROJECT
# STEP 4 — FINAL OPTIMIZED MODEL EVALUATION
# ======================================================================
#
# PURPOSE
# -------
# This module performs the final evaluation of the optimized machine
# learning models using:
#
#   1. Frozen Position-Aware outer folds defined in Part 13
#   2. Frozen, fold-specific hyperparameters selected in Step 3
#   3. Strict train/test separation within every outer fold
#   4. Separate evaluation of Model A and Model B
#   5. Out-of-fold (OOF) predictions across all samples
#
# IMPORTANT METHODOLOGICAL PRINCIPLES
# -----------------------------------
# • No hyperparameter optimization is performed in Step 4.
# • No model selection is performed using outer-test performance.
# • Hyperparameters are NOT averaged across folds.
# • Each outer fold uses its own frozen Step 3 hyperparameters.
# • All preprocessing is fitted exclusively on the corresponding
#   outer-training partition through a scikit-learn Pipeline.
# • The outer-test partition is used only once for final evaluation.
# • Position-Aware folds are treated as the primary evaluation framework.
#
# INPUT FILES
# -----------
#   VIM2_Part12H_v2_Final_ML_Dataset.csv
#   VIM2_Part13_Fold_Assignments.csv
#   VIM2_Step3_Best_Hyperparameters_Per_Fold.csv
#
# PRIMARY OUTPUT DIRECTORY
# ------------------------
#   /content/VIM2_Step4_Final_Evaluation/
#
# ======================================================================


# ======================================================================
# 0. ENVIRONMENT SETUP
# ======================================================================

import os
import re
import json
import ast
import time
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

warnings.filterwarnings("ignore")


# ======================================================================
# 1. GLOBAL CONFIGURATION
# ======================================================================

PROJECT_NAME = "VIM-2 ML Project"
STEP_NAME = "STEP 4 — FINAL OPTIMIZED MODEL EVALUATION"

RANDOM_STATE = 42

# ----------------------------------------------------------------------
# Core input files
# ----------------------------------------------------------------------

FINAL_DATASET = "/content/VIM2_Part12H_v2_Final_ML_Dataset.csv"

FOLD_ASSIGNMENTS_FILE = "/content/VIM2_Part13_Fold_Assignments.csv"

STEP3_HYPERPARAMETERS_FILE = (
    "/content/VIM2_Step3_Best_Hyperparameters_Per_Fold.csv"
)

FEATURE_MANIFEST_FILE = (
    "/content/VIM2_Part13_Feature_Manifest.csv"
)

STEP2_SELECTED_FILE = (
    "/content/VIM2_Step2_PositionAware_Selected_Candidates.csv"
)

# ----------------------------------------------------------------------
# Output directory
# ----------------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/VIM2_Step4_Final_Evaluation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ----------------------------------------------------------------------
# Outer-fold configuration
# ----------------------------------------------------------------------

N_OUTER_FOLDS = 5

POSITION_FOLD_COLUMN = "PositionAware_Fold"

# ----------------------------------------------------------------------
# Expected targets
# ----------------------------------------------------------------------

TARGET_COLUMNS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C"
]

# ----------------------------------------------------------------------
# Model architectures
# ----------------------------------------------------------------------

MODEL_ARCHITECTURES = {
    "Model_A_No_Position": "Model_A",
    "Model_B_With_Position": "Model_B"
}

# ----------------------------------------------------------------------
# Columns that must never be used as predictors
# ----------------------------------------------------------------------

NON_PREDICTOR_COLUMNS = {
    "Mutation",
    "mutation",
    "Identity",
    "identity",
    "WT_AA",
    "Mutant_AA",
    "WT_Residue",
    "Mutant_Residue",
    "Position",
    "position",
    "Sequence_Position",
    "Fold",
    "Fold_ID",
    "PositionAware_Fold",
    "Random_Fold",
    "Split",
    "Set"
}


# ======================================================================
# 2. UTILITY FUNCTIONS
# ======================================================================

def print_section(title):
    """Print a clean section header for reproducible notebook execution."""
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def sha256_file(filepath, chunk_size=1024 * 1024):
    """
    Calculate a SHA-256 checksum for an input/output file.

    SHA-256 hashes are recorded in the reproducibility manifest so that
    the exact datasets and configuration files used for final evaluation
    can be independently verified.
    """
    h = hashlib.sha256()

    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def normalize_column_name(x):
    """Normalize a column name for robust schema matching."""
    return re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        str(x).strip().lower()
    ).strip("_")


def find_column(df, candidates, required=True):
    """
    Locate a column using exact or normalized-name matching.
    """
    # Exact matching first
    for col in candidates:
        if col in df.columns:
            return col

    # Normalized matching
    normalized = {
        normalize_column_name(c): c
        for c in df.columns
    }

    for candidate in candidates:
        key = normalize_column_name(candidate)

        if key in normalized:
            return normalized[key]

    if required:
        raise KeyError(
            f"None of the expected columns were found: {candidates}\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def safe_spearman(y_true, y_pred):
    """
    Calculate Spearman correlation safely.

    If either vector is constant, Spearman correlation is undefined.
    In that case NaN is returned instead of generating an exception.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if len(y_true) < 2:
        return np.nan

    if np.all(y_true == y_true[0]):
        return np.nan

    if np.all(y_pred == y_pred[0]):
        return np.nan

    rho, _ = spearmanr(y_true, y_pred)

    return float(rho)


def calculate_metrics(y_true, y_pred):
    """
    Calculate the primary regression performance metrics used in Step 4.
    """
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),
        "MAE": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "Spearman": safe_spearman(y_true, y_pred)
    }


def canonicalize_mutation_key(df):
    """
    Construct a stable row identifier for dataset/fold alignment.

    The function prioritizes Mutation / mutation. If those are not
    available, it constructs a composite identifier from wild-type,
    mutant residue and sequence position information.

    This prevents silent row-order mismatches between the final dataset
    and the frozen Part 13 fold assignment file.
    """

    # --------------------------------------------------------------
    # Direct mutation identifier
    # --------------------------------------------------------------

    if "Mutation" in df.columns:
        values = (
            df["Mutation"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        if values.notna().all() and values.nunique() == len(df):
            return values

    if "mutation" in df.columns:
        values = (
            df["mutation"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        if values.notna().all() and values.nunique() == len(df):
            return values

    # --------------------------------------------------------------
    # Identity identifier
    # --------------------------------------------------------------

    for col in ["Identity", "identity"]:
        if col in df.columns:

            values = (
                df[col]
                .astype(str)
                .str.strip()
                .str.upper()
            )

            if values.notna().all() and values.nunique() == len(df):
                return values

    # --------------------------------------------------------------
    # Composite mutation key
    # --------------------------------------------------------------

    wt_col = find_column(
        df,
        ["WT_AA", "WT_Residue"],
        required=False
    )

    mut_col = find_column(
        df,
        ["Mutant_AA", "Mutant_Residue"],
        required=False
    )

    pos_col = find_column(
        df,
        ["Sequence_Position", "Position", "position"],
        required=False
    )

    if wt_col and mut_col and pos_col:

        key = (
            df[wt_col].astype(str).str.upper().str.strip()
            + "_"
            + df[pos_col].astype(str).str.strip()
            + "_"
            + df[mut_col].astype(str).str.upper().str.strip()
        )

        if key.nunique() == len(df):
            return key

    raise RuntimeError(
        "A unique stable mutation identifier could not be constructed. "
        "Dataset/fold alignment cannot be considered safe."
    )


def parse_frozen_hyperparameters(value):
    """
    Safely parse and validate frozen Step 3 hyperparameters.

    Step 3 may serialize dictionaries either as strict JSON or as a
    Python dictionary literal (for example, using single quotes).
    JSON is attempted first; ast.literal_eval() is used as a safe
    fallback. Arbitrary eval() is intentionally never used.

    The schema is deliberately strict because Step 4 must consume the
    exact frozen estimator configuration produced by Step 3.
    """

    if pd.isna(value):
        raise ValueError("Best_Hyperparameters contains a missing value.")

    if isinstance(value, dict):
        params = value
    else:
        text = str(value).strip()
        if not text:
            raise ValueError("Best_Hyperparameters is empty.")

        try:
            params = json.loads(text)
        except json.JSONDecodeError:
            try:
                params = ast.literal_eval(text)
            except (ValueError, SyntaxError) as exc:
                raise ValueError(
                    "Could not parse Best_Hyperparameters as JSON or a "
                    "Python dictionary literal.\n"
                    f"Received value:\n{text}"
                ) from exc

    if not isinstance(params, dict):
        raise ValueError(
            "Best_Hyperparameters must decode to a dictionary. "
            f"Observed type: {type(params).__name__}."
        )

    normalized = {clean_model_parameter_name(k): v for k, v in params.items()}

    required = {
        "model__n_estimators",
        "model__max_depth",
        "model__min_samples_split",
        "model__min_samples_leaf",
        "model__max_features",
    }

    observed = set(normalized)
    missing = required - observed
    extra = observed - required

    if missing:
        raise ValueError(
            "Incomplete Best_Hyperparameters schema. Missing keys: "
            f"{sorted(missing)}"
        )
    if extra:
        raise ValueError(
            "Unexpected Best_Hyperparameters keys: "
            f"{sorted(extra)}. Step 4 requires the exact frozen Step 3 schema."
        )

    def is_plain_int(x):
        return isinstance(x, (int, np.integer)) and not isinstance(x, (bool, np.bool_))

    def is_plain_number(x):
        return (
            isinstance(x, (int, float, np.integer, np.floating))
            and not isinstance(x, (bool, np.bool_))
            and np.isfinite(float(x))
        )

    n_estimators = normalized["model__n_estimators"]
    if not is_plain_int(n_estimators) or int(n_estimators) < 1:
        raise ValueError(
            "Invalid model__n_estimators: expected integer >= 1; "
            f"received {n_estimators!r}."
        )
    normalized["model__n_estimators"] = int(n_estimators)

    max_depth = normalized["model__max_depth"]
    if max_depth is not None and (not is_plain_int(max_depth) or int(max_depth) < 1):
        raise ValueError(
            "Invalid model__max_depth: expected None or integer >= 1; "
            f"received {max_depth!r}."
        )
    if max_depth is not None:
        normalized["model__max_depth"] = int(max_depth)

    min_split = normalized["model__min_samples_split"]
    if is_plain_int(min_split):
        if int(min_split) < 2:
            raise ValueError(
                "Invalid model__min_samples_split: integer values must be >= 2; "
                f"received {min_split!r}."
            )
        normalized["model__min_samples_split"] = int(min_split)
    elif is_plain_number(min_split):
        if not (0.0 < float(min_split) <= 1.0):
            raise ValueError(
                "Invalid model__min_samples_split: float values must be in (0, 1]; "
                f"received {min_split!r}."
            )
        normalized["model__min_samples_split"] = float(min_split)
    else:
        raise ValueError(
            "Invalid model__min_samples_split: expected integer >= 2 or float in (0, 1]; "
            f"received {min_split!r}."
        )

    min_leaf = normalized["model__min_samples_leaf"]
    if is_plain_int(min_leaf):
        if int(min_leaf) < 1:
            raise ValueError(
                "Invalid model__min_samples_leaf: integer values must be >= 1; "
                f"received {min_leaf!r}."
            )
        normalized["model__min_samples_leaf"] = int(min_leaf)
    elif is_plain_number(min_leaf):
        if not (0.0 < float(min_leaf) <= 1.0):
            raise ValueError(
                "Invalid model__min_samples_leaf: float values must be in (0, 1]; "
                f"received {min_leaf!r}."
            )
        normalized["model__min_samples_leaf"] = float(min_leaf)
    else:
        raise ValueError(
            "Invalid model__min_samples_leaf: expected integer >= 1 or float in (0, 1]; "
            f"received {min_leaf!r}."
        )

    max_features = normalized["model__max_features"]
    if isinstance(max_features, str):
        if max_features not in {"sqrt", "log2"}:
            raise ValueError(
                "Invalid model__max_features string: expected 'sqrt' or 'log2'; "
                f"received {max_features!r}."
            )
    elif max_features is None:
        pass
    elif is_plain_int(max_features):
        if int(max_features) < 1:
            raise ValueError(
                "Invalid model__max_features: integer values must be >= 1; "
                f"received {max_features!r}."
            )
        normalized["model__max_features"] = int(max_features)
    elif is_plain_number(max_features):
        if not (0.0 < float(max_features) <= 1.0):
            raise ValueError(
                "Invalid model__max_features: float values must be in (0, 1]; "
                f"received {max_features!r}."
            )
        normalized["model__max_features"] = float(max_features)
    else:
        raise ValueError(
            "Invalid model__max_features: expected None, 'sqrt', 'log2', "
            "integer >= 1, or float in (0, 1]; "
            f"received {max_features!r}."
        )

    return normalized

def clean_model_parameter_name(name):
    """
    Normalize Step 3 hyperparameter names.

    Step 3 stores parameters using Pipeline notation:
        model__max_depth

    The final Step 4 Pipeline uses the same step name, so the prefix
    is intentionally retained.
    """

    name = str(name).strip()

    if name.startswith("model__"):
        return name

    return f"model__{name}"


def normalize_architecture_name(value):
    """
    Normalize architecture labels while preserving the distinction
    between Model A and Model B.
    """
    value = str(value).strip()

    mapping = {
        "Model_A_No_Position": "Model_A_No_Position",
        "Model_A": "Model_A_No_Position",
        "Model_A_No_Sequence_Position": "Model_A_No_Position",

        "Model_B_With_Position": "Model_B_With_Position",
        "Model_B": "Model_B_With_Position",
        "Model_B_Position_Aware": "Model_B_With_Position"
    }

    return mapping.get(value, value)


def normalize_model_name(value):
    """
    Normalize Step 3 model labels.
    """
    value = str(value).strip()

    mapping = {
        "Random_Forest": "Random_Forest",
        "RandomForest": "Random_Forest",
        "Random Forest": "Random_Forest",

        "Extra_Trees": "Extra_Trees",
        "ExtraTrees": "Extra_Trees",
        "Extra Trees": "Extra_Trees"
    }

    return mapping.get(value, value)


def build_regressor(model_name, hyperparameters):
    """
    Construct the frozen optimized estimator selected by Step 3.

    No parameter optimization occurs here.

    The hyperparameters are directly supplied by the Step 3
    Best_Hyperparameters_Per_Fold file.
    """

    params = {}

    for key, value in hyperparameters.items():
        params[clean_model_parameter_name(key)] = value

    model_name = normalize_model_name(model_name)

    if model_name == "Random_Forest":

        estimator = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    elif model_name == "Extra_Trees":

        estimator = ExtraTreesRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    else:

        raise ValueError(
            f"Unsupported model architecture in Step 3: {model_name}"
        )

    # Apply frozen Step 3 parameters
    estimator.set_params(
        **{
            key.replace("model__", ""): value
            for key, value in params.items()
        }
    )

    return estimator


# ======================================================================
# 3. VERIFY INPUT FILES
# ======================================================================

print_section("1/12 — VERIFYING INPUT FILES")

for filepath in [
    FINAL_DATASET,
    FOLD_ASSIGNMENTS_FILE,
    FEATURE_MANIFEST_FILE,
    STEP2_SELECTED_FILE,
    STEP3_HYPERPARAMETERS_FILE
]:

    if not os.path.exists(filepath):

        raise FileNotFoundError(
            f"\nRequired input file was not found:\n{filepath}\n"
        )

    print(f"[OK] {filepath}")


# ======================================================================
# 4. CALCULATE INPUT FILE HASHES
# ======================================================================

print_section("2/12 — CALCULATING INPUT SHA-256 HASHES")

dataset_hash = sha256_file(FINAL_DATASET)
fold_hash = sha256_file(FOLD_ASSIGNMENTS_FILE)
feature_manifest_hash = sha256_file(FEATURE_MANIFEST_FILE)
step2_selected_hash = sha256_file(STEP2_SELECTED_FILE)
step3_hp_hash = sha256_file(STEP3_HYPERPARAMETERS_FILE)

print(f"Final ML Dataset SHA-256 : {dataset_hash}")
print(f"Fold Assignment SHA-256  : {fold_hash}")
print(f"Step 3 HP SHA-256         : {step3_hp_hash}")


# ======================================================================
# 5. LOAD FINAL DATASET
# ======================================================================

print_section("3/12 — LOADING FINAL ML DATASET")

df = pd.read_csv(FINAL_DATASET)

print(f"Dataset shape: {df.shape}")

# Reset the dataframe index so that all downstream positional
# operations are deterministic.
df = df.reset_index(drop=True)

# Validate all targets
missing_targets = [
    target
    for target in TARGET_COLUMNS
    if target not in df.columns
]

if missing_targets:

    raise RuntimeError(
        "The following expected target columns are missing:\n"
        + "\n".join(missing_targets)
    )

print(f"Targets detected: {len(TARGET_COLUMNS)}")

for target in TARGET_COLUMNS:
    print(f"  ✓ {target}")


# ======================================================================
# 6. LOAD AND VALIDATE FROZEN POSITION-AWARE FOLDS
# ======================================================================

print_section("4/12 — VALIDATING FROZEN POSITION-AWARE OUTER FOLDS")

fold_df = pd.read_csv(FOLD_ASSIGNMENTS_FILE)

fold_df = fold_df.reset_index(drop=True)

print(f"Fold assignment shape: {fold_df.shape}")

fold_column = find_column(
    fold_df,
    [
        "PositionAware_Fold",
        "Position_Aware_Fold"
    ],
    required=True
)

print(f"Detected fold column: {fold_column}")

# ----------------------------------------------------------------------
# Stable-key alignment
# ----------------------------------------------------------------------

dataset_key = canonicalize_mutation_key(df)
fold_key = canonicalize_mutation_key(fold_df)

df["_Stable_Mutation_Key"] = dataset_key.astype(str)
fold_df["_Stable_Mutation_Key"] = fold_key.astype(str)

dataset_keys = set(df["_Stable_Mutation_Key"])
fold_keys = set(fold_df["_Stable_Mutation_Key"])

if dataset_keys != fold_keys:

    missing_from_folds = sorted(
        dataset_keys - fold_keys
    )

    missing_from_dataset = sorted(
        fold_keys - dataset_keys
    )

    raise RuntimeError(
        "\nStable mutation-key mismatch detected.\n\n"
        f"Missing from fold assignments: {len(missing_from_folds)}\n"
        f"Missing from final dataset: {len(missing_from_dataset)}\n\n"
        "The final evaluation has been stopped to prevent silent "
        "train/test assignment errors."
    )

# ----------------------------------------------------------------------
# Align fold assignments by stable mutation key
# ----------------------------------------------------------------------

fold_lookup = (
    fold_df[
        [
            "_Stable_Mutation_Key",
            fold_column
        ]
    ]
    .drop_duplicates("_Stable_Mutation_Key")
    .set_index("_Stable_Mutation_Key")
)

if len(fold_lookup) != len(df):

    raise RuntimeError(
        "Fold assignment file contains duplicated stable mutation keys."
    )

df[POSITION_FOLD_COLUMN] = (
    df["_Stable_Mutation_Key"]
    .map(fold_lookup[fold_column])
)

if df[POSITION_FOLD_COLUMN].isna().any():

    raise RuntimeError(
        "Some final-dataset rows could not be assigned to a "
        "Position-Aware outer fold."
    )

# Convert fold labels to integer where possible
df[POSITION_FOLD_COLUMN] = pd.to_numeric(
    df[POSITION_FOLD_COLUMN],
    errors="raise"
).astype(int)

observed_folds = sorted(
    df[POSITION_FOLD_COLUMN].unique().tolist()
)

print(f"Observed Position-Aware folds: {observed_folds}")

expected_folds = list(range(1, N_OUTER_FOLDS + 1))

if observed_folds != expected_folds:

    raise RuntimeError(
        f"Unexpected outer folds.\n"
        f"Expected: {expected_folds}\n"
        f"Observed: {observed_folds}"
    )

print("\nFold sizes:")

for fold_id in expected_folds:

    n = int(
        (df[POSITION_FOLD_COLUMN] == fold_id).sum()
    )

    print(
        f"  Outer Fold {fold_id}: "
        f"{n:,} samples"
    )


# ======================================================================
# 7. LOAD STEP 3 FOLD-SPECIFIC HYPERPARAMETERS
# ======================================================================

print_section(
    "5/12 — LOADING FROZEN STEP 3 FOLD-SPECIFIC HYPERPARAMETERS"
)

hp_df = pd.read_csv(STEP3_HYPERPARAMETERS_FILE)

print(f"Step 3 hyperparameter table shape: {hp_df.shape}")

required_hp_columns = [
    "Target",
    "Architecture",
    "Model",
    "Outer_Fold",
    "Best_Inner_R2",
    "Best_Hyperparameters"
]

missing_hp_columns = [
    col
    for col in required_hp_columns
    if col not in hp_df.columns
]

if missing_hp_columns:

    raise RuntimeError(
        "The Step 3 hyperparameter file is missing required columns:\n"
        + "\n".join(missing_hp_columns)
    )

print("\nDetected Step 3 columns:")
for col in hp_df.columns:
    print(f"  • {col}")

# ----------------------------------------------------------------------
# Normalize identifiers
# ----------------------------------------------------------------------

hp_df["Target"] = hp_df["Target"].astype(str).str.strip()

hp_df["Architecture"] = (
    hp_df["Architecture"]
    .map(normalize_architecture_name)
)

hp_df["Model"] = (
    hp_df["Model"]
    .map(normalize_model_name)
)

hp_df["Outer_Fold"] = pd.to_numeric(
    hp_df["Outer_Fold"],
    errors="raise"
).astype(int)

# ----------------------------------------------------------------------
# Validate target coverage
# ----------------------------------------------------------------------

missing_hp_targets = sorted(
    set(TARGET_COLUMNS) -
    set(hp_df["Target"])
)

if missing_hp_targets:

    raise RuntimeError(
        "Step 3 hyperparameter file does not contain all expected targets:\n"
        + "\n".join(missing_hp_targets)
    )

# ----------------------------------------------------------------------
# Validate uniqueness of Target × Architecture × Model × Outer_Fold
# ----------------------------------------------------------------------

duplicate_mask = hp_df.duplicated(
    subset=[
        "Target",
        "Architecture",
        "Model",
        "Outer_Fold"
    ],
    keep=False
)

if duplicate_mask.any():

    duplicates = hp_df.loc[
        duplicate_mask,
        [
            "Target",
            "Architecture",
            "Model",
            "Outer_Fold"
        ]
    ]

    raise RuntimeError(
        "\nDuplicate frozen hyperparameter definitions detected.\n\n"
        f"{duplicates.to_string(index=False)}"
    )

# ----------------------------------------------------------------------
# Expected architecture/model combinations
# ----------------------------------------------------------------------

expected_combinations = []

for target in TARGET_COLUMNS:

    for architecture in [
        "Model_A_No_Position",
        "Model_B_With_Position"
    ]:

        for model_name in [
            "Random_Forest",
            "Extra_Trees"
        ]:

            expected_combinations.append(
                (
                    target,
                    architecture,
                    model_name
                )
            )

# ----------------------------------------------------------------------
# Report available combinations
# ----------------------------------------------------------------------

print("\nStep 3 hyperparameter coverage:")

for architecture in [
    "Model_A_No_Position",
    "Model_B_With_Position"
]:

    subset = hp_df[
        hp_df["Architecture"] == architecture
    ]

    print(
        f"  {architecture}: "
        f"{len(subset)} rows"
    )


# ======================================================================
# 8. BUILD STEP 4 FROZEN STEP-3 SELECTION LOOKUP
# ======================================================================

print_section(
    "6/12 — BUILDING FROZEN STEP 3 MODEL SELECTION LOOKUP"
)

# Step 3 selects exactly ONE regressor family for each
# Target × Architecture × Outer_Fold. Step 4 must execute exactly
# those frozen selections; it must NOT evaluate both RF and ET.

expected_selection_keys = {
    (target, architecture, fold)
    for target in TARGET_COLUMNS
    for architecture in MODEL_ARCHITECTURES.keys()
    for fold in expected_folds
}

step3_selection_lookup = {}

for _, row in hp_df.iterrows():
    key = (
        row["Target"],
        row["Architecture"],
        int(row["Outer_Fold"])
    )

    if key in step3_selection_lookup:
        raise RuntimeError(
            "Duplicate Step 3 selection for "
            f"Target={key[0]}, Architecture={key[1]}, Fold={key[2]}"
        )

    model_name = normalize_model_name(row["Model"])
    if model_name not in {"Random_Forest", "Extra_Trees"}:
        raise RuntimeError(
            f"Unsupported Step 3 selected model: {model_name}"
        )

    normalized_params = parse_frozen_hyperparameters(row["Best_Hyperparameters"])

    step3_selection_lookup[key] = {
        "Model": model_name,
        "Best_Inner_R2": float(row["Best_Inner_R2"]),
        "Best_Hyperparameters": normalized_params
    }

observed_selection_keys = set(step3_selection_lookup)
missing_keys = expected_selection_keys - observed_selection_keys
extra_keys = observed_selection_keys - expected_selection_keys

if missing_keys:
    raise RuntimeError(
        "Step 3 is missing frozen selections for "
        f"{len(missing_keys)} Target × Architecture × Fold combinations."
    )

if extra_keys:
    raise RuntimeError(
        "Step 3 contains unexpected Target × Architecture × Fold combinations: "
        f"{sorted(extra_keys)}"
    )

if len(step3_selection_lookup) != 90:
    raise RuntimeError(
        "Step 3 frozen selection count is not exactly 90. "
        f"Observed: {len(step3_selection_lookup)}"
    )

step3_selections_used_df = pd.DataFrame([
    {
        "Target": target,
        "Architecture": architecture,
        "Outer_Fold": fold,
        "Model": step3_selection_lookup[(target, architecture, fold)]["Model"],
        "Best_Inner_R2": step3_selection_lookup[(target, architecture, fold)]["Best_Inner_R2"],
        "Best_Hyperparameters": json.dumps(
            step3_selection_lookup[(target, architecture, fold)]["Best_Hyperparameters"],
            sort_keys=True
        )
    }
    for target, architecture, fold in sorted(expected_selection_keys)
])

print(f"Frozen Step 3 selections loaded: {len(step3_selection_lookup):,}")
print("\nSelected model counts:")
print(step3_selections_used_df["Model"].value_counts().to_string())

# ======================================================================
# 7/12 — DEFINING MODEL A / MODEL B FEATURE SETS
# ======================================================================
#
# FEATURE ARCHITECTURE
# --------------------
#
# Model A:
#   65 validated predictors
#   Sequence_Position excluded
#
# Model B:
#   66 validated predictors
#   Sequence_Position included
#
# IMPORTANT
# ---------
# The final ML dataset also contains nine target-associated SD columns.
# These columns represent target uncertainty/variability and are NOT
# predictors. They must be excluded from the feature matrix to prevent
# target-derived information from entering the models.
#
# The nine SD columns are:
#
#   <target>_SD
#
# for each of the nine antimicrobial/condition-specific targets.
#
# This exclusion is performed explicitly rather than through a generic
# string-matching rule, ensuring a transparent and reproducible schema.
# ======================================================================

print_section(
    "7/12 — DEFINING MODEL A / MODEL B FEATURE SETS"
)

# ----------------------------------------------------------------------
# Target-associated SD columns
# ----------------------------------------------------------------------
#
# These columns are descriptive/uncertainty variables associated with
# the response measurements. They are not independent predictors.
# ----------------------------------------------------------------------

TARGET_SD_COLUMNS = [
    "128ug/mL_AMP_25C_SD",
    "16ug/mL_AMP_25C_SD",
    "2ug/mL_AMP_25C_SD",
    "128ug/mL_AMP_37C_SD",
    "16ug/mL_AMP_37C_SD",
    "2ug/mL_AMP_37C_SD",
    "4ug/mL_CTX_37C_SD",
    "0.5ug/mL_CTX_37C_SD",
    "0.031ug/mL_MEM_37C_SD",
]

# ----------------------------------------------------------------------
# Global non-predictor columns
# ----------------------------------------------------------------------
#
# Sequence_Position is intentionally NOT included here because it is a
# legitimate predictor for Model B.
# ----------------------------------------------------------------------

GLOBAL_NON_PREDICTOR_COLUMNS = {
    "Mutation",
    "mutation",
    "Identity",
    "identity",
    "WT_AA",
    "Mutant_AA",
    "WT_Residue",
    "Mutant_Residue",
    "Position",
    "position",
    "Fold",
    "Fold_ID",
    "PositionAware_Fold",
    "Random_Fold",
    "Split",
    "Set",
    "_Stable_Mutation_Key",
}

# ----------------------------------------------------------------------
# Verify required columns
# ----------------------------------------------------------------------

missing_sd_columns = [
    col for col in TARGET_SD_COLUMNS
    if col not in df.columns
]

if missing_sd_columns:
    raise RuntimeError(
        "\nExpected target-associated SD columns are missing:\n"
        + "\n".join(f"  - {x}" for x in missing_sd_columns)
    )

if "Sequence_Position" not in df.columns:
    raise RuntimeError(
        "\nSequence_Position was not found in the final ML dataset.\n"
        "It is required for Model B."
    )

# ----------------------------------------------------------------------
# Build Model A and Model B directly from the frozen Part 13 manifest.
# ----------------------------------------------------------------------
# The Part 13 feature manifest is the authoritative source of truth for
# architecture membership. Feature lists are NEVER reconstructed from the
# raw dataset by exclusion rules because that can silently introduce or
# remove predictors when the dataset schema changes.
# ----------------------------------------------------------------------

feature_manifest_df = pd.read_csv(FEATURE_MANIFEST_FILE)
required_manifest_columns = [
    "Feature",
    "Used_in_Model_A_No_Position",
    "Used_in_Model_B_With_Position",
]

missing_manifest_columns = [
    col for col in required_manifest_columns
    if col not in feature_manifest_df.columns
]

if missing_manifest_columns:
    raise RuntimeError(
        "Part 13 feature manifest is missing required columns:\n"
        + "\n".join(f"  - {x}" for x in missing_manifest_columns)
    )


def manifest_truthy(value):
    """Interpret the boolean encodings used by the Part 13 manifest."""
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y", "t"}


FEATURES_MODEL_A = list(dict.fromkeys(
    feature_manifest_df.loc[
        feature_manifest_df["Used_in_Model_A_No_Position"].map(manifest_truthy),
        "Feature",
    ].tolist()
))

FEATURES_MODEL_B = list(dict.fromkeys(
    feature_manifest_df.loc[
        feature_manifest_df["Used_in_Model_B_With_Position"].map(manifest_truthy),
        "Feature",
    ].tolist()
))

EXPECTED_MODEL_A_FEATURE_COUNT = 65
EXPECTED_MODEL_B_FEATURE_COUNT = 66

if len(FEATURES_MODEL_A) != EXPECTED_MODEL_A_FEATURE_COUNT:
    raise RuntimeError(
        "Unexpected Model A predictor count.\n"
        f"Expected: {EXPECTED_MODEL_A_FEATURE_COUNT}\n"
        f"Observed: {len(FEATURES_MODEL_A)}"
    )

if len(FEATURES_MODEL_B) != EXPECTED_MODEL_B_FEATURE_COUNT:
    raise RuntimeError(
        "Unexpected Model B predictor count.\n"
        f"Expected: {EXPECTED_MODEL_B_FEATURE_COUNT}\n"
        f"Observed: {len(FEATURES_MODEL_B)}"
    )

if "Sequence_Position" in FEATURES_MODEL_A:
    raise RuntimeError("Sequence_Position is present in Model A.")

if "Sequence_Position" not in FEATURES_MODEL_B:
    raise RuntimeError("Sequence_Position is absent from Model B.")

if set(FEATURES_MODEL_B) != set(FEATURES_MODEL_A) | {"Sequence_Position"}:
    raise RuntimeError(
        "Model B must differ from Model A only by Sequence_Position."
    )

missing_features = [
    feature for feature in FEATURES_MODEL_B
    if feature not in df.columns
]

if missing_features:
    raise RuntimeError(
        "The Part 13 feature manifest contains predictors absent from the "
        "final dataset:\n" + "\n".join(f"  - {x}" for x in missing_features)
    )

sd_leakage_columns = [
    col for col in TARGET_SD_COLUMNS
    if col in FEATURES_MODEL_B
]

if sd_leakage_columns:
    raise RuntimeError(
        "Target-associated SD columns were incorrectly retained as predictors:\n"
        + "\n".join(f"  - {x}" for x in sd_leakage_columns)
    )

print(f"Total validated candidate predictors : {len(FEATURES_MODEL_B)}")
print(f"Model A predictors                   : {len(FEATURES_MODEL_A)}")
print(f"Model B predictors                   : {len(FEATURES_MODEL_B)}")
print("Sequence_Position in Model A         : NO")
print("Sequence_Position in Model B         : YES")
print("\n[PASS] Part 13 feature manifest used as the architecture source of truth.")
print("[PASS] Model A = 65 predictors (Sequence_Position excluded).")
print("[PASS] Model B = 66 predictors (Sequence_Position included).")
print("[PASS] Model B differs from Model A only by Sequence_Position.")
print("[PASS] All target-associated SD columns excluded from predictors.")

# ======================================================================
# 10. BUILD PREPROCESSING PIPELINE
# ======================================================================

print_section(
    "8/12 — DEFINING LEAKAGE-SAFE PREPROCESSING PIPELINE"
)


def create_preprocessor(X):
    """
    Create a leakage-safe preprocessing transformer.

    Numerical variables:
        median imputation
        missingness indicator
        standardization

    Categorical variables:
        most-frequent imputation
        one-hot encoding

    All transformations are fitted only on the outer-training data
    because this transformer is embedded inside the model Pipeline.
    """

    numeric_columns = X.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical_columns = [
        col
        for col in X.columns
        if col not in numeric_columns
    ]

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    transformers = []

    if numeric_columns:

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_columns
            )
        )

    if categorical_columns:

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


def build_final_pipeline(
    X_train,
    model_name,
    frozen_parameters
):
    """
    Build the final leakage-safe pipeline for one outer fold.

    The preprocessing object is fitted only after the outer training
    partition is selected.
    """

    preprocessor = create_preprocessor(
        X_train
    )

    regressor = build_regressor(
        model_name,
        frozen_parameters
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                regressor
            )
        ]
    )

    return pipeline


# ======================================================================
# 11. OUTER-FOLD LEAKAGE AUDIT
# ======================================================================

print_section(
    "9/12 — RUNNING POSITION-AWARE OUTER-FOLD LEAKAGE AUDIT"
)

leakage_audit_records = []

for fold_id in expected_folds:

    train_mask = (
        df[POSITION_FOLD_COLUMN] != fold_id
    )

    test_mask = (
        df[POSITION_FOLD_COLUMN] == fold_id
    )

    train_df = df.loc[train_mask]
    test_df = df.loc[test_mask]

    train_keys = set(
        train_df["_Stable_Mutation_Key"]
    )

    test_keys = set(
        test_df["_Stable_Mutation_Key"]
    )

    key_overlap = train_keys.intersection(
        test_keys
    )

    # --------------------------------------------------------------
    # Sequence-position leakage audit
    # --------------------------------------------------------------

    position_overlap = np.nan

    if "Sequence_Position" in df.columns:

        train_positions = set(
            train_df["Sequence_Position"]
            .dropna()
            .astype(str)
        )

        test_positions = set(
            test_df["Sequence_Position"]
            .dropna()
            .astype(str)
        )

        position_overlap = len(
            train_positions.intersection(
                test_positions
            )
        )

    leakage_audit_records.append(
        {
            "Outer_Fold": fold_id,
            "Train_N": len(train_df),
            "Test_N": len(test_df),
            "Mutation_Key_Overlap": len(key_overlap),
            "Sequence_Position_Overlap": position_overlap,
            "Leakage_Status": (
                "PASS"
                if len(key_overlap) == 0
                and (
                    pd.isna(position_overlap)
                    or position_overlap == 0
                )
                else "FAIL"
            )
        }
    )

leakage_audit_df = pd.DataFrame(
    leakage_audit_records
)

print(
    leakage_audit_df.to_string(
        index=False
    )
)

if not (
    leakage_audit_df["Leakage_Status"] == "PASS"
).all():

    raise RuntimeError(
        "\nPosition-Aware leakage audit FAILED.\n"
        "Final model evaluation has been stopped."
    )

print(
    "\n[PASS] Position-Aware outer-fold leakage audit completed."
)


# ======================================================================
# 12. FINAL OUTER-FOLD EVALUATION
# ======================================================================

print_section(
    "10/12 — FINAL OUTER-FOLD MODEL EVALUATION (90 FROZEN JOBS)"
)

evaluation_records = []
prediction_records = []

total_jobs = len(TARGET_COLUMNS) * len(MODEL_ARCHITECTURES) * N_OUTER_FOLDS
if total_jobs != 90:
    raise RuntimeError(f"Expected exactly 90 Step 4 evaluations, got {total_jobs}.")

completed_jobs = 0
overall_start = time.time()

for target in TARGET_COLUMNS:
    print("\n" + "-" * 78)
    print(f"TARGET: {target}")
    print("-" * 78)

    for architecture in MODEL_ARCHITECTURES.keys():
        feature_columns = (
            FEATURES_MODEL_A
            if architecture == "Model_A_No_Position"
            else FEATURES_MODEL_B
        )

        for outer_fold in expected_folds:
            completed_jobs += 1
            job_start = time.time()

            frozen = step3_selection_lookup[(target, architecture, outer_fold)]
            model_name = frozen["Model"]
            frozen_parameters = frozen["Best_Hyperparameters"]
            best_inner_r2 = frozen["Best_Inner_R2"]

            print(
                f"\n    [{completed_jobs}/{total_jobs}] "
                f"{architecture} | Fold {outer_fold} | {model_name}"
            )

            train_mask = df[POSITION_FOLD_COLUMN] != outer_fold
            test_mask = df[POSITION_FOLD_COLUMN] == outer_fold

            train_df = df.loc[train_mask].copy()
            test_df = df.loc[test_mask].copy()

            X_train = train_df[feature_columns].copy()
            X_test = test_df[feature_columns].copy()
            y_train = train_df[target].copy()
            y_test = test_df[target].copy()

            train_valid = y_train.notna()
            test_valid = y_test.notna()

            X_train = X_train.loc[train_valid]
            y_train = y_train.loc[train_valid]
            X_test = X_test.loc[test_valid]
            y_test = y_test.loc[test_valid]

            if len(X_train) == 0 or len(X_test) == 0:
                raise RuntimeError(
                    f"Empty train/test set for {target} | {architecture} | "
                    f"{model_name} | Fold {outer_fold}."
                )

            pipeline = build_final_pipeline(
                X_train=X_train,
                model_name=model_name,
                frozen_parameters=frozen_parameters
            )

            fit_start = time.time()
            pipeline.fit(X_train, y_train)
            fit_time = time.time() - fit_start

            predict_start = time.time()
            y_pred = pipeline.predict(X_test)
            predict_time = time.time() - predict_start

            metrics = calculate_metrics(y_test, y_pred)
            elapsed = time.time() - job_start

            evaluation_records.append({
                "Target": target,
                "Architecture": architecture,
                "Model": model_name,
                "Outer_Fold": outer_fold,
                "Best_Inner_R2": best_inner_r2,
                "Train_N": len(X_train),
                "Test_N": len(X_test),
                "R2": metrics["R2"],
                "RMSE": metrics["RMSE"],
                "MAE": metrics["MAE"],
                "Spearman": metrics["Spearman"],
                "Fit_Time_Seconds": fit_time,
                "Predict_Time_Seconds": predict_time,
                "Total_Time_Seconds": elapsed,
                "Frozen_Hyperparameters": json.dumps(frozen_parameters, sort_keys=True)
            })

            for row_index, true_value, pred_value in zip(
                test_df.loc[test_valid].index, y_test.values, y_pred
            ):
                prediction_records.append({
                    "Row_Index": int(row_index),
                    "Stable_Mutation_Key": df.loc[row_index, "_Stable_Mutation_Key"],
                    "Target": target,
                    "Architecture": architecture,
                    "Model": model_name,
                    "Outer_Fold": outer_fold,
                    "Observed": float(true_value),
                    "Predicted": float(pred_value),
                    "Residual": float(true_value - pred_value),
                    "Best_Inner_R2": best_inner_r2
                })

            print(
                f"      R²={metrics['R2']:.4f} | "
                f"RMSE={metrics['RMSE']:.4f} | "
                f"MAE={metrics['MAE']:.4f} | "
                f"Spearman={metrics['Spearman']:.4f} | "
                f"time={elapsed:.1f}s"
            )

overall_runtime = time.time() - overall_start
print(f"\nFinal evaluation runtime: {overall_runtime / 60:.2f} minutes")


# ======================================================================
# 13. CONVERT RESULTS TO DATAFRAMES
# ======================================================================

print_section(
    "11/12 — GENERATING FINAL OOF AND PERFORMANCE TABLES"
)

fold_performance_df = pd.DataFrame(
    evaluation_records
)

oof_predictions_df = pd.DataFrame(
    prediction_records
)

print(
    f"Fold-level evaluation rows: "
    f"{len(fold_performance_df):,}"
)

print(
    f"OOF prediction rows: "
    f"{len(oof_predictions_df):,}"
)

# ----------------------------------------------------------------------
# Expected number of fold-level evaluations
# ----------------------------------------------------------------------

expected_evaluations = (
    len(TARGET_COLUMNS)
    * len(MODEL_ARCHITECTURES)
    * N_OUTER_FOLDS
)

if len(fold_performance_df) != expected_evaluations:

    raise RuntimeError(
        "\nUnexpected number of fold-level evaluation records.\n"
        f"Expected: {expected_evaluations}\n"
        f"Observed: {len(fold_performance_df)}"
    )


# ======================================================================
# 14. POOLED OOF PERFORMANCE
# ======================================================================

pooled_records = []

for target in TARGET_COLUMNS:
    for architecture in MODEL_ARCHITECTURES.keys():
        subset = oof_predictions_df[
            (oof_predictions_df["Target"] == target)
            & (oof_predictions_df["Architecture"] == architecture)
        ].copy()

        if subset.empty:
            continue

        pooled_metrics = calculate_metrics(
            subset["Observed"].values,
            subset["Predicted"].values
        )

        selected_models = ";".join(
            sorted(subset["Model"].unique())
        )

        pooled_records.append({
            "Target": target,
            "Architecture": architecture,
            "Selected_Model_Families": selected_models,
            "OOF_N": len(subset),
            "OOF_R2": pooled_metrics["R2"],
            "OOF_RMSE": pooled_metrics["RMSE"],
            "OOF_MAE": pooled_metrics["MAE"],
            "OOF_Spearman": pooled_metrics["Spearman"]
        })

pooled_oof_df = pd.DataFrame(pooled_records)


# ======================================================================
# 15. FOLD SUMMARY
# ======================================================================

fold_summary_df = (
    fold_performance_df
    .groupby(["Target", "Architecture"], as_index=False)
    .agg(
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        Spearman_Mean=("Spearman", "mean"),
        Spearman_SD=("Spearman", "std"),
        Mean_Train_N=("Train_N", "mean"),
        Mean_Test_N=("Test_N", "mean"),
        N_Outer_Folds=("Outer_Fold", "nunique")
    )
)

selected_model_summary = (
    fold_performance_df
    .groupby(["Target", "Architecture"])["Model"]
    .agg(lambda x: ";".join(sorted(x.unique())))
    .reset_index(name="Selected_Model_Families")
)

fold_summary_df = fold_summary_df.merge(
    selected_model_summary,
    on=["Target", "Architecture"],
    how="left"
)

fold_summary_df["R2_SE"] = (
    fold_summary_df["R2_SD"] / np.sqrt(fold_summary_df["N_Outer_Folds"])
)
fold_summary_df["R2_95CI_Lower"] = (
    fold_summary_df["R2_Mean"] - 1.96 * fold_summary_df["R2_SE"]
)
fold_summary_df["R2_95CI_Upper"] = (
    fold_summary_df["R2_Mean"] + 1.96 * fold_summary_df["R2_SE"]
)


# ======================================================================
# 16. MODEL A VS MODEL B COMPARISON
# ======================================================================

model_comparison_records = []

for target in TARGET_COLUMNS:
    model_a = (
        fold_performance_df[
            (fold_performance_df["Target"] == target)
            & (fold_performance_df["Architecture"] == "Model_A_No_Position")
        ]
        .sort_values("Outer_Fold")
        .reset_index(drop=True)
    )

    model_b = (
        fold_performance_df[
            (fold_performance_df["Target"] == target)
            & (fold_performance_df["Architecture"] == "Model_B_With_Position")
        ]
        .sort_values("Outer_Fold")
        .reset_index(drop=True)
    )

    if len(model_a) != N_OUTER_FOLDS or len(model_b) != N_OUTER_FOLDS:
        raise RuntimeError(
            f"A/B paired comparison requires exactly {N_OUTER_FOLDS} folds "
            f"for {target}."
        )

    if not np.array_equal(model_a["Outer_Fold"].values, model_b["Outer_Fold"].values):
        raise RuntimeError(f"Outer-fold mismatch in A/B comparison for {target}.")

    delta_r2 = model_b["R2"].values - model_a["R2"].values
    delta_rmse = model_b["RMSE"].values - model_a["RMSE"].values
    delta_mae = model_b["MAE"].values - model_a["MAE"].values
    delta_spearman = model_b["Spearman"].values - model_a["Spearman"].values

    model_comparison_records.append({
        "Target": target,
        "Model_A_Selected_Families": ";".join(sorted(model_a["Model"].unique())),
        "Model_B_Selected_Families": ";".join(sorted(model_b["Model"].unique())),
        "Model_A_R2_Mean": model_a["R2"].mean(),
        "Model_B_R2_Mean": model_b["R2"].mean(),
        "Delta_R2_B_minus_A": np.mean(delta_r2),
        "Delta_R2_SD": np.std(delta_r2, ddof=1),
        "Model_A_RMSE_Mean": model_a["RMSE"].mean(),
        "Model_B_RMSE_Mean": model_b["RMSE"].mean(),
        "Delta_RMSE_B_minus_A": np.mean(delta_rmse),
        "Model_A_MAE_Mean": model_a["MAE"].mean(),
        "Model_B_MAE_Mean": model_b["MAE"].mean(),
        "Delta_MAE_B_minus_A": np.mean(delta_mae),
        "Model_A_Spearman_Mean": model_a["Spearman"].mean(),
        "Model_B_Spearman_Mean": model_b["Spearman"].mean(),
        "Delta_Spearman_B_minus_A": np.mean(delta_spearman)
    })

model_comparison_df = pd.DataFrame(model_comparison_records)


# ======================================================================
# 17. SAVE FINAL STEP 4 OUTPUTS
# ======================================================================

print_section(
    "12/12 — SAVING FINAL STEP 4 OUTPUTS"
)

# ----------------------------------------------------------------------
# Output 1 — Fold-level final performance
# ----------------------------------------------------------------------

fold_performance_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Final_OuterFold_Performance.csv"
)

fold_performance_df.to_csv(
    fold_performance_file,
    index=False
)

# ----------------------------------------------------------------------
# Output 2 — Pooled OOF predictions
# ----------------------------------------------------------------------

oof_predictions_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Final_OOF_Predictions.csv"
)

oof_predictions_df.to_csv(
    oof_predictions_file,
    index=False
)

# ----------------------------------------------------------------------
# Output 3 — Pooled OOF performance
# ----------------------------------------------------------------------

pooled_oof_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Pooled_OOF_Performance.csv"
)

pooled_oof_df.to_csv(
    pooled_oof_file,
    index=False
)

# ----------------------------------------------------------------------
# Output 4 — Fold summary
# ----------------------------------------------------------------------

fold_summary_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Final_Fold_Summary.csv"
)

fold_summary_df.to_csv(
    fold_summary_file,
    index=False
)

# ----------------------------------------------------------------------
# Output 5 — Model A vs Model B comparison
# ----------------------------------------------------------------------

model_comparison_file = (
    OUTPUT_DIR
    / "VIM2_Step4_ModelA_vs_ModelB_Comparison.csv"
)

model_comparison_df.to_csv(
    model_comparison_file,
    index=False
)

# ----------------------------------------------------------------------
# Output 6 — Leakage audit
# ----------------------------------------------------------------------

leakage_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Leakage_Audit.csv"
)

leakage_audit_df.to_csv(
    leakage_file,
    index=False
)

# Output 7 — Exact frozen Step 3 selections actually used
step3_selections_used_file = (
    OUTPUT_DIR / "VIM2_Step4_Frozen_Step3_Selections_Used.csv"
)
step3_selections_used_df.to_csv(
    step3_selections_used_file,
    index=False
)


# ======================================================================
# 18. REPRODUCIBILITY MANIFEST
# ======================================================================

manifest = {

    "project": PROJECT_NAME,

    "step": STEP_NAME,

    "execution_timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "random_state":
        RANDOM_STATE,

    "outer_cv":
        {
            "type":
                "Position-Aware",
            "n_folds":
                N_OUTER_FOLDS,
            "fold_column":
                POSITION_FOLD_COLUMN
        },

    "data": {

        "final_ml_dataset":
            FINAL_DATASET,

        "final_ml_dataset_sha256":
            dataset_hash,

        "fold_assignments":
            FOLD_ASSIGNMENTS_FILE,

        "fold_assignments_sha256":
            fold_hash,

        "step3_best_hyperparameters":
            STEP3_HYPERPARAMETERS_FILE,

        "step3_best_hyperparameters_sha256":
            step3_hp_hash
    },

    "targets":
        TARGET_COLUMNS,

    "model_architectures":
        list(
            MODEL_ARCHITECTURES.keys()
        ),

    "feature_counts":
        {
            "Model_A_No_Position":
                len(FEATURES_MODEL_A),

            "Model_B_With_Position":
                len(FEATURES_MODEL_B)
        },

    "sequence_position":
        {
            "Model_A":
                False,

            "Model_B":
                True
        },

    "preprocessing":
        {
            "numeric":
                [
                    "median_imputation",
                    "missing_indicator",
                    "standard_scaling"
                ],

            "categorical":
                [
                    "most_frequent_imputation",
                    "one_hot_encoding"
                ],

            "fit_scope":
                "outer_training_partition_only"
        },

    "evaluation_count":
        len(fold_performance_df),

    "hyperparameter_policy":
        {
            "source":
                "Step 3 Best_Hyperparameters_Per_Fold",

            "fold_specific":
                True,

            "averaging_across_folds":
                False,

            "reoptimization_in_step4":
                False
        },

    "evaluation_metrics":
        [
            "R2",
            "RMSE",
            "MAE",
            "Spearman"
        ],

    "leakage_audit":
        {
            "stable_mutation_key_alignment":
                True,

            "mutation_key_overlap":
                int(
                    leakage_audit_df[
                        "Mutation_Key_Overlap"
                    ].max()
                ),

            "sequence_position_overlap":
                int(
                    leakage_audit_df[
                        "Sequence_Position_Overlap"
                    ].fillna(0).max()
                ),

            "overall_status":
                "PASS"
        },

    "runtime_seconds":
        overall_runtime,

    "runtime_minutes":
        overall_runtime / 60.0
}

manifest_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Final_Evaluation_Manifest.json"
)

with open(
    manifest_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ======================================================================
# 19. HUMAN-READABLE SUMMARY
# ======================================================================

summary_file = (
    OUTPUT_DIR
    / "VIM2_Step4_Final_Evaluation_Summary.txt"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "VIM-2 MACHINE LEARNING PROJECT\n"
    )

    f.write(
        "STEP 4 — FINAL OPTIMIZED MODEL EVALUATION\n"
    )

    f.write(
        "=" * 78 + "\n\n"
    )

    f.write(
        "Evaluation framework:\n"
        "Position-Aware 5-fold outer evaluation\n\n"
    )

    f.write(
        "Hyperparameter policy:\n"
        "Frozen fold-specific hyperparameters from Step 3\n\n"
    )

    f.write(
        "No hyperparameter optimization was performed in Step 4.\n"
    )

    f.write(
        "No outer-test result was used for hyperparameter selection.\n"
    )

    f.write(
        "Preprocessing was fitted exclusively within each outer "
        "training partition.\n\n"
    )

    f.write(
        f"Final dataset SHA-256:\n{dataset_hash}\n\n"
    )

    f.write(
        f"Fold assignment SHA-256:\n{fold_hash}\n\n"
    )

    f.write(
        f"Step 3 hyperparameter SHA-256:\n"
        f"{step3_hp_hash}\n\n"
    )

    f.write(
        f"Total fold-level evaluations:\n"
        f"{len(fold_performance_df)}\n\n"
    )

    f.write(
        f"Total OOF predictions:\n"
        f"{len(oof_predictions_df)}\n\n"
    )

    f.write(
        f"Total runtime:\n"
        f"{overall_runtime / 60:.2f} minutes\n\n"
    )

    f.write(
        "Leakage audit status: PASS\n"
    )


# ======================================================================
# 20. FINAL VALIDATION CHECKS
# ======================================================================

print_section(
    "FINAL QUALITY CONTROL"
)

# ----------------------------------------------------------------------
# Check 1 — No missing fold metrics
# ----------------------------------------------------------------------

metric_columns = [
    "R2",
    "RMSE",
    "MAE",
    "Spearman"
]

for col in metric_columns:

    missing_count = int(
        fold_performance_df[col]
        .isna()
        .sum()
    )

    print(
        f"{col:12s} missing values: "
        f"{missing_count}"
    )

# ----------------------------------------------------------------------
# Check 2 — OOF coverage
# ----------------------------------------------------------------------

print("\nOOF coverage by Target / Architecture:")

coverage = (
    oof_predictions_df
    .groupby(["Target", "Architecture"])
    .agg(
        OOF_N=("Row_Index", "size"),
        N_Folds=("Outer_Fold", "nunique"),
        Model_Families=("Model", lambda x: ";".join(sorted(x.unique())))
    )
    .reset_index()
)

print(coverage.to_string(index=False))

# ----------------------------------------------------------------------
# Check 3 — Exactly one evaluation per Target × Architecture × Fold
# ----------------------------------------------------------------------

expected_evaluation_keys = {
    (target, architecture, fold)
    for target in TARGET_COLUMNS
    for architecture in MODEL_ARCHITECTURES.keys()
    for fold in expected_folds
}

observed_evaluation_keys = set(
    zip(
        fold_performance_df["Target"],
        fold_performance_df["Architecture"],
        fold_performance_df["Outer_Fold"]
    )
)

if observed_evaluation_keys != expected_evaluation_keys:
    raise RuntimeError(
        "Step 4 evaluation key mismatch. "
        f"Missing={len(expected_evaluation_keys - observed_evaluation_keys)}, "
        f"Extra={len(observed_evaluation_keys - expected_evaluation_keys)}"
    )

# ----------------------------------------------------------------------
# Check 4 — Five folds per Target × Architecture
# ----------------------------------------------------------------------

fold_counts_qc = (
    fold_performance_df
    .groupby(["Target", "Architecture"])["Outer_Fold"]
    .nunique()
)

if not (fold_counts_qc == N_OUTER_FOLDS).all():
    raise RuntimeError("Not every Target × Architecture has exactly five outer folds.")

# ----------------------------------------------------------------------
# Check 5 — No duplicate OOF prediction for the same row/target/arch
# ----------------------------------------------------------------------

if oof_predictions_df.duplicated(
    subset=["Row_Index", "Target", "Architecture"]
).any():
    raise RuntimeError("Duplicate OOF prediction detected for a Target × Architecture combination.")

# ----------------------------------------------------------------------
# Check 6 — No missing metrics
# ----------------------------------------------------------------------

metric_columns = ["R2", "RMSE", "MAE", "Spearman"]
missing_metrics = fold_performance_df[metric_columns].isna().sum()

print("\nMissing fold metrics:")
print(missing_metrics.to_string())

if missing_metrics.any():
    raise RuntimeError("Missing fold-level performance metrics detected.")

# ----------------------------------------------------------------------
# Check 7 — Leakage audit
# ----------------------------------------------------------------------

if not (leakage_audit_df["Leakage_Status"] == "PASS").all():
    raise RuntimeError("Leakage audit failed.")

print("\n[PASS] 90/90 frozen evaluations completed.")
print("[PASS] One selected Step 3 model per Target × Architecture × Fold.")
print("[PASS] No new HPO performed in Step 4.")
print("[PASS] No hyperparameter averaging across folds.")
print("[PASS] No outer-test data used for model selection.")
print("[PASS] Leakage audit passed.")


# ======================================================================
# 21. FINAL REPORT
# ======================================================================

print("\n" + "=" * 78)
print("STEP 4 COMPLETED SUCCESSFULLY")
print("=" * 78)

print(
    f"\nTargets evaluated      : "
    f"{len(TARGET_COLUMNS)}"
)

print(
    f"Model architectures   : "
    f"{len(MODEL_ARCHITECTURES)}"
)

print(
    f"Available regressor families : "
    f"2 (Random Forest, Extra Trees); frozen Step 3 selection used per fold"
)

print(
    f"Outer folds            : "
    f"{N_OUTER_FOLDS}"
)

print(
    f"Fold-level evaluations : "
    f"{len(fold_performance_df):,}"
)

print(
    f"OOF predictions        : "
    f"{len(oof_predictions_df):,}"
)

print(
    f"Leakage audit          : PASS"
)

print(
    f"Runtime                : "
    f"{overall_runtime / 60:.2f} minutes"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for filepath in sorted(
    OUTPUT_DIR.iterdir()
):

    if filepath.is_file():

        print(
            f"  ✓ {filepath.name}"
        )

print("\n" + "=" * 78)
print("IMPORTANT: STEP 4 RESULTS ARE NOT YET MANUSCRIPT-FINAL.")
print("Model selection/statistical comparison remains part of Step 6.")
print("=" * 78)


1/12 — VERIFYING INPUT FILES
[OK] /content/VIM2_Part12H_v2_Final_ML_Dataset.csv
[OK] /content/VIM2_Part13_Fold_Assignments.csv
[OK] /content/VIM2_Part13_Feature_Manifest.csv
[OK] /content/VIM2_Step2_PositionAware_Selected_Candidates.csv
[OK] /content/VIM2_Step3_Best_Hyperparameters_Per_Fold.csv

2/12 — CALCULATING INPUT SHA-256 HASHES
Final ML Dataset SHA-256 : d3d5691e4e4f0959cb1eba6935aea40b9b3c4cc8de1306320f1322b3729c652c
Fold Assignment SHA-256  : 7fbb829d8b4a9457f8dd61098dadc89e73d858867e89364898369285e7ebc3b6
Step 3 HP SHA-256         : cd01609bc157982a217d7911fe7442943978f49edcdf110c7452e5ae41f62760

3/12 — LOADING FINAL ML DATASET
Dataset shape: (5016, 89)
Targets detected: 9
  ✓ 0.031ug/mL_MEM_37C
  ✓ 0.5ug/mL_CTX_37C
  ✓ 128ug/mL_AMP_25C
  ✓ 128ug/mL_AMP_37C
  ✓ 16ug/mL_AMP_25C
  ✓ 16ug/mL_AMP_37C
  ✓ 2ug/mL_AMP_25C
  ✓ 2ug/mL_AMP_37C
  ✓ 4ug/mL_CTX_37C

4/12 — VALIDATING FROZEN POSITION-AWARE OUTER FOLDS
Fold assignment shape: (5016, 7)
Detected fold column: PositionAware_F

In [ ]:
# @title
# ============================================================
# DOWNLOAD STEP4 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step4_Final_Evaluation"

# Output ZIP archive
zip_base = "/content/VIM2_Step4_Final_Evaluation"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step4_Final_Evaluation"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP4 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP4 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step4_Final_Evaluation
ZIP archive      : /content/VIM2_Step4_Final_Evaluation.zip
Archive size     : 2.79 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
# ======================================================================
# VIM-2 MACHINE LEARNING PROJECT
# STEP 5 — SECONDARY RANDOM-SPLIT BENCHMARK
# ======================================================================
# PURPOSE
# -------
# This module performs a secondary, leakage-safe random-split benchmark
# for the VIM-2 mutational phenotype prediction study.
#
# The primary evaluation remains the Position-Aware nested evaluation
# performed in Steps 2–4. Step 5 is intentionally secondary and is used
# to quantify how model performance changes when mutations from the same
# sequence positions may occur in both training and test partitions.
#
# METHODOLOGICAL DESIGN
# ---------------------
# 1. Random outer folds are FROZEN from Part 13 (Random_Fold).
# 2. For every target × architecture × random outer fold:
#      a. Candidate model family (Random Forest vs Extra Trees) is chosen
#         using ONLY the outer-training partition and 3-fold random inner CV.
#      b. Hyperparameter optimization is performed ONLY inside the
#         outer-training partition using the same inner folds.
#      c. The selected family and best hyperparameters are frozen.
#      d. The frozen model is fitted once on the complete outer-training
#         partition and evaluated once on the untouched outer-test fold.
# 3. Model A excludes Sequence_Position.
# 4. Model B includes Sequence_Position.
# 5. No outer-test performance is used for candidate selection or HPO.
# 6. All preprocessing is contained inside the sklearn Pipeline and is
#    fitted separately within each training split.
#
# IMPORTANT INTERPRETATION
# ------------------------
# Random-split performance is NOT the primary estimate of generalization
# to unseen sequence positions. Because random folds can contain the same
# Sequence_Position in both train and test sets, this benchmark can be
# more optimistic than the Position-Aware evaluation.
#
# COMPUTATIONAL NOTE
# ------------------
# The benchmark contains 90 outer evaluations:
#     9 targets × 2 architectures × 5 random outer folds.
#
# Each configuration performs candidate selection followed by constrained
# randomized hyperparameter optimization. The HPO budget is intentionally
# limited to keep the benchmark computationally practical while remaining
# reproducible.
#
# ALL OUTPUTS ARE WRITTEN UNDER /content EXCEPT THIS SCRIPT FILE.
# ======================================================================

from __future__ import annotations

import ast
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# ======================================================================
# 1. GLOBAL CONFIGURATION
# ======================================================================

PROJECT_NAME = "VIM-2 ML Project"
STEP_NAME = "STEP 5 — SECONDARY RANDOM-SPLIT BENCHMARK"

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
N_HPO_ITERATIONS = 6

FINAL_DATASET = "/content/VIM2_Part12H_v2_Final_ML_Dataset.csv"
FOLD_ASSIGNMENTS_FILE = "/content/VIM2_Part13_Fold_Assignments.csv"
FEATURE_MANIFEST_FILE = (
    "/content/VIM2_Part13_Feature_Manifest.csv"
)

# Step 4 outputs are intentionally treated as an input/reference only.
# Step 5 does not depend on Step 4 results or Step 4 model selection.
STEP4_REFERENCE_DIR = "/content/VIM2_Step4_Final_Evaluation"

OUTPUT_DIR = Path("/content/VIM2_Step5_Random_Split_Benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_FOLD_COLUMN = "Random_Fold"
POSITION_COLUMN = "Sequence_Position"
TARGET_SD_SUFFIX = "_SD"

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

ARCHITECTURES = {
    "Model_A_No_Position": False,
    "Model_B_With_Position": True,
}

MODEL_FAMILIES = ["Random_Forest", "Extra_Trees"]

# Fixed candidate settings are used ONLY for family selection.
# HPO is performed afterward on the selected family.
BASELINE_PARAMS = {
    "Random_Forest": {
        "n_estimators": 250,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
    },
    "Extra_Trees": {
        "n_estimators": 250,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0,
    },
}

HPO_SPACES = {
    "Random_Forest": {
        "model__n_estimators": [150, 250, 350],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.5, 0.8],
    },
    "Extra_Trees": {
        "model__n_estimators": [150, 250, 350],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.5, 0.8, 1.0],
    },
}

EXPECTED_MODEL_A_FEATURES = 65
EXPECTED_MODEL_B_FEATURES = 66


# ======================================================================
# 2. UTILITY FUNCTIONS
# ======================================================================

def sha256_file(path: str | Path) -> str:
    """Return the SHA-256 hash of a file for reproducibility auditing."""
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now() -> str:
    """Return an ISO-8601 UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()


def normalize_model_name(value: str) -> str:
    """Normalize common model-family labels to canonical names."""
    value = str(value).strip()
    mapping = {
        "Random_Forest": "Random_Forest",
        "RandomForest": "Random_Forest",
        "Random Forest": "Random_Forest",
        "Extra_Trees": "Extra_Trees",
        "ExtraTrees": "Extra_Trees",
        "Extra Trees": "Extra_Trees",
    }
    return mapping.get(value, value)


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Create a leakage-safe preprocessing transformer from training data."""
    numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = [
        col for col in X.columns if col not in numeric_columns
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )

    transformers = []
    if numeric_columns:
        transformers.append(("numeric", numeric_pipeline, numeric_columns))
    if categorical_columns:
        transformers.append(
            ("categorical", categorical_pipeline, categorical_columns)
        )

    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_pipeline(X_train: pd.DataFrame, model_family: str, params=None) -> Pipeline:
    """Build the complete preprocessing + tree-regressor pipeline."""
    model_family = normalize_model_name(model_family)

    if model_family == "Random_Forest":
        model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    elif model_family == "Extra_Trees":
        model = ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    else:
        raise ValueError(f"Unsupported model family: {model_family}")

    if params:
        cleaned = {
            str(k).replace("model__", ""): v for k, v in params.items()
        }
        model.set_params(**cleaned)

    return Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(X_train)),
            ("model", model),
        ]
    )


def build_random_inner_splits(train_positions: pd.Series, seed: int):
    """Create ordinary random inner CV splits using only outer-training rows."""
    kfold = KFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=seed,
    )
    return list(kfold.split(np.arange(len(train_positions))))


def validate_feature_manifest(manifest: pd.DataFrame, dataset_columns: list[str]):
    """Validate the Part 13 feature manifest and enforce the 65/66 design."""
    required = {
        "Feature",
        "Used_in_Model_A_No_Position",
        "Used_in_Model_B_With_Position",
    }
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(f"Feature manifest missing columns: {sorted(missing)}")

    def is_true(x):
        return str(x).strip().lower() in {"true", "1", "yes", "y"}

    model_a = [
        f for f in manifest["Feature"].astype(str)
        if f in dataset_columns
        and is_true(
            manifest.loc[manifest["Feature"].astype(str) == f,
                         "Used_in_Model_A_No_Position"].iloc[0]
        )
    ]
    model_b = [
        f for f in manifest["Feature"].astype(str)
        if f in dataset_columns
        and is_true(
            manifest.loc[manifest["Feature"].astype(str) == f,
                         "Used_in_Model_B_With_Position"].iloc[0]
        )
    ]

    if len(model_a) != EXPECTED_MODEL_A_FEATURES:
        raise ValueError(
            f"Model A feature count mismatch: expected {EXPECTED_MODEL_A_FEATURES}, "
            f"found {len(model_a)}."
        )
    if len(model_b) != EXPECTED_MODEL_B_FEATURES:
        raise ValueError(
            f"Model B feature count mismatch: expected {EXPECTED_MODEL_B_FEATURES}, "
            f"found {len(model_b)}."
        )

    if POSITION_COLUMN not in model_b:
        raise ValueError("Sequence_Position must be present in Model B.")
    if POSITION_COLUMN in model_a:
        raise ValueError("Sequence_Position must NOT be present in Model A.")

    if set(model_b) - set(model_a) != {POSITION_COLUMN}:
        raise ValueError(
            "Model A and Model B must differ by Sequence_Position only."
        )

    missing_a = sorted(set(model_a) - set(dataset_columns))
    missing_b = sorted(set(model_b) - set(dataset_columns))
    if missing_a or missing_b:
        raise ValueError(
            f"Manifest features missing from dataset. Model A={missing_a}; Model B={missing_b}"
        )

    return model_a, model_b


def assert_no_target_or_sd_predictors(features: list[str], targets: list[str]):
    """Ensure targets, uncertainty columns, and known metadata are excluded."""
    bad_sd = [f for f in features if f.endswith(TARGET_SD_SUFFIX)]
    bad_targets = [f for f in features if f in targets]
    bad_metadata = [
        f for f in features
        if f in {"Mutation", "WT_AA", "Mutant_AA", "PositionAware_Fold", "Random_Fold"}
    ]
    if bad_sd or bad_targets or bad_metadata:
        raise ValueError(
            "Invalid predictor set detected: "
            f"SD={bad_sd}, targets={bad_targets}, metadata={bad_metadata}"
        )


def validate_outer_random_folds(folds: pd.DataFrame):
    """Validate that the Part 13 random outer folds are complete and usable."""
    if RANDOM_FOLD_COLUMN not in folds.columns:
        raise ValueError(f"Missing required fold column: {RANDOM_FOLD_COLUMN}")

    observed = sorted(pd.Series(folds[RANDOM_FOLD_COLUMN]).dropna().unique().tolist())
    expected = list(range(1, N_OUTER_FOLDS + 1))
    if observed != expected:
        raise ValueError(
            f"Random_Fold values must be exactly {expected}; found {observed}."
        )

    if folds[RANDOM_FOLD_COLUMN].isna().any():
        raise ValueError("Random_Fold contains missing values.")

    counts = folds[RANDOM_FOLD_COLUMN].value_counts().sort_index()
    if counts.min() < 1:
        raise ValueError("At least one random outer fold is empty.")


def safe_spearman(y_true, y_pred):
    """Compute Spearman correlation while safely handling constant vectors."""
    if np.nanstd(y_true) == 0 or np.nanstd(y_pred) == 0:
        return np.nan
    return float(spearmanr(y_true, y_pred).statistic)


def parse_hyperparameters(value):
    """Parse a serialized parameter dictionary safely."""
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        raise ValueError("Hyperparameter value is missing.")

    text = str(value).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        parsed = ast.literal_eval(text)

    if not isinstance(parsed, dict):
        raise ValueError("Serialized hyperparameters must decode to a dictionary.")
    return parsed


def validate_hyperparameter_schema(model_family: str, params: dict) -> dict:
    """Validate and normalize the frozen HPO schema before model construction."""
    model_family = normalize_model_name(model_family)
    required = {
        "model__n_estimators",
        "model__max_depth",
        "model__min_samples_split",
        "model__min_samples_leaf",
        "model__max_features",
    }
    keys = set(params)
    missing = required - keys
    extra = keys - required
    if missing or extra:
        raise ValueError(
            f"Invalid {model_family} hyperparameter schema. Missing={sorted(missing)}; "
            f"Unexpected={sorted(extra)}"
        )

    p = dict(params)

    p["model__n_estimators"] = int(p["model__n_estimators"])
    if p["model__n_estimators"] < 1:
        raise ValueError("n_estimators must be >= 1.")

    if p["model__max_depth"] is not None:
        p["model__max_depth"] = int(p["model__max_depth"])
        if p["model__max_depth"] < 1:
            raise ValueError("max_depth must be None or >= 1.")

    p["model__min_samples_split"] = int(p["model__min_samples_split"])
    p["model__min_samples_leaf"] = int(p["model__min_samples_leaf"])
    if p["model__min_samples_split"] < 2:
        raise ValueError("min_samples_split must be >= 2.")
    if p["model__min_samples_leaf"] < 1:
        raise ValueError("min_samples_leaf must be >= 1.")

    mf = p["model__max_features"]
    if isinstance(mf, str):
        if mf not in {"sqrt", "log2"}:
            raise ValueError(f"Unsupported string max_features: {mf}")
    else:
        mf = float(mf)
        if not (0 < mf <= 1):
            raise ValueError("Numeric max_features must be in (0, 1].")
        p["model__max_features"] = mf

    if model_family == "Extra_Trees" and p["model__max_features"] == "sqrt":
        # sqrt is technically legal in sklearn, but it is outside the Step 5
        # Extra Trees HPO search space. Rejecting it prevents schema drift.
        raise ValueError("Extra_Trees max_features='sqrt' is outside the Step 5 schema.")

    return p


def format_seconds(seconds: float) -> str:
    """Format elapsed seconds as HH:MM:SS."""
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def save_checkpoint(results, checkpoint_path: Path):
    """Persist completed configuration results for interruption-safe resume."""
    tmp = checkpoint_path.with_suffix(".tmp")
    pd.DataFrame(results).to_csv(tmp, index=False)
    os.replace(tmp, checkpoint_path)


# ======================================================================
# 3. LOAD AND VALIDATE INPUTS
# ======================================================================

print("=" * 78)
print("VIM-2 STEP 5 — SECONDARY RANDOM-SPLIT BENCHMARK")
print("=" * 78)
print(f"Started: {utc_now()}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print()

for path in [FINAL_DATASET, FOLD_ASSIGNMENTS_FILE, FEATURE_MANIFEST_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required input file not found: {path}")

print("[1/10] Loading dataset, folds, and feature manifest...")
df = pd.read_csv(FINAL_DATASET)
folds = pd.read_csv(FOLD_ASSIGNMENTS_FILE)
manifest = pd.read_csv(FEATURE_MANIFEST_FILE)

if len(df) != len(folds):
    raise ValueError(
        f"Dataset/fold row mismatch: dataset={len(df)}, folds={len(folds)}"
    )

if "Row_Index" in folds.columns:
    if not np.array_equal(folds["Row_Index"].to_numpy(), np.arange(len(folds))):
        raise ValueError("Part 13 Row_Index is not aligned with dataset row order.")

validate_outer_random_folds(folds)

model_a_features, model_b_features = validate_feature_manifest(
    manifest, df.columns.tolist()
)
assert_no_target_or_sd_predictors(model_a_features, TARGETS)
assert_no_target_or_sd_predictors(model_b_features, TARGETS)

print(f"Dataset rows: {len(df):,}")
print(f"Model A predictors: {len(model_a_features)}")
print(f"Model B predictors: {len(model_b_features)}")
print(f"Targets: {len(TARGETS)}")
print(f"Random outer folds: {N_OUTER_FOLDS}")
print()


# ======================================================================
# 4. ALIGN FOLD ASSIGNMENTS WITH DATA
# ======================================================================

print("[2/10] Aligning frozen Part 13 random folds...")

# Part 13 stores fold assignments in dataset row order. This explicit copy
# avoids accidental dependence on a mutation string that may not be unique.
df = df.copy()
df["__Random_Fold"] = folds[RANDOM_FOLD_COLUMN].astype(int).to_numpy()

if POSITION_COLUMN in folds.columns:
    df["__Sequence_Position_Check"] = folds[POSITION_COLUMN].to_numpy()
    if POSITION_COLUMN in df.columns:
        left = pd.to_numeric(df[POSITION_COLUMN], errors="coerce")
        right = pd.to_numeric(df["__Sequence_Position_Check"], errors="coerce")
        if not np.array_equal(left.fillna(-999999).to_numpy(), right.fillna(-999999).to_numpy()):
            raise ValueError("Sequence_Position mismatch between dataset and Part 13 folds.")

print("Random fold alignment: PASS")
print()


# ======================================================================
# 5. RESUME / CHECKPOINT MANAGEMENT
# ======================================================================

print("[3/10] Preparing checkpoint/resume system...")

CHECKPOINT_FILE = OUTPUT_DIR / "VIM2_Step5_Random_Split_Checkpoint.csv"

if CHECKPOINT_FILE.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
    checkpoint_records = checkpoint_df.to_dict("records")
    completed_keys = {
        (
            str(r["Target"]),
            str(r["Architecture"]),
            int(r["Outer_Fold"]),
        )
        for r in checkpoint_records
    }
    print(f"Existing checkpoint found: {len(completed_keys)} completed configurations.")
else:
    checkpoint_records = []
    completed_keys = set()
    print("No checkpoint found; starting from 0/90.")

TOTAL_CONFIGS = len(TARGETS) * len(ARCHITECTURES) * N_OUTER_FOLDS
print(f"Total outer configurations: {TOTAL_CONFIGS}")
print()


# ======================================================================
# 6. MAIN NESTED RANDOM-SPLIT BENCHMARK
# ======================================================================

print("[4/10] Running nested random-split benchmark...")
print()

start_time = time.time()
completed_this_run = 0

for target in TARGETS:
    if target not in df.columns:
        raise ValueError(f"Target not found in dataset: {target}")

    # Rows with a missing target cannot contribute to supervised learning.
    valid_target_mask = df[target].notna().to_numpy()

    for architecture, include_position in ARCHITECTURES.items():
        feature_list = model_b_features if include_position else model_a_features

        for outer_fold in range(1, N_OUTER_FOLDS + 1):
            key = (target, architecture, outer_fold)
            if key in completed_keys:
                continue

            config_start = time.time()
            mask = valid_target_mask
            outer_test_mask = df["__Random_Fold"].to_numpy() == outer_fold
            outer_train_mask = ~outer_test_mask
            outer_train_mask &= mask
            outer_test_mask &= mask

            if outer_train_mask.sum() < N_INNER_FOLDS:
                raise ValueError(
                    f"Insufficient outer-training rows for {target}, {architecture}, fold {outer_fold}."
                )
            if outer_test_mask.sum() < 1:
                raise ValueError(
                    f"Empty outer-test fold for {target}, {architecture}, fold {outer_fold}."
                )

            X_train = df.loc[outer_train_mask, feature_list].copy()
            y_train = pd.to_numeric(df.loc[outer_train_mask, target], errors="coerce")
            X_test = df.loc[outer_test_mask, feature_list].copy()
            y_test = pd.to_numeric(df.loc[outer_test_mask, target], errors="coerce")

            # ----------------------------------------------------------
            # Candidate family selection using ONLY outer training data.
            # ----------------------------------------------------------
            inner_splits = build_random_inner_splits(
                df.loc[outer_train_mask, POSITION_COLUMN]
                if POSITION_COLUMN in df.columns
                else pd.Series(np.arange(len(X_train))),
                seed=RANDOM_STATE + outer_fold,
            )

            candidate_scores = {}
            for family in MODEL_FAMILIES:
                candidate_pipeline = make_pipeline(
                    X_train,
                    family,
                    BASELINE_PARAMS[family],
                )
                candidate_cv = []
                for inner_train_idx, inner_valid_idx in inner_splits:
                    X_inner_train = X_train.iloc[inner_train_idx]
                    y_inner_train = y_train.iloc[inner_train_idx]
                    X_inner_valid = X_train.iloc[inner_valid_idx]
                    y_inner_valid = y_train.iloc[inner_valid_idx]

                    candidate_pipeline.fit(X_inner_train, y_inner_train)
                    pred = candidate_pipeline.predict(X_inner_valid)
                    candidate_cv.append(r2_score(y_inner_valid, pred))

                candidate_scores[family] = float(np.mean(candidate_cv))

            selected_family = max(
                MODEL_FAMILIES,
                key=lambda fam: (candidate_scores[fam], fam == "Random_Forest"),
            )

            # ----------------------------------------------------------
            # Constrained HPO on the selected family using ONLY outer
            # training data and the same random inner CV splits.
            # ----------------------------------------------------------
            hpo_pipeline = make_pipeline(X_train, selected_family)
            search = RandomizedSearchCV(
                estimator=hpo_pipeline,
                param_distributions=HPO_SPACES[selected_family],
                n_iter=N_HPO_ITERATIONS,
                scoring="r2",
                cv=inner_splits,
                refit=True,
                random_state=RANDOM_STATE + outer_fold,
                n_jobs=-1,
            )
            search.fit(X_train, y_train)

            best_params = validate_hyperparameter_schema(
                selected_family,
                search.best_params_,
            )

            # ----------------------------------------------------------
            # Final outer-test fit: the test partition is untouched until
            # this point and is used exactly once for evaluation.
            # ----------------------------------------------------------
            final_pipeline = make_pipeline(
                X_train,
                selected_family,
                best_params,
            )
            final_pipeline.fit(X_train, y_train)
            y_pred = final_pipeline.predict(X_test)

            spearman = safe_spearman(y_test.to_numpy(), y_pred)
            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae = float(mean_absolute_error(y_test, y_pred))
            r2 = float(r2_score(y_test, y_pred))

            record = {
                "Target": target,
                "Architecture": architecture,
                "Outer_Fold": outer_fold,
                "Model_Family": selected_family,
                "Inner_Baseline_R2_Random_Forest": candidate_scores["Random_Forest"],
                "Inner_Baseline_R2_Extra_Trees": candidate_scores["Extra_Trees"],
                "Best_Inner_R2": float(search.best_score_),
                "Best_Hyperparameters": json.dumps(best_params, sort_keys=True),
                "N_Train": int(len(X_train)),
                "N_Test": int(len(X_test)),
                "Outer_Test_R2": r2,
                "Outer_Test_RMSE": rmse,
                "Outer_Test_MAE": mae,
                "Outer_Test_Spearman": spearman,
                "HPO_Iterations": N_HPO_ITERATIONS,
                "Inner_CV_Folds": N_INNER_FOLDS,
                "Random_State": RANDOM_STATE,
                "Elapsed_Seconds": round(time.time() - config_start, 3),
            }

            checkpoint_records.append(record)
            completed_keys.add(key)
            completed_this_run += 1
            save_checkpoint(checkpoint_records, CHECKPOINT_FILE)

            elapsed = time.time() - start_time
            done = len(completed_keys)
            avg = elapsed / completed_this_run if completed_this_run else np.nan
            remaining = max(TOTAL_CONFIGS - done, 0)
            eta = avg * remaining if np.isfinite(avg) else np.nan

            print(
                f"[{done:02d}/{TOTAL_CONFIGS}] "
                f"Target={target} | {architecture} | Fold={outer_fold} | "
                f"Selected={selected_family} | Test R2={r2:.4f} | "
                f"Elapsed={format_seconds(elapsed)} | "
                f"ETA={format_seconds(eta) if np.isfinite(eta) else 'N/A'}"
            )

print()
print("Nested random-split benchmark complete.")
print()


# ======================================================================
# 7. SAVE FOLD-LEVEL PERFORMANCE
# ======================================================================

print("[5/10] Writing fold-level performance...")

results_df = pd.DataFrame(checkpoint_records)
results_df = results_df.sort_values(
    ["Target", "Architecture", "Outer_Fold"]
).reset_index(drop=True)

fold_output = OUTPUT_DIR / "VIM2_Step5_Random_Split_Fold_Performance.csv"
results_df.to_csv(fold_output, index=False)


# ======================================================================
# 8. POOLED / OOF SUMMARY
# ======================================================================

print("[6/10] Computing pooled benchmark summaries...")

# Reconstruct pooled OOF predictions by rerunning only the frozen models.
# This stage does not perform selection or HPO; all model decisions are
# already frozen in results_df. It is intentionally separate from the
# outer evaluation logic for transparent auditability.

oof_rows = []

for row in results_df.to_dict("records"):
    target = row["Target"]
    architecture = row["Architecture"]
    outer_fold = int(row["Outer_Fold"])
    family = normalize_model_name(row["Model_Family"])
    params = validate_hyperparameter_schema(
        family,
        parse_hyperparameters(row["Best_Hyperparameters"]),
    )
    feature_list = (
        model_b_features
        if architecture == "Model_B_With_Position"
        else model_a_features
    )

    valid = df[target].notna().to_numpy()
    test_mask = (df["__Random_Fold"].to_numpy() == outer_fold) & valid
    train_mask = (df["__Random_Fold"].to_numpy() != outer_fold) & valid

    X_train = df.loc[train_mask, feature_list].copy()
    y_train = pd.to_numeric(df.loc[train_mask, target], errors="coerce")
    X_test = df.loc[test_mask, feature_list].copy()
    y_test = pd.to_numeric(df.loc[test_mask, target], errors="coerce")

    pipeline = make_pipeline(X_train, family, params)
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    row_index_values = df.index[test_mask].to_numpy()
    for idx, true_value, prediction in zip(
        row_index_values, y_test.to_numpy(), pred
    ):
        oof_rows.append(
            {
                "Dataset_Row_Index": int(idx),
                "Target": target,
                "Architecture": architecture,
                "Random_Fold": outer_fold,
                "Observed": float(true_value),
                "Predicted": float(prediction),
                "Model_Family": family,
            }
        )

oof_df = pd.DataFrame(oof_rows)
oof_output = OUTPUT_DIR / "VIM2_Step5_Random_Split_OOF_Predictions.csv"
oof_df.to_csv(oof_output, index=False)

summary_rows = []
for (target, architecture), group in oof_df.groupby(
    ["Target", "Architecture"], sort=True
):
    y_true = group["Observed"].to_numpy()
    y_pred = group["Predicted"].to_numpy()
    summary_rows.append(
        {
            "Target": target,
            "Architecture": architecture,
            "N_OOF": int(len(group)),
            "Pooled_OOF_R2": float(r2_score(y_true, y_pred)),
            "Pooled_OOF_RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "Pooled_OOF_MAE": float(mean_absolute_error(y_true, y_pred)),
            "Pooled_OOF_Spearman": safe_spearman(y_true, y_pred),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_output = OUTPUT_DIR / "VIM2_Step5_Random_Split_Pooled_OOF_Performance.csv"
summary_df.to_csv(summary_output, index=False)


# ======================================================================
# 9. MODEL A VS MODEL B COMPARISON
# ======================================================================

print("[7/10] Computing Model A vs Model B comparison...")

pivot = summary_df.pivot(
    index="Target",
    columns="Architecture",
    values=["Pooled_OOF_R2", "Pooled_OOF_RMSE", "Pooled_OOF_MAE", "Pooled_OOF_Spearman"],
)

comparison_rows = []
for target in TARGETS:
    row = {"Target": target}
    for metric in ["Pooled_OOF_R2", "Pooled_OOF_RMSE", "Pooled_OOF_MAE", "Pooled_OOF_Spearman"]:
        a_col = (metric, "Model_A_No_Position")
        b_col = (metric, "Model_B_With_Position")
        if a_col in pivot.columns and b_col in pivot.columns:
            a = pivot.loc[target, a_col]
            b = pivot.loc[target, b_col]
            row[f"Model_A_{metric}"] = a
            row[f"Model_B_{metric}"] = b
            row[f"Model_B_minus_A_{metric}"] = b - a
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_output = OUTPUT_DIR / "VIM2_Step5_Model_A_vs_B_Random_Split_Comparison.csv"
comparison_df.to_csv(comparison_output, index=False)


# ======================================================================
# 10. LEAKAGE AUDIT AND REPRODUCIBILITY MANIFEST
# ======================================================================

print("[8/10] Writing leakage audit...")

leakage_rows = [
    {
        "Check": "Frozen random outer folds from Part 13",
        "Status": "PASS",
        "Details": "Random_Fold assignments are read directly from Part 13 and never regenerated.",
    },
    {
        "Check": "Outer-test data excluded from candidate selection",
        "Status": "PASS",
        "Details": "RF vs Extra Trees family selection uses only outer-training rows.",
    },
    {
        "Check": "Outer-test data excluded from HPO",
        "Status": "PASS",
        "Details": "RandomizedSearchCV uses only inner CV splits generated from outer training data.",
    },
    {
        "Check": "Preprocessing leakage protection",
        "Status": "PASS",
        "Details": "Imputation, scaling, and one-hot encoding are inside the sklearn Pipeline.",
    },
    {
        "Check": "Model A excludes Sequence_Position",
        "Status": "PASS",
        "Details": f"Model A contains exactly {len(model_a_features)} predictors.",
    },
    {
        "Check": "Model B includes Sequence_Position",
        "Status": "PASS",
        "Details": f"Model B contains exactly {len(model_b_features)} predictors.",
    },
    {
        "Check": "Model A/B differ only by Sequence_Position",
        "Status": "PASS",
        "Details": "Feature manifest was used as the source of truth.",
    },
    {
        "Check": "Target SD columns excluded",
        "Status": "PASS",
        "Details": "No *_SD target uncertainty column is included as a predictor.",
    },
    {
        "Check": "Step 4 results used for model selection",
        "Status": "PASS",
        "Details": "Step 5 performs its own nested random-split selection and does not select models using Step 4 test metrics.",
    },
    {
        "Check": "Random-split interpretation",
        "Status": "PASS",
        "Details": "Benchmark is explicitly secondary because random folds may share Sequence_Position across train and test.",
    },
]

leakage_df = pd.DataFrame(leakage_rows)
leakage_output = OUTPUT_DIR / "VIM2_Step5_Random_Split_Leakage_Audit.csv"
leakage_df.to_csv(leakage_output, index=False)


print("[9/10] Writing reproducibility manifest...")

manifest_out = {
    "project": PROJECT_NAME,
    "step": STEP_NAME,
    "status": "COMPLETE",
    "timestamp_utc": utc_now(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "random_state": RANDOM_STATE,
    "n_outer_folds": N_OUTER_FOLDS,
    "n_inner_folds": N_INNER_FOLDS,
    "hpo_iterations": N_HPO_ITERATIONS,
    "dataset_rows": int(len(df)),
    "targets": TARGETS,
    "model_a_predictor_count": len(model_a_features),
    "model_b_predictor_count": len(model_b_features),
    "primary_evaluation": "Position-Aware nested outer-test evaluation in Step 4",
    "secondary_evaluation": "Frozen Part 13 Random_Fold nested benchmark in Step 5",
    "interpretation_note": "Random-split results may be optimistic for unseen-position generalization.",
    "input_files": {
        "final_dataset": {
            "path": FINAL_DATASET,
            "sha256": sha256_file(FINAL_DATASET),
        },
        "fold_assignments": {
            "path": FOLD_ASSIGNMENTS_FILE,
            "sha256": sha256_file(FOLD_ASSIGNMENTS_FILE),
        },
        "feature_manifest": {
            "path": FEATURE_MANIFEST_FILE,
            "sha256": sha256_file(FEATURE_MANIFEST_FILE),
        },
    },
    "step4_reference_directory": STEP4_REFERENCE_DIR,
    "outputs": {
        "fold_performance": str(fold_output),
        "pooled_oof_performance": str(summary_output),
        "oof_predictions": str(oof_output),
        "model_a_vs_b": str(comparison_output),
        "leakage_audit": str(leakage_output),
        "checkpoint": str(CHECKPOINT_FILE),
    },
}

manifest_output = OUTPUT_DIR / "VIM2_Step5_Random_Split_Reproducibility_Manifest.json"
with open(manifest_output, "w", encoding="utf-8") as handle:
    json.dump(manifest_out, handle, indent=2)


# ======================================================================
# 11. FINAL VALIDATION
# ======================================================================

print("[10/10] Running final output validation...")

expected_rows = TOTAL_CONFIGS
if len(results_df) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} fold-level evaluations, found {len(results_df)}."
    )

if results_df[["Target", "Architecture", "Outer_Fold"]].duplicated().any():
    raise RuntimeError("Duplicate target × architecture × random fold configuration detected.")

if results_df["Model_Family"].map(normalize_model_name).isin(MODEL_FAMILIES).all() is False:
    raise RuntimeError("Invalid model family found in Step 5 results.")

if not summary_df.shape[0] == len(TARGETS) * len(ARCHITECTURES):
    raise RuntimeError("Unexpected pooled OOF summary size.")

for path in [
    fold_output,
    summary_output,
    oof_output,
    comparison_output,
    leakage_output,
    manifest_output,
]:
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Expected output is missing or empty: {path}")

print()
print("=" * 78)
print("STEP 5 COMPLETE — SECONDARY RANDOM-SPLIT BENCHMARK")
print("=" * 78)
print(f"Fold-level evaluations: {len(results_df)} / {TOTAL_CONFIGS}")
print(f"Model A pooled summaries: {sum(summary_df.Architecture == 'Model_A_No_Position')}")
print(f"Model B pooled summaries: {sum(summary_df.Architecture == 'Model_B_With_Position')}")
print(f"Output directory: {OUTPUT_DIR}")
print()
print("IMPORTANT: Step 5 is a secondary benchmark; Step 4 remains the")
print("primary Position-Aware generalization evaluation.")
print("Do not make manuscript-final model-selection claims until Step 6.")
print("=" * 78)


VIM-2 STEP 5 — SECONDARY RANDOM-SPLIT BENCHMARK
Started: 2026-09-03T10:21:25.432852+00:00
Python: 3.13.15
pandas: 2.2.3
scikit-learn: 1.6.1

[1/10] Loading dataset, folds, and feature manifest...
Dataset rows: 5,016
Model A predictors: 65
Model B predictors: 66
Targets: 9
Random outer folds: 5

[2/10] Aligning frozen Part 13 random folds...
Random fold alignment: PASS

[3/10] Preparing checkpoint/resume system...
No checkpoint found; starting from 0/90.
Total outer configurations: 90

[4/10] Running nested random-split benchmark...

[01/90] Target=0.031ug/mL_MEM_37C | Model_A_No_Position | Fold=1 | Selected=Extra_Trees | Test R2=0.8170 | Elapsed=00:02:36 | ETA=03:52:51
[02/90] Target=0.031ug/mL_MEM_37C | Model_A_No_Position | Fold=2 | Selected=Extra_Trees | Test R2=0.8100 | Elapsed=00:04:21 | ETA=03:12:04
[03/90] Target=0.031ug/mL_MEM_37C | Model_A_No_Position | Fold=3 | Selected=Extra_Trees | Test R2=0.8398 | Elapsed=00:05:56 | ETA=02:52:11
[04/90] Target=0.031ug/mL_MEM_37C | Model_A_

In [ ]:
# @title

# ============================================================
# DOWNLOAD STEP5 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step5_Random_Split_Benchmark"

# Output ZIP archive
zip_base = "/content/VIM2_Step5_Random_Split_Benchmark"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step5_Random_Split_Benchmark"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP5 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP5 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step5_Random_Split_Benchmark
ZIP archive      : /content/VIM2_Step5_Random_Split_Benchmark.zip
Archive size     : 1.76 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title

"""
VIM-2 ML PROJECT
STEP 6 — STATISTICAL COMPARISON AND PRIMARY MODEL SELECTION
Q1 FINAL / SCHEMA-ROBUST / LEAKAGE-SAFE

Purpose
-------
Integrate the FROZEN results of:
  Step 4 = primary Position-Aware nested outer-test evaluation
  Step 5 = secondary Random-Split nested outer-test benchmark

Step 6:
  1. Validates Step 4 and Step 5 schemas and completeness.
  2. Canonicalizes metric names without modifying the source CSVs.
  3. Compares Model A vs Model B using paired outer-fold statistics.
  4. Quantifies Position-Aware vs Random-Split differences as supporting evidence.
  5. Integrates pooled OOF metrics.
  6. Selects the final architecture (A vs B) using a pre-specified
     Position-Aware-only decision hierarchy.
  7. Applies Benjamini-Hochberg FDR correction across the nine phenotype
     A-vs-B primary tests.
  8. Produces 10,000-iteration percentile bootstrap CIs.
  9. Writes a detailed leakage/decision audit and reproducibility manifest.

Critical safeguards
-------------------
* NO model fitting.
* NO HPO.
* NO new folds.
* NO use of Part 14.
* Step 4 and Step 5 are treated as frozen evidence.
* Random-Split cannot override the primary Position-Aware decision.
* RF vs Extra Trees is NOT re-selected here; the Step 3 frozen family remains
  descriptive evidence only.
* Model A = 65 predictors, no Sequence_Position.
* Model B = 66 predictors, with Sequence_Position.
* Statistical tests are supporting evidence; n=5 outer folds is low-power.

Final architecture-selection hierarchy
---------------------------------------
For each target:
  1. Higher pooled Position-Aware OOF R2.
  2. If exactly tied, higher mean Position-Aware outer-fold R2.
  3. If still tied, lower pooled Position-Aware RMSE.
  4. If still tied, retain simpler Model A.

This is an architecture decision only. It does not select RF vs Extra Trees.
"""

from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, ttest_rel, wilcoxon


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

PROJECT_NAME = "VIM-2 ML Project"
STEP_NAME = "STEP 6 — STATISTICAL COMPARISON AND PRIMARY MODEL SELECTION"

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
ALPHA = 0.05
BOOTSTRAP_ITERATIONS = 10_000

MODEL_A = "Model_A_No_Position"
MODEL_B = "Model_B_With_Position"
ARCHITECTURES = [MODEL_A, MODEL_B]

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

# Actual completed Step 4 / Step 5 directories.
STEP4_DIR = Path("/content")
STEP5_DIR = Path("/content")
OUTPUT_DIR = Path("/content/VIM2_Step6_Statistical_Comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STEP4_FOLD_FILENAME = "VIM2_Step4_Final_OuterFold_Performance.csv"
STEP4_POOLED_FILENAME = "VIM2_Step4_Pooled_OOF_Performance.csv"
STEP5_FOLD_FILENAME = "VIM2_Step5_Random_Split_Fold_Performance.csv"
STEP5_POOLED_FILENAME = "VIM2_Step5_Random_Split_Pooled_OOF_Performance.csv"

MODEL_FAMILIES = {"Random_Forest", "Extra_Trees"}

# Canonical metric names used internally by Step 6.
METRICS = ["R2", "RMSE", "MAE", "Spearman"]

EXPECTED_FEATURE_COUNTS = {
    MODEL_A: 65,
    MODEL_B: 66,
}


# ======================================================================
# 2. HELPERS
# ======================================================================

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def require_file(path: str | Path) -> Path:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Required input is missing or empty: {path}")
    return path


def require_columns(df: pd.DataFrame, columns: list[str], name: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def normalize_architecture(value: object) -> str:
    s = str(value).strip().lower().replace("-", "_").replace(" ", "")
    if "model_a" in s or "modela" in s or "no_position" in s:
        return MODEL_A
    if "model_b" in s or "modelb" in s or "with_position" in s:
        return MODEL_B
    raise ValueError(f"Unknown architecture label: {value!r}")


def normalize_model_family(value: object) -> str:
    s = str(value).strip().lower().replace("-", "_").replace(" ", "")
    if "random_forest" in s or "randomforest" in s:
        return "Random_Forest"
    if "extra_trees" in s or "extratrees" in s:
        return "Extra_Trees"
    raise ValueError(f"Unknown model family: {value!r}")


def canonicalize_fold_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Convert source-specific fold metric names into the canonical internal
    names. The original DataFrame is not modified.
    """
    out = df.copy()

    # Step 5 actual schema:
    # Outer_Test_R2 / Outer_Test_RMSE / Outer_Test_MAE / Outer_Test_Spearman
    # Step 4 actual schema:
    # R2 / RMSE / MAE / Spearman
    aliases = {
        "R2": ["R2", "Outer_Test_R2"],
        "RMSE": ["RMSE", "Outer_Test_RMSE"],
        "MAE": ["MAE", "Outer_Test_MAE"],
        "Spearman": ["Spearman", "Outer_Test_Spearman"],
    }

    require_columns(out, ["Target", "Architecture", "Outer_Fold"], name)

    for canonical, candidates in aliases.items():
        present = [c for c in candidates if c in out.columns]
        if len(present) == 0:
            raise ValueError(
                f"{name} has no usable column for {canonical}. "
                f"Expected one of: {candidates}"
            )
        if len(present) > 1:
            # If both exist, they must agree numerically.
            left = pd.to_numeric(out[present[0]], errors="coerce")
            right = pd.to_numeric(out[present[1]], errors="coerce")
            if not np.allclose(left.to_numpy(float), right.to_numpy(float),
                               equal_nan=True):
                raise ValueError(
                    f"{name} contains conflicting duplicate representations "
                    f"for {canonical}: {present}"
                )
        out[canonical] = pd.to_numeric(out[present[0]], errors="coerce")

    out["Target"] = out["Target"].astype(str).str.strip()
    out["Architecture"] = out["Architecture"].map(normalize_architecture)
    out["Outer_Fold"] = pd.to_numeric(
        out["Outer_Fold"], errors="raise"
    ).astype(int)

    if "Model_Family" in out.columns:
        out["Model_Family"] = out["Model_Family"].map(normalize_model_family)

    return out


def canonicalize_pooled_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Canonicalize pooled Step 4 / Step 5 OOF summaries.

    Step 4 actual frozen schema:
      Target, Architecture, Selected_Model_Families, OOF_N,
      OOF_R2, OOF_RMSE, OOF_MAE, OOF_Spearman
    Step 5 actual frozen schema may use:
      Target, Architecture, ..., Pooled_OOF_R2, Pooled_OOF_RMSE,
      Pooled_OOF_MAE, Pooled_OOF_Spearman
    """
    out = df.copy()

    require_columns(out, ["Target", "Architecture"], name)

    # Actual frozen Step 4 pooled schema:
    # OOF_R2 / OOF_RMSE / OOF_MAE / OOF_Spearman
    # Some future/alternate outputs may use R2 or Pooled_OOF_*; all are
    # accepted, but the source values are never modified on disk.
    aliases = {
        "R2": ["R2", "OOF_R2", "Pooled_OOF_R2"],
        "RMSE": ["RMSE", "OOF_RMSE", "Pooled_OOF_RMSE"],
        "MAE": ["MAE", "OOF_MAE", "Pooled_OOF_MAE"],
        "Spearman": ["Spearman", "OOF_Spearman", "Pooled_OOF_Spearman"],
    }

    for canonical, candidates in aliases.items():
        present = [c for c in candidates if c in out.columns]
        if len(present) == 0:
            raise ValueError(
                f"{name} has no usable pooled column for {canonical}. "
                f"Expected one of: {candidates}"
            )
        if len(present) > 1:
            left = pd.to_numeric(out[present[0]], errors="coerce")
            right = pd.to_numeric(out[present[1]], errors="coerce")
            if not np.allclose(left.to_numpy(float), right.to_numpy(float),
                               equal_nan=True):
                raise ValueError(
                    f"{name} contains conflicting duplicate representations "
                    f"for {canonical}: {present}"
                )
        out[canonical] = pd.to_numeric(out[present[0]], errors="coerce")

    out["Target"] = out["Target"].astype(str).str.strip()
    out["Architecture"] = out["Architecture"].map(normalize_architecture)

    # Step 4 uses OOF_N; normalize it to the internal N_OOF name.
    if "N_OOF" in out.columns:
        out["N_OOF"] = pd.to_numeric(out["N_OOF"], errors="coerce")
    elif "OOF_N" in out.columns:
        out["N_OOF"] = pd.to_numeric(out["OOF_N"], errors="coerce")

    if "N_OOF" in out.columns:
        if out["N_OOF"].isna().any() or (out["N_OOF"] <= 0).any():
            raise ValueError(f"{name} contains invalid OOF_N/N_OOF values.")

    # Preserve and validate the frozen family summary when supplied.
    if "Selected_Model_Families" in out.columns:
        if out["Selected_Model_Families"].astype(str).str.strip().eq("").any():
            raise ValueError(
                f"{name} contains empty Selected_Model_Families values."
            )

    return out


def validate_common_targets_architectures(
    df: pd.DataFrame,
    name: str,
    expect_outer_fold: bool,
) -> None:
    unknown_targets = sorted(set(df["Target"]) - set(TARGETS))
    missing_targets = sorted(set(TARGETS) - set(df["Target"]))

    if unknown_targets:
        raise ValueError(f"{name} contains unknown targets: {unknown_targets}")
    if missing_targets:
        raise ValueError(f"{name} is missing targets: {missing_targets}")

    unknown_arch = sorted(set(df["Architecture"]) - set(ARCHITECTURES))
    missing_arch = sorted(set(ARCHITECTURES) - set(df["Architecture"]))

    if unknown_arch:
        raise ValueError(f"{name} contains unknown architectures: {unknown_arch}")
    if missing_arch:
        raise ValueError(f"{name} is missing architectures: {missing_arch}")

    for metric in METRICS:
        if df[metric].isna().any():
            bad = int(df[metric].isna().sum())
            raise ValueError(f"{name} contains {bad} missing {metric} values.")
        if not np.isfinite(df[metric].to_numpy(float)).all():
            raise ValueError(f"{name} contains non-finite {metric} values.")

    if expect_outer_fold:
        invalid_folds = sorted(
            set(df["Outer_Fold"]) - set(range(1, N_OUTER_FOLDS + 1))
        )
        if invalid_folds:
            raise ValueError(
                f"{name} contains invalid outer-fold IDs: {invalid_folds}"
            )


def validate_fold_completeness(df: pd.DataFrame, name: str) -> None:
    expected_rows = len(TARGETS) * len(ARCHITECTURES) * N_OUTER_FOLDS

    if len(df) != expected_rows:
        raise RuntimeError(
            f"{name}: expected exactly {expected_rows} rows "
            f"(9 targets × 2 architectures × 5 folds); found {len(df)}."
        )

    if df.duplicated(
        ["Target", "Architecture", "Outer_Fold"]
    ).any():
        dup = df.loc[
            df.duplicated(
                ["Target", "Architecture", "Outer_Fold"], keep=False
            ),
            ["Target", "Architecture", "Outer_Fold"],
        ]
        raise RuntimeError(
            f"{name}: duplicate Target × Architecture × Outer_Fold records:\n"
            f"{dup.to_string(index=False)}"
        )

    counts = (
        df.groupby(["Target", "Architecture"])["Outer_Fold"]
        .nunique()
        .reindex(
            pd.MultiIndex.from_product(
                [TARGETS, ARCHITECTURES],
                names=["Target", "Architecture"],
            )
        )
    )

    if counts.isna().any() or not (counts == N_OUTER_FOLDS).all():
        raise RuntimeError(
            f"{name}: every target × architecture combination must contain "
            f"exactly {N_OUTER_FOLDS} distinct folds."
        )

    if set(df["Outer_Fold"].unique()) != set(range(1, N_OUTER_FOLDS + 1)):
        raise RuntimeError(
            f"{name}: expected outer folds 1..{N_OUTER_FOLDS}."
        )


def validate_pooled_completeness(df: pd.DataFrame, name: str) -> None:
    expected_rows = len(TARGETS) * len(ARCHITECTURES)

    if len(df) != expected_rows:
        raise RuntimeError(
            f"{name}: expected exactly {expected_rows} rows "
            f"(9 targets × 2 architectures); found {len(df)}."
        )

    if df.duplicated(["Target", "Architecture"]).any():
        raise RuntimeError(
            f"{name}: duplicate Target × Architecture pooled records."
        )


def validate_step5_family_information(df: pd.DataFrame) -> None:
    if "Model_Family" not in df.columns:
        raise ValueError(
            "Step 5 fold results must contain Model_Family so that Step 6 "
            "can verify that frozen Step 3 family decisions remain intact."
        )

    invalid = sorted(set(df["Model_Family"]) - MODEL_FAMILIES)
    if invalid:
        raise ValueError(
            f"Step 5 contains invalid frozen model families: {invalid}"
        )

    # Candidate-selection columns are not used for Step 6 decisions, but
    # their presence verifies that the expected Step 5 output was supplied.
    required_step5 = [
        "Best_Inner_R2",
        "Best_Hyperparameters",
        "N_Train",
        "N_Test",
        "HPO_Iterations",
        "Inner_CV_Folds",
        "Random_State",
    ]
    require_columns(df, required_step5, "Step 5 fold results")

    for col in ["N_Train", "N_Test", "HPO_Iterations", "Inner_CV_Folds"]:
        vals = pd.to_numeric(df[col], errors="coerce")
        if vals.isna().any():
            raise ValueError(f"Step 5 contains invalid values in {col}.")
        if (vals <= 0).any():
            raise ValueError(f"Step 5 contains non-positive values in {col}.")

    if not np.isfinite(
        pd.to_numeric(df["Best_Inner_R2"], errors="coerce").to_numpy(float)
    ).all():
        raise ValueError("Step 5 contains non-finite Best_Inner_R2 values.")

    if df["Best_Hyperparameters"].astype(str).str.strip().eq("").any():
        raise ValueError("Step 5 contains empty Best_Hyperparameters values.")


def validate_paired_ab(df: pd.DataFrame, benchmark_name: str) -> None:
    for target in TARGETS:
        sub = df[df["Target"] == target]
        a = sub[sub["Architecture"] == MODEL_A].sort_values("Outer_Fold")
        b = sub[sub["Architecture"] == MODEL_B].sort_values("Outer_Fold")

        if len(a) != N_OUTER_FOLDS or len(b) != N_OUTER_FOLDS:
            raise RuntimeError(
                f"{benchmark_name}: incomplete A/B fold pairing for {target}."
            )

        if not np.array_equal(
            a["Outer_Fold"].to_numpy(int),
            b["Outer_Fold"].to_numpy(int),
        ):
            raise RuntimeError(
                f"{benchmark_name}: Model A and B are not paired on identical "
                f"outer-fold IDs for {target}."
            )


def safe_mean(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    return float(np.mean(x)) if len(x) else np.nan


def bootstrap_mean_ci(
    values: np.ndarray,
    seed: int,
) -> tuple[float, float]:
    """
    Percentile bootstrap 95% CI for the mean across outer folds.

    With n=5 this is descriptive/supporting evidence, not a substitute
    for independent large-sample inference.
    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    indices = rng.integers(
        0,
        len(x),
        size=(BOOTSTRAP_ITERATIONS, len(x)),
    )
    means = x[indices].mean(axis=1)
    return (
        float(np.percentile(means, 2.5)),
        float(np.percentile(means, 97.5)),
    )


def bootstrap_paired_delta_ci(
    a: np.ndarray,
    b: np.ndarray,
    seed: int,
) -> tuple[float, float]:
    """Percentile bootstrap 95% CI for paired B-minus-A fold differences."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    delta = b[mask] - a[mask]

    if len(delta) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    indices = rng.integers(
        0,
        len(delta),
        size=(BOOTSTRAP_ITERATIONS, len(delta)),
    )
    means = delta[indices].mean(axis=1)
    return (
        float(np.percentile(means, 2.5)),
        float(np.percentile(means, 97.5)),
    )


def paired_tests(a: np.ndarray, b: np.ndarray) -> dict[str, float]:
    """
    Paired tests for Model B versus Model A across identical outer folds.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]

    if len(a) < 2:
        return {
            "Paired_t": np.nan,
            "Paired_t_p": np.nan,
            "Wilcoxon": np.nan,
            "Wilcoxon_p": np.nan,
        }

    t_stat, t_p = ttest_rel(b, a)

    if np.allclose(b - a, 0.0):
        w_stat, w_p = 0.0, 1.0
    else:
        try:
            w_stat, w_p = wilcoxon(
                b,
                a,
                alternative="two-sided",
                method="auto",
            )
        except Exception:
            w_stat, w_p = np.nan, np.nan

    return {
        "Paired_t": float(t_stat),
        "Paired_t_p": float(t_p),
        "Wilcoxon": float(w_stat),
        "Wilcoxon_p": float(w_p),
    }


def paired_cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    """Paired Cohen's d for B minus A."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    d = (b - a)[np.isfinite(a) & np.isfinite(b)]

    if len(d) < 2:
        return np.nan

    sd = np.std(d, ddof=1)
    if sd == 0:
        if np.mean(d) == 0:
            return 0.0
        return math.copysign(np.inf, np.mean(d))

    return float(np.mean(d) / sd)


def welch_test(a: np.ndarray, b: np.ndarray) -> dict[str, float]:
    """
    Supporting Welch test for Position-Aware vs Random-Split fold R2.

    The two split strategies are not naturally paired because their frozen
    outer partitions differ, so this is descriptive/supporting only.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) < 2 or len(b) < 2:
        return {"Welch_t": np.nan, "Welch_p": np.nan}

    stat, p = ttest_ind(b, a, equal_var=False)
    return {"Welch_t": float(stat), "Welch_p": float(p)}


def benjamini_hochberg(
    p_values: np.ndarray,
) -> np.ndarray:
    """
    Benjamini-Hochberg FDR-adjusted q-values.

    NaN p-values remain NaN.
    """
    p = np.asarray(p_values, dtype=float)
    q = np.full(len(p), np.nan)

    valid = np.isfinite(p)
    idx = np.where(valid)[0]
    if len(idx) == 0:
        return q

    order = idx[np.argsort(p[idx])]
    m = len(order)

    running = 1.0
    for rank, original_idx in reversed(
        list(enumerate(order, start=1))
    ):
        running = min(
            running,
            p[original_idx] * m / rank,
        )
        q[original_idx] = min(1.0, running)

    return q


def add_fdr(
    df: pd.DataFrame,
    p_column: str,
    q_column: str,
) -> pd.DataFrame:
    out = df.copy()
    out[q_column] = benjamini_hochberg(
        pd.to_numeric(out[p_column], errors="coerce").to_numpy(float)
    )
    return out


def selection_with_tie_hierarchy(
    a_pooled_r2: float,
    b_pooled_r2: float,
    a_mean_fold_r2: float,
    b_mean_fold_r2: float,
    a_pooled_rmse: float,
    b_pooled_rmse: float,
) -> tuple[str, str]:
    """
    Pre-specified primary Position-Aware architecture-selection rule.
    """
    if b_pooled_r2 > a_pooled_r2:
        return MODEL_B, "Higher pooled Position-Aware OOF R2"
    if b_pooled_r2 < a_pooled_r2:
        return MODEL_A, "Higher pooled Position-Aware OOF R2"

    if b_mean_fold_r2 > a_mean_fold_r2:
        return MODEL_B, (
            "Pooled Position-Aware OOF R2 tie; "
            "higher mean outer-fold R2"
        )
    if b_mean_fold_r2 < a_mean_fold_r2:
        return MODEL_A, (
            "Pooled Position-Aware OOF R2 tie; "
            "higher mean outer-fold R2"
        )

    if b_pooled_rmse < a_pooled_rmse:
        return MODEL_B, (
            "R2 tie; lower pooled Position-Aware RMSE"
        )
    if b_pooled_rmse > a_pooled_rmse:
        return MODEL_A, (
            "R2 tie; lower pooled Position-Aware RMSE"
        )

    return MODEL_A, (
        "Effectively tied; simpler Model A retained"
    )


# ======================================================================
# 3. LOAD FROZEN INPUTS
# ======================================================================

print("=" * 86)
print("STEP 6 — STATISTICAL COMPARISON AND PRIMARY MODEL SELECTION")
print("=" * 86)
print("Loading frozen Step 4 and Step 5 outputs...\n")

step4_fold_file = require_file(STEP4_DIR / STEP4_FOLD_FILENAME)
step4_pooled_file = require_file(STEP4_DIR / STEP4_POOLED_FILENAME)
step5_fold_file = require_file(STEP5_DIR / STEP5_FOLD_FILENAME)
step5_pooled_file = require_file(STEP5_DIR / STEP5_POOLED_FILENAME)

step4_fold_raw = pd.read_csv(step4_fold_file)
step4_pooled_raw = pd.read_csv(step4_pooled_file)
step5_fold_raw = pd.read_csv(step5_fold_file)
step5_pooled_raw = pd.read_csv(step5_pooled_file)


# ======================================================================
# 4. VALIDATE + CANONICALIZE INPUT SCHEMAS
# ======================================================================

print("[1/10] Validating frozen input schemas and completeness...")

step4_fold = canonicalize_fold_df(
    step4_fold_raw,
    "Step 4 fold results",
)
step5_fold = canonicalize_fold_df(
    step5_fold_raw,
    "Step 5 fold results",
)
step4_pooled = canonicalize_pooled_df(
    step4_pooled_raw,
    "Step 4 pooled results",
)
step5_pooled = canonicalize_pooled_df(
    step5_pooled_raw,
    "Step 5 pooled results",
)

validate_common_targets_architectures(
    step4_fold,
    "Step 4 fold results",
    expect_outer_fold=True,
)
validate_common_targets_architectures(
    step5_fold,
    "Step 5 fold results",
    expect_outer_fold=True,
)
validate_common_targets_architectures(
    step4_pooled,
    "Step 4 pooled results",
    expect_outer_fold=False,
)
validate_common_targets_architectures(
    step5_pooled,
    "Step 5 pooled results",
    expect_outer_fold=False,
)

validate_fold_completeness(step4_fold, "Step 4")
validate_fold_completeness(step5_fold, "Step 5")
validate_pooled_completeness(step4_pooled, "Step 4 pooled")
validate_pooled_completeness(step5_pooled, "Step 5 pooled")

validate_step5_family_information(step5_fold)

validate_paired_ab(step4_fold, "Step 4 Position-Aware")
validate_paired_ab(step5_fold, "Step 5 Random-Split")

# Pooled N_OOF validation where available.
if "N_OOF" in step5_pooled.columns:
    bad_n = (
        pd.to_numeric(step5_pooled["N_OOF"], errors="coerce").isna()
        | (pd.to_numeric(step5_pooled["N_OOF"], errors="coerce") <= 0)
    )
    if bad_n.any():
        raise ValueError(
            "Step 5 pooled results contain invalid N_OOF values."
        )

print("[PASS] Step 4 fold schema canonicalized.")
print("[PASS] Step 5 fold schema canonicalized "
      "(Outer_Test_* → canonical metric names).")
print("[PASS] Step 4 pooled schema validated.")
print("[PASS] Step 5 pooled schema validated "
      "(Pooled_OOF_* → canonical metric names).")
print("[PASS] Step 4 = 90 frozen Position-Aware outer-test evaluations.")
print("[PASS] Step 5 = 90 frozen Random-Split outer-test evaluations.")
print("[PASS] Both benchmarks contain all 9 targets × 2 architectures × 5 folds.")
print("[PASS] Model A/B outer-fold pairing validated.")
print("[PASS] Step 5 frozen RF/ET family information validated.\n")


# ======================================================================
# 5. PRIMARY MODEL A VS B — POSITION-AWARE
# ======================================================================

print("[2/10] Testing Model A vs Model B under the PRIMARY Position-Aware benchmark...")

ab_records = []

for target_index, target in enumerate(TARGETS):
    a = (
        step4_fold[
            (step4_fold["Target"] == target)
            & (step4_fold["Architecture"] == MODEL_A)
        ]
        .sort_values("Outer_Fold")
    )
    b = (
        step4_fold[
            (step4_fold["Target"] == target)
            & (step4_fold["Architecture"] == MODEL_B)
        ]
        .sort_values("Outer_Fold")
    )

    a_r2 = a["R2"].to_numpy(float)
    b_r2 = b["R2"].to_numpy(float)
    a_rmse = a["RMSE"].to_numpy(float)
    b_rmse = b["RMSE"].to_numpy(float)
    a_mae = a["MAE"].to_numpy(float)
    b_mae = b["MAE"].to_numpy(float)
    a_sp = a["Spearman"].to_numpy(float)
    b_sp = b["Spearman"].to_numpy(float)

    tests = paired_tests(a_r2, b_r2)
    delta_ci = bootstrap_paired_delta_ci(
        a_r2,
        b_r2,
        RANDOM_STATE + 1000 + target_index,
    )
    a_ci = bootstrap_mean_ci(
        a_r2,
        RANDOM_STATE + 2000 + target_index,
    )
    b_ci = bootstrap_mean_ci(
        b_r2,
        RANDOM_STATE + 3000 + target_index,
    )

    # Fold-level secondary metric deltas.
    delta_rmse = b_rmse - a_rmse
    delta_mae = b_mae - a_mae
    delta_sp = b_sp - a_sp

    ab_records.append({
        "Target": target,
        "N_Paired_Folds": N_OUTER_FOLDS,

        "Model_A_Mean_Fold_R2": safe_mean(a_r2),
        "Model_B_Mean_Fold_R2": safe_mean(b_r2),
        "Delta_Mean_Fold_R2_B_minus_A": safe_mean(b_r2 - a_r2),

        "Model_A_Mean_Fold_R2_CI95_Lower": a_ci[0],
        "Model_A_Mean_Fold_R2_CI95_Upper": a_ci[1],
        "Model_B_Mean_Fold_R2_CI95_Lower": b_ci[0],
        "Model_B_Mean_Fold_R2_CI95_Upper": b_ci[1],

        "Delta_Mean_Fold_R2_CI95_Lower": delta_ci[0],
        "Delta_Mean_Fold_R2_CI95_Upper": delta_ci[1],

        "Model_A_Mean_Fold_RMSE": safe_mean(a_rmse),
        "Model_B_Mean_Fold_RMSE": safe_mean(b_rmse),
        "Delta_Mean_Fold_RMSE_B_minus_A": safe_mean(delta_rmse),

        "Model_A_Mean_Fold_MAE": safe_mean(a_mae),
        "Model_B_Mean_Fold_MAE": safe_mean(b_mae),
        "Delta_Mean_Fold_MAE_B_minus_A": safe_mean(delta_mae),

        "Model_A_Mean_Fold_Spearman": safe_mean(a_sp),
        "Model_B_Mean_Fold_Spearman": safe_mean(b_sp),
        "Delta_Mean_Fold_Spearman_B_minus_A": safe_mean(delta_sp),

        "Paired_Cohens_d_R2": paired_cohens_d(a_r2, b_r2),
        **tests,
    })

model_ab_df = pd.DataFrame(ab_records)

# FDR across the 9 primary paired t-tests.
model_ab_df = add_fdr(
    model_ab_df,
    "Paired_t_p",
    "Paired_t_FDR_q",
)
model_ab_df["Paired_t_FDR_Significant"] = (
    model_ab_df["Paired_t_FDR_q"] < ALPHA
)

# Also provide FDR-adjusted Wilcoxon q-values as supplementary information.
model_ab_df = add_fdr(
    model_ab_df,
    "Wilcoxon_p",
    "Wilcoxon_FDR_q",
)
model_ab_df["Wilcoxon_FDR_Significant"] = (
    model_ab_df["Wilcoxon_FDR_q"] < ALPHA
)

print("[PASS] Paired t-tests completed for all 9 phenotypes.")
print("[PASS] Wilcoxon signed-rank tests completed for all 9 phenotypes.")
print("[PASS] 10,000-iteration bootstrap CIs completed.")
print("[PASS] Benjamini-Hochberg FDR correction completed.\n")


# ======================================================================
# 6. POSITION-AWARE VS RANDOM-SPLIT — SUPPORTING BENCHMARK
# ======================================================================

print("[3/10] Comparing Position-Aware and Random-Split performance...")

split_records = []

for target_index, target in enumerate(TARGETS):
    for architecture_index, architecture in enumerate(ARCHITECTURES):
        pa = (
            step4_fold[
                (step4_fold["Target"] == target)
                & (step4_fold["Architecture"] == architecture)
            ]
            .sort_values("Outer_Fold")
        )
        rs = (
            step5_fold[
                (step5_fold["Target"] == target)
                & (step5_fold["Architecture"] == architecture)
            ]
            .sort_values("Outer_Fold")
        )

        pa_r2 = pa["R2"].to_numpy(float)
        rs_r2 = rs["R2"].to_numpy(float)

        welch = welch_test(pa_r2, rs_r2)
        pa_ci = bootstrap_mean_ci(
            pa_r2,
            RANDOM_STATE + 4000
            + target_index * 10
            + architecture_index,
        )
        rs_ci = bootstrap_mean_ci(
            rs_r2,
            RANDOM_STATE + 5000
            + target_index * 10
            + architecture_index,
        )

        split_records.append({
            "Target": target,
            "Architecture": architecture,
            "N_PositionAware_Folds": len(pa_r2),
            "N_RandomSplit_Folds": len(rs_r2),

            "PositionAware_Mean_Fold_R2": safe_mean(pa_r2),
            "RandomSplit_Mean_Fold_R2": safe_mean(rs_r2),
            "Delta_Random_minus_PositionAware_R2":
                safe_mean(rs_r2) - safe_mean(pa_r2),

            "PositionAware_R2_CI95_Lower": pa_ci[0],
            "PositionAware_R2_CI95_Upper": pa_ci[1],
            "RandomSplit_R2_CI95_Lower": rs_ci[0],
            "RandomSplit_R2_CI95_Upper": rs_ci[1],

            "Welch_t": welch["Welch_t"],
            "Welch_p": welch["Welch_p"],

            "Interpretation":
                "Supporting/descriptive only; split strategies are not "
                "naturally paired and Random-Split may be optimistic for "
                "unseen-position generalization.",
        })

split_df = pd.DataFrame(split_records)

print("[PASS] Position-Aware vs Random-Split comparison completed.")
print("[NOTE] Welch statistics are SUPPORTING only; Random-Split cannot "
      "override the primary Position-Aware architecture decision.\n")


# ======================================================================
# 7. POOLED OOF INTEGRATION
# ======================================================================

print("[4/10] Integrating pooled OOF metrics...")

pooled_records = []

for target in TARGETS:
    for architecture in ARCHITECTURES:
        pa = step4_pooled[
            (step4_pooled["Target"] == target)
            & (step4_pooled["Architecture"] == architecture)
        ]
        rs = step5_pooled[
            (step5_pooled["Target"] == target)
            & (step5_pooled["Architecture"] == architecture)
        ]

        if len(pa) != 1 or len(rs) != 1:
            raise RuntimeError(
                f"Pooled result mismatch for {target} / {architecture}."
            )

        pa = pa.iloc[0]
        rs = rs.iloc[0]

        pooled_records.append({
            "Target": target,
            "Architecture": architecture,

            "PositionAware_R2": float(pa["R2"]),
            "PositionAware_RMSE": float(pa["RMSE"]),
            "PositionAware_MAE": float(pa["MAE"]),
            "PositionAware_Spearman": float(pa["Spearman"]),

            "RandomSplit_R2": float(rs["R2"]),
            "RandomSplit_RMSE": float(rs["RMSE"]),
            "RandomSplit_MAE": float(rs["MAE"]),
            "RandomSplit_Spearman": float(rs["Spearman"]),

            "Delta_R2_Random_minus_PositionAware":
                float(rs["R2"] - pa["R2"]),
            "Delta_RMSE_Random_minus_PositionAware":
                float(rs["RMSE"] - pa["RMSE"]),
            "Delta_MAE_Random_minus_PositionAware":
                float(rs["MAE"] - pa["MAE"]),
            "Delta_Spearman_Random_minus_PositionAware":
                float(rs["Spearman"] - pa["Spearman"]),
        })

pooled_df = pd.DataFrame(pooled_records)

if len(pooled_df) != len(TARGETS) * len(ARCHITECTURES):
    raise RuntimeError("Integrated pooled table is incomplete.")

print("[PASS] 18 pooled target × architecture records integrated.\n")


# ======================================================================
# 8. FINAL ARCHITECTURE SELECTION — POSITION-AWARE ONLY
# ======================================================================

print("[5/10] Applying the pre-specified PRIMARY Position-Aware "
      "architecture-selection rule...")

selection_records = []

for target in TARGETS:
    pa = pooled_df[pooled_df["Target"] == target]

    a = pa[pa["Architecture"] == MODEL_A].iloc[0]
    b = pa[pa["Architecture"] == MODEL_B].iloc[0]

    fold = model_ab_df[model_ab_df["Target"] == target].iloc[0]

    selected, basis = selection_with_tie_hierarchy(
        a_pooled_r2=float(a["PositionAware_R2"]),
        b_pooled_r2=float(b["PositionAware_R2"]),
        a_mean_fold_r2=float(fold["Model_A_Mean_Fold_R2"]),
        b_mean_fold_r2=float(fold["Model_B_Mean_Fold_R2"]),
        a_pooled_rmse=float(a["PositionAware_RMSE"]),
        b_pooled_rmse=float(b["PositionAware_RMSE"]),
    )

    selection_records.append({
        "Target": target,

        "Model_A_Predictors": EXPECTED_FEATURE_COUNTS[MODEL_A],
        "Model_B_Predictors": EXPECTED_FEATURE_COUNTS[MODEL_B],

        "Model_A_PositionAware_Pooled_R2":
            float(a["PositionAware_R2"]),
        "Model_B_PositionAware_Pooled_R2":
            float(b["PositionAware_R2"]),
        "Delta_PositionAware_R2_B_minus_A":
            float(b["PositionAware_R2"] - a["PositionAware_R2"]),

        "Model_A_PositionAware_Pooled_RMSE":
            float(a["PositionAware_RMSE"]),
        "Model_B_PositionAware_Pooled_RMSE":
            float(b["PositionAware_RMSE"]),
        "Delta_PositionAware_RMSE_B_minus_A":
            float(b["PositionAware_RMSE"] - a["PositionAware_RMSE"]),

        "Model_A_Mean_PositionAware_Fold_R2":
            float(fold["Model_A_Mean_Fold_R2"]),
        "Model_B_Mean_PositionAware_Fold_R2":
            float(fold["Model_B_Mean_Fold_R2"]),
        "Delta_Mean_PositionAware_Fold_R2_B_minus_A":
            float(fold["Delta_Mean_Fold_R2_B_minus_A"]),

        "Paired_t_p": float(fold["Paired_t_p"]),
        "Paired_t_FDR_q": float(fold["Paired_t_FDR_q"]),
        "Wilcoxon_p": float(fold["Wilcoxon_p"]),
        "Wilcoxon_FDR_q": float(fold["Wilcoxon_FDR_q"]),
        "Paired_Cohens_d_R2": float(fold["Paired_Cohens_d_R2"]),

        "Selected_Architecture": selected,
        "Selection_Basis": basis,

        "PositionAware_Is_Primary_Criterion": True,
        "RandomSplit_Is_Not_Selection_Criterion": True,
        "RF_ET_Is_Not_ReSelected_in_Step6": True,
    })

selection_df = pd.DataFrame(selection_records)

if len(selection_df) != len(TARGETS):
    raise RuntimeError(
        "Final architecture selection must contain exactly 9 targets."
    )

if set(selection_df["Selected_Architecture"]) - set(ARCHITECTURES):
    raise RuntimeError("Invalid selected architecture detected.")

print("[PASS] Final architecture decision generated for all 9 targets.\n")


# ======================================================================
# 9. MANUSCRIPT-READY INTEGRATED TABLE
# ======================================================================

print("[6/10] Building integrated manuscript-results table...")

integrated = pooled_df.merge(
    selection_df[
        [
            "Target",
            "Selected_Architecture",
            "Selection_Basis",
            "Delta_PositionAware_R2_B_minus_A",
            "Delta_PositionAware_RMSE_B_minus_A",
        ]
    ],
    on="Target",
    how="left",
)

integrated = integrated.merge(
    model_ab_df[
        [
            "Target",
            "Model_A_Mean_Fold_R2",
            "Model_B_Mean_Fold_R2",
            "Delta_Mean_Fold_R2_B_minus_A",
            "Delta_Mean_Fold_R2_CI95_Lower",
            "Delta_Mean_Fold_R2_CI95_Upper",
            "Paired_Cohens_d_R2",
            "Paired_t",
            "Paired_t_p",
            "Paired_t_FDR_q",
            "Wilcoxon",
            "Wilcoxon_p",
            "Wilcoxon_FDR_q",
        ]
    ],
    on="Target",
    how="left",
)

integrated["Primary_Evaluation"] = "Position-Aware"
integrated["Secondary_Evaluation"] = "Random-Split"
integrated["Model_A_Predictors"] = EXPECTED_FEATURE_COUNTS[MODEL_A]
integrated["Model_B_Predictors"] = EXPECTED_FEATURE_COUNTS[MODEL_B]

if integrated["Selected_Architecture"].isna().any():
    raise RuntimeError("Integrated manuscript table has missing selections.")

print("[PASS] Integrated manuscript-results table created.\n")


# ======================================================================
# 10. EXPLICIT LEAKAGE / DECISION AUDIT
# ======================================================================

print("[7/10] Running explicit Step 6 leakage and decision audit...")

audit_rows = []


def add_audit(check: str, passed: bool, detail: str) -> None:
    audit_rows.append({
        "Check": check,
        "Status": "PASS" if passed else "FAIL",
        "Detail": detail,
    })


add_audit(
    "Step 4 completeness",
    len(step4_fold) == 90,
    "Exactly 90 frozen Position-Aware outer-test evaluations.",
)

add_audit(
    "Step 5 completeness",
    len(step5_fold) == 90,
    "Exactly 90 frozen Random-Split outer-test evaluations.",
)

add_audit(
    "Step 4 pooled completeness",
    len(step4_pooled) == 18,
    "Exactly 18 pooled target × architecture records.",
)

add_audit(
    "Step 5 pooled completeness",
    len(step5_pooled) == 18,
    "Exactly 18 pooled target × architecture records.",
)

add_audit(
    "All 9 targets present",
    set(step4_fold["Target"]) == set(TARGETS)
    and set(step5_fold["Target"]) == set(TARGETS),
    "All prespecified phenotypes are present in both benchmarks.",
)

add_audit(
    "Model A predictor count",
    EXPECTED_FEATURE_COUNTS[MODEL_A] == 65,
    "Model A = 65 predictors; Sequence_Position excluded.",
)

add_audit(
    "Model B predictor count",
    EXPECTED_FEATURE_COUNTS[MODEL_B] == 66,
    "Model B = 66 predictors; Sequence_Position included.",
)

add_audit(
    "A/B fold pairing",
    True,
    "Model A and Model B are paired on the same five outer folds "
    "within each benchmark.",
)

add_audit(
    "No model fitting",
    True,
    "Step 6 performs no model fitting.",
)

add_audit(
    "No HPO",
    True,
    "Step 6 performs no hyperparameter optimization.",
)

add_audit(
    "No new folds",
    True,
    "Step 6 creates no train/test or CV folds.",
)

add_audit(
    "No Part 14 use",
    True,
    "Step 6 reads only frozen Step 4 and Step 5 outputs.",
)

add_audit(
    "Position-Aware primary",
    True,
    "Final architecture selection is determined only by "
    "Position-Aware evidence.",
)

add_audit(
    "Random-Split secondary",
    True,
    "Random-Split is supporting evidence and cannot override "
    "Position-Aware selection.",
)

add_audit(
    "No RF/ET reselection",
    True,
    "Step 3 frozen RF/ET family decisions are not changed in Step 6.",
)

add_audit(
    "No outer-test reuse for fitting",
    True,
    "Step 6 does not fit, optimize, or tune against outer-test results.",
)

add_audit(
    "Primary decision hierarchy pre-specified",
    True,
    "Pooled Position-Aware R2 → mean fold R2 → pooled RMSE → "
    "simpler Model A.",
)

add_audit(
    "FDR scope",
    True,
    "BH-FDR is applied across the 9 phenotype-level primary A/B tests.",
)

add_audit(
    "Low-power inference acknowledged",
    True,
    "Only five outer folds are available; p-values are supporting evidence.",
)

audit_df = pd.DataFrame(audit_rows)

if not (audit_df["Status"] == "PASS").all():
    failed = audit_df[audit_df["Status"] != "PASS"]
    raise RuntimeError(
        "Step 6 audit failed:\n"
        + failed.to_string(index=False)
    )

print("[PASS] Step 6 leakage/decision audit passed.\n")


# ======================================================================
# 11. SAVE OUTPUTS
# ======================================================================

print("[8/10] Saving statistical and selection outputs...")

outputs = {
    "VIM2_Step6_ModelA_vs_ModelB_PositionAware.csv": model_ab_df,
    "VIM2_Step6_PositionAware_vs_RandomSplit.csv": split_df,
    "VIM2_Step6_Pooled_Integrated_Performance.csv": pooled_df,
    "VIM2_Step6_Final_Model_Selection.csv": selection_df,
    "VIM2_Step6_Integrated_Manuscript_Results.csv": integrated,
    "VIM2_Step6_Leakage_Audit.csv": audit_df,
}

for filename, frame in outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

print("[PASS] CSV outputs saved.\n")


# ======================================================================
# 12. FINAL DECISION JSON + REPRODUCIBILITY MANIFEST
# ======================================================================

print("[9/10] Writing final decision and reproducibility metadata...")

decision_payload = {
    "project": PROJECT_NAME,
    "step": STEP_NAME,
    "timestamp_utc": utc_now(),

    "primary_evaluation": "Position-Aware",
    "secondary_benchmark": "Random-Split",

    "model_a": {
        "name": MODEL_A,
        "predictors": 65,
        "sequence_position": False,
    },
    "model_b": {
        "name": MODEL_B,
        "predictors": 66,
        "sequence_position": True,
    },

    "selection_rule": [
        "Primary: higher pooled Position-Aware OOF R2",
        "Secondary tie-break: higher mean Position-Aware outer-fold R2",
        "Tertiary tie-break: lower pooled Position-Aware RMSE",
        "Final tie-break: simpler Model A",
        "Random-Split cannot override Position-Aware selection",
        "RF vs Extra Trees is not re-selected in Step 6",
    ],

    "statistical_analysis": {
        "primary_tests": "paired t-test across identical five Position-Aware outer folds",
        "nonparametric_support": "Wilcoxon signed-rank test",
        "effect_size": "paired Cohen's d",
        "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        "bootstrap_ci": "percentile 95% CI",
        "fdr_method": "Benjamini-Hochberg",
        "fdr_family": "9 phenotype-level primary Model A vs Model B tests",
        "split_strategy_test": "Welch t-test, supporting/descriptive only",
        "low_power_note": "n=5 outer folds per phenotype; inferential p-values are low-power",
    },

    "selected_architecture_by_target": {
        row["Target"]: row["Selected_Architecture"]
        for _, row in selection_df.iterrows()
    },

    "selection_basis_by_target": {
        row["Target"]: row["Selection_Basis"]
        for _, row in selection_df.iterrows()
    },
}

decision_file = OUTPUT_DIR / "VIM2_Step6_Final_Model_Selection.json"
with open(decision_file, "w", encoding="utf-8") as handle:
    json.dump(decision_payload, handle, indent=2)


manifest = {
    "project": PROJECT_NAME,
    "step": STEP_NAME,
    "status": "COMPLETE",
    "execution_timestamp_utc": utc_now(),

    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,

    "random_state": RANDOM_STATE,
    "n_outer_folds": N_OUTER_FOLDS,
    "alpha": ALPHA,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,

    "primary_evaluation": "Position-Aware",
    "secondary_benchmark": "Random-Split",

    "model_a_predictor_count": 65,
    "model_b_predictor_count": 66,

    "input_files": {
        str(step4_fold_file): sha256_file(step4_fold_file),
        str(step4_pooled_file): sha256_file(step4_pooled_file),
        str(step5_fold_file): sha256_file(step5_fold_file),
        str(step5_pooled_file): sha256_file(step5_pooled_file),
    },

    "input_row_counts": {
        "step4_fold": int(len(step4_fold)),
        "step4_pooled": int(len(step4_pooled)),
        "step5_fold": int(len(step5_fold)),
        "step5_pooled": int(len(step5_pooled)),
    },

    "outputs": [
        str(OUTPUT_DIR / filename)
        for filename in sorted(outputs.keys())
    ] + [
        str(decision_file),
    ],

    "interpretation_notes": [
        "Position-Aware is the primary generalization benchmark.",
        "Random-Split is secondary and may be optimistic for unseen-position generalization.",
        "Step 6 does not refit or tune any model.",
        "Step 6 does not re-select Random Forest vs Extra Trees.",
        "Statistical inference is low-power because each phenotype has five outer folds.",
        "Architecture selection is based on pre-specified Position-Aware performance hierarchy.",
    ],
}

manifest_file = OUTPUT_DIR / "VIM2_Step6_Reproducibility_Manifest.json"
with open(manifest_file, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print("[PASS] Final decision JSON written.")
print("[PASS] Reproducibility manifest written.\n")


# ======================================================================
# 13. FINAL OUTPUT VALIDATION
# ======================================================================

print("[10/10] Final output validation...")

required_outputs = [
    "VIM2_Step6_ModelA_vs_ModelB_PositionAware.csv",
    "VIM2_Step6_PositionAware_vs_RandomSplit.csv",
    "VIM2_Step6_Pooled_Integrated_Performance.csv",
    "VIM2_Step6_Final_Model_Selection.csv",
    "VIM2_Step6_Integrated_Manuscript_Results.csv",
    "VIM2_Step6_Leakage_Audit.csv",
    "VIM2_Step6_Final_Model_Selection.json",
    "VIM2_Step6_Reproducibility_Manifest.json",
]

for filename in required_outputs:
    require_file(OUTPUT_DIR / filename)

# Reload output tables to make sure serialization succeeded.
reloaded_selection = pd.read_csv(
    OUTPUT_DIR / "VIM2_Step6_Final_Model_Selection.csv"
)
reloaded_ab = pd.read_csv(
    OUTPUT_DIR / "VIM2_Step6_ModelA_vs_ModelB_PositionAware.csv"
)
reloaded_split = pd.read_csv(
    OUTPUT_DIR / "VIM2_Step6_PositionAware_vs_RandomSplit.csv"
)
reloaded_audit = pd.read_csv(
    OUTPUT_DIR / "VIM2_Step6_Leakage_Audit.csv"
)

if len(reloaded_selection) != 9:
    raise RuntimeError("Final selection CSV does not contain exactly 9 targets.")

if len(reloaded_ab) != 9:
    raise RuntimeError("A/B Position-Aware table does not contain 9 targets.")

if len(reloaded_split) != 18:
    raise RuntimeError(
        "Position-Aware vs Random-Split table does not contain 18 records."
    )

if len(reloaded_audit) == 0 or not (
    reloaded_audit["Status"] == "PASS"
).all():
    raise RuntimeError("Reloaded leakage audit is not PASS.")

print("[PASS] All required Step 6 outputs exist and reload successfully.")
print()
print("=" * 86)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 86)
print("Primary criterion : Position-Aware generalization")
print("Secondary         : Random-Split benchmark")
print("Model A           : 65 predictors")
print("Model B           : 66 predictors")
print("Targets           : 9")
print("Outer folds       : 5")
print("Bootstrap         : 10,000 iterations")
print("FDR               : Benjamini-Hochberg across 9 primary A/B tests")
print("Model fitting     : NONE")
print("HPO               : NONE")
print("New folds         : NONE")
print("Part 14 used      : NO")
print("Leakage audit     : PASS")
print()
print("FINAL ARCHITECTURE SELECTION")
print("-" * 86)

for row in selection_df.itertuples(index=False):
    print(
        f"{row.Target:<28} -> "
        f"{row.Selected_Architecture:<24} | "
        f"ΔR²(B-A)={row.Delta_PositionAware_R2_B_minus_A:+.6f} | "
        f"{row.Selection_Basis}"
    )

print()
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 86)


STEP 6 — STATISTICAL COMPARISON AND PRIMARY MODEL SELECTION
Loading frozen Step 4 and Step 5 outputs...

[1/10] Validating frozen input schemas and completeness...
[PASS] Step 4 fold schema canonicalized.
[PASS] Step 5 fold schema canonicalized (Outer_Test_* → canonical metric names).
[PASS] Step 4 pooled schema validated.
[PASS] Step 5 pooled schema validated (Pooled_OOF_* → canonical metric names).
[PASS] Step 4 = 90 frozen Position-Aware outer-test evaluations.
[PASS] Step 5 = 90 frozen Random-Split outer-test evaluations.
[PASS] Both benchmarks contain all 9 targets × 2 architectures × 5 folds.
[PASS] Model A/B outer-fold pairing validated.
[PASS] Step 5 frozen RF/ET family information validated.

[2/10] Testing Model A vs Model B under the PRIMARY Position-Aware benchmark...
[PASS] Paired t-tests completed for all 9 phenotypes.
[PASS] Wilcoxon signed-rank tests completed for all 9 phenotypes.
[PASS] 10,000-iteration bootstrap CIs completed.
[PASS] Benjamini-Hochberg FDR correction

In [ ]:
# @title

# ============================================================
# DOWNLOAD STEP6 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step6_Statistical_Comparison"

# Output ZIP archive
zip_base = "/content/VIM2_Step6_Statistical_Comparison"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step6_Statistical_Comparison"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP 6 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP 6 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step6_Statistical_Comparison
ZIP archive      : /content/VIM2_Step6_Statistical_Comparison.zip
Archive size     : 0.02 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
VIM-2 Project — Step 7: Final Model Interpretation
===================================================

Purpose
-------
Interpret ONLY the final, pre-specified architectures selected in Step 6,
using the frozen hyperparameters selected in Step 3 and the frozen
Position-Aware outer folds from Part13.

Scientific design
-----------------
Primary interpretation:
    1. Fold-specific impurity-based feature importance from the fitted
       final RF/Extra-Trees models.
    2. Fold-specific permutation importance evaluated on the untouched
       Position-Aware outer test folds.
    3. Across-fold mean, SD, median and stability summaries.
    4. Target-specific architecture and model-family audit.\n    5. Leakage-safe one-hot encoding of the categorical Secondary_Structure feature.

Important leakage safeguards
-----------------------------
- Part13 frozen Position-Aware folds are used unchanged.
- Step6 decides Model A vs Model B; Step7 NEVER re-selects architecture.
- Step3 decides RF vs Extra Trees and hyperparameters; Step7 NEVER re-HPOs.
- Outer test folds are used only for post-hoc interpretation/permutation
  scoring. They are never used to fit, tune, or select models.
- No target-associated *_SD columns are predictors.
- Feature membership comes from the Part13 feature manifest.
- No new folds are created.
- No preprocessing is learned from the outer test data.

Outputs
-------
/content/VIM2_Step7_Model_Interpretation/
    VIM2_Step7_Fold_Feature_Importance.csv
    VIM2_Step7_Feature_Importance_Summary.csv
    VIM2_Step7_Top_Features_By_Target.csv
    VIM2_Step7_Target_Architecture_Audit.csv
    VIM2_Step7_Leakage_Audit.csv
    VIM2_Step7_Reproducibility_Manifest.json
    VIM2_Step7_SHA256.csv

Notes
-----
Secondary_Structure is handled with leakage-safe one-hot encoding; all reported importances are returned to the original Part13 feature level.\n\nThis implementation deliberately does NOT generate SHAP by default.
SHAP can be added later as a secondary sensitivity/interpretation analysis,
but the primary Step7 result here is model-native importance plus
out-of-fold permutation importance. This avoids introducing a new
interpretation dependency or changing the fitted models.
"""

from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import platform
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

DATA_FILE = Path("/content/VIM2_Part12H_v2_Final_ML_Dataset.csv")
FOLD_FILE = Path("/content/VIM2_Part13_Fold_Assignments.csv")
FEATURE_MANIFEST_FILE = Path(
    "/content/VIM2_Part13_Feature_Manifest.csv"
)

STEP3_DIR = Path("/content/VIM2_Step3_PositionAware_Nested_HPO")
STEP3_FROZEN_FILE = STEP3_DIR / "VIM2_Step3_Best_Hyperparameters_Per_Fold.csv"
STEP4_DIR = Path("/content/VIM2_Step4_Final_Evaluation")
STEP6_DIR = Path("/content/VIM2_Step6_Statistical_Comparison")
STEP6_FINAL_SELECTION_FILE = STEP6_DIR / "VIM2_Step6_Final_Model_Selection.csv"

STEP2_DIR = Path("/content/VIM2_Step2_PositionAware_Candidate_Selection")

OUTPUT_DIR = Path("/content/VIM2_Step7_Model_Interpretation")

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

ARCHITECTURE_A = "Model_A_No_Position"
ARCHITECTURE_B = "Model_B_With_Position"
ARCHITECTURES = [ARCHITECTURE_A, ARCHITECTURE_B]

MODEL_FAMILIES = ["Random_Forest", "Extra_Trees"]

OUTER_FOLDS = 5

# Permutation settings are deliberately fixed and are NOT tuned.
PERMUTATION_REPEATS = 20
PERMUTATION_RANDOM_STATE_BASE = 17001
PERMUTATION_SCORING = "r2"

# Runtime / reproducibility
N_JOBS = -1
GLOBAL_RANDOM_STATE = 42

# Minimum expected predictor counts.
EXPECTED_PREDICTORS = {
    ARCHITECTURE_A: 65,
    ARCHITECTURE_B: 66,
}


# ---------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def require_file(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def require_dir(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def normalize_model_name(x: Any) -> str:
    s = str(x).strip().lower().replace("-", "_").replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    aliases = {
        "randomforest": "Random_Forest",
        "random_forest": "Random_Forest",
        "rf": "Random_Forest",
        "extratrees": "Extra_Trees",
        "extra_trees": "Extra_Trees",
        "et": "Extra_Trees",
    }
    return aliases.get(s, str(x).strip())


def normalize_architecture(x: Any) -> str:
    s = str(x).strip().lower().replace("-", "_").replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    if "model_a" in s or "no_position" in s or s in {"a", "modela"}:
        return ARCHITECTURE_A
    if "model_b" in s or "with_position" in s or s in {"b", "modelb"}:
        return ARCHITECTURE_B
    return str(x).strip()


def find_first_existing(directory: Path, candidates: Sequence[str]) -> Path | None:
    for name in candidates:
        p = directory / name
        if p.exists():
            return p
    return None


def find_csv_containing(directory: Path, required_tokens: Sequence[str]) -> Path | None:
    if not directory.exists():
        return None
    csvs = sorted(directory.glob("*.csv"))
    for p in csvs:
        name = p.name.lower()
        if all(tok.lower() in name for tok in required_tokens):
            return p
    return None


def find_json_containing(directory: Path, required_tokens: Sequence[str]) -> Path | None:
    if not directory.exists():
        return None
    js = sorted(directory.glob("*.json"))
    for p in js:
        name = p.name.lower()
        if all(tok.lower() in name for tok in required_tokens):
            return p
    return None


def parse_frozen_hyperparameters(value: Any) -> Dict[str, Any]:
    if isinstance(value, dict):
        obj = dict(value)
    else:
        s = str(value).strip()
        if not s:
            raise ValueError("Empty Best_Hyperparameters value.")
        try:
            obj = json.loads(s)
        except Exception:
            try:
                obj = ast.literal_eval(s)
            except Exception as exc:
                raise ValueError(
                    f"Cannot parse frozen hyperparameters: {s[:300]}"
                ) from exc

    if not isinstance(obj, dict):
        raise ValueError("Frozen hyperparameters are not a dictionary.")

    # Step 3 stores RandomizedSearchCV pipeline parameters using the
    # pipeline step prefix, e.g. ``model__n_estimators``. Step 4 explicitly
    # confirms these are the frozen parameter names exported by Step 3.
    # Normalize both prefixed and unprefixed forms to the estimator-level
    # names used by Step 7. This is a schema normalization only; no tuning
    # or parameter modification occurs.
    allowed = {
        "n_estimators",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "max_features",
        "model__n_estimators",
        "model__max_depth",
        "model__min_samples_split",
        "model__min_samples_leaf",
        "model__max_features",
    }
    unknown = set(obj) - allowed
    if unknown:
        raise ValueError(f"Unexpected hyperparameter keys: {sorted(unknown)}")

    normalized = {}
    for key, value_item in obj.items():
        clean = key[7:] if str(key).startswith("model__") else str(key)
        if clean in normalized:
            raise ValueError(f"Duplicate frozen hyperparameter after normalization: {clean}")
        normalized[clean] = value_item

    required = {
        "n_estimators",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "max_features",
    }
    missing = required - set(normalized)
    if missing:
        raise ValueError(f"Missing frozen hyperparameters: {sorted(missing)}")

    out = dict(normalized)

    out["n_estimators"] = int(out["n_estimators"])
    if out["n_estimators"] < 1:
        raise ValueError("n_estimators must be >= 1.")

    if out["max_depth"] is not None:
        out["max_depth"] = int(out["max_depth"])
        if out["max_depth"] < 1:
            raise ValueError("max_depth must be >= 1 or None.")

    out["min_samples_split"] = int(out["min_samples_split"])
    if out["min_samples_split"] < 2:
        raise ValueError("min_samples_split must be >= 2.")

    out["min_samples_leaf"] = int(out["min_samples_leaf"])
    if out["min_samples_leaf"] < 1:
        raise ValueError("min_samples_leaf must be >= 1.")

    mf = out["max_features"]
    if isinstance(mf, str):
        if mf not in {"sqrt", "log2"}:
            raise ValueError(f"Unsupported string max_features: {mf}")
    elif mf is not None:
        mf = float(mf)
        if not (0 < mf <= 1):
            raise ValueError("Numeric max_features must be in (0, 1].")
        out["max_features"] = mf

    return out


def make_estimator(model_family: str, params: Mapping[str, Any], random_state: int):
    family = normalize_model_name(model_family)
    common = dict(
        n_estimators=int(params["n_estimators"]),
        max_depth=params["max_depth"],
        min_samples_split=int(params["min_samples_split"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_features=params["max_features"],
        random_state=int(random_state),
        n_jobs=N_JOBS,
    )
    if family == "Random_Forest":
        return RandomForestRegressor(**common)
    if family == "Extra_Trees":
        return ExtraTreesRegressor(**common)
    raise ValueError(f"Unsupported model family: {model_family}")


# ---------------------------------------------------------------------
# Feature manifest / feature selection
# ---------------------------------------------------------------------

def load_feature_manifest() -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    require_file(FEATURE_MANIFEST_FILE, "Part13 feature manifest")
    mf = pd.read_csv(FEATURE_MANIFEST_FILE)

    required = {
        "Feature",
        "Used_in_Model_A_No_Position",
        "Used_in_Model_B_With_Position",
    }
    missing = required - set(mf.columns)
    if missing:
        raise ValueError(f"Feature manifest missing columns: {sorted(missing)}")

    if mf["Feature"].duplicated().any():
        dup = mf.loc[mf["Feature"].duplicated(), "Feature"].tolist()
        raise ValueError(f"Duplicate features in manifest: {dup[:10]}")

    def truth(v: Any) -> bool:
        return str(v).strip().lower() in {"true", "1", "yes", "y"}

    features = {}
    for arch, col in [
        (ARCHITECTURE_A, "Used_in_Model_A_No_Position"),
        (ARCHITECTURE_B, "Used_in_Model_B_With_Position"),
    ]:
        f = mf.loc[mf[col].map(truth), "Feature"].astype(str).tolist()
        features[arch] = f

    if len(features[ARCHITECTURE_A]) != 65:
        raise ValueError(
            f"Model A manifest count is {len(features[ARCHITECTURE_A])}, expected 65."
        )
    if len(features[ARCHITECTURE_B]) != 66:
        raise ValueError(
            f"Model B manifest count is {len(features[ARCHITECTURE_B])}, expected 66."
        )

    if set(features[ARCHITECTURE_B]) != (
        set(features[ARCHITECTURE_A]) | {"Sequence_Position"}
    ):
        raise ValueError(
            "Model B is not exactly Model A plus Sequence_Position."
        )

    return mf, features


def validate_predictors(
    df: pd.DataFrame,
    features: Dict[str, List[str]],
):
    """
    Validate predictor membership and data types.

    Most predictors are numeric. Secondary_Structure is an intentional
    categorical predictor and is handled by the leakage-safe preprocessing
    pipeline below. It is therefore NOT treated as a numeric raw column.

    This validation does not change the conceptual predictor counts:
        Model A = 65
        Model B = 66
    """
    allowed_categorical = {"Secondary_Structure"}

    for arch, cols in features.items():
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise ValueError(
                f"{arch} has {len(missing)} missing dataset columns: {missing[:15]}"
            )

        # Ensure target-associated SD columns are not in predictors.
        bad_sd = [c for c in cols if c.endswith("_SD")]
        if bad_sd:
            raise ValueError(
                f"Target-associated SD columns were included as predictors: {bad_sd}"
            )

        for c in cols:
            if c in allowed_categorical:
                # Step 3 explicitly allows missing Secondary_Structure and
                # handles it with train-only most-frequent imputation inside
                # the pipeline. Do not reject these rows here.
                continue

            if not pd.api.types.is_numeric_dtype(df[c]):
                raise TypeError(
                    f"Predictor {c} is not numeric and is not an approved "
                    f"categorical feature."
                )


def make_preprocessor(predictor_cols: Sequence[str]) -> ColumnTransformer:
    """
    Reproduce the Step 3 preprocessing exactly.

    Step 3 uses:
      * numeric predictors -> median imputation + missing indicators +
        StandardScaler
      * Secondary_Structure -> most-frequent imputation + OneHotEncoder
        (handle_unknown='ignore')

    The preprocessing is fitted only on the training partition because it is
    embedded inside the final sklearn Pipeline.
    """
    categorical_cols = [
        c for c in predictor_cols if c == "Secondary_Structure"
    ]
    numeric_cols = [
        c for c in predictor_cols if c != "Secondary_Structure"
    ]

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    transformers = [
        (
            "numeric",
            numeric_pipeline,
            numeric_cols,
        )
    ]

    if categorical_cols:
        try:
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            )
        except TypeError:
            # Compatibility with older scikit-learn versions.
            encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse=False,
            )

        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent",
                    ),
                ),
                (
                    "onehot",
                    encoder,
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_cols,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        # Match the actual Step 3 sklearn naming convention explicitly.
        # This affects feature names only, not model fitting or predictions.
        verbose_feature_names_out=True,
    )


def make_final_pipeline(
    model_family: str,
    params: Mapping[str, Any],
    random_state: int,
    predictor_cols: Sequence[str],
) -> Pipeline:
    """
    Final frozen model = leakage-safe preprocessing + frozen RF/ET.

    Hyperparameters remain exactly those frozen in Step3. The preprocessing
    is fit on X_train only via Pipeline.fit().
    """
    preprocessor = make_preprocessor(predictor_cols)
    estimator = make_estimator(model_family, params, random_state)

    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )


def aggregate_transformed_importance(
    predictor_cols: Sequence[str],
    transformed_feature_names: Sequence[str],
    importance: np.ndarray,
) -> Dict[str, float]:
    """
    Map sklearn model-space importance back to the original Part13 predictors.

    IMPORTANT:
    Step 3/Step 7 use a ColumnTransformer with sklearn's default
    ``verbose_feature_names_out=True``. Therefore transformed names can be
    emitted as, for example:

        numeric__WT_AA_A
        numeric__missingindicator_WT_AA_A
        categorical__Secondary_Structure_alpha

    The previous implementation assumed unprefixed names and consequently
    failed on valid transformed names such as ``numeric__WT_AA_A``.

    This mapper is deliberately strict but robust to the supported sklearn
    naming variants. It never invents a conceptual predictor: every transformed
    feature must resolve to exactly one member of ``predictor_cols``.
    """
    predictor_set = set(map(str, predictor_cols))
    out = {feature: 0.0 for feature in predictor_cols}

    transformed_feature_names = list(map(str, transformed_feature_names))
    importance = np.asarray(importance, dtype=float)

    if len(transformed_feature_names) != len(importance):
        raise ValueError(
            "Transformed feature-name / importance length mismatch: "
            f"{len(transformed_feature_names)} names vs {len(importance)} values."
        )

    def resolve_conceptual_feature(name: str) -> str | None:
        # 1) Exact conceptual predictor name.
        if name in predictor_set:
            return name

        # 2) Explicit ColumnTransformer prefixes used by Step 3.
        #    Split only once so category names containing underscores remain
        #    untouched.
        if "__" in name:
            transformer, remainder = name.split("__", 1)

            if transformer == "numeric":
                name = remainder
            elif transformer == "categorical":
                if remainder == "Secondary_Structure":
                    return "Secondary_Structure"
                if remainder.startswith("Secondary_Structure_"):
                    return "Secondary_Structure"
                return None
            else:
                # Do not silently strip an unknown transformer prefix.
                return None

        # 3) Numeric SimpleImputer(add_indicator=True) naming variants.
        #    sklearn commonly emits ``missingindicator_<feature>``.
        if name.startswith("missingindicator_"):
            base = name[len("missingindicator_"):]
            if base in predictor_set:
                return base
            return None

        # Older/custom naming variants can emit ``<feature>_missing``.
        if name.endswith("_missing"):
            base = name[:-len("_missing")]
            if base in predictor_set:
                return base
            return None

        # 4) Direct numeric feature after removing ``numeric__``.
        if name in predictor_set:
            return name

        return None

    unresolved = []

    for name, value in zip(transformed_feature_names, importance):
        feature = resolve_conceptual_feature(name)

        if feature is None:
            unresolved.append(name)
            continue

        out[feature] += float(value)

    if unresolved:
        known = ", ".join(map(str, predictor_cols))
        raise ValueError(
            "Could not map transformed feature(s) back to a Part13 "
            "conceptual predictor.\n"
            f"Unresolved transformed names: {unresolved[:20]}"
            + (" ..." if len(unresolved) > 20 else "")
            + "\nKnown conceptual predictors: "
            + known
        )

    # A successful mapping must conserve the total native importance.
    if not np.isclose(
        sum(out.values()),
        float(np.sum(importance)),
        rtol=1e-10,
        atol=1e-12,
    ):
        raise ValueError(
            "Native importance was not conserved during conceptual mapping: "
            f"mapped={sum(out.values()):.16g}, "
            f"transformed={float(np.sum(importance)):.16g}."
        )

    return out


def get_transformed_feature_names(fitted_pipeline: Pipeline) -> List[str]:
    preprocessor = fitted_pipeline.named_steps["preprocess"]
    names = preprocessor.get_feature_names_out()
    return [str(x) for x in names]


# ---------------------------------------------------------------------
# Load Step 6 final architecture decisions
# ---------------------------------------------------------------------

def load_step6_architecture_selection() -> pd.DataFrame:
    require_dir(STEP6_DIR, "Step6 output directory")

    candidates = [
        "VIM2_Step6_Statistical_Comparison.csv",
        "VIM2_Step6_Final_Architecture_Selection.csv",
        "VIM2_Step6_Statistical_Comparison_Q1.csv",
    ]
    p = find_first_existing(STEP6_DIR, candidates)

    if p is None:
        # Search all CSVs for the decisive column.
        for q in sorted(STEP6_DIR.glob("*.csv")):
            try:
                tmp = pd.read_csv(q, nrows=2)
            except Exception:
                continue
            if "Selected_Architecture" in tmp.columns:
                p = q
                break

    if p is None:
        raise FileNotFoundError(
            "Could not locate Step6 CSV containing Selected_Architecture."
        )

    df = pd.read_csv(p)

    required = {
        "Target",
        "Selected_Architecture",
        "PositionAware_Is_Primary_Criterion",
        "RandomSplit_Is_Not_Selection_Criterion",
        "RF_ET_Is_Not_ReSelected_in_Step6",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Step6 selection file missing columns: {sorted(missing)}")

    df["Target"] = df["Target"].astype(str)
    df["Selected_Architecture"] = df["Selected_Architecture"].map(
        normalize_architecture
    )

    if set(df["Target"]) != set(TARGETS) or len(df) != len(TARGETS):
        raise ValueError("Step6 selection does not contain exactly the 9 expected targets.")

    if df["Target"].duplicated().any():
        raise ValueError("Duplicate target rows in Step6 architecture selection.")

    if not df["Selected_Architecture"].isin(ARCHITECTURES).all():
        raise ValueError("Unexpected Selected_Architecture values in Step6.")

    for c in [
        "PositionAware_Is_Primary_Criterion",
        "RandomSplit_Is_Not_Selection_Criterion",
        "RF_ET_Is_Not_ReSelected_in_Step6",
    ]:
        vals = df[c].astype(str).str.strip().str.lower()
        if not vals.isin({"true", "1", "yes"}).all():
            raise ValueError(
                f"Step6 audit column {c} is not uniformly TRUE/yes."
            )

    return df[
        [
            "Target",
            "Selected_Architecture",
            "PositionAware_Is_Primary_Criterion",
            "RandomSplit_Is_Not_Selection_Criterion",
            "RF_ET_Is_Not_ReSelected_in_Step6",
        ]
    ].copy()


# ---------------------------------------------------------------------
# Load Step3 frozen HPO results
# ---------------------------------------------------------------------

def load_step3_frozen_hpo() -> pd.DataFrame:
    require_dir(STEP3_DIR, "Step3 output directory")

    preferred = [
        "VIM2_Step3_Best_Hyperparameters_Per_Fold.csv",
        "VIM2_Step3_PositionAware_Nested_HPO.csv",
        "VIM2_Step3_PositionAware_Nested_HPO_Results.csv",
        "VIM2_Step3_Nested_HPO_Fold_Performance.csv",
        "VIM2_Step3_PositionAware_Nested_HPO_Fold_Performance.csv",
    ]

    p = find_first_existing(STEP3_DIR, preferred)
    if p is None:
        for q in sorted(STEP3_DIR.glob("*.csv")):
            try:
                tmp = pd.read_csv(q, nrows=2)
            except Exception:
                continue
            cols = set(tmp.columns)
            if (
                {"Target", "Architecture", "Outer_Fold"}
                <= cols
                and "Best_Hyperparameters" in cols
                and ("Model" in cols or "Model_Family" in cols)
            ):
                p = q
                break

    if p is None:
        raise FileNotFoundError(
            "Could not locate Step3 CSV containing frozen Best_Hyperparameters."
        )

    df = pd.read_csv(p)

    # Step 3's actual frozen-output schema uses `Model`, not `Model_Family`.
    # Internally Step 7 standardizes this to `Model_Family` so downstream
    # interpretation logic remains explicit and unchanged.
    if "Model_Family" not in df.columns and "Model" in df.columns:
        df["Model_Family"] = df["Model"]

    required = {
        "Target",
        "Architecture",
        "Outer_Fold",
        "Model_Family",
        "Best_Hyperparameters",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Step3 HPO file missing columns: {sorted(missing)}")

    df["Target"] = df["Target"].astype(str)
    df["Architecture"] = df["Architecture"].map(normalize_architecture)
    df["Outer_Fold"] = pd.to_numeric(df["Outer_Fold"], errors="coerce")
    df["Model_Family"] = df["Model_Family"].map(normalize_model_name)

    if df["Outer_Fold"].isna().any():
        raise ValueError("Step3 contains invalid Outer_Fold values.")
    df["Outer_Fold"] = df["Outer_Fold"].astype(int)

    if not df["Target"].isin(TARGETS).all():
        raise ValueError("Step3 contains unexpected targets.")
    if not df["Architecture"].isin(ARCHITECTURES).all():
        raise ValueError("Step3 contains unexpected architectures.")
    if not df["Model_Family"].isin(MODEL_FAMILIES).all():
        raise ValueError("Step3 contains unexpected model families.")

    expected = len(TARGETS) * len(ARCHITECTURES) * OUTER_FOLDS
    if len(df) != expected:
        raise ValueError(
            f"Step3 frozen HPO rows = {len(df)}, expected {expected}."
        )

    if df.duplicated(["Target", "Architecture", "Outer_Fold"]).any():
        raise ValueError("Duplicate Step3 target/architecture/fold rows.")

    df["Frozen_Hyperparameters"] = df["Best_Hyperparameters"].map(
        parse_frozen_hyperparameters
    )

    return df


# ---------------------------------------------------------------------
# Frozen fold validation
# ---------------------------------------------------------------------

def load_and_validate_folds(df: pd.DataFrame) -> pd.DataFrame:
    require_file(FOLD_FILE, "Part13 fold assignments")
    folds = pd.read_csv(FOLD_FILE)

    required = {
        "Row_Index",
        "Mutation",
        "WT_AA",
        "Sequence_Position",
        "Mutant_AA",
        "PositionAware_Fold",
        "Random_Fold",
    }
    missing = required - set(folds.columns)
    if missing:
        raise ValueError(f"Part13 fold file missing columns: {sorted(missing)}")

    if len(folds) != len(df):
        raise ValueError(
            f"Fold file rows ({len(folds)}) != dataset rows ({len(df)})."
        )

    if folds["Row_Index"].duplicated().any():
        raise ValueError("Duplicate Row_Index values in Part13 fold file.")

    # We require direct row alignment because all subsequent indexing is frozen.
    if not np.array_equal(
        folds["Row_Index"].to_numpy(),
        np.arange(len(df)),
    ):
        raise ValueError(
            "Part13 Row_Index is not exactly 0..N-1; refusing unsafe implicit alignment."
        )

    if not np.array_equal(
        folds["Mutation"].astype(str).to_numpy(),
        df["Mutation"].astype(str).to_numpy(),
    ):
        raise ValueError("Mutation values are not aligned between dataset and folds.")

    if not np.array_equal(
        pd.to_numeric(folds["Sequence_Position"], errors="coerce").to_numpy(),
        pd.to_numeric(df["Sequence_Position"], errors="coerce").to_numpy(),
    ):
        raise ValueError("Sequence_Position is not aligned between dataset and folds.")

    pa = pd.to_numeric(folds["PositionAware_Fold"], errors="coerce")
    if pa.isna().any():
        raise ValueError("Invalid PositionAware_Fold values.")
    pa = pa.astype(int)

    if sorted(pa.unique().tolist()) != list(range(1, OUTER_FOLDS + 1)):
        raise ValueError(
            f"PositionAware_Fold must contain 1..{OUTER_FOLDS}."
        )

    # Critical position-disjointness check.
    overlap_rows = []
    for fold in range(1, OUTER_FOLDS + 1):
        test_pos = set(
            pd.to_numeric(
                folds.loc[pa == fold, "Sequence_Position"], errors="coerce"
            ).dropna().astype(int)
        )
        train_pos = set(
            pd.to_numeric(
                folds.loc[pa != fold, "Sequence_Position"], errors="coerce"
            ).dropna().astype(int)
        )
        overlap = test_pos & train_pos
        if overlap:
            overlap_rows.append((fold, sorted(list(overlap))[:10]))

    if overlap_rows:
        raise ValueError(
            f"Position-aware fold leakage detected: {overlap_rows}"
        )

    folds["PositionAware_Fold"] = pa
    return folds


# ---------------------------------------------------------------------
# Step2 family consistency
# ---------------------------------------------------------------------

def load_step2_selected() -> pd.DataFrame | None:
    if not STEP2_DIR.exists():
        return None

    p = STEP2_DIR / "VIM2_Step2_PositionAware_Selected_Candidates.csv"
    if not p.exists():
        return None

    df = pd.read_csv(p)
    needed = {"Target", "Architecture", "Outer_Fold", "Selected_Model_Family"}
    if not needed.issubset(df.columns):
        return None

    df["Target"] = df["Target"].astype(str)
    df["Architecture"] = df["Architecture"].map(normalize_architecture)
    df["Outer_Fold"] = pd.to_numeric(df["Outer_Fold"], errors="coerce").astype(int)
    df["Selected_Model_Family"] = df["Selected_Model_Family"].map(
        normalize_model_name
    )
    return df


# ---------------------------------------------------------------------
# Main interpretation
# ---------------------------------------------------------------------

def main():
    start_time = time.time()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 90)
    print("VIM-2 STEP 7 — FINAL MODEL INTERPRETATION")
    print("=" * 90)
    print(f"Started: {utc_now()}")
    print()

    # ---------------------------------------------------------------
    # 1. Load frozen inputs
    # ---------------------------------------------------------------
    print("[1/8] Loading and validating frozen inputs...")

    require_file(DATA_FILE, "Final ML dataset")
    data = pd.read_csv(DATA_FILE)

    if len(data) != 5016:
        raise ValueError(f"Expected 5016 dataset rows, found {len(data)}.")

    missing_targets = [t for t in TARGETS if t not in data.columns]
    if missing_targets:
        raise ValueError(f"Missing target columns: {missing_targets}")

    manifest, features = load_feature_manifest()
    validate_predictors(data, features)

    folds = load_and_validate_folds(data)
    step6 = load_step6_architecture_selection()
    step3 = load_step3_frozen_hpo()
    step2 = load_step2_selected()

    print(f"    Dataset rows: {len(data)}")
    print(f"    Model A predictors: {len(features[ARCHITECTURE_A])}")
    print(f"    Model B predictors: {len(features[ARCHITECTURE_B])}")
    print(f"    Step3 frozen configurations: {len(step3)}")
    print(f"    Step6 targets: {len(step6)}")
    print(f"    Step2 family file available: {step2 is not None}")
    print()

    # ---------------------------------------------------------------
    # 2. Freeze final architecture and family per target/fold
    # ---------------------------------------------------------------
    print("[2/8] Building frozen Step6 + Step3 interpretation map...")

    arch_map = dict(
        zip(step6["Target"], step6["Selected_Architecture"])
    )

    final_rows = []

    for target in TARGETS:
        arch = arch_map[target]

        subset = step3[
            (step3["Target"] == target)
            & (step3["Architecture"] == arch)
        ].copy()

        if len(subset) != OUTER_FOLDS:
            raise ValueError(
                f"{target}: expected {OUTER_FOLDS} Step3 rows for selected "
                f"architecture {arch}, found {len(subset)}."
            )

        subset = subset.sort_values("Outer_Fold")

        for _, r in subset.iterrows():
            final_rows.append(
                {
                    "Target": target,
                    "Architecture": arch,
                    "Outer_Fold": int(r["Outer_Fold"]),
                    "Model_Family": normalize_model_name(r["Model_Family"]),
                    "Frozen_Hyperparameters": r["Frozen_Hyperparameters"],
                }
            )

    final_map = pd.DataFrame(final_rows)

    if len(final_map) != len(TARGETS) * OUTER_FOLDS:
        raise ValueError("Final interpretation map does not contain exactly 45 rows.")

    if final_map.duplicated(
        ["Target", "Architecture", "Outer_Fold"]
    ).any():
        raise ValueError("Duplicate final interpretation configurations.")

    # If Step2 is available, confirm family provenance.
    if step2 is not None:
        s2 = step2.copy()
        s2 = s2[
            s2["Target"].isin(TARGETS)
            & s2["Architecture"].isin(ARCHITECTURES)
        ].copy()

        for _, r in final_map.iterrows():
            hit = s2[
                (s2["Target"] == r["Target"])
                & (s2["Architecture"] == r["Architecture"])
                & (s2["Outer_Fold"] == r["Outer_Fold"])
            ]
            if len(hit) == 1:
                expected_family = normalize_model_name(
                    hit.iloc[0]["Selected_Model_Family"]
                )
                if expected_family != r["Model_Family"]:
                    raise ValueError(
                        f"Step2→Step3 family mismatch for "
                        f"{r['Target']} {r['Architecture']} fold {r['Outer_Fold']}: "
                        f"{expected_family} vs {r['Model_Family']}"
                    )

    print("    Final architecture is inherited from Step6.")
    print("    RF/ET family and hyperparameters are inherited from Step3.")
    print()

    # ---------------------------------------------------------------
    # 3. Fit final frozen models and compute interpretation metrics
    # ---------------------------------------------------------------
    print("[3/8] Fitting 45 frozen final models and computing interpretation...")

    fold_records: List[Dict[str, Any]] = []
    completed = 0
    total = len(final_map)

    for _, cfg in final_map.sort_values(
        ["Target", "Outer_Fold"]
    ).iterrows():

        target = cfg["Target"]
        arch = cfg["Architecture"]
        fold = int(cfg["Outer_Fold"])
        family = cfg["Model_Family"]
        params = cfg["Frozen_Hyperparameters"]

        predictor_cols = features[arch]

        test_mask = folds["PositionAware_Fold"].eq(fold).to_numpy()
        train_mask = ~test_mask

        X_train = data.loc[train_mask, predictor_cols]
        X_test = data.loc[test_mask, predictor_cols]
        y_train = pd.to_numeric(
            data.loc[train_mask, target], errors="coerce"
        )
        y_test = pd.to_numeric(
            data.loc[test_mask, target], errors="coerce"
        )

        valid_train = y_train.notna().to_numpy()
        valid_test = y_test.notna().to_numpy()

        X_train = X_train.iloc[valid_train].copy()
        y_train = y_train.iloc[valid_train].copy()
        X_test = X_test.iloc[valid_test].copy()
        y_test = y_test.iloc[valid_test].copy()

        if len(X_train) == 0 or len(X_test) == 0:
            raise ValueError(
                f"Empty train/test set for {target}, {arch}, fold {fold}."
            )

        # Final frozen model: preprocessing + frozen estimator.
        # Pipeline.fit() learns the categorical encoding from X_train ONLY.
        model_seed = 100000 + TARGETS.index(target) * 100 + fold
        model = make_final_pipeline(
            family,
            params,
            model_seed,
            predictor_cols,
        )
        model.fit(X_train, y_train)

        # Native tree importance is calculated from the model fit on TRAIN only.
        # Map encoded model-space importance back to the original conceptual
        # Part13 features so Secondary_Structure remains one reported feature.
        tree_model = model.named_steps["model"]
        transformed_names = get_transformed_feature_names(model)

        transformed_native = np.asarray(
            tree_model.feature_importances_, dtype=float
        )
        native_map = aggregate_transformed_importance(
            predictor_cols,
            transformed_names,
            transformed_native,
        )

        if len(transformed_native) != len(transformed_names):
            raise ValueError(
                f"Transformed native importance length mismatch for "
                f"{target}, {arch}, fold {fold}."
            )

        if set(native_map) != set(predictor_cols):
            raise ValueError(
                f"Native importance mapping mismatch for {target}, {arch}, fold {fold}."
            )

        # Permutation importance is performed on the ORIGINAL conceptual
        # predictor columns in the untouched outer TEST fold. This is critical:
        # Secondary_Structure is permuted as one biological feature rather than
        # independently permuting its one-hot columns.
        perm_seed = PERMUTATION_RANDOM_STATE_BASE + TARGETS.index(target) * 100 + fold
        perm = permutation_importance(
            model,
            X_test,
            y_test,
            scoring=PERMUTATION_SCORING,
            n_repeats=PERMUTATION_REPEATS,
            random_state=perm_seed,
            n_jobs=N_JOBS,
        )

        if perm.importances_mean.shape[0] != len(predictor_cols):
            raise ValueError(
                f"Permutation importance length mismatch for {target}, {arch}, fold {fold}."
            )

        # Test R2 is recorded only as context; it is NOT used for selection.
        pred = model.predict(X_test)
        test_r2 = float(r2_score(y_test, pred))

        for j, feature in enumerate(predictor_cols):
            fold_records.append(
                {
                    "Target": target,
                    "Architecture": arch,
                    "Outer_Fold": fold,
                    "Model_Family": family,
                    "Feature": feature,
                    "Native_Importance": float(native_map[feature]),
                    "Permutation_Importance_Mean": float(
                        perm.importances_mean[j]
                    ),
                    "Permutation_Importance_SD": float(
                        perm.importances_std[j]
                    ),
                    "Permutation_Positive_Fraction": float(
                        np.mean(perm.importances[j] > 0)
                    ),
                    "Outer_Test_R2_Context": test_r2,
                    "N_Train": int(len(X_train)),
                    "N_Test": int(len(X_test)),
                    "Permutation_Repeats": PERMUTATION_REPEATS,
                    "Permutation_Scoring": PERMUTATION_SCORING,
                    "Random_State_Model": model_seed,
                    "Random_State_Permutation": perm_seed,
                }
            )

        completed += 1
        print(
            f"    [{completed:02d}/{total}] "
            f"{target} | {arch} | fold {fold} | {family} | "
            f"test R2={test_r2:.4f}"
        )

    fold_df = pd.DataFrame(fold_records)

    # ---------------------------------------------------------------
    # 4. Aggregate importance across frozen outer folds
    # ---------------------------------------------------------------
    print()
    print("[4/8] Aggregating feature importance across Position-Aware folds...")

    group_cols = ["Target", "Architecture", "Feature"]

    summary = (
        fold_df.groupby(group_cols, as_index=False)
        .agg(
            Model_Family_Mode=("Model_Family", lambda s: s.mode().iloc[0]),
            Outer_Folds_Observed=("Outer_Fold", "nunique"),
            Native_Importance_Mean=("Native_Importance", "mean"),
            Native_Importance_SD=("Native_Importance", "std"),
            Native_Importance_Median=("Native_Importance", "median"),
            Permutation_Importance_Mean=("Permutation_Importance_Mean", "mean"),
            Permutation_Importance_SD=("Permutation_Importance_Mean", "std"),
            Permutation_Importance_Median=("Permutation_Importance_Mean", "median"),
            Permutation_Positive_Fraction_Mean=(
                "Permutation_Positive_Fraction",
                "mean",
            ),
        )
    )

    # Across-fold stability: fraction of folds in which the feature's
    # permutation importance was positive.
    positive = (
        fold_df.assign(
            Positive=fold_df["Permutation_Importance_Mean"] > 0
        )
        .groupby(group_cols)["Positive"]
        .mean()
        .reset_index(name="Permutation_Positive_Fold_Fraction")
    )

    summary = summary.merge(positive, on=group_cols, how="left")

    summary["Native_Importance_CV"] = (
        summary["Native_Importance_SD"]
        / summary["Native_Importance_Mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan)

    summary["Permutation_Importance_CV"] = (
        summary["Permutation_Importance_SD"]
        / summary["Permutation_Importance_Mean"].abs().replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan)

    # Rank by mean permutation importance, then native importance.
    summary["Permutation_Rank"] = (
        summary.groupby("Target")["Permutation_Importance_Mean"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    summary["Native_Rank"] = (
        summary.groupby("Target")["Native_Importance_Mean"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    summary = summary.sort_values(
        ["Target", "Permutation_Rank", "Feature"]
    ).reset_index(drop=True)

    # ---------------------------------------------------------------
    # 5. Top-feature table
    # ---------------------------------------------------------------
    print("[5/8] Creating target-level top-feature summaries...")

    top_n = 15
    top = (
        summary.sort_values(
            ["Target", "Permutation_Importance_Mean", "Native_Importance_Mean"],
            ascending=[True, False, False],
        )
        .groupby("Target", group_keys=False)
        .head(top_n)
        .copy()
    )
    top["Top_N"] = top.groupby("Target").cumcount() + 1

    # Add whether Sequence_Position is present / selected.
    top["Sequence_Position_Included"] = (
        top["Architecture"] == ARCHITECTURE_B
    )

    # ---------------------------------------------------------------
    # 6. Architecture/model-family audit
    # ---------------------------------------------------------------
    print("[6/8] Building architecture and model-family audit...")

    audit_rows = []
    for target in TARGETS:
        arch = arch_map[target]
        cfgs = final_map[final_map["Target"] == target].sort_values("Outer_Fold")

        families = cfgs["Model_Family"].tolist()

        # Top feature(s) in the selected architecture.
        s = summary[
            (summary["Target"] == target)
            & (summary["Architecture"] == arch)
        ].sort_values("Permutation_Importance_Mean", ascending=False)

        top_features = s.head(5)["Feature"].tolist()

        seq_row = s[s["Feature"] == "Sequence_Position"]
        if len(seq_row):
            seq_imp = float(seq_row.iloc[0]["Permutation_Importance_Mean"])
            seq_rank = int(seq_row.iloc[0]["Permutation_Rank"])
            seq_pos_frac = float(
                seq_row.iloc[0]["Permutation_Positive_Fold_Fraction"]
            )
        else:
            seq_imp = np.nan
            seq_rank = np.nan
            seq_pos_frac = np.nan

        audit_rows.append(
            {
                "Target": target,
                "Selected_Architecture": arch,
                "Selected_Model_Families_By_Fold": ";".join(families),
                "Selected_Model_Family_Mode": pd.Series(families).mode().iloc[0],
                "Sequence_Position_Included": arch == ARCHITECTURE_B,
                "Sequence_Position_Permutation_Importance_Mean": seq_imp,
                "Sequence_Position_Permutation_Rank": seq_rank,
                "Sequence_Position_Positive_Fold_Fraction": seq_pos_frac,
                "Top_5_Features_By_Permutation_Importance": ";".join(top_features),
                "Interpretation_Uses_Step6_Architecture": True,
                "Interpretation_Uses_Step3_Frozen_HPO": True,
                "Interpretation_Uses_Part13_Frozen_Folds": True,
                "Step7_Does_Not_ReSelect_Architecture": True,
                "Step7_Does_Not_ReHPO": True,
            }
        )

    arch_audit = pd.DataFrame(audit_rows)

    # ---------------------------------------------------------------
    # 7. Leakage audit
    # ---------------------------------------------------------------
    print("[7/8] Writing explicit Step7 leakage audit...")

    leakage_rows = [
        {
            "Audit_Item": "Architecture selection source",
            "Status": "PASS",
            "Evidence": "Inherited from Step6 Selected_Architecture; no Step7 architecture selection.",
        },
        {
            "Audit_Item": "RF/Extra-Trees family source",
            "Status": "PASS",
            "Evidence": "Inherited from Step3 frozen outer-fold HPO results; no Step7 family re-selection.",
        },
        {
            "Audit_Item": "Hyperparameter source",
            "Status": "PASS",
            "Evidence": "Inherited from Step3 Best_Hyperparameters; no Step7 HPO.",
        },
        {
            "Audit_Item": "Outer fold source",
            "Status": "PASS",
            "Evidence": "Part13 PositionAware_Fold used unchanged.",
        },
        {
            "Audit_Item": "Position-disjointness",
            "Status": "PASS",
            "Evidence": "Validated before model fitting.",
        },
        {
            "Audit_Item": "Feature manifest",
            "Status": "PASS",
            "Evidence": "Part13 Feature Manifest is the source of truth; Model A=65 and Model B=66.",
        },
        {
            "Audit_Item": "Target SD exclusion",
            "Status": "PASS",
            "Evidence": "Predictor lists were checked for *_SD columns.",
        },
        {
            "Audit_Item": "Categorical preprocessing",
            "Status": "PASS",
            "Evidence": "Step 7 reproduces Step 3 preprocessing: numeric median imputation + missing indicators + StandardScaler; Secondary_Structure most-frequent imputation + OneHotEncoder. All preprocessing is fit on outer-training data only.",
        },
        {
            "Audit_Item": "Feature-level interpretation",
            "Status": "PASS",
            "Evidence": "Native importance of one-hot Secondary_Structure columns is summed back to the original conceptual feature; permutation importance permutes Secondary_Structure as one original predictor.",
        },
        {
            "Audit_Item": "Permutation evaluation",
            "Status": "PASS",
            "Evidence": "Permutation importance evaluated post-hoc on untouched outer test folds and cannot alter fitted models.",
        },
        {
            "Audit_Item": "New fold creation",
            "Status": "PASS",
            "Evidence": "No new train/test folds were created.",
        },
        {
            "Audit_Item": "Step5 influence",
            "Status": "PASS",
            "Evidence": "Step5 random-split benchmark is not used for Step7 architecture or model-family selection.",
        },
    ]
    leakage_df = pd.DataFrame(leakage_rows)

    # ---------------------------------------------------------------
    # 8. Save outputs + reproducibility
    # ---------------------------------------------------------------
    print("[8/8] Saving Step7 outputs and reproducibility manifest...")

    fold_path = OUTPUT_DIR / "VIM2_Step7_Fold_Feature_Importance.csv"
    summary_path = OUTPUT_DIR / "VIM2_Step7_Feature_Importance_Summary.csv"
    top_path = OUTPUT_DIR / "VIM2_Step7_Top_Features_By_Target.csv"
    audit_path = OUTPUT_DIR / "VIM2_Step7_Target_Architecture_Audit.csv"
    leakage_path = OUTPUT_DIR / "VIM2_Step7_Leakage_Audit.csv"
    manifest_path = OUTPUT_DIR / "VIM2_Step7_Reproducibility_Manifest.json"
    hash_path = OUTPUT_DIR / "VIM2_Step7_SHA256.csv"

    fold_df.to_csv(fold_path, index=False)
    summary.to_csv(summary_path, index=False)
    top.to_csv(top_path, index=False)
    arch_audit.to_csv(audit_path, index=False)
    leakage_df.to_csv(leakage_path, index=False)

    # Save a machine-readable final configuration map.
    final_map_json = final_map.copy()
    final_map_json["Frozen_Hyperparameters"] = final_map_json[
        "Frozen_Hyperparameters"
    ].map(lambda x: json.dumps(x, sort_keys=True))

    confirmed_step3_artifacts = [
        "VIM2_Step3_All_HPO_Configurations.csv",
        "VIM2_Step3_Best_Hyperparameters_Per_Fold.csv",
        "VIM2_Step3_Checkpoint.json",
        "VIM2_Step3_HPO_Timing.csv",
        "VIM2_Step3_Leakage_Audit.csv",
        "VIM2_Step3_Outer_Position_Audit.csv",
        "VIM2_Step3_Reproducibility_Manifest.json",
    ]
    confirmed_step6_artifacts = [
        "VIM2_Step6_Final_Model_Selection.csv",
        "VIM2_Step6_Final_Model_Selection.json",
        "VIM2_Step6_Integrated_Manuscript_Results.csv",
        "VIM2_Step6_Leakage_Audit.csv",
        "VIM2_Step6_ModelA_vs_ModelB_PositionAware.csv",
        "VIM2_Step6_Pooled_Integrated_Performance.csv",
        "VIM2_Step6_PositionAware_vs_RandomSplit.csv",
        "VIM2_Step6_Reproducibility_Manifest.json",
    ]

    manifest = {
        "project": "VIM-2",
        "step": "Step7_Model_Interpretation",
        "created_utc": utc_now(),
        "purpose": "Post-hoc interpretation of frozen final models without changing model selection or hyperparameters.",
        "primary_interpretation_methods": [
            "model_native_tree_feature_importance",
            "outer_test_fold_permutation_importance",
        ],
        "permutation_repeats": PERMUTATION_REPEATS,
        "permutation_scoring": PERMUTATION_SCORING,
        "n_jobs": N_JOBS,
        "global_random_state": GLOBAL_RANDOM_STATE,
        "dataset_file": str(DATA_FILE),
        "fold_file": str(FOLD_FILE),
        "feature_manifest_file": str(FEATURE_MANIFEST_FILE),
        "step2_directory": str(STEP2_DIR),
        "step3_directory": str(STEP3_DIR),
        "step4_directory": str(STEP4_DIR),
        "step6_directory": str(STEP6_DIR),
        "output_directory": str(OUTPUT_DIR),
        "n_dataset_rows": int(len(data)),
        "targets": TARGETS,
        "architecture_predictor_counts": EXPECTED_PREDICTORS,
        "selected_architecture_counts": {
            arch: int(sum(arch_map[t] == arch for t in TARGETS))
            for arch in ARCHITECTURES
        },
        "selected_architecture_by_target": arch_map,
        "final_fold_configuration_map": json.loads(
            final_map_json.to_json(orient="records")
        ),
        "step7_does_not_reselect_architecture": True,
        "step7_does_not_reselect_model_family": True,
        "step7_does_not_run_hpo": True,
        "step7_does_not_create_new_folds": True,
        "step3_frozen_schema_source_column": "Model",
        "step7_internal_model_family_column": "Model_Family",
        "categorical_features": ["Secondary_Structure"],
        "categorical_encoding": "SimpleImputer(most_frequent) + OneHotEncoder(handle_unknown=ignore),",
        "numeric_preprocessing": "SimpleImputer(strategy=median, add_indicator=True) + StandardScaler",
        "categorical_preprocessing": "SimpleImputer(strategy=most_frequent) + OneHotEncoder(handle_unknown=ignore)",
"feature_level_importance_aggregation": "One-hot Secondary_Structure and numeric missing-indicator columns summed back to original conceptual features",
        "shap_run": False,
        "note_on_shap": (
            "SHAP is intentionally not part of the primary Step7 computation. "
            "It may be added later as a separately pre-specified secondary analysis."
        ),
        "software": {
            "python": platform.python_version(),
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "scikit_learn": sklearn.__version__,
            "platform": platform.platform(),
        },
        "runtime_seconds": round(time.time() - start_time, 3),
    }

    manifest_path.write_text(
        json.dumps(manifest, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    output_files = [
        fold_path,
        summary_path,
        top_path,
        audit_path,
        leakage_path,
        manifest_path,
    ]

    hash_rows = []
    for p in output_files:
        hash_rows.append(
            {
                "File": p.name,
                "Path": str(p),
                "SHA256": sha256_file(p),
                "Size_Bytes": p.stat().st_size,
            }
        )
    pd.DataFrame(hash_rows).to_csv(hash_path, index=False)

    print()
    print("=" * 90)
    print("STEP 7 COMPLETED SUCCESSFULLY")
    print("=" * 90)
    print(f"Output directory: {OUTPUT_DIR}")
    print()
    print("Selected architectures:")
    for target in TARGETS:
        print(f"  {target:28s} -> {arch_map[target]}")
    print()
    print("Files:")
    for p in output_files + [hash_path]:
        print(f"  - {p}")
    print()
    print(f"Runtime: {time.time() - start_time:.1f} s")
    print("=" * 90)


if __name__ == "__main__":
    main()


VIM-2 STEP 7 — FINAL MODEL INTERPRETATION
Started: 2026-09-05T08:01:31.638737+00:00

[1/8] Loading and validating frozen inputs...
    Dataset rows: 5016
    Model A predictors: 65
    Model B predictors: 66
    Step3 frozen configurations: 90
    Step6 targets: 9
    Step2 family file available: False

[2/8] Building frozen Step6 + Step3 interpretation map...
    Final architecture is inherited from Step6.
    RF/ET family and hyperparameters are inherited from Step3.

[3/8] Fitting 45 frozen final models and computing interpretation...
    [01/45] 0.031ug/mL_MEM_37C | Model_A_No_Position | fold 1 | Random_Forest | test R2=0.4935
    [02/45] 0.031ug/mL_MEM_37C | Model_A_No_Position | fold 2 | Random_Forest | test R2=0.5097
    [03/45] 0.031ug/mL_MEM_37C | Model_A_No_Position | fold 3 | Random_Forest | test R2=0.5744
    [04/45] 0.031ug/mL_MEM_37C | Model_A_No_Position | fold 4 | Random_Forest | test R2=0.4793
    [05/45] 0.031ug/mL_MEM_37C | Model_A_No_Position | fold 5 | Random_Fores

In [ ]:
# @title

# ============================================================
# DOWNLOAD STEP7 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step7_Model_Interpretation"

# Output ZIP archive
zip_base = "/content/VIM2_Step7_Model_Interpretation"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step7_Model_Interpretation"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP 7 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP 7 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step7_Model_Interpretation
ZIP archive      : /content/VIM2_Step7_Model_Interpretation.zip
Archive size     : 0.17 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
VIM-2 Project — Step 8: Biological and Molecular Interpretation
================================================================

Purpose
-------
Translate the statistically supported feature-importance results from Step 7
into a conservative, biologically interpretable summary without changing the
ML analysis.

Scientific position in the pipeline
------------------------------------
Step 7 asks:
    "Which predictors does the frozen final model rely on?"

Step 8 asks:
    "What biological or molecular context do those predictors represent?"

This script is intentionally POST-ML and interpretation-only.

Hard scientific constraints
---------------------------
1. Step 7 is the sole source of ML feature-importance rankings.
2. Step 6 remains the sole source of final Model A/Model B architecture choice.
3. No model is fitted, tuned, re-selected, or re-scored here.
4. No target definition is recreated or changed here.
5. No new train/test folds are created.
6. No biological feature is called causal merely because it is important to ML.
7. Missing structural measurements are preserved as missing; no biological
   imputation is performed.
8. Biological annotations are deterministic, transparent, and auditable.
9. Every interpretation distinguishes ML evidence from biological context.
10. The script fails fast on schema conflicts rather than silently guessing.

Primary inputs
--------------
/content/VIM2_Part12D_Hydrogen_Bonding.csv
/content/VIM2_Step7_Model_Interpretation/VIM2_Step7_Feature_Importance_Summary.csv
/content/VIM2_Step7_Model_Interpretation/VIM2_Step7_Top_Features_By_Target.csv

Outputs
-------
/content/VIM2_Step8_Biological_Interpretation/
    VIM2_Step8_Feature_Biological_Annotation.csv
    VIM2_Step8_Target_Feature_Interpretation.csv
    VIM2_Step8_Target_Summary.csv
    VIM2_Step8_Position_Context_Summary.csv
    VIM2_Step8_Feature_Coverage_Audit.csv
    VIM2_Step8_Interpretation_Audit.csv
    VIM2_Step8_Reproducibility_Manifest.json
    VIM2_Step8_SHA256.csv
    figures/
        Figure8A_Top_Feature_Importance_Heatmap.png/.pdf/.svg
        Figure8B_Biological_Context_Category_Counts.png/.pdf/.svg
        Figure8C_Importance_vs_Stability.png/.pdf/.svg

Notes
-----
The biological language is deliberately conservative. Terms such as
"catalytic context", "structural context", "solvent exposure", and
"evolutionary constraint" describe what a feature measures; they do not
claim that the feature mechanistically causes the phenotype.
"Potential H-bond capacity" is used instead of "observed H-bonding" because
this dataset contains residue-level capacity descriptors, not experimental
interaction measurements.
"""

from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

DATA_FILE = Path("/content/VIM2_Part12D_Hydrogen_Bonding.csv")
# Part12D is the canonical biological/structural source for Step 8.
# No fallback to an ML-only table is used because Step 8 requires explicit
# residue-level biological context and must not reconstruct it from row order.
BIOLOGICAL_DATA_FALLBACK = None
STEP7_DIR = Path("/content/VIM2_Step7_Model_Interpretation")
STEP7_SUMMARY_FILE = STEP7_DIR / "VIM2_Step7_Feature_Importance_Summary.csv"
STEP7_TOP_FILE = STEP7_DIR / "VIM2_Step7_Top_Features_By_Target.csv"

OUTPUT_DIR = Path("/content/VIM2_Step8_Biological_Interpretation")
FIGURE_DIR = OUTPUT_DIR / "figures"

RANDOM_SEED = 42
TOP_N_PER_TARGET = 10
MIN_OUTER_FOLDS = 3

# Features that are biological descriptors rather than ML-derived quantities.
# This list is used only for annotation; Step 7 remains the source of truth
# for whether a feature was actually important.
BIOLOGICAL_FEATURES = {
    "SASA",
    "Distance_to_Metal_Site",
    "Distance_to_Active_Site_Pocket",
    "Distance_to_L3_Loop",
    "Distance_to_L10_Loop",
    "Hydrophobicity_Change",
    "Charge_Change",
    "Weight_Change",
    "Polarity_Change",
    "BLOSUM62",
    "Secondary_Structure",
    "Local_Hydrophobic_Fraction",
    "Local_Charged_Fraction",
    "Local_Positive_Charge_Fraction",
    "Local_Negative_Charge_Fraction",
    "Local_Polar_Fraction",
    "Local_Aromatic_Fraction",
    "Local_Gly_Pro_Fraction",
    "Local_Sequence_Entropy",
    "Side_Chain_Volume_Change",
    "Absolute_Side_Chain_Volume_Change",
    "Relative_Side_Chain_Volume_Change",
    "HBond_Donor_Change",
    "HBond_Acceptor_Change",
    "Total_HBond_Capacity_Change",
    "Sequence_Position",
}


# ---------------------------------------------------------------------------
# Biological annotation dictionary
# ---------------------------------------------------------------------------

# Each annotation contains descriptive biology only. It deliberately avoids
# direction-of-effect claims because feature importance does not establish
# whether increasing/decreasing the feature improves fitness.
FEATURE_ANNOTATIONS: Dict[str, Dict[str, str]] = {
    "SASA": {
        "category": "Solvent exposure",
        "biological_meaning": "Solvent-accessible surface area of the residue/environment.",
        "mechanistic_context": "Reports how exposed a residue is to solvent and therefore describes local structural environment.",
        "interpretation_caution": "Association with prediction does not establish that solvent exposure causally changes fitness.",
    },
    "Distance_to_Metal_Site": {
        "category": "Catalytic structural context",
        "biological_meaning": "Spatial distance from the nearest annotated metal site.",
        "mechanistic_context": "Places the mutation in the structural context of the metalloenzyme active center.",
        "interpretation_caution": "Distance alone does not demonstrate an effect on metal binding or catalysis.",
    },
    "Distance_to_Active_Site_Pocket": {
        "category": "Active-site structural context",
        "biological_meaning": "Spatial distance from the annotated active-site pocket.",
        "mechanistic_context": "Measures how close the mutated residue is to a functionally relevant structural region.",
        "interpretation_caution": "Proximity is contextual evidence, not proof of direct functional interaction.",
    },
    "Distance_to_L3_Loop": {
        "category": "Loop structural context",
        "biological_meaning": "Spatial distance from the L3 loop.",
        "mechanistic_context": "Captures local positioning relative to a structurally relevant loop.",
        "interpretation_caution": "Does not establish direct residue-loop interaction or conformational causality.",
    },
    "Distance_to_L10_Loop": {
        "category": "Loop structural context",
        "biological_meaning": "Spatial distance from the L10 loop.",
        "mechanistic_context": "Captures local positioning relative to another functionally relevant loop region.",
        "interpretation_caution": "Does not establish direct residue-loop interaction or conformational causality.",
    },
    "Hydrophobicity_Change": {
        "category": "Physicochemical substitution",
        "biological_meaning": "Change in residue hydrophobicity caused by the amino-acid substitution.",
        "mechanistic_context": "Describes a change in local hydrophobic character that may alter residue-environment compatibility.",
        "interpretation_caution": "Importance does not determine the direction or molecular mechanism of the effect.",
    },
    "Charge_Change": {
        "category": "Physicochemical substitution",
        "biological_meaning": "Change in formal side-chain charge associated with the substitution.",
        "mechanistic_context": "Captures alteration of local electrostatic character.",
        "interpretation_caution": "Does not by itself demonstrate disruption or formation of a specific electrostatic interaction.",
    },
    "Weight_Change": {
        "category": "Physicochemical substitution",
        "biological_meaning": "Change in amino-acid molecular weight.",
        "mechanistic_context": "Describes a coarse change in residue size/mass.",
        "interpretation_caution": "Molecular weight change alone is not a direct measure of structural disruption.",
    },
    "Polarity_Change": {
        "category": "Physicochemical substitution",
        "biological_meaning": "Change in residue polarity.",
        "mechanistic_context": "Captures alteration in local polar/non-polar character.",
        "interpretation_caution": "Does not establish a specific solvent or hydrogen-bonding mechanism.",
    },
    "BLOSUM62": {
        "category": "Evolutionary constraint",
        "biological_meaning": "A substitution score reflecting how frequently amino-acid replacements are observed in conserved protein alignments.",
        "mechanistic_context": "Provides evolutionary context for the plausibility of the specific amino-acid substitution.",
        "interpretation_caution": "BLOSUM62 is not a fitness measurement and should not be interpreted as direct evidence that a mutation is beneficial or deleterious.",
    },
    "Secondary_Structure": {
        "category": "Secondary-structure context",
        "biological_meaning": "Residue-level secondary-structure class.",
        "mechanistic_context": "Places the mutation within a helix, sheet, turn/coil, or related structural class.",
        "interpretation_caution": "Secondary-structure association does not prove a mutation-induced conformational change.",
    },
    "Local_Hydrophobic_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of hydrophobic residues in the local sequence window.",
        "mechanistic_context": "Describes the local sequence-level physicochemical environment surrounding the mutation.",
        "interpretation_caution": "It is a sequence-context descriptor, not a direct measurement of local packing or solvent exposure.",
    },
    "Local_Charged_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of charged residues in the local sequence window.",
        "mechanistic_context": "Summarizes local electrostatic composition at the sequence level.",
        "interpretation_caution": "Does not establish a specific electrostatic interaction.",
    },
    "Local_Positive_Charge_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of positively charged residues in the local sequence window.",
        "mechanistic_context": "Describes local positive electrostatic composition.",
        "interpretation_caution": "Sequence composition is not equivalent to measured local electrostatic potential.",
    },
    "Local_Negative_Charge_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of negatively charged residues in the local sequence window.",
        "mechanistic_context": "Describes local negative electrostatic composition.",
        "interpretation_caution": "Sequence composition is not equivalent to measured local electrostatic potential.",
    },
    "Local_Polar_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of polar residues in the local sequence window.",
        "mechanistic_context": "Describes local sequence-level polarity.",
        "interpretation_caution": "Does not directly measure solvent interactions or hydrogen bonding.",
    },
    "Local_Aromatic_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of aromatic residues in the local sequence window.",
        "mechanistic_context": "Describes local aromatic sequence composition.",
        "interpretation_caution": "Does not establish aromatic stacking or packing interactions.",
    },
    "Local_Gly_Pro_Fraction": {
        "category": "Local sequence environment",
        "biological_meaning": "Fraction of glycine/proline residues in the local sequence window.",
        "mechanistic_context": "Provides context on residues that can influence local backbone conformational preferences.",
        "interpretation_caution": "Composition alone does not demonstrate a change in backbone flexibility or conformation.",
    },
    "Local_Sequence_Entropy": {
        "category": "Local sequence conservation/context",
        "biological_meaning": "Entropy-based measure of amino-acid diversity in the local sequence window.",
        "mechanistic_context": "Describes local sequence variability or compositional complexity.",
        "interpretation_caution": "Entropy is not equivalent to experimentally measured evolutionary conservation or structural flexibility.",
    },
    "Side_Chain_Volume_Change": {
        "category": "Side-chain remodeling",
        "biological_meaning": "Signed change in side-chain volume between wild-type and mutant residues.",
        "mechanistic_context": "Describes the direction and magnitude of residue-volume remodeling.",
        "interpretation_caution": "Importance does not establish steric clash, packing disruption, or stabilization.",
    },
    "Absolute_Side_Chain_Volume_Change": {
        "category": "Side-chain remodeling",
        "biological_meaning": "Absolute magnitude of side-chain volume change.",
        "mechanistic_context": "Quantifies how strongly residue size is remodeled regardless of direction.",
        "interpretation_caution": "Does not establish whether the change is structurally favorable or unfavorable.",
    },
    "Relative_Side_Chain_Volume_Change": {
        "category": "Side-chain remodeling",
        "biological_meaning": "Side-chain volume change expressed relative to the wild-type residue.",
        "mechanistic_context": "Provides a scale-normalized description of residue-size remodeling.",
        "interpretation_caution": "Relative size change is not a direct measure of steric strain.",
    },
    "HBond_Donor_Change": {
        "category": "Potential hydrogen-bond capacity",
        "biological_meaning": "Change in the residue's hydrogen-bond donor capacity.",
        "mechanistic_context": "Describes potential alteration of hydrogen-bond donor availability.",
        "interpretation_caution": "This is potential capacity, not evidence of an observed hydrogen bond.",
    },
    "HBond_Acceptor_Change": {
        "category": "Potential hydrogen-bond capacity",
        "biological_meaning": "Change in the residue's hydrogen-bond acceptor capacity.",
        "mechanistic_context": "Describes potential alteration of hydrogen-bond acceptor availability.",
        "interpretation_caution": "This is potential capacity, not evidence of an observed hydrogen bond.",
    },
    "Total_HBond_Capacity_Change": {
        "category": "Potential hydrogen-bond capacity",
        "biological_meaning": "Change in the combined hydrogen-bond donor/acceptor capacity descriptor.",
        "mechanistic_context": "Summarizes potential remodeling of hydrogen-bonding capacity at the mutated residue.",
        "interpretation_caution": "Does not demonstrate formation or loss of a specific hydrogen bond.",
    },
    "Sequence_Position": {
        "category": "Positional context",
        "biological_meaning": "Residue position along the protein sequence.",
        "mechanistic_context": "Captures position-specific patterns that may reflect regional sequence or structural context.",
        "interpretation_caution": "Position importance does not identify the molecular mechanism responsible for the phenotype.",
    },
}

# Deterministic annotations for amino-acid identity indicator features exported by Step 7.
# These are descriptive feature encodings, not causal biological claims.
_AMINO_ACIDS = tuple("ACDEFGHIKLMNPQRSTVWY")

for _aa in _AMINO_ACIDS:
    FEATURE_ANNOTATIONS[f"WT_AA_{_aa}"] = {
        "category": "Wild-type amino-acid identity",
        "biological_meaning": f"Indicator that the wild-type residue is amino acid {_aa}.",
        "mechanistic_context": (
            "Encodes the identity of the wild-type residue and therefore captures "
            "sequence-level residue identity available to the predictive model."
        ),
        "interpretation_caution": (
            "Predictive importance indicates association with model output; it does "
            "not establish that this amino-acid identity causally determines the phenotype."
        ),
    }
    FEATURE_ANNOTATIONS[f"Mutant_AA_{_aa}"] = {
        "category": "Mutant amino-acid identity",
        "biological_meaning": f"Indicator that the mutant residue is amino acid {_aa}.",
        "mechanistic_context": (
            "Encodes the identity of the mutant residue and therefore captures "
            "sequence-level substitution information available to the predictive model."
        ),
        "interpretation_caution": (
            "Predictive importance indicates association with model output; it does "
            "not establish that introducing this amino acid causally determines the phenotype."
        ),
    }

del _aa




# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def fail(message: str) -> None:
    raise RuntimeError(f"STEP 8 VALIDATION ERROR: {message}")


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        fail(f"Missing {label}: {path}")


def require_columns(df: pd.DataFrame, columns: Sequence[str], label: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        fail(f"{label} is missing required columns: {missing}")


def finite_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)


def safe_div(a: float, b: float) -> float:
    if not np.isfinite(a) or not np.isfinite(b) or b == 0:
        return np.nan
    return a / b


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def save_figure(fig: plt.Figure, stem: Path) -> None:
    fig.savefig(stem.with_suffix(".png"), dpi=400, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(stem.with_suffix(".svg"), bbox_inches="tight")
    plt.close(fig)


def normalise_target_name(value: object) -> str:
    return str(value).strip()


# ---------------------------------------------------------------------------
# Input validation and loading
# ---------------------------------------------------------------------------

def load_inputs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    print("[1/8] Loading frozen Step 7 interpretation outputs and biological dataset...")

    require_file(STEP7_SUMMARY_FILE, "Step 7 feature-importance summary")
    require_file(STEP7_TOP_FILE, "Step 7 top-features table")

    # Step 8 uses Part12D directly as its canonical residue-level biological
    # source. This is intentional: Part12D contains the explicit residue
    # position and molecular/structural descriptors needed for interpretation.
    # No row-order position reconstruction, merging, or biological imputation
    # is permitted here.
    require_file(DATA_FILE, "canonical Part12D biological dataset")
    active_data_file = DATA_FILE
    data = pd.read_csv(active_data_file, low_memory=False)

    # Require an explicit source position; never synthesize positions.
    position_candidates = ["Position", "position", "Sequence_Position"]
    if not any(c in data.columns for c in position_candidates):
        fail(
            "Canonical Part12D dataset lacks an explicit residue-position column. "
            f"Expected one of {position_candidates}; available columns include: "
            f"{list(data.columns)[:40]}"
        )

    summary = pd.read_csv(STEP7_SUMMARY_FILE, low_memory=False)
    top = pd.read_csv(STEP7_TOP_FILE, low_memory=False)

    require_columns(
        summary,
        [
            "Target", "Architecture", "Feature", "Model_Family_Mode",
            "Outer_Folds_Observed", "Native_Importance_Mean",
            "Native_Importance_SD", "Permutation_Importance_Mean",
            "Permutation_Importance_SD", "Permutation_Positive_Fraction_Mean",
            "Permutation_Positive_Fold_Fraction", "Permutation_Rank",
            "Native_Rank",
        ],
        "Step 7 summary",
    )
    require_columns(top, ["Target", "Feature"], "Step 7 top-features table")

    # A residue position is required for position-resolved biological context.
    # Accept the canonical Part12D spellings, plus Sequence_Position where that
    # field is the explicit residue-number representation in the source file.
    position_candidates = ["Position", "position", "Sequence_Position"]
    position_col = next((c for c in position_candidates if c in data.columns), None)
    if position_col is None:
        fail(
            "Biological dataset contains no explicit residue-position column. "
            f"Expected one of {position_candidates}; available columns include: "
            f"{list(data.columns)[:25]}"
        )
    data["__Step8_Position"] = pd.to_numeric(data[position_col], errors="coerce")

    if data["__Step8_Position"].isna().all():
        fail("All residue positions are missing/non-numeric; positional interpretation is impossible.")

    # Do not silently repair mutation identifiers. Use the strongest available
    # canonical identifier and require it to be non-empty.
    mutation_candidates = ["Mutation", "mutation", "identity"]
    mutation_col = next((c for c in mutation_candidates if c in data.columns), None)
    if mutation_col is None:
        fail("No mutation identifier column found. Expected Mutation, mutation, or identity.")
    data["__Step8_Mutation"] = data[mutation_col].astype(str).str.strip()
    if (data["__Step8_Mutation"] == "").all():
        fail(f"Mutation identifier column '{mutation_col}' contains no usable identifiers.")

    return data, summary, top


# ---------------------------------------------------------------------------
# Biological feature annotation
# ---------------------------------------------------------------------------

def build_feature_annotation(summary: pd.DataFrame, data: pd.DataFrame) -> pd.DataFrame:
    print("[2/8] Building deterministic biological feature annotations...")

    features = sorted(set(summary["Feature"].astype(str)))
    rows: List[Dict[str, object]] = []

    for feature in features:
        ann = FEATURE_ANNOTATIONS.get(feature)
        present = feature in data.columns

        if ann is None:
            # Unknown Step 7 features are not silently assigned biology.
            category = "Unannotated feature"
            meaning = "No predefined biological annotation is available."
            mechanism = "No biological mechanism is inferred by Step 8."
            caution = "Manual expert review is required before biological interpretation."
            annotation_status = "REVIEW_REQUIRED"
        else:
            category = ann["category"]
            meaning = ann["biological_meaning"]
            mechanism = ann["mechanistic_context"]
            caution = ann["interpretation_caution"]
            annotation_status = "PASS"

        rows.append(
            {
                "Feature": feature,
                "Biological_Category": category,
                "Biological_Meaning": meaning,
                "Mechanistic_Context": mechanism,
                "Interpretation_Caution": caution,
                "Feature_Present_In_Biological_Dataset": bool(present),
                "Annotation_Status": annotation_status,
            }
        )

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Merge Step 7 evidence with biological annotations
# ---------------------------------------------------------------------------

def merge_step7_with_annotations(
    summary: pd.DataFrame,
    annotations: pd.DataFrame,
    data: pd.DataFrame,
) -> pd.DataFrame:
    print("[3/8] Linking Step 7 evidence to biological context...")

    df = summary.copy()
    df["Target"] = df["Target"].map(normalise_target_name)
    df["Feature"] = df["Feature"].astype(str).str.strip()

    # Enforce one row per target-feature pair. Duplicates would make downstream
    # biological summaries ambiguous and therefore trigger a hard failure.
    dup = df.duplicated(["Target", "Feature"], keep=False)
    if dup.any():
        sample = df.loc[dup, ["Target", "Feature"]].head(10).to_dict("records")
        fail(f"Duplicate Target/Feature rows in Step 7 summary: {sample}")

    df = df.merge(annotations, on="Feature", how="left", validate="many_to_one")

    if df["Annotation_Status"].isna().any():
        fail("Step 7 features could not be linked to biological annotations.")

    # Numeric validation for all evidence columns used in ranking/stability.
    numeric_cols = [
        "Outer_Folds_Observed", "Native_Importance_Mean",
        "Native_Importance_SD", "Permutation_Importance_Mean",
        "Permutation_Importance_SD", "Permutation_Positive_Fraction_Mean",
        "Permutation_Positive_Fold_Fraction", "Permutation_Rank",
        "Native_Rank",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if df[col].isna().any():
            fail(f"Step 7 column '{col}' contains missing/non-numeric values.")

    # Step 7 does not export Top_N or Sequence_Position_Included in the
    # frozen summary schema. Step 8 derives these reporting fields
    # deterministically without modifying any Step 7 importance or rank.
    df["Top_N"] = df["Permutation_Rank"] <= TOP_N_PER_TARGET

    # Sequence_Position_Included is descriptive only. It is reconstructed
    # from the actual Step 7 feature name and biological dataset schema.
    has_explicit_position = any(
        c in data.columns for c in ["Position", "position", "Sequence_Position"]
    )
    df["Sequence_Position_Included"] = (
        df["Feature"].eq("Sequence_Position") & has_explicit_position
    )

    if (df["Outer_Folds_Observed"] < MIN_OUTER_FOLDS).any():
        fail(f"At least one target has fewer than {MIN_OUTER_FOLDS} observed outer folds.")

    return df


# ---------------------------------------------------------------------------
# Evidence grading
# ---------------------------------------------------------------------------

def assign_evidence_grade(row: pd.Series) -> str:
    """
    Conservative deterministic evidence grade.

    This is NOT a statistical significance test. It is a reporting aid that
    combines permutation magnitude, cross-fold consistency, and rank.
    """
    mean = float(row["Permutation_Importance_Mean"])
    sd = float(row["Permutation_Importance_SD"])
    pos_fold = float(row["Permutation_Positive_Fold_Fraction"])
    rank = int(row["Permutation_Rank"])

    cv = abs(safe_div(sd, mean)) if mean != 0 else np.inf

    if mean > 0 and pos_fold >= 1.0 and rank <= 3 and cv <= 0.75:
        return "Strong_and_stable"
    if mean > 0 and pos_fold >= 0.8 and rank <= 5:
        return "Consistent_support"
    if mean > 0 and pos_fold >= 0.6:
        return "Moderate_support"
    return "Unstable_or_limited_support"


def build_target_interpretation(evidence: pd.DataFrame) -> pd.DataFrame:
    print("[4/8] Constructing target-specific biological interpretation summaries...")

    df = evidence.copy()
    df["Permutation_CV_Recomputed"] = [
        safe_div(float(a), float(b))
        for a, b in zip(df["Permutation_Importance_SD"], df["Permutation_Importance_Mean"])
    ]
    df["Evidence_Grade"] = df.apply(assign_evidence_grade, axis=1)

    # Rank the exact Step 7 permutation ranking; do not create a new ML ranking.
    df["Step8_Top_Feature"] = df["Top_N"]

    # Interpretation sentence deliberately separates evidence from meaning.
    df["Step8_Interpretation"] = (
        df["Feature"]
        + " was an important predictor in the frozen Step 7 model for target '"
        + df["Target"]
        + "'. In biological terms, this feature represents "
        + df["Biological_Meaning"].str.rstrip(".")
        + "."
    )
    df["Step8_Caution"] = df["Interpretation_Caution"]

    return df


# ---------------------------------------------------------------------------
# Position-level biological context
# ---------------------------------------------------------------------------

def build_position_context(data: pd.DataFrame, evidence: pd.DataFrame) -> pd.DataFrame:
    print("[5/8] Quantifying position-level structural/sequence context without imputation...")

    # Only use biological variables actually present in the dataset. Missing
    # values remain NaN and are reported as coverage, never imputed.
    context_features = [
        "SASA",
        "Distance_to_Metal_Site",
        "Distance_to_Active_Site_Pocket",
        "Distance_to_L3_Loop",
        "Distance_to_L10_Loop",
        "Local_Hydrophobic_Fraction",
        "Local_Charged_Fraction",
        "Local_Polar_Fraction",
        "Local_Sequence_Entropy",
        "Side_Chain_Volume_Change",
        "Total_HBond_Capacity_Change",
    ]
    context_features = [c for c in context_features if c in data.columns]

    rows: List[Dict[str, object]] = []
    for pos, group in data.groupby("__Step8_Position", dropna=True, sort=True):
        row: Dict[str, object] = {
            "Position": int(pos),
            "Mutation_Count": int(len(group)),
        }
        for feature in context_features:
            values = finite_series(group[feature])
            row[f"{feature}_N"] = int(values.notna().sum())
            row[f"{feature}_Coverage"] = float(values.notna().mean())
            row[f"{feature}_Median"] = float(values.median()) if values.notna().any() else np.nan
            row[f"{feature}_IQR"] = (
                float(values.quantile(0.75) - values.quantile(0.25))
                if values.notna().any() else np.nan
            )
        rows.append(row)

    position_df = pd.DataFrame(rows)

    # Add whether the position contains a Step 7 top predictor's mutations only
    # as a contextual flag; no prediction is performed at the position level.
    top_features = set(evidence.loc[evidence["Step8_Top_Feature"], "Feature"])
    position_df["Top_Feature_Count"] = 0
    position_df["Top_Feature_Present"] = False

    if top_features:
        available = [f for f in top_features if f in data.columns]
        if available:
            # A position is flagged when at least one top feature is observed.
            # This is descriptive coverage, not feature importance.
            for idx, pos in position_df["Position"].items():
                g = data.loc[data["__Step8_Position"] == pos]
                count = sum(int(finite_series(g[f]).notna().any()) for f in available)
                position_df.loc[idx, "Top_Feature_Count"] = count
                position_df.loc[idx, "Top_Feature_Present"] = count > 0

    return position_df


# ---------------------------------------------------------------------------
# Audit tables
# ---------------------------------------------------------------------------

def build_coverage_audit(data: pd.DataFrame, evidence: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    features = sorted(evidence["Feature"].unique())

    for feature in features:
        if feature not in data.columns:
            coverage = 0.0
            n = 0
            status = "NOT_PRESENT"
        else:
            s = finite_series(data[feature])
            n = int(s.notna().sum())
            coverage = float(s.notna().mean())
            status = "PASS" if n > 0 else "NO_VALID_VALUES"

        rows.append(
            {
                "Feature": feature,
                "N_Total": int(len(data)),
                "N_Valid": n,
                "Coverage_Fraction": coverage,
                "Coverage_Percent": 100.0 * coverage,
                "Status": status,
                "No_Imputation_Performed": True,
            }
        )

    return pd.DataFrame(rows)


def build_interpretation_audit(evidence: pd.DataFrame) -> pd.DataFrame:
    checks = [
        (
            "Step7_source_of_feature_importance",
            "PASS",
            "Feature ranks and importance values are inherited directly from Step 7.",
        ),
        (
            "No_model_fitting",
            "PASS",
            "Step 8 contains no model fitting, HPO, architecture selection, or prediction step.",
        ),
        (
            "No_target_redefinition",
            "PASS",
            "Targets are read from Step 7 outputs and are not reconstructed in Step 8.",
        ),
        (
            "No_new_folds",
            "PASS",
            "Step 8 creates no train/test folds and does not rescore models.",
        ),
        (
            "No_biological_imputation",
            "PASS",
            "Missing structural/biological values remain missing and are reported through coverage metrics.",
        ),
        (
            "Causal_language_control",
            "PASS",
            "Interpretation language describes predictive association and biological context, not causality.",
        ),
        (
            "H_bond_language_control",
            "PASS",
            "Hydrogen-bond descriptors are reported as potential capacity rather than observed interactions.",
        ),
        (
            "BLOSUM62_language_control",
            "PASS",
            "BLOSUM62 is described as evolutionary substitution context, not direct fitness.",
        ),
        (
            "Step7_schema_compatibility",
            "PASS",
            "Step 8 uses the frozen Step 7 summary schema; absent reporting fields are derived deterministically in Step 8.",
        ),
        (
            "Feature_annotation_completeness",
            "PASS" if (evidence["Annotation_Status"] == "PASS").all() else "REVIEW_REQUIRED",
            "Every Step 7 feature must have an explicit annotation before manuscript-level interpretation.",
        ),
    ]
    return pd.DataFrame(checks, columns=["Audit_Item", "Status", "Evidence"])


# ---------------------------------------------------------------------------
# Publication-quality figures
# ---------------------------------------------------------------------------

def plot_top_feature_heatmap(evidence: pd.DataFrame) -> None:
    print("[6/8] Generating publication-quality biological interpretation figures...")

    top = evidence[evidence["Permutation_Rank"] <= TOP_N_PER_TARGET].copy()
    if top.empty:
        return

    pivot = top.pivot_table(
        index="Feature",
        columns="Target",
        values="Permutation_Importance_Mean",
        aggfunc="first",
    )

    # Keep the most frequently top-ranked features visible while limiting
    # extreme figure dimensions for many targets.
    feature_order = (
        top.groupby("Feature")["Permutation_Rank"]
        .mean()
        .sort_values()
        .index.tolist()
    )
    pivot = pivot.reindex(feature_order)

    fig_w = max(9.0, 0.65 * len(pivot.columns) + 4.0)
    fig_h = max(5.5, 0.34 * len(pivot.index) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(pivot.values, aspect="auto", interpolation="nearest")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=55, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel("Target")
    ax.set_ylabel("Step 7 top feature")
    ax.set_title("Biological Context of Top Predictive Features Across Targets")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Mean permutation importance")
    fig.tight_layout()
    save_figure(fig, FIGURE_DIR / "Figure8A_Top_Feature_Importance_Heatmap")


def plot_category_counts(evidence: pd.DataFrame) -> None:
    top = evidence[evidence["Permutation_Rank"] <= TOP_N_PER_TARGET].copy()
    counts = top["Biological_Category"].value_counts().sort_values(ascending=True)
    if counts.empty:
        return

    fig, ax = plt.subplots(figsize=(8.5, max(4.5, 0.45 * len(counts) + 1.5)))
    ax.barh(counts.index, counts.values)
    ax.set_xlabel("Number of target-feature occurrences")
    ax.set_ylabel("Biological context category")
    ax.set_title("Biological Context Categories Among Step 7 Top Features")
    fig.tight_layout()
    save_figure(fig, FIGURE_DIR / "Figure8B_Biological_Context_Category_Counts")


def plot_importance_stability(evidence: pd.DataFrame) -> None:
    df = evidence.copy()
    df["Permutation_CV"] = [
        safe_div(float(sd), float(mean))
        for sd, mean in zip(df["Permutation_Importance_SD"], df["Permutation_Importance_Mean"])
    ]
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["Permutation_Importance_Mean", "Permutation_CV"])
    if df.empty:
        return

    fig, ax = plt.subplots(figsize=(8.5, 6.0))
    ax.scatter(df["Permutation_CV"], df["Permutation_Importance_Mean"], alpha=0.65, s=28)
    top = df[df["Permutation_Rank"] <= 3]
    for _, r in top.iterrows():
        ax.annotate(
            str(r["Feature"]),
            (r["Permutation_CV"], r["Permutation_Importance_Mean"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )
    ax.set_xlabel("Permutation importance CV (SD / mean)")
    ax.set_ylabel("Mean permutation importance")
    ax.set_title("Predictive Importance Versus Cross-Fold Stability")
    fig.tight_layout()
    save_figure(fig, FIGURE_DIR / "Figure8C_Importance_vs_Stability")


# ---------------------------------------------------------------------------
# Reproducibility and outputs
# ---------------------------------------------------------------------------

def write_outputs(
    data: pd.DataFrame,
    annotations: pd.DataFrame,
    evidence: pd.DataFrame,
    position_df: pd.DataFrame,
    coverage_df: pd.DataFrame,
    audit_df: pd.DataFrame,
) -> None:
    print("[7/8] Writing Step 8 tables, audits, and reproducibility metadata...")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    annotations.to_csv(OUTPUT_DIR / "VIM2_Step8_Feature_Biological_Annotation.csv", index=False)
    evidence.to_csv(OUTPUT_DIR / "VIM2_Step8_Target_Feature_Interpretation.csv", index=False)
    position_df.to_csv(OUTPUT_DIR / "VIM2_Step8_Position_Context_Summary.csv", index=False)
    coverage_df.to_csv(OUTPUT_DIR / "VIM2_Step8_Feature_Coverage_Audit.csv", index=False)
    audit_df.to_csv(OUTPUT_DIR / "VIM2_Step8_Interpretation_Audit.csv", index=False)

    # Concise target-level summary for Results drafting.
    summary_rows: List[Dict[str, object]] = []
    for target, g in evidence.groupby("Target", sort=False):
        g = g.sort_values("Permutation_Rank")
        topg = g.head(TOP_N_PER_TARGET)
        strongest = g.iloc[0]
        summary_rows.append(
            {
                "Target": target,
                "Architecture": str(g["Architecture"].iloc[0]),
                "Model_Family_Mode": str(g["Model_Family_Mode"].iloc[0]),
                "Sequence_Position_Included": bool(g["Sequence_Position_Included"].iloc[0]),
                "Top_Feature": str(strongest["Feature"]),
                "Top_Feature_Biological_Category": str(strongest["Biological_Category"]),
                "Top_Feature_Permutation_Mean": float(strongest["Permutation_Importance_Mean"]),
                "Top_Feature_Permutation_SD": float(strongest["Permutation_Importance_SD"]),
                "Top_Feature_Positive_Fold_Fraction": float(strongest["Permutation_Positive_Fold_Fraction"]),
                "Top_Feature_Evidence_Grade": str(strongest["Evidence_Grade"]),
                "N_Top_Features": int(len(topg)),
                "N_Biological_Categories_Among_Top10": int(topg["Biological_Category"].nunique()),
            }
        )

    pd.DataFrame(summary_rows).to_csv(
        OUTPUT_DIR / "VIM2_Step8_Target_Summary.csv", index=False
    )

    # Hash generated tabular outputs for reproducibility. Exclude the manifest
    # hash table itself to avoid self-referential content.
    sha_path = OUTPUT_DIR / "VIM2_Step8_SHA256.csv"
    hash_rows = []
    for path in sorted(OUTPUT_DIR.glob("*.csv")):
        if path.resolve() == sha_path.resolve():
            continue
        hash_rows.append({"File": path.name, "SHA256": sha256_file(path)})
    pd.DataFrame(hash_rows).to_csv(sha_path, index=False)

    manifest = {
        "project": "VIM-2",
        "step": 8,
        "purpose": "Biological and molecular interpretation of frozen Step 7 feature-importance results",
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "python_version": sys.version,
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "random_seed": RANDOM_SEED,
        "top_n_per_target": TOP_N_PER_TARGET,
        "input_data_configured": str(DATA_FILE),
        "biological_data_source": str(DATA_FILE),
        "biological_data_source_type": "canonical_Part12D",
        "step7_summary": str(STEP7_SUMMARY_FILE),
        "step7_top_features": str(STEP7_TOP_FILE),
        "n_mutations": int(len(data)),
        "n_targets": int(evidence["Target"].nunique()),
        "n_features_interpreted": int(evidence["Feature"].nunique()),
        "no_model_fitting": True,
        "no_hpo": True,
        "no_architecture_selection": True,
        "no_target_redefinition": True,
        "no_new_folds": True,
        "no_biological_imputation": True,
        "step7_is_source_of_ml_importance": True,
        "step7_summary_schema": "Frozen Step 7 schema using Permutation_Rank and Native_Rank; Top_N and Sequence_Position_Included are Step 8-derived reporting fields.",
        "step8_derived_reporting_fields": ["Top_N", "Sequence_Position_Included"],
        "biological_interpretation_is_non_causal": True,
    }
    with (OUTPUT_DIR / "VIM2_Step8_Reproducibility_Manifest.json").open("w", encoding="utf-8") as fh:
        json.dump(manifest, fh, indent=2, ensure_ascii=False)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    print("=" * 78)
    print("VIM-2 STEP 8 — BIOLOGICAL / MOLECULAR INTERPRETATION")
    print("Post-ML, leakage-safe, non-causal, Q1-oriented implementation")
    print("=" * 78)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    data, step7_summary, step7_top = load_inputs()
    annotations = build_feature_annotation(step7_summary, data)
    evidence = merge_step7_with_annotations(step7_summary, annotations, data)

    # Fail immediately if any Step 7 feature lacks a deterministic annotation.
    # This prevents downstream biological summaries/figures from being generated
    # from an incomplete interpretation schema.
    if (evidence["Annotation_Status"] != "PASS").any():
        unknown = sorted(
            evidence.loc[evidence["Annotation_Status"] != "PASS", "Feature"]
            .astype(str)
            .unique()
        )
        fail(f"Unannotated Step 7 features remain: {unknown}")

    evidence = build_target_interpretation(evidence)
    position_df = build_position_context(data, evidence)
    coverage_df = build_coverage_audit(data, evidence)
    audit_df = build_interpretation_audit(evidence)

    plot_top_feature_heatmap(evidence)
    plot_category_counts(evidence)
    plot_importance_stability(evidence)
    write_outputs(data, annotations, evidence, position_df, coverage_df, audit_df)

    print("[8/8] Final validation...")
    required_outputs = [
        "VIM2_Step8_Feature_Biological_Annotation.csv",
        "VIM2_Step8_Target_Feature_Interpretation.csv",
        "VIM2_Step8_Target_Summary.csv",
        "VIM2_Step8_Position_Context_Summary.csv",
        "VIM2_Step8_Feature_Coverage_Audit.csv",
        "VIM2_Step8_Interpretation_Audit.csv",
        "VIM2_Step8_Reproducibility_Manifest.json",
        "VIM2_Step8_SHA256.csv",
    ]
    missing = [name for name in required_outputs if not (OUTPUT_DIR / name).is_file()]
    if missing:
        fail(f"Expected output files were not created: {missing}")

    if not audit_df["Status"].isin(["PASS"]).all():
        fail("One or more Step 8 audit checks did not PASS.")

    print("\nSTEP 8 COMPLETE — all validation checks passed.")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Targets interpreted: {evidence['Target'].nunique()}")
    print(f"Features interpreted: {evidence['Feature'].nunique()}")
    print(f"Mutations/context rows: {len(data)}")
    print("No model fitting, HPO, architecture selection, target redefinition, or biological imputation was performed.")


if __name__ == "__main__":
    main()


VIM-2 STEP 8 — BIOLOGICAL / MOLECULAR INTERPRETATION
Post-ML, leakage-safe, non-causal, Q1-oriented implementation
[1/8] Loading frozen Step 7 interpretation outputs and biological dataset...
[2/8] Building deterministic biological feature annotations...
[3/8] Linking Step 7 evidence to biological context...
[4/8] Constructing target-specific biological interpretation summaries...
[5/8] Quantifying position-level structural/sequence context without imputation...
[6/8] Generating publication-quality biological interpretation figures...
[7/8] Writing Step 8 tables, audits, and reproducibility metadata...
[8/8] Final validation...

STEP 8 COMPLETE — all validation checks passed.
Output directory: /content/VIM2_Step8_Biological_Interpretation
Targets interpreted: 9
Features interpreted: 66
Mutations/context rows: 5016
No model fitting, HPO, architecture selection, target redefinition, or biological imputation was performed.


In [ ]:
# @title

# ============================================================
# DOWNLOAD STEP8 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step8_Biological_Interpretation"

# Output ZIP archive
zip_base = "/content/VIM2_Step8_Biological_Interpretation"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step8_Biological_Interpretation"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP 8 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP 8 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step8_Biological_Interpretation
ZIP archive      : /content/VIM2_Step8_Biological_Interpretation.zip
Archive size     : 1.03 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
VIM-2 ML PROJECT
STEP 9 — ROBUSTNESS / SENSITIVITY ANALYSIS
Q1-ORIENTED / LEAKAGE-SAFE / FROZEN-DESIGN

Purpose
-------
Step 9 evaluates whether the main conclusions from Steps 3–8 are robust to
pre-specified sensitivity analyses.

PRIMARY PRINCIPLES
------------------
1. Position-Aware nested CV remains the primary validation framework.
2. The frozen Part 13 outer folds are reused unchanged.
3. Step 3 hyperparameters are reused unchanged; NO new HPO is performed.
4. Step 6 remains the sole source of the final Model A / Model B architecture
   decision.
5. Step 3 / Step 2 frozen model-family decisions are reused; RF vs Extra Trees
   is NOT re-selected by Step 9.
6. No outer-test information is used for model selection or tuning.
7. Sensitivity analyses are reported as robustness evidence, not as a new
   primary model-selection procedure.
8. No target is redefined.
9. No biological values are imputed.
10. Position-disjoint structure is preserved in every fold-level analysis.
11. Every perturbation is deterministic and reproducible.
12. Fail-fast validation is used for schema, alignment, fold, and hyperparameter
    inconsistencies.

Sensitivity domains
-------------------
A. Fold robustness:
   - fold-level dispersion and leave-one-fold-out summaries.
B. Feature-group robustness:
   - structural / sequence / biochemical / molecular subsets.
   - position is isolated as its own group.
C. Model-choice robustness:
   - compare frozen RF and Extra-Trees behavior without selecting a new family.
D. Input perturbation robustness:
   - deterministic feature-column permutation on the outer test data for
     the frozen final model, quantified by R2 degradation.
E. Position contribution:
   - direct comparison of frozen Model A vs Model B using the same outer folds.
F. Prediction-error robustness:
   - fold-level R2 / RMSE / MAE / Spearman dispersion and outlier-resistant
     summaries.

IMPORTANT
---------
This script does NOT:
- run HPO;
- create new train/test folds;
- change Step 6 architecture selection;
- use Step 5 Random-Split as the primary analysis;
- claim statistical significance from sensitivity scores;
- infer causality.

Expected project inputs
-----------------------
/content/VIM2_Part12H_v2_Final_ML_Dataset.csv
/content/VIM2_Part13_Fold_Assignments.csv
/content/VIM2_Part13_ML_Preparation/VIM2_Part13_Feature_Manifest.csv
/content/VIM2_Step2_PositionAware_Candidate_Selection/
    VIM2_Step2_PositionAware_Selected_Candidates.csv
/content/VIM2_Step3_PositionAware_Nested_HPO/
    VIM2_Step3_Best_Hyperparameters_Per_Fold.csv
/content/VIM2_Step4_Final_Evaluation/
    VIM2_Step4_Final_OuterFold_Performance.csv
    VIM2_Step4_Pooled_OOF_Performance.csv
/content/VIM2_Step6_Statistical_Comparison/
    VIM2_Step6_Final_Model_Selection.csv
/content/VIM2_Step7_Model_Interpretation/
    VIM2_Step7_Feature_Importance_Summary.csv
/content/VIM2_Step8_Biological_Interpretation/
    VIM2_Step8_Position_Context_Summary.csv

Outputs
-------
/content/VIM2_Step9_Robustness_Sensitivity/
    VIM2_Step9_Fold_Robustness.csv
    VIM2_Step9_LeaveOneFoldOut.csv
    VIM2_Step9_FeatureGroup_Robustness.csv
    VIM2_Step9_ModelFamily_Robustness.csv
    VIM2_Step9_Permutation_Sensitivity.csv
    VIM2_Step9_Position_Contribution.csv
    VIM2_Step9_Error_Robustness.csv
    VIM2_Step9_Integrated_Robustness_Summary.csv
    VIM2_Step9_Validation_Audit.csv
    VIM2_Step9_Reproducibility_Manifest.json
    VIM2_Step9_SHA256.csv
    figures/
        Figure9A_Fold_Robustness.png/.pdf/.svg
        Figure9B_FeatureGroup_Robustness.png/.pdf/.svg
        Figure9C_Position_Contribution.png/.pdf/.svg
        Figure9D_Permutation_Sensitivity.png/.pdf/.svg
"""

from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


# ============================================================================
# 1. CONFIGURATION
# ============================================================================

PROJECT_NAME = "VIM-2 ML Project"
STEP_NAME = "STEP 9 — ROBUSTNESS / SENSITIVITY ANALYSIS"

RANDOM_STATE = 42
N_OUTER_FOLDS = 5

MODEL_A = "Model_A_No_Position"
MODEL_B = "Model_B_With_Position"
ARCHITECTURES = [MODEL_A, MODEL_B]
MODEL_FAMILIES = ["Random_Forest", "Extra_Trees"]

POSITION_COLUMN = "Sequence_Position"
FOLD_COLUMN = "PositionAware_Fold"

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

DATA_FILE = Path("/content/VIM2_Part12H_v2_Final_ML_Dataset.csv")
FOLD_FILE = Path("/content/VIM2_Part13_Fold_Assignments.csv")
FEATURE_MANIFEST_FILE = Path(
    "/content/VIM2_Part13_Feature_Manifest.csv"
)

STEP2_FILE = Path(
    "/content/VIM2_Step2_PositionAware_Selected_Candidates.csv"
)
STEP3_FILE = Path(
    "/content/VIM2_Step3_Best_Hyperparameters_Per_Fold.csv"
)
STEP4_FOLD_FILE = Path(
    "/content/VIM2_Step4_Final_OuterFold_Performance.csv"
)
STEP4_POOLED_FILE = Path(
    "/content/VIM2_Step4_Pooled_OOF_Performance.csv"
)
STEP6_SELECTION_FILE = Path(
    "/content/VIM2_Step6_Final_Model_Selection.csv"
)
STEP7_SUMMARY_FILE = Path(
    "/content/VIM2_Step7_Feature_Importance_Summary.csv"
)
STEP8_POSITION_FILE = Path(
    "/content/VIM2_Step8_Position_Context_Summary.csv"
)

OUTPUT_DIR = Path("/content/VIM2_Step9_Robustness_Sensitivity")
FIGURE_DIR = OUTPUT_DIR / "figures"

N_JOBS = -1

# Deterministic perturbation settings.
PERMUTATION_REPEATS = 20
PERMUTATION_SEED_BASE = 91001

# Leave-one-fold-out is descriptive sensitivity only.
MIN_REMAINING_FOLDS = 3

EXPECTED_FEATURE_COUNTS = {
    MODEL_A: 65,
    MODEL_B: 66,
}

# Canonical feature groups.
# These groups are descriptive sensitivity subsets, not new feature-selection
# procedures. Any feature not explicitly assigned is placed in "Other".
FEATURE_GROUPS = {
    "Structural": {
        "Distance_to_Metal_Site",
        "Distance_to_Active_Site_Pocket",
        "Distance_to_L3_Loop",
        "Distance_to_L10_Loop",
        "SASA",
        "Residue_Surface_Depth_Proxy",
        "CA_Contact_Count",
        "CA_Contact_Density",
        "Local_Packing_Density",
        "CA_BFactor",
        "Normalized_CA_BFactor",
        "Local_Flexibility_Relative",
    },
    "Local_Sequence": {
        "Local_Hydrophobic_Fraction",
        "Local_Charged_Fraction",
        "Local_Positive_Charge_Fraction",
        "Local_Negative_Charge_Fraction",
        "Local_Polar_Fraction",
        "Local_Aromatic_Fraction",
        "Local_Gly_Pro_Fraction",
        "Local_Sequence_Entropy",
    },
    "Biochemical": {
        "Hydrophobicity_Change",
        "Charge_Change",
        "Weight_Change",
        "Polarity_Change",
        "BLOSUM62",
    },
    "Side_Chain": {
        "Side_Chain_Volume_Change",
        "Absolute_Side_Chain_Volume_Change",
        "Relative_Side_Chain_Volume_Change",
    },
    "Hydrogen_Bond": {
        "HBond_Donor_Change",
        "HBond_Acceptor_Change",
        "Total_HBond_Capacity_Change",
    },
    "Position": {
        POSITION_COLUMN,
    },
}


# ============================================================================
# 2. GENERIC HELPERS
# ============================================================================

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def require_file(path: Path, label: str) -> None:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"Required {label} is missing or empty:\n{path}"
        )


def fail(message: str) -> None:
    raise RuntimeError(f"STEP 9 VALIDATION ERROR: {message}")


def require_columns(
    df: pd.DataFrame,
    columns: Iterable[str],
    label: str,
) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        fail(f"{label} is missing required columns: {missing}")


def normalize_architecture(value: object) -> str:
    s = str(value).strip().lower().replace("-", "_").replace(" ", "")
    if "model_a" in s or "modela" in s or "no_position" in s:
        return MODEL_A
    if "model_b" in s or "modelb" in s or "with_position" in s:
        return MODEL_B
    raise ValueError(f"Unknown architecture label: {value!r}")


def normalize_model_family(value: object) -> str:
    s = str(value).strip().lower().replace("-", "_").replace(" ", "")
    if "random_forest" in s or "randomforest" in s or s == "rf":
        return "Random_Forest"
    if "extra_trees" in s or "extratrees" in s or s == "et":
        return "Extra_Trees"
    raise ValueError(f"Unknown model-family label: {value!r}")


def normalize_target(value: object) -> str:
    return str(value).strip()


def manifest_bool(series: pd.Series) -> pd.Series:
    values = series.astype(str).str.strip().str.lower()
    true_values = {"true", "1", "yes", "y", "t"}
    false_values = {"false", "0", "no", "n", "f"}
    unknown = set(values.unique()) - true_values - false_values
    if unknown:
        fail(f"Unexpected boolean values in feature manifest: {sorted(unknown)}")
    return values.isin(true_values)


def build_onehot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def safe_float(value: object) -> float:
    try:
        return float(value)
    except Exception:
        return np.nan


def parse_params(value: object) -> Dict[str, Any]:
    if isinstance(value, dict):
        return dict(value)

    text = str(value).strip()
    try:
        obj = ast.literal_eval(text)
    except Exception:
        try:
            obj = json.loads(text)
        except Exception as exc:
            fail(f"Unable to parse frozen hyperparameters: {value!r}; {exc}")

    if not isinstance(obj, dict):
        fail(f"Frozen hyperparameters are not a dictionary: {obj!r}")

    return dict(obj)


def canonical_param_dict(params: Mapping[str, Any]) -> Dict[str, Any]:
    out = {}
    for key, value in params.items():
        k = str(key).strip()
        if k.startswith("model__"):
            k = k[len("model__"):]
        out[k] = value
    return out


def numeric_series(df: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(df[column], errors="coerce")


def finite_or_fail(series: pd.Series, label: str) -> None:
    values = pd.to_numeric(series, errors="coerce").to_numpy(float)
    if not np.isfinite(values).all():
        fail(f"{label} contains missing or non-finite numeric values.")


def metric_bundle(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if len(y_true) == 0:
        return {
            "R2": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan,
            "Spearman": np.nan,
        }

    r2 = r2_score(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    if len(y_true) >= 2 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        sp = spearmanr(y_true, y_pred).statistic
    else:
        sp = np.nan

    return {
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "Spearman": float(sp) if np.isfinite(sp) else np.nan,
    }


def robust_location(values: Sequence[float]) -> float:
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan


def iqr(values: Sequence[float]) -> float:
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    return float(np.percentile(x, 75) - np.percentile(x, 25))


def make_feature_group_map(features: Sequence[str]) -> Dict[str, str]:
    mapping = {}
    for feature in features:
        matches = [
            group
            for group, members in FEATURE_GROUPS.items()
            if feature in members
        ]
        if len(matches) > 1:
            fail(f"Feature belongs to multiple sensitivity groups: {feature}")
        mapping[feature] = matches[0] if matches else "Other"
    return mapping


# ============================================================================
# 3. PREPROCESSING / MODEL RECONSTRUCTION
# ============================================================================

def build_preprocessor(features: Sequence[str]) -> ColumnTransformer:
    features = list(features)

    categorical = [
        f for f in features
        if f == "Secondary_Structure"
    ]
    numeric = [
        f for f in features
        if f != "Secondary_Structure"
    ]

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ]
    )

    transformers = [
        ("numeric", numeric_pipeline, numeric),
    ]

    if categorical:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                ("onehot", build_onehot_encoder()),
            ]
        )
        transformers.append(
            ("categorical", categorical_pipeline, categorical)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


def build_estimator(
    model_family: str,
    params: Mapping[str, Any],
) -> object:
    params = canonical_param_dict(params)

    if model_family == "Random_Forest":
        estimator = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    elif model_family == "Extra_Trees":
        estimator = ExtraTreesRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    else:
        fail(f"Unsupported frozen model family: {model_family}")

    valid = estimator.get_params(deep=False)
    unknown = sorted(set(params) - set(valid))
    if unknown:
        fail(
            f"Frozen Step 3 hyperparameters contain unsupported estimator "
            f"parameters for {model_family}: {unknown}"
        )

    estimator.set_params(**params)
    return estimator


def build_pipeline(
    features: Sequence[str],
    model_family: str,
    params: Mapping[str, Any],
) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(features)),
            ("model", build_estimator(model_family, params)),
        ]
    )


# ============================================================================
# 4. LOAD + VALIDATE FROZEN INPUTS
# ============================================================================

def load_inputs() -> Dict[str, pd.DataFrame]:
    print("=" * 88)
    print("VIM-2 STEP 9 — ROBUSTNESS / SENSITIVITY ANALYSIS")
    print("=" * 88)
    print("Loading frozen project inputs...\n")

    paths = {
        "data": DATA_FILE,
        "folds": FOLD_FILE,
        "manifest": FEATURE_MANIFEST_FILE,
        "step2": STEP2_FILE,
        "step3": STEP3_FILE,
        "step4_fold": STEP4_FOLD_FILE,
        "step4_pooled": STEP4_POOLED_FILE,
        "step6": STEP6_SELECTION_FILE,
        "step7": STEP7_SUMMARY_FILE,
        "step8_position": STEP8_POSITION_FILE,
    }

    for label, path in paths.items():
        require_file(path, label)

    frames = {
        key: pd.read_csv(path, low_memory=False)
        for key, path in paths.items()
    }

    print(f"Dataset                  : {frames['data'].shape}")
    print(f"Frozen outer folds       : {frames['folds'].shape}")
    print(f"Feature manifest         : {frames['manifest'].shape}")
    print(f"Step 2 frozen selection  : {frames['step2'].shape}")
    print(f"Step 3 frozen HPO        : {frames['step3'].shape}")
    print(f"Step 4 fold performance  : {frames['step4_fold'].shape}")
    print(f"Step 4 pooled performance: {frames['step4_pooled'].shape}")
    print(f"Step 6 final selection   : {frames['step6'].shape}")
    print(f"Step 7 importance        : {frames['step7'].shape}")
    print(f"Step 8 position context  : {frames['step8_position'].shape}")
    print()

    return frames


def validate_dataset(
    df: pd.DataFrame,
    folds: pd.DataFrame,
    manifest: pd.DataFrame,
) -> Dict[str, List[str]]:
    print("[1/9] Validating dataset, frozen folds, and feature manifest...")

    require_columns(
        df,
        ["Mutation", POSITION_COLUMN] + TARGETS,
        "ML dataset",
    )

    require_columns(
        folds,
        [
            "Row_Index",
            "Mutation",
            "WT_AA",
            POSITION_COLUMN,
            "Mutant_AA",
            FOLD_COLUMN,
        ],
        "Part 13 fold assignments",
    )

    if len(df) != len(folds):
        fail(
            f"Dataset/fold row count mismatch: {len(df)} vs {len(folds)}"
        )

    if not np.array_equal(
        folds["Row_Index"].to_numpy(),
        np.arange(len(df)),
    ):
        fail("Part 13 Row_Index does not exactly match dataset row order.")

    for col in ["Mutation", POSITION_COLUMN]:
        if not np.array_equal(
            folds[col].astype(str).to_numpy(),
            df[col].astype(str).to_numpy(),
        ):
            fail(f"Part 13 column '{col}' is not aligned with dataset.")

    observed_folds = sorted(
        pd.to_numeric(
            folds[FOLD_COLUMN],
            errors="raise",
        ).astype(int).unique().tolist()
    )
    if observed_folds != list(range(1, N_OUTER_FOLDS + 1)):
        fail(f"Unexpected outer fold labels: {observed_folds}")

    if folds[FOLD_COLUMN].isna().any():
        fail("Frozen outer-fold assignments contain missing fold IDs.")

    require_columns(
        manifest,
        [
            "Feature",
            "Used_in_Model_A_No_Position",
            "Used_in_Model_B_With_Position",
        ],
        "Part 13 feature manifest",
    )

    features_a = manifest.loc[
        manifest_bool(manifest["Used_in_Model_A_No_Position"]),
        "Feature",
    ].astype(str).tolist()

    features_b = manifest.loc[
        manifest_bool(manifest["Used_in_Model_B_With_Position"]),
        "Feature",
    ].astype(str).tolist()

    if len(features_a) != EXPECTED_FEATURE_COUNTS[MODEL_A]:
        fail(
            f"Model A feature count must be 65; found {len(features_a)}."
        )

    if len(features_b) != EXPECTED_FEATURE_COUNTS[MODEL_B]:
        fail(
            f"Model B feature count must be 66; found {len(features_b)}."
        )

    if POSITION_COLUMN in features_a:
        fail("Sequence_Position is incorrectly present in Model A.")

    if POSITION_COLUMN not in features_b:
        fail("Sequence_Position is missing from Model B.")

    if set(features_b) != set(features_a) | {POSITION_COLUMN}:
        fail("Model B is not exactly Model A + Sequence_Position.")

    missing_data_features = sorted(
        set(features_b) - set(df.columns)
    )
    if missing_data_features:
        fail(
            "Manifest features missing from ML dataset: "
            f"{missing_data_features}"
        )

    if df[POSITION_COLUMN].isna().any():
        fail("Sequence_Position contains missing values.")

    architectures = {
        MODEL_A: features_a,
        MODEL_B: features_b,
    }

    print("[PASS] Dataset/fold row alignment validated.")
    print("[PASS] Frozen Position-Aware folds = 5.")
    print("[PASS] Model A = 65 predictors.")
    print("[PASS] Model B = 66 predictors.")
    print("[PASS] Model B = Model A + Sequence_Position.\n")

    return architectures


def validate_step2(
    step2: pd.DataFrame,
) -> pd.DataFrame:
    print("[2/9] Validating Step 2 frozen model-family selections...")

    require_columns(
        step2,
        ["Target", "Architecture", "Outer_Fold", "Selected_Model"],
        "Step 2 frozen selection",
    )

    out = step2.copy()
    out["Target"] = out["Target"].map(normalize_target)
    out["Architecture"] = out["Architecture"].map(normalize_architecture)
    out["Outer_Fold"] = pd.to_numeric(
        out["Outer_Fold"],
        errors="raise",
    ).astype(int)
    out["Selected_Model"] = out["Selected_Model"].map(
        normalize_model_family
    )

    expected = len(TARGETS) * len(ARCHITECTURES) * N_OUTER_FOLDS
    if len(out) != expected:
        fail(
            f"Step 2 selection must contain {expected} rows; "
            f"found {len(out)}."
        )

    key = ["Target", "Architecture", "Outer_Fold"]
    if out.duplicated(key).any():
        fail("Step 2 contains duplicate Target × Architecture × Outer_Fold.")

    if set(out["Target"]) != set(TARGETS):
        fail("Step 2 target coverage does not match the frozen 9 targets.")

    if set(out["Architecture"]) != set(ARCHITECTURES):
        fail("Step 2 architecture coverage is incomplete.")

    if set(out["Outer_Fold"]) != set(range(1, N_OUTER_FOLDS + 1)):
        fail("Step 2 outer-fold IDs are not exactly 1..5.")

    print("[PASS] Step 2 frozen RF/Extra-Trees decisions validated.\n")
    return out


def validate_step3(
    step3: pd.DataFrame,
) -> pd.DataFrame:
    print("[3/9] Validating Step 3 frozen hyperparameters...")

    # Step 3's actual frozen output uses the column name "Model".
    # "Selected_Model" belongs to the Step 2 frozen model-family table.
    # Accept the canonical Step 3 schema and normalize it internally to
    # "Selected_Model" so the downstream Step 2 ↔ Step 3 audit remains
    # explicit and deterministic.
    required = [
        "Target",
        "Architecture",
        "Outer_Fold",
        "Model",
        "Best_Hyperparameters",
    ]
    require_columns(step3, required, "Step 3 frozen HPO output")

    out = step3.copy()
    out = out.rename(columns={"Model": "Selected_Model"})
    out["Target"] = out["Target"].map(normalize_target)
    out["Architecture"] = out["Architecture"].map(normalize_architecture)
    out["Outer_Fold"] = pd.to_numeric(
        out["Outer_Fold"],
        errors="raise",
    ).astype(int)
    out["Selected_Model"] = out["Selected_Model"].map(
        normalize_model_family
    )

    expected = len(TARGETS) * len(ARCHITECTURES) * N_OUTER_FOLDS
    if len(out) != expected:
        fail(
            f"Step 3 frozen HPO output must contain {expected} rows; "
            f"found {len(out)}."
        )

    key = ["Target", "Architecture", "Outer_Fold"]
    if out.duplicated(key).any():
        fail("Step 3 contains duplicate Target × Architecture × Outer_Fold.")

    if set(out["Target"]) != set(TARGETS):
        fail("Step 3 target coverage is incomplete or incorrect.")

    if set(out["Architecture"]) != set(ARCHITECTURES):
        fail("Step 3 architecture coverage is incomplete.")

    if set(out["Outer_Fold"]) != set(range(1, N_OUTER_FOLDS + 1)):
        fail("Step 3 outer-fold IDs are not exactly 1..5.")

    if out["Best_Hyperparameters"].astype(str).str.strip().eq("").any():
        fail("Step 3 contains empty Best_Hyperparameters values.")

    if "Best_Inner_R2" in out.columns:
        out["Best_Inner_R2"] = pd.to_numeric(
            out["Best_Inner_R2"],
            errors="raise",
        )
        if not np.isfinite(out["Best_Inner_R2"]).all():
            fail("Step 3 contains non-finite Best_Inner_R2 values.")

    # Ensure Step 3 family agrees with Step 2 at every frozen configuration.
    print("[PASS] Step 3 frozen hyperparameters are structurally complete.\n")
    return out


def validate_step2_step3_agreement(
    step2: pd.DataFrame,
    step3: pd.DataFrame,
) -> None:
    print("[4/9] Checking Step 2 ↔ Step 3 frozen family agreement...")

    key = ["Target", "Architecture", "Outer_Fold"]

    a = step2[key + ["Selected_Model"]].rename(
        columns={"Selected_Model": "Step2_Model"}
    )
    b = step3[key + ["Selected_Model"]].rename(
        columns={"Selected_Model": "Step3_Model"}
    )

    merged = a.merge(b, on=key, how="outer", validate="one_to_one")

    if merged.isna().any().any():
        fail("Step 2 and Step 3 configuration keys do not align.")

    mismatch = merged[
        merged["Step2_Model"] != merged["Step3_Model"]
    ]
    if not mismatch.empty:
        fail(
            "Step 2 / Step 3 frozen model-family mismatch:\n"
            + mismatch.to_string(index=False)
        )

    print("[PASS] Step 2 and Step 3 model-family decisions agree exactly.\n")


def validate_step4_step6_step7_step8(
    step4_fold: pd.DataFrame,
    step4_pooled: pd.DataFrame,
    step6: pd.DataFrame,
    step7: pd.DataFrame,
    step8_position: pd.DataFrame,
) -> None:
    print("[5/9] Validating frozen downstream evidence from Steps 4–8...")

    require_columns(
        step4_fold,
        ["Target", "Architecture", "Outer_Fold", "R2", "RMSE", "MAE"],
        "Step 4 fold performance",
    )
    require_columns(
        step4_pooled,
        ["Target", "Architecture", "OOF_R2", "OOF_RMSE", "OOF_MAE"],
        "Step 4 pooled performance",
    )
    require_columns(
        step6,
        ["Target"],
        "Step 6 final selection",
    )
    require_columns(
        step7,
        [
            "Target",
            "Architecture",
            "Feature",
            "Permutation_Rank",
            "Permutation_Importance_Mean",
        ],
        "Step 7 feature-importance summary",
    )
    require_columns(
        step8_position,
        ["Position"],
        "Step 8 position-context summary",
    )

    fold = step4_fold.copy()
    fold["Architecture"] = fold["Architecture"].map(normalize_architecture)
    fold["Target"] = fold["Target"].map(normalize_target)
    fold["Outer_Fold"] = pd.to_numeric(
        fold["Outer_Fold"], errors="raise"
    ).astype(int)

    expected_fold_rows = len(TARGETS) * 2 * N_OUTER_FOLDS
    if len(fold) != expected_fold_rows:
        fail(
            f"Step 4 fold performance must contain {expected_fold_rows} rows; "
            f"found {len(fold)}."
        )

    expected_keys = pd.MultiIndex.from_product(
        [TARGETS, ARCHITECTURES, range(1, N_OUTER_FOLDS + 1)],
        names=["Target", "Architecture", "Outer_Fold"],
    )
    observed_keys = pd.MultiIndex.from_frame(
        fold[["Target", "Architecture", "Outer_Fold"]]
    )
    if not expected_keys.equals(observed_keys.sort_values()):
        fail("Step 4 fold performance does not contain exactly the expected Target × Architecture × Outer_Fold grid.")

    if fold.duplicated(
        ["Target", "Architecture", "Outer_Fold"]
    ).any():
        fail("Step 4 fold performance contains duplicate configurations.")

    for col in ["R2", "RMSE", "MAE"]:
        finite_or_fail(fold[col], f"Step 4 {col}")

    pooled = step4_pooled.copy()
    pooled["Architecture"] = pooled["Architecture"].map(
        normalize_architecture
    )
    pooled["Target"] = pooled["Target"].map(normalize_target)

    expected_pooled_rows = len(TARGETS) * 2
    if len(pooled) != expected_pooled_rows:
        fail(
            f"Step 4 pooled performance must contain {expected_pooled_rows} "
            f"rows; found {len(pooled)}."
        )

    expected_pooled_keys = pd.MultiIndex.from_product(
        [TARGETS, ARCHITECTURES],
        names=["Target", "Architecture"],
    )
    observed_pooled_keys = pd.MultiIndex.from_frame(
        pooled[["Target", "Architecture"]]
    )
    if not expected_pooled_keys.equals(observed_pooled_keys.sort_values()):
        fail("Step 4 pooled performance does not contain exactly the expected Target × Architecture grid.")

    if pooled.duplicated(["Target", "Architecture"]).any():
        fail("Step 4 pooled performance contains duplicate configurations.")

    for col in ["OOF_R2", "OOF_RMSE", "OOF_MAE"]:
        finite_or_fail(pooled[col], f"Step 4 {col}")

    # Step 6 must contain exactly one final architecture per target.
    step6_target_cols = [
        c for c in [
            "Final_Architecture",
            "Selected_Architecture",
            "Architecture_Selected",
        ]
        if c in step6.columns
    ]
    if not step6_target_cols:
        fail(
            "Step 6 final-selection file has no recognized architecture "
            "selection column."
        )

    selection_col = step6_target_cols[0]
    s6 = step6.copy()
    s6["Target"] = s6["Target"].map(normalize_target)
    s6["Final_Architecture_Normalized"] = s6[selection_col].map(
        normalize_architecture
    )

    if set(s6["Target"]) != set(TARGETS):
        fail("Step 6 final-selection target coverage is incomplete.")

    if s6["Target"].duplicated().any():
        fail("Step 6 must contain exactly one final architecture per target.")

    # Step 7 source is checked for valid targets/architectures/ranks.
    s7 = step7.copy()
    s7["Target"] = s7["Target"].map(normalize_target)
    s7["Architecture"] = s7["Architecture"].map(normalize_architecture)
    s7["Permutation_Rank"] = pd.to_numeric(
        s7["Permutation_Rank"], errors="raise"
    )
    s7["Permutation_Importance_Mean"] = pd.to_numeric(
        s7["Permutation_Importance_Mean"], errors="coerce"
    )
    if s7["Permutation_Rank"].isna().any():
        fail("Step 7 contains missing Permutation_Rank values.")
    if s7["Permutation_Importance_Mean"].isna().any():
        fail("Step 7 contains missing/non-numeric Permutation_Importance_Mean values.")
    if not set(s7["Target"]).issubset(set(TARGETS)):
        fail("Step 7 contains targets outside the frozen target list.")
    if not set(s7["Architecture"]).issubset(set(ARCHITECTURES)):
        fail("Step 7 contains architectures outside the frozen architecture list.")
    if s7.duplicated(["Target", "Architecture", "Feature"]).any():
        fail("Step 7 contains duplicate Target × Architecture × Feature rows.")

    # Step 8 must retain explicit positions; Step 9 never synthesizes them.
    if pd.to_numeric(
        step8_position["Position"],
        errors="coerce",
    ).isna().all():
        fail("Step 8 position-context file contains no usable positions.")

    print("[PASS] Step 4 fold/pooled schemas validated.")
    print("[PASS] Step 6 contains one frozen architecture per target.")
    print("[PASS] Step 7 feature-importance schema validated.")
    print("[PASS] Step 8 position context contains explicit positions.\n")


# ============================================================================
# 5. FROZEN MODEL-FAMILY / PARAMETER RECOVERY
# ============================================================================

def make_frozen_configuration_table(
    step2: pd.DataFrame,
    step3: pd.DataFrame,
) -> pd.DataFrame:
    merged = step3.copy()

    # Step 2 agreement has already been validated.
    merged["Frozen_Model_Family"] = merged["Selected_Model"]
    merged["Frozen_Hyperparameters"] = merged["Best_Hyperparameters"].map(
        parse_params
    )
    merged["Canonical_Hyperparameters"] = merged[
        "Frozen_Hyperparameters"
    ].map(canonical_param_dict)

    return merged[
        [
            "Target",
            "Architecture",
            "Outer_Fold",
            "Frozen_Model_Family",
            "Frozen_Hyperparameters",
            "Canonical_Hyperparameters",
        ]
    ].copy()


# ============================================================================
# 6. FOLD ROBUSTNESS
# ============================================================================

def compute_fold_robustness(
    step4_fold: pd.DataFrame,
    step6: pd.DataFrame,
) -> pd.DataFrame:
    records = []

    s6_col = next(
        c for c in [
            "Final_Architecture",
            "Selected_Architecture",
            "Architecture_Selected",
        ]
        if c in step6.columns
    )
    final_arch = (
        step6[["Target", s6_col]]
        .copy()
        .assign(
            Target=lambda x: x["Target"].map(normalize_target),
            Final_Architecture=lambda x: x[s6_col].map(
                normalize_architecture
            ),
        )
        [["Target", "Final_Architecture"]]
    )

    fold = step4_fold.copy()
    fold["Target"] = fold["Target"].map(normalize_target)
    fold["Architecture"] = fold["Architecture"].map(normalize_architecture)
    fold["Outer_Fold"] = pd.to_numeric(
        fold["Outer_Fold"], errors="raise"
    ).astype(int)

    for _, sel in final_arch.iterrows():
        target = sel["Target"]
        architecture = sel["Final_Architecture"]

        sub = fold[
            (fold["Target"] == target)
            & (fold["Architecture"] == architecture)
        ].sort_values("Outer_Fold")

        if len(sub) != N_OUTER_FOLDS:
            fail(
                f"Final architecture {target}/{architecture} does not have "
                f"exactly 5 Step 4 outer folds."
            )

        for metric in ["R2", "RMSE", "MAE"]:
            values = pd.to_numeric(sub[metric], errors="coerce").to_numpy(float)
            records.append(
                {
                    "Target": target,
                    "Architecture": architecture,
                    "Metric": metric,
                    "N_Folds": len(values),
                    "Mean": float(np.mean(values)),
                    "SD": float(np.std(values, ddof=1)),
                    "Median": float(np.median(values)),
                    "IQR": iqr(values),
                    "Min": float(np.min(values)),
                    "Max": float(np.max(values)),
                    "CV": (
                        float(np.std(values, ddof=1) / abs(np.mean(values)))
                        if abs(np.mean(values)) > 1e-12
                        else np.nan
                    ),
                }
            )

    return pd.DataFrame(records)


def compute_leave_one_fold_out(
    step4_fold: pd.DataFrame,
    step6: pd.DataFrame,
) -> pd.DataFrame:
    s6_col = next(
        c for c in [
            "Final_Architecture",
            "Selected_Architecture",
            "Architecture_Selected",
        ]
        if c in step6.columns
    )

    selections = step6[["Target", s6_col]].copy()
    selections["Target"] = selections["Target"].map(normalize_target)
    selections["Architecture"] = selections[s6_col].map(
        normalize_architecture
    )

    fold = step4_fold.copy()
    fold["Target"] = fold["Target"].map(normalize_target)
    fold["Architecture"] = fold["Architecture"].map(normalize_architecture)
    fold["Outer_Fold"] = pd.to_numeric(
        fold["Outer_Fold"], errors="raise"
    ).astype(int)

    records = []

    for _, row in selections.iterrows():
        target = row["Target"]
        architecture = row["Architecture"]

        sub = fold[
            (fold["Target"] == target)
            & (fold["Architecture"] == architecture)
        ].sort_values("Outer_Fold")

        for omitted in range(1, N_OUTER_FOLDS + 1):
            remain = sub[sub["Outer_Fold"] != omitted]

            if len(remain) < MIN_REMAINING_FOLDS:
                fail(
                    f"Leave-one-fold-out would retain fewer than "
                    f"{MIN_REMAINING_FOLDS} folds."
                )

            records.append(
                {
                    "Target": target,
                    "Architecture": architecture,
                    "Omitted_Outer_Fold": omitted,
                    "Remaining_Folds": len(remain),
                    "R2_Mean": float(remain["R2"].mean()),
                    "R2_Median": float(remain["R2"].median()),
                    "RMSE_Mean": float(remain["RMSE"].mean()),
                    "MAE_Mean": float(remain["MAE"].mean()),
                }
            )

    return pd.DataFrame(records)


# ============================================================================
# 7. FEATURE-GROUP ROBUSTNESS
# ============================================================================

def compute_feature_group_robustness(
    architectures: Dict[str, List[str]],
    step7: pd.DataFrame,
) -> pd.DataFrame:
    """
    Quantify whether the frozen Step 7 top-feature signal is concentrated in
    one biological/feature family.

    This is intentionally based on Step 7 importance, not on refitting.
    It therefore cannot introduce a new model-selection step.
    """
    s7 = step7.copy()
    s7["Target"] = s7["Target"].map(normalize_target)
    s7["Architecture"] = s7["Architecture"].map(normalize_architecture)
    s7["Feature"] = s7["Feature"].astype(str).str.strip()

    records = []

    for architecture, features in architectures.items():
        group_map = make_feature_group_map(features)

        sub = s7[s7["Architecture"] == architecture].copy()
        if sub.empty:
            continue

        sub["Sensitivity_Group"] = sub["Feature"].map(
            lambda x: group_map.get(x, "Not_in_manifest")
        )

        for target in TARGETS:
            t = sub[sub["Target"] == target].copy()
            if t.empty:
                continue

            total_abs = np.nansum(
                np.abs(
                    pd.to_numeric(
                        t["Permutation_Importance_Mean"],
                        errors="coerce",
                    ).to_numpy(float)
                )
            )

            for group, g in t.groupby("Sensitivity_Group"):
                imp = pd.to_numeric(
                    g["Permutation_Importance_Mean"],
                    errors="coerce",
                ).to_numpy(float)
                ranks = pd.to_numeric(
                    g["Permutation_Rank"],
                    errors="coerce",
                ).to_numpy(float)

                records.append(
                    {
                        "Target": target,
                        "Architecture": architecture,
                        "Feature_Group": group,
                        "N_Features": int(len(g)),
                        "Permutation_Importance_Sum": float(
                            np.nansum(imp)
                        ),
                        "Permutation_Importance_Absolute_Share": (
                            float(np.nansum(np.abs(imp)) / total_abs)
                            if total_abs > 0
                            else np.nan
                        ),
                        "Best_Permutation_Rank": (
                            float(np.nanmin(ranks))
                            if len(ranks)
                            else np.nan
                        ),
                        "Median_Permutation_Rank": (
                            float(np.nanmedian(ranks))
                            if len(ranks)
                            else np.nan
                        ),
                    }
                )

    return pd.DataFrame(records)


# ============================================================================
# 8. MODEL-FAMILY ROBUSTNESS
# ============================================================================

def compute_model_family_robustness(
    frozen_config: pd.DataFrame,
    step4_fold: pd.DataFrame,
) -> pd.DataFrame:
    """
    Summarize the frozen Step 3 family actually used in each configuration and
    compare family-level performance descriptively.

    No family is selected here.
    """
    fold = step4_fold.copy()
    fold["Target"] = fold["Target"].map(normalize_target)
    fold["Architecture"] = fold["Architecture"].map(normalize_architecture)
    fold["Outer_Fold"] = pd.to_numeric(
        fold["Outer_Fold"], errors="raise"
    ).astype(int)

    if "Model_Family" in fold.columns:
        fold["Model_Family"] = fold["Model_Family"].map(
            normalize_model_family
        )
    else:
        # Recover the family from frozen Step 3, configuration by configuration.
        fold = fold.merge(
            frozen_config[
                [
                    "Target",
                    "Architecture",
                    "Outer_Fold",
                    "Frozen_Model_Family",
                ]
            ],
            on=["Target", "Architecture", "Outer_Fold"],
            how="left",
            validate="one_to_one",
        )
        fold["Model_Family"] = fold["Frozen_Model_Family"]

    # A Target × Architecture configuration always has exactly 5 outer folds,
    # but the frozen Step 3 family may legitimately differ across those folds.
    # Therefore an individual family can occur on 1–5 folds. Requiring five
    # folds per family would incorrectly reject valid frozen selections.
    config_key = ["Target", "Architecture", "Outer_Fold"]
    if fold.duplicated(config_key).any():
        fail(
            "Step 4 fold table contains duplicate Target × Architecture × Outer_Fold "
            "rows before model-family robustness aggregation."
        )

    config_counts = (
        fold.groupby(["Target", "Architecture"])
        .size()
        .reset_index(name="N_Folds")
    )
    bad_configs = config_counts[
        config_counts["N_Folds"] != N_OUTER_FOLDS
    ]
    if not bad_configs.empty:
        fail(
            "Step 4 fold table does not contain exactly 5 outer folds for every "
            "Target × Architecture configuration:\n"
            + bad_configs.to_string(index=False)
        )

    records = []

    for (target, architecture, family), sub in fold.groupby(
        ["Target", "Architecture", "Model_Family"]
    ):
        if sub["Outer_Fold"].nunique() != len(sub):
            fail(
                f"Duplicate outer-fold observations in frozen family configuration "
                f"{target}/{architecture}/{family}."
            )

        records.append(
            {
                "Target": target,
                "Architecture": architecture,
                "Frozen_Model_Family": family,
                "N_Outer_Folds": len(sub),
                "Mean_R2": float(sub["R2"].mean()),
                "SD_R2": float(sub["R2"].std(ddof=1)),
                "Median_R2": float(sub["R2"].median()),
                "Mean_RMSE": float(sub["RMSE"].mean()),
                "Mean_MAE": float(sub["MAE"].mean()),
                "Interpretation": (
                    "Descriptive robustness evidence only; Step 9 does not "
                    "re-select RF vs Extra Trees."
                ),
            }
        )

    return pd.DataFrame(records)


# ============================================================================
# 9. TRUE FROZEN-MODEL PERMUTATION SENSITIVITY
# ============================================================================

def run_permutation_sensitivity(
    data: pd.DataFrame,
    folds: pd.DataFrame,
    architectures: Dict[str, List[str]],
    frozen_config: pd.DataFrame,
    step6: pd.DataFrame,
) -> pd.DataFrame:
    """
    Reconstruct each frozen Step 3 model on its outer-training partition,
    evaluate on the untouched outer test partition, then permute ONE predictor
    at a time on the outer test data.

    IMPORTANT PERFORMANCE IMPLEMENTATION
    ------------------------------------
    The statistical procedure is unchanged from Step 9 v5:
      * same frozen Step 3 model family and hyperparameters;
      * same frozen Step 6 architecture;
      * same outer folds;
      * same 20 deterministic permutation repeats;
      * same raw-column permutation seeds.

    To avoid tens of thousands of repeated pipeline transformations, the fitted
    preprocessor transforms the untouched outer-test set ONCE. Because the
    preprocessing is row-wise and fitted only on the outer-training partition,
    permuting a raw predictor is exactly equivalent to permuting that
    predictor's transformed column block (including its missingness indicator
    and, for Secondary_Structure, its one-hot block). Twenty repeats for one
    feature are then predicted in a single batched estimator call.
    """
    print("[6/9] Running deterministic outer-test permutation sensitivity...")
    print(
        "       Frozen family + frozen Step 3 hyperparameters; "
        "no HPO is performed."
    )
    print(
        "       FAST MODE: preprocessing once + batched 20-repeat prediction "
        "per feature."
    )

    s6_col = next(
        c for c in [
            "Final_Architecture",
            "Selected_Architecture",
            "Architecture_Selected",
        ]
        if c in step6.columns
    )
    final_arch = step6[["Target", s6_col]].copy()
    final_arch["Target"] = final_arch["Target"].map(normalize_target)
    final_arch["Architecture"] = final_arch[s6_col].map(
        normalize_architecture
    )

    fold_assign = folds[FOLD_COLUMN].astype(int).to_numpy()
    records = []

    for target_index, target in enumerate(TARGETS):
        selected_arch = final_arch.loc[
            final_arch["Target"] == target,
            "Architecture",
        ].iloc[0]

        features = architectures[selected_arch]

        for outer_fold in range(1, N_OUTER_FOLDS + 1):
            fold_start = time.perf_counter()

            train_mask = fold_assign != outer_fold
            test_mask = fold_assign == outer_fold

            train = data.loc[train_mask, features + [target]].copy()
            test = data.loc[test_mask, features + [target]].copy()

            frozen = frozen_config[
                (frozen_config["Target"] == target)
                & (frozen_config["Architecture"] == selected_arch)
                & (frozen_config["Outer_Fold"] == outer_fold)
            ]

            if len(frozen) != 1:
                fail(
                    f"Missing unique frozen Step 3 configuration for "
                    f"{target}/{selected_arch}/fold{outer_fold}."
                )

            frozen_row = frozen.iloc[0]
            family = frozen_row["Frozen_Model_Family"]
            params = frozen_row["Canonical_Hyperparameters"]

            # Match Step 4: rows with missing target values are excluded
            # from the relevant training/test partition. No target value is
            # imputed. Feature missingness remains handled by the pipeline.
            train_valid = pd.to_numeric(train[target], errors="coerce").notna()
            test_valid = pd.to_numeric(test[target], errors="coerce").notna()

            train_eval = train.loc[train_valid].copy()
            test_eval = test.loc[test_valid].copy()

            y_train = pd.to_numeric(
                train_eval[target], errors="coerce"
            ).to_numpy(float)
            y_test = pd.to_numeric(
                test_eval[target], errors="coerce"
            ).to_numpy(float)

            if len(train_eval) == 0 or len(test_eval) == 0:
                fail(
                    f"Empty valid train/test set for {target} | fold "
                    f"{outer_fold} after excluding missing target values."
                )

            if not np.isfinite(y_train).all() or not np.isfinite(y_test).all():
                fail(
                    f"Non-finite target values remain for {target} in fold "
                    f"{outer_fold} after target-validity filtering."
                )

            model = build_pipeline(features, family, params)
            model.fit(train_eval[features], y_train)

            # Transform the untouched test partition exactly once. All later
            # permutations operate directly in this fitted feature space.
            preprocessor = model.named_steps["preprocessor"]
            estimator = model.named_steps["model"]
            X_test_transformed = np.asarray(
                preprocessor.transform(test_eval[features]),
                dtype=float,
            )

            baseline_pred = estimator.predict(X_test_transformed)
            baseline = metric_bundle(y_test, baseline_pred)

            # Build exact transformed-column blocks for each raw predictor.
            # Numeric preprocessing = median imputation + missing indicator +
            # scaling, all row-wise. Secondary_Structure = one-hot encoding,
            # also row-wise. Therefore shuffling the transformed block with
            # the same row permutation is mathematically equivalent to
            # shuffling the corresponding raw predictor before preprocessing.
            numeric_features = [
                f for f in features if f != "Secondary_Structure"
            ]
            categorical_features = [
                f for f in features if f == "Secondary_Structure"
            ]

            numeric_imputer = preprocessor.named_transformers_[
                "numeric"
            ].named_steps["imputer"]
            missing_numeric_indices = (
                list(numeric_imputer.indicator_.features_)
                if getattr(numeric_imputer, "indicator_", None) is not None
                else []
            )
            n_numeric = len(numeric_features)
            n_indicator = len(missing_numeric_indices)
            categorical_start = n_numeric + n_indicator

            transformed_blocks = {}
            for feature_index, feature in enumerate(features):
                if feature == "Secondary_Structure":
                    if not categorical_features:
                        fail(
                            "Internal transformed-feature mapping error: "
                            "Secondary_Structure is present but categorical "
                            "feature list is empty."
                        )
                    cols = np.arange(
                        categorical_start,
                        X_test_transformed.shape[1],
                        dtype=int,
                    )
                else:
                    numeric_index = numeric_features.index(feature)
                    cols_list = [numeric_index]
                    if numeric_index in missing_numeric_indices:
                        indicator_offset = missing_numeric_indices.index(
                            numeric_index
                        )
                        cols_list.append(n_numeric + indicator_offset)
                    cols = np.asarray(cols_list, dtype=int)

                if len(cols) == 0 or np.any(cols >= X_test_transformed.shape[1]):
                    fail(
                        f"Could not map transformed columns for feature "
                        f"{feature} ({target}, fold {outer_fold})."
                    )
                transformed_blocks[feature] = cols

            n_test = len(test_eval)
            total_features = len(features)

            for feature_index, feature in enumerate(features):
                feature_start = time.perf_counter()
                cols = transformed_blocks[feature]
                batch = np.empty(
                    (
                        PERMUTATION_REPEATS,
                        n_test,
                        X_test_transformed.shape[1],
                    ),
                    dtype=float,
                )
                batch[:] = X_test_transformed

                seeds = []
                for permutation_repeat in range(1, PERMUTATION_REPEATS + 1):
                    permutation_seed = (
                        PERMUTATION_SEED_BASE
                        + target_index * 1_000_000
                        + outer_fold * 10_000
                        + feature_index * 100
                        + permutation_repeat
                    )
                    rng = np.random.default_rng(permutation_seed)
                    permutation_index = rng.permutation(n_test)
                    # Use row-slice assignment here. NumPy advanced indexing
                    # with batch[repeat, :, cols] reorders the indexed axes and
                    # produces shape (n_cols, n_test), causing a broadcast
                    # mismatch for single-column features.
                    batch[permutation_repeat - 1][:, cols] = (
                        X_test_transformed[permutation_index][:, cols]
                    )
                    seeds.append(permutation_seed)

                # One estimator call for all 20 repeats of this feature.
                perm_pred_all = estimator.predict(
                    batch.reshape(
                        PERMUTATION_REPEATS * n_test,
                        X_test_transformed.shape[1],
                    )
                ).reshape(PERMUTATION_REPEATS, n_test)

                for permutation_repeat, permutation_seed in enumerate(
                    seeds, start=1
                ):
                    perm_metrics = metric_bundle(
                        y_test,
                        perm_pred_all[permutation_repeat - 1],
                    )
                    records.append(
                        {
                            "Target": target,
                            "Architecture": selected_arch,
                            "Outer_Fold": outer_fold,
                            "Model_Family": family,
                            "Feature": feature,
                            "Permutation_Repeat": permutation_repeat,
                            "Baseline_R2": baseline["R2"],
                            "Permuted_R2": perm_metrics["R2"],
                            "R2_Degradation": baseline["R2"] - perm_metrics["R2"],
                            "Baseline_RMSE": baseline["RMSE"],
                            "Permuted_RMSE": perm_metrics["RMSE"],
                            "RMSE_Increase": (
                                perm_metrics["RMSE"] - baseline["RMSE"]
                            ),
                            "Baseline_MAE": baseline["MAE"],
                            "Permuted_MAE": perm_metrics["MAE"],
                            "MAE_Increase": (
                                perm_metrics["MAE"] - baseline["MAE"]
                            ),
                            "Baseline_Spearman": baseline["Spearman"],
                            "Permuted_Spearman": perm_metrics["Spearman"],
                            "Permutation_Seed": permutation_seed,
                        }
                    )

                elapsed = time.perf_counter() - feature_start
                print(
                    f"       {target} | fold {outer_fold}/{N_OUTER_FOLDS} | "
                    f"feature {feature_index + 1}/{total_features}: "
                    f"{feature} | {elapsed:.1f}s"
                )

            fold_elapsed = time.perf_counter() - fold_start
            print(
                f"       [FOLD PASS] {target} | fold {outer_fold}/"
                f"{N_OUTER_FOLDS} | {len(features)} features × "
                f"{PERMUTATION_REPEATS} repeats | {fold_elapsed / 60:.2f} min"
            )

    out = pd.DataFrame(records)

    if out.empty:
        fail("Permutation sensitivity produced no records.")

    expected_records = sum(
        len(architectures[
            final_arch.loc[final_arch["Target"] == target, "Architecture"].iloc[0]
        ])
        * N_OUTER_FOLDS
        * PERMUTATION_REPEATS
        for target in TARGETS
    )
    if len(out) != expected_records:
        fail(
            f"Permutation record count mismatch: got {len(out):,}, "
            f"expected {expected_records:,}."
        )

    print(
        f"[PASS] Generated {len(out):,} feature-level outer-test "
        "permutation records."
    )
    return out

# ============================================================================
# 10. POSITION CONTRIBUTION
# ============================================================================

def compute_position_contribution(
    step4_fold: pd.DataFrame,
    step4_pooled: pd.DataFrame,
    step6: pd.DataFrame,
) -> pd.DataFrame:
    s6_col = next(
        c for c in [
            "Final_Architecture",
            "Selected_Architecture",
            "Architecture_Selected",
        ]
        if c in step6.columns
    )

    selections = step6[["Target", s6_col]].copy()
    selections["Target"] = selections["Target"].map(normalize_target)
    selections["Selected_Architecture"] = selections[s6_col].map(
        normalize_architecture
    )

    fold = step4_fold.copy()
    fold["Target"] = fold["Target"].map(normalize_target)
    fold["Architecture"] = fold["Architecture"].map(normalize_architecture)
    fold["Outer_Fold"] = pd.to_numeric(
        fold["Outer_Fold"], errors="raise"
    ).astype(int)

    pooled = step4_pooled.copy()
    pooled["Target"] = pooled["Target"].map(normalize_target)
    pooled["Architecture"] = pooled["Architecture"].map(
        normalize_architecture
    )

    records = []

    for _, sel in selections.iterrows():
        target = sel["Target"]
        selected = sel["Selected_Architecture"]

        a = fold[
            (fold["Target"] == target)
            & (fold["Architecture"] == MODEL_A)
        ].sort_values("Outer_Fold")

        b = fold[
            (fold["Target"] == target)
            & (fold["Architecture"] == MODEL_B)
        ].sort_values("Outer_Fold")

        pa = pooled[
            (pooled["Target"] == target)
            & (pooled["Architecture"] == MODEL_A)
        ]
        pb = pooled[
            (pooled["Target"] == target)
            & (pooled["Architecture"] == MODEL_B)
        ]

        if len(a) != N_OUTER_FOLDS or len(b) != N_OUTER_FOLDS:
            fail(f"Incomplete A/B pairing for {target}.")

        if len(pa) != 1 or len(pb) != 1:
            fail(f"Missing pooled A/B results for {target}.")

        delta_fold = (
            b["R2"].to_numpy(float)
            - a["R2"].to_numpy(float)
        )

        records.append(
            {
                "Target": target,
                "Selected_Architecture": selected,
                "Model_A_Pooled_OOF_R2": float(pa.iloc[0]["OOF_R2"]),
                "Model_B_Pooled_OOF_R2": float(pb.iloc[0]["OOF_R2"]),
                "Pooled_Delta_B_minus_A_R2": (
                    float(pb.iloc[0]["OOF_R2"] - pa.iloc[0]["OOF_R2"])
                ),
                "Mean_Fold_Delta_B_minus_A_R2": float(
                    np.mean(delta_fold)
                ),
                "SD_Fold_Delta_B_minus_A_R2": float(
                    np.std(delta_fold, ddof=1)
                ),
                "N_Folds_Better": int(np.sum(delta_fold > 0)),
                "N_Folds_Worse": int(np.sum(delta_fold < 0)),
                "N_Folds_Tied": int(np.sum(np.isclose(delta_fold, 0.0))),
                "Position_Contribution_Conclusion": (
                    "Position-aware architecture selected by frozen Step 6"
                    if selected == MODEL_B
                    else "No additional Sequence_Position contribution "
                         "under the frozen Step 6 decision"
                ),
            }
        )

    return pd.DataFrame(records)


# ============================================================================
# 11. ERROR ROBUSTNESS
# ============================================================================

def compute_error_robustness(
    step4_fold: pd.DataFrame,
    step6: pd.DataFrame,
) -> pd.DataFrame:
    s6_col = next(
        c for c in [
            "Final_Architecture",
            "Selected_Architecture",
            "Architecture_Selected",
        ]
        if c in step6.columns
    )

    selections = step6[["Target", s6_col]].copy()
    selections["Target"] = selections["Target"].map(normalize_target)
    selections["Architecture"] = selections[s6_col].map(
        normalize_architecture
    )

    fold = step4_fold.copy()
    fold["Target"] = fold["Target"].map(normalize_target)
    fold["Architecture"] = fold["Architecture"].map(normalize_architecture)
    fold["Outer_Fold"] = pd.to_numeric(
        fold["Outer_Fold"], errors="raise"
    ).astype(int)

    records = []

    for _, sel in selections.iterrows():
        target = sel["Target"]
        architecture = sel["Architecture"]

        sub = fold[
            (fold["Target"] == target)
            & (fold["Architecture"] == architecture)
        ].sort_values("Outer_Fold")

        for metric in ["R2", "RMSE", "MAE", "Spearman"]:
            if metric not in sub.columns:
                if metric == "Spearman":
                    continue
                fail(f"Step 4 missing metric required for robustness: {metric}")

            values = pd.to_numeric(
                sub[metric], errors="coerce"
            ).to_numpy(float)

            records.append(
                {
                    "Target": target,
                    "Architecture": architecture,
                    "Metric": metric,
                    "Mean": float(np.nanmean(values)),
                    "Median": float(np.nanmedian(values)),
                    "SD": float(np.nanstd(values, ddof=1)),
                    "IQR": iqr(values),
                    "Minimum": float(np.nanmin(values)),
                    "Maximum": float(np.nanmax(values)),
                    "Worst_Fold_Value": (
                        float(np.nanmin(values))
                        if metric in ["R2", "Spearman"]
                        else float(np.nanmax(values))
                    ),
                }
            )

    return pd.DataFrame(records)


# ============================================================================
# 12. INTEGRATED ROBUSTNESS SUMMARY
# ============================================================================

def compute_integrated_summary(
    fold_df: pd.DataFrame,
    loo_df: pd.DataFrame,
    group_df: pd.DataFrame,
    perm_df: pd.DataFrame,
    position_df: pd.DataFrame,
) -> pd.DataFrame:
    records = []

    for target in TARGETS:
        fold_t = fold_df[fold_df["Target"] == target]
        loo_t = loo_df[loo_df["Target"] == target]
        group_t = group_df[group_df["Target"] == target]
        perm_t = perm_df[perm_df["Target"] == target]
        pos_t = position_df[position_df["Target"] == target]

        r2_row = fold_t[fold_t["Metric"] == "R2"]
        if r2_row.empty:
            continue
        r2_row = r2_row.iloc[0]

        loo_values = pd.to_numeric(
            loo_t["R2_Mean"], errors="coerce"
        ).to_numpy(float)

        perm_degrad = pd.to_numeric(
            perm_t["R2_Degradation"], errors="coerce"
        ).to_numpy(float)

        positive_perm_fraction = (
            float(np.mean(perm_degrad > 0))
            if len(perm_degrad)
            else np.nan
        )

        strongest_group = "None"
        strongest_share = np.nan
        if not group_t.empty:
            g = group_t.sort_values(
                "Permutation_Importance_Absolute_Share",
                ascending=False,
            ).iloc[0]
            strongest_group = str(g["Feature_Group"])
            strongest_share = safe_float(
                g["Permutation_Importance_Absolute_Share"]
            )

        position_selected = (
            str(pos_t.iloc[0]["Selected_Architecture"])
            if not pos_t.empty
            else np.nan
        )

        robustness_flags = []

        if safe_float(r2_row["CV"]) <= 0.75:
            robustness_flags.append("stable_fold_R2")
        else:
            robustness_flags.append("heterogeneous_fold_R2")

        if len(loo_values):
            loo_range = float(
                np.nanmax(loo_values) - np.nanmin(loo_values)
            )
            if loo_range <= 0.15:
                robustness_flags.append("stable_leave_one_fold_out")
            else:
                robustness_flags.append("sensitive_to_fold_omission")

        if np.isfinite(positive_perm_fraction):
            if positive_perm_fraction >= 0.75:
                robustness_flags.append("permutation_supported")
            else:
                robustness_flags.append("permutation_mixed")

        if np.isfinite(strongest_share) and strongest_share >= 0.50:
            robustness_flags.append("group_concentrated")
        else:
            robustness_flags.append("group_distributed")

        records.append(
            {
                "Target": target,
                "Final_Architecture": position_selected,
                "OuterFold_R2_Mean": safe_float(r2_row["Mean"]),
                "OuterFold_R2_SD": safe_float(r2_row["SD"]),
                "OuterFold_R2_CV": safe_float(r2_row["CV"]),
                "LeaveOneFoldOut_R2_Median": (
                    robust_location(loo_values)
                    if len(loo_values)
                    else np.nan
                ),
                "LeaveOneFoldOut_R2_IQR": (
                    iqr(loo_values)
                    if len(loo_values)
                    else np.nan
                ),
                "Permutation_Positive_Fraction": positive_perm_fraction,
                "Median_R2_Degradation": (
                    robust_location(perm_degrad)
                    if len(perm_degrad)
                    else np.nan
                ),
                "Strongest_Feature_Group": strongest_group,
                "Strongest_Group_Importance_Share": strongest_share,
                "Robustness_Flags": ";".join(robustness_flags),
                "Scientific_Interpretation": (
                    "Robustness evidence is supportive/descriptive and does "
                    "not replace the primary Position-Aware nested-CV results."
                ),
            }
        )

    return pd.DataFrame(records)


# ============================================================================
# 13. FIGURES
# ============================================================================

def save_figure(fig: plt.Figure, stem: str) -> None:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIGURE_DIR / f"{stem}.png",
        dpi=600,
        bbox_inches="tight",
    )
    fig.savefig(
        FIGURE_DIR / f"{stem}.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        FIGURE_DIR / f"{stem}.svg",
        bbox_inches="tight",
    )
    plt.close(fig)


def plot_fold_robustness(fold_df: pd.DataFrame) -> None:
    sub = fold_df[fold_df["Metric"] == "R2"].copy()
    if sub.empty:
        return

    pivot = sub.pivot(
        index="Target",
        columns="Architecture",
        values="Mean",
    )

    fig, ax = plt.subplots(
        figsize=(12, 7),
        constrained_layout=True,
    )
    pivot.plot(kind="bar", ax=ax)
    ax.axhline(0, linewidth=0.8)
    ax.set_title("Step 9A — Outer-Fold R² Robustness")
    ax.set_xlabel("Phenotype")
    ax.set_ylabel("Mean Position-Aware outer-fold R²")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Architecture")
    save_figure(fig, "Figure9A_Fold_Robustness")


def plot_feature_group_robustness(group_df: pd.DataFrame) -> None:
    if group_df.empty:
        return

    top = (
        group_df.groupby("Feature_Group")[
            "Permutation_Importance_Absolute_Share"
        ]
        .mean()
        .sort_values(ascending=False)
    )

    fig, ax = plt.subplots(
        figsize=(10, 6),
        constrained_layout=True,
    )
    top.plot(kind="bar", ax=ax)
    ax.set_title("Step 9B — Feature-Group Importance Distribution")
    ax.set_xlabel("Feature group")
    ax.set_ylabel("Mean absolute permutation-importance share")
    ax.tick_params(axis="x", rotation=35)
    save_figure(fig, "Figure9B_FeatureGroup_Robustness")


def plot_position_contribution(position_df: pd.DataFrame) -> None:
    if position_df.empty:
        return

    sub = position_df.set_index("Target")[
        "Pooled_Delta_B_minus_A_R2"
    ]

    fig, ax = plt.subplots(
        figsize=(12, 6),
        constrained_layout=True,
    )
    sub.plot(kind="bar", ax=ax)
    ax.axhline(0, linewidth=0.8)
    ax.set_title(
        "Step 9C — Sequence-Position Contribution "
        "(Model B − Model A)"
    )
    ax.set_xlabel("Phenotype")
    ax.set_ylabel("Pooled Position-Aware ΔR²")
    ax.tick_params(axis="x", rotation=45)
    save_figure(fig, "Figure9C_Position_Contribution")


def plot_permutation_sensitivity(perm_df: pd.DataFrame) -> None:
    if perm_df.empty:
        return

    top = (
        perm_df.groupby("Feature")["R2_Degradation"]
        .mean()
        .sort_values(ascending=False)
        .head(15)
        .sort_values()
    )

    fig, ax = plt.subplots(
        figsize=(10, 8),
        constrained_layout=True,
    )
    top.plot(kind="barh", ax=ax)
    ax.axvline(0, linewidth=0.8)
    ax.set_title(
        "Step 9D — Most Consistently Sensitive Predictors"
    )
    ax.set_xlabel("Mean outer-test R² degradation after permutation")
    ax.set_ylabel("Feature")
    save_figure(fig, "Figure9D_Permutation_Sensitivity")


# ============================================================================
# 14. VALIDATION AUDIT
# ============================================================================

def build_validation_audit(
    architectures: Dict[str, List[str]],
    frozen_config: pd.DataFrame,
    step4_fold: pd.DataFrame,
    step6: pd.DataFrame,
    perm_df: pd.DataFrame,
    step7: pd.DataFrame,
    step8_position: pd.DataFrame,
) -> pd.DataFrame:
    checks = []

    def add(name: str, status: str, detail: str) -> None:
        checks.append(
            {
                "Check": name,
                "Status": status,
                "Detail": detail,
            }
        )

    add(
        "Primary_validation_framework",
        "PASS",
        "Frozen Position-Aware 5-fold outer partitions retained.",
    )

    add(
        "No_new_HPO",
        "PASS",
        "Step 9 reconstructs models only from frozen Step 3 hyperparameters.",
    )

    add(
        "No_architecture_reselection",
        "PASS",
        "Step 6 remains the sole final architecture-selection source.",
    )

    add(
        "Model_A_feature_count",
        "PASS" if len(architectures[MODEL_A]) == 65 else "FAIL",
        f"Observed {len(architectures[MODEL_A])}; expected 65.",
    )

    add(
        "Model_B_feature_count",
        "PASS" if len(architectures[MODEL_B]) == 66 else "FAIL",
        f"Observed {len(architectures[MODEL_B])}; expected 66.",
    )

    add(
        "Frozen_configuration_count",
        "PASS"
        if len(frozen_config)
        == len(TARGETS) * 2 * N_OUTER_FOLDS
        else "FAIL",
        f"Observed {len(frozen_config)}; expected 90.",
    )

    add(
        "Step4_fold_count",
        "PASS"
        if len(step4_fold)
        == len(TARGETS) * 2 * N_OUTER_FOLDS
        else "FAIL",
        f"Observed {len(step4_fold)}; expected 90.",
    )

    add(
        "Step6_target_selection_count",
        "PASS"
        if len(step6) == len(TARGETS)
        else "FAIL",
        f"Observed {len(step6)}; expected 9.",
    )

    add(
        "Permutation_output_nonempty",
        "PASS" if not perm_df.empty else "FAIL",
        f"Generated {len(perm_df):,} records.",
    )

    add(
        "Step7_source_present",
        "PASS" if not step7.empty else "FAIL",
        "Step 7 remains the frozen source for feature-importance evidence.",
    )

    add(
        "Step8_position_context_present",
        "PASS" if not step8_position.empty else "FAIL",
        "Step 8 position context is available for biological interpretation.",
    )

    add(
        "No_biological_imputation",
        "PASS",
        "Step 9 does not impute biological/structural measurements.",
    )

    add(
        "Sensitivity_is_supporting_only",
        "PASS",
        "Step 9 sensitivity results are not used to alter Step 6 decisions.",
    )

    return pd.DataFrame(checks)


# ============================================================================
# 15. REPRODUCIBILITY
# ============================================================================

def write_manifest(
    frozen_config: pd.DataFrame,
    audit: pd.DataFrame,
) -> None:
    input_paths = [
        DATA_FILE,
        FOLD_FILE,
        FEATURE_MANIFEST_FILE,
        STEP2_FILE,
        STEP3_FILE,
        STEP4_FOLD_FILE,
        STEP4_POOLED_FILE,
        STEP6_SELECTION_FILE,
        STEP7_SUMMARY_FILE,
        STEP8_POSITION_FILE,
    ]

    manifest = {
        "project": PROJECT_NAME,
        "step": STEP_NAME,
        "created_at_utc": utc_now(),
        "python_version": sys.version,
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "scikit_learn_version": sklearn.__version__,
        "random_state": RANDOM_STATE,
        "outer_folds": N_OUTER_FOLDS,
        "permutation_repeats": PERMUTATION_REPEATS,
        "permutation_seed_base": PERMUTATION_SEED_BASE,
        "targets": TARGETS,
        "architectures": ARCHITECTURES,
        "expected_feature_counts": EXPECTED_FEATURE_COUNTS,
        "sensitivity_domains": [
            "fold_robustness",
            "leave_one_fold_out",
            "feature_group_robustness",
            "frozen_model_family_robustness",
            "outer_test_permutation_sensitivity",
            "sequence_position_contribution",
            "error_robustness",
        ],
        "hard_constraints": {
            "position_aware_primary": True,
            "new_hpo": False,
            "new_folds": False,
            "architecture_reselection": False,
            "target_redefinition": False,
            "biological_imputation": False,
            "step6_remains_architecture_source": True,
            "step3_remains_hyperparameter_source": True,
            "step7_remains_importance_source": True,
            "step9_results_are_supporting_only": True,
        },
        "frozen_configuration_records": int(len(frozen_config)),
        "validation_all_pass": bool(
            audit["Status"].eq("PASS").all()
        ),
        "input_files": {
            str(path): sha256_file(path)
            for path in input_paths
            if path.is_file()
        },
    }

    with (
        OUTPUT_DIR / "VIM2_Step9_Reproducibility_Manifest.json"
    ).open("w", encoding="utf-8") as fh:
        json.dump(manifest, fh, indent=2, ensure_ascii=False)


def clean_previous_outputs() -> None:
    """Remove only Step 9 artifacts that this script owns, preventing stale-file contamination."""
    known_files = [
        "VIM2_Step9_Fold_Robustness.csv",
        "VIM2_Step9_LeaveOneFoldOut.csv",
        "VIM2_Step9_FeatureGroup_Robustness.csv",
        "VIM2_Step9_ModelFamily_Robustness.csv",
        "VIM2_Step9_Permutation_Sensitivity.csv",
        "VIM2_Step9_Position_Contribution.csv",
        "VIM2_Step9_Error_Robustness.csv",
        "VIM2_Step9_Integrated_Robustness_Summary.csv",
        "VIM2_Step9_Validation_Audit.csv",
        "VIM2_Step9_Reproducibility_Manifest.json",
        "VIM2_Step9_SHA256.csv",
    ]
    figure_stems = [
        "Figure9A_Fold_Robustness",
        "Figure9B_FeatureGroup_Robustness",
        "Figure9C_Position_Contribution",
        "Figure9D_Permutation_Sensitivity",
    ]
    for name in known_files:
        path = OUTPUT_DIR / name
        if path.exists():
            path.unlink()
    for stem in figure_stems:
        for suffix in [".png", ".pdf", ".svg"]:
            path = FIGURE_DIR / f"{stem}{suffix}"
            if path.exists():
                path.unlink()


def write_sha256() -> None:
    expected = [
        OUTPUT_DIR / "VIM2_Step9_Fold_Robustness.csv",
        OUTPUT_DIR / "VIM2_Step9_LeaveOneFoldOut.csv",
        OUTPUT_DIR / "VIM2_Step9_FeatureGroup_Robustness.csv",
        OUTPUT_DIR / "VIM2_Step9_ModelFamily_Robustness.csv",
        OUTPUT_DIR / "VIM2_Step9_Permutation_Sensitivity.csv",
        OUTPUT_DIR / "VIM2_Step9_Position_Contribution.csv",
        OUTPUT_DIR / "VIM2_Step9_Error_Robustness.csv",
        OUTPUT_DIR / "VIM2_Step9_Integrated_Robustness_Summary.csv",
        OUTPUT_DIR / "VIM2_Step9_Validation_Audit.csv",
        OUTPUT_DIR / "VIM2_Step9_Reproducibility_Manifest.json",
        FIGURE_DIR / "Figure9A_Fold_Robustness.png",
        FIGURE_DIR / "Figure9A_Fold_Robustness.pdf",
        FIGURE_DIR / "Figure9A_Fold_Robustness.svg",
        FIGURE_DIR / "Figure9B_FeatureGroup_Robustness.png",
        FIGURE_DIR / "Figure9B_FeatureGroup_Robustness.pdf",
        FIGURE_DIR / "Figure9B_FeatureGroup_Robustness.svg",
        FIGURE_DIR / "Figure9C_Position_Contribution.png",
        FIGURE_DIR / "Figure9C_Position_Contribution.pdf",
        FIGURE_DIR / "Figure9C_Position_Contribution.svg",
        FIGURE_DIR / "Figure9D_Permutation_Sensitivity.png",
        FIGURE_DIR / "Figure9D_Permutation_Sensitivity.pdf",
        FIGURE_DIR / "Figure9D_Permutation_Sensitivity.svg",
    ]
    files = sorted(p for p in expected if p.is_file())

    rows = [
        {
            "File": str(p.relative_to(OUTPUT_DIR)),
            "SHA256": sha256_file(p),
            "Size_Bytes": p.stat().st_size,
        }
        for p in files
    ]

    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / "VIM2_Step9_SHA256.csv",
        index=False,
    )


# ============================================================================
# 16. MAIN
# ============================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    clean_previous_outputs()

    frames = load_inputs()

    data = frames["data"]
    folds = frames["folds"]
    manifest = frames["manifest"]

    architectures = validate_dataset(
        data,
        folds,
        manifest,
    )

    step2 = validate_step2(frames["step2"])
    step3 = validate_step3(frames["step3"])

    validate_step2_step3_agreement(
        step2,
        step3,
    )

    validate_step4_step6_step7_step8(
        frames["step4_fold"],
        frames["step4_pooled"],
        frames["step6"],
        frames["step7"],
        frames["step8_position"],
    )

    frozen_config = make_frozen_configuration_table(
        step2,
        step3,
    )

    print("[PASS] Frozen Step 3 configurations recovered.\n")

    print("[7/9] Computing fold / leave-one-fold-out robustness...")
    fold_robustness = compute_fold_robustness(
        frames["step4_fold"],
        frames["step6"],
    )
    loo = compute_leave_one_fold_out(
        frames["step4_fold"],
        frames["step6"],
    )
    print("[PASS] Fold robustness summaries completed.\n")

    print("[8/9] Computing feature-group, family, position, and error robustness...")
    group_robustness = compute_feature_group_robustness(
        architectures,
        frames["step7"],
    )
    family_robustness = compute_model_family_robustness(
        frozen_config,
        frames["step4_fold"],
    )
    position_contribution = compute_position_contribution(
        frames["step4_fold"],
        frames["step4_pooled"],
        frames["step6"],
    )
    error_robustness = compute_error_robustness(
        frames["step4_fold"],
        frames["step6"],
    )
    print("[PASS] Supporting robustness summaries completed.\n")

    permutation_sensitivity = run_permutation_sensitivity(
        data,
        folds,
        architectures,
        frozen_config,
        frames["step6"],
    )

    integrated = compute_integrated_summary(
        fold_robustness,
        loo,
        group_robustness,
        permutation_sensitivity,
        position_contribution,
    )

    print("[9/9] Writing figures, audits, and reproducibility outputs...")

    plot_fold_robustness(fold_robustness)
    plot_feature_group_robustness(group_robustness)
    plot_position_contribution(position_contribution)
    plot_permutation_sensitivity(permutation_sensitivity)

    audit = build_validation_audit(
        architectures,
        frozen_config,
        frames["step4_fold"],
        frames["step6"],
        permutation_sensitivity,
        frames["step7"],
        frames["step8_position"],
    )

    if not audit["Status"].eq("PASS").all():
        failed = audit.loc[
            audit["Status"] != "PASS"
        ].to_string(index=False)
        fail(f"One or more validation checks failed:\n{failed}")

    fold_robustness.to_csv(
        OUTPUT_DIR / "VIM2_Step9_Fold_Robustness.csv",
        index=False,
    )
    loo.to_csv(
        OUTPUT_DIR / "VIM2_Step9_LeaveOneFoldOut.csv",
        index=False,
    )
    group_robustness.to_csv(
        OUTPUT_DIR / "VIM2_Step9_FeatureGroup_Robustness.csv",
        index=False,
    )
    family_robustness.to_csv(
        OUTPUT_DIR / "VIM2_Step9_ModelFamily_Robustness.csv",
        index=False,
    )
    permutation_sensitivity.to_csv(
        OUTPUT_DIR / "VIM2_Step9_Permutation_Sensitivity.csv",
        index=False,
    )
    position_contribution.to_csv(
        OUTPUT_DIR / "VIM2_Step9_Position_Contribution.csv",
        index=False,
    )
    error_robustness.to_csv(
        OUTPUT_DIR / "VIM2_Step9_Error_Robustness.csv",
        index=False,
    )
    integrated.to_csv(
        OUTPUT_DIR / "VIM2_Step9_Integrated_Robustness_Summary.csv",
        index=False,
    )
    audit.to_csv(
        OUTPUT_DIR / "VIM2_Step9_Validation_Audit.csv",
        index=False,
    )

    write_manifest(
        frozen_config,
        audit,
    )
    write_sha256()

    print("[PASS] All Step 9 validation checks passed.")
    print()
    print("=" * 88)
    print("STEP 9 COMPLETED SUCCESSFULLY")
    print("=" * 88)
    print(f"Output directory: {OUTPUT_DIR}")
    print()
    print("Primary interpretation:")
    print(
        "Step 9 provides robustness/sensitivity evidence around the frozen "
        "Position-Aware pipeline; it does not replace Steps 4–6."
    )
    print()
    print("Generated:")
    for path in sorted(
        p for p in OUTPUT_DIR.rglob("*")
        if p.is_file()
    ):
        print(f"  - {path}")
    print("=" * 88)


if __name__ == "__main__":
    main()


VIM-2 STEP 9 — ROBUSTNESS / SENSITIVITY ANALYSIS
Loading frozen project inputs...

Dataset                  : (5016, 89)
Frozen outer folds       : (5016, 7)
Feature manifest         : (91, 7)
Step 2 frozen selection  : (90, 11)
Step 3 frozen HPO        : (90, 14)
Step 4 fold performance  : (90, 15)
Step 4 pooled performance: (18, 8)
Step 6 final selection   : (9, 22)
Step 7 importance        : (591, 17)
Step 8 position context  : (266, 48)

[1/9] Validating dataset, frozen folds, and feature manifest...
[PASS] Dataset/fold row alignment validated.
[PASS] Frozen Position-Aware folds = 5.
[PASS] Model A = 65 predictors.
[PASS] Model B = 66 predictors.
[PASS] Model B = Model A + Sequence_Position.

[2/9] Validating Step 2 frozen model-family selections...
[PASS] Step 2 frozen RF/Extra-Trees decisions validated.

[3/9] Validating Step 3 frozen hyperparameters...
[PASS] Step 3 frozen hyperparameters are structurally complete.

[4/9] Checking Step 2 ↔ Step 3 frozen family agreement...
[PASS

In [ ]:
# @title

# ============================================================
# DOWNLOAD STEP 9 OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step9_Robustness_Sensitivity"

# Output ZIP archive
zip_base = "/content/VIM2_Step9_Robustness_Sensitivity"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step9_Robustness_Sensitivity"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP 9 DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP 9 DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step9_Robustness_Sensitivity
ZIP archive      : /content/VIM2_Step9_Robustness_Sensitivity.zip
Archive size     : 5.52 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
# ============================================================
# VIM-2 STEP 9 — PROFESSIONAL ROBUSTNESS RESULT AUDIT
# Reads already-generated Step 9 outputs and gives:
#   1) target-level verdict
#   2) detailed robustness diagnostics
#   3) red/yellow/green classification
#   4) overall project-level assessment
#   5) exportable CSV reports
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

STEP9_DIR = Path("/content")

INTEGRATED = STEP9_DIR / "VIM2_Step9_Integrated_Robustness_Summary.csv"
FOLD = STEP9_DIR / "VIM2_Step9_Fold_Robustness.csv"
LOO = STEP9_DIR / "VIM2_Step9_LeaveOneFoldOut.csv"
GROUP = STEP9_DIR / "VIM2_Step9_FeatureGroup_Robustness.csv"
MODEL = STEP9_DIR / "VIM2_Step9_ModelFamily_Robustness.csv"
PERM = STEP9_DIR / "VIM2_Step9_Permutation_Sensitivity.csv"
POSITION = STEP9_DIR / "VIM2_Step9_Position_Contribution.csv"
ERROR = STEP9_DIR / "VIM2_Step9_Error_Robustness.csv"

OUTPUT = STEP9_DIR / "VIM2_Step9_Professional_Result_Audit"

OUTPUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

required = {
    "Integrated": INTEGRATED,
    "Fold": FOLD,
    "LeaveOneFoldOut": LOO,
    "FeatureGroup": GROUP,
    "ModelFamily": MODEL,
    "Permutation": PERM,
    "Position": POSITION,
    "Error": ERROR,
}

print("=" * 90)
print("VIM-2 STEP 9 — PROFESSIONAL ROBUSTNESS RESULT AUDIT")
print("=" * 90)

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"\nMissing Step 9 output:\n{name}\n{path}"
        )

print("\nAll Step 9 output files found.\n")

integrated = pd.read_csv(INTEGRATED)
fold = pd.read_csv(FOLD)
loo = pd.read_csv(LOO)
group = pd.read_csv(GROUP)
model = pd.read_csv(MODEL)
perm = pd.read_csv(PERM)
position = pd.read_csv(POSITION)
error = pd.read_csv(ERROR)

print(f"Integrated summary : {integrated.shape}")
print(f"Fold robustness    : {fold.shape}")
print(f"LOO robustness     : {loo.shape}")
print(f"Feature groups     : {group.shape}")
print(f"Model family       : {model.shape}")
print(f"Permutation        : {perm.shape}")
print(f"Position           : {position.shape}")
print(f"Error robustness   : {error.shape}")

# ------------------------------------------------------------
# 3. TARGET ORDER
# ------------------------------------------------------------

TARGETS = [
    "0.031ug/mL_MEM_37C",
    "0.5ug/mL_CTX_37C",
    "128ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_25C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_25C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
]

LABELS = {
    "0.031ug/mL_MEM_37C": "MEM 0.031",
    "0.5ug/mL_CTX_37C": "CTX 0.5",
    "128ug/mL_AMP_25C": "AMP 128 / 25°C",
    "128ug/mL_AMP_37C": "AMP 128 / 37°C",
    "16ug/mL_AMP_25C": "AMP 16 / 25°C",
    "16ug/mL_AMP_37C": "AMP 16 / 37°C",
    "2ug/mL_AMP_25C": "AMP 2 / 25°C",
    "2ug/mL_AMP_37C": "AMP 2 / 37°C",
    "4ug/mL_CTX_37C": "CTX 4",
}

# ------------------------------------------------------------
# 4. NUMERIC CLEANUP
# ------------------------------------------------------------

numeric_cols = [
    "OuterFold_R2_Mean",
    "OuterFold_R2_SD",
    "OuterFold_R2_CV",
    "LeaveOneFoldOut_R2_Median",
    "LeaveOneFoldOut_R2_IQR",
    "Permutation_Positive_Fraction",
    "Median_R2_Degradation",
    "Strongest_Group_Importance_Share",
]

for col in numeric_cols:
    if col in integrated.columns:
        integrated[col] = pd.to_numeric(
            integrated[col], errors="coerce"
        )

# ------------------------------------------------------------
# 5. RE-COMPUTE EXPLICIT ROBUSTNESS FLAGS
# ------------------------------------------------------------

def classify_row(row):

    cv = row["OuterFold_R2_CV"]
    loo_iqr = row["LeaveOneFoldOut_R2_IQR"]
    perm_frac = row["Permutation_Positive_Fraction"]
    degradation = row["Median_R2_Degradation"]

    flags = []

    # Fold stability
    if pd.notna(cv):
        if cv <= 0.75:
            flags.append("FOLD_STABLE")
        else:
            flags.append("FOLD_HETEROGENEOUS")

    # LOO stability
    # Use the actual integrated LOO spread if available.
    target = row["Target"]

    loo_target = loo[
        loo["Target"].astype(str) == str(target)
    ]

    if not loo_target.empty and "R2_Mean" in loo_target.columns:
        vals = pd.to_numeric(
            loo_target["R2_Mean"], errors="coerce"
        ).dropna()

        if len(vals):
            loo_range = vals.max() - vals.min()

            if loo_range <= 0.15:
                flags.append("LOO_STABLE")
            else:
                flags.append("LOO_SENSITIVE")

    # Permutation support
    if pd.notna(perm_frac):
        if perm_frac >= 0.75:
            flags.append("PERMUTATION_SUPPORTED")
        else:
            flags.append("PERMUTATION_MIXED")

    # Feature-group concentration
    share = row["Strongest_Group_Importance_Share"]

    if pd.notna(share):
        if share >= 0.50:
            flags.append("GROUP_CONCENTRATED")
        else:
            flags.append("GROUP_DISTRIBUTED")

    # --------------------------------------------------------
    # Overall classification
    # --------------------------------------------------------

    positive = sum(
        x in flags
        for x in [
            "FOLD_STABLE",
            "LOO_STABLE",
            "PERMUTATION_SUPPORTED",
            "GROUP_DISTRIBUTED",
        ]
    )

    negative = sum(
        x in flags
        for x in [
            "FOLD_HETEROGENEOUS",
            "LOO_SENSITIVE",
            "PERMUTATION_MIXED",
        ]
    )

    # Strong robustness
    if positive >= 3 and negative == 0:
        verdict = "STRONGLY ROBUST"
        grade = "GREEN"

    # Acceptable robustness
    elif positive >= 2 and negative <= 1:
        verdict = "ACCEPTABLY ROBUST"
        grade = "GREEN"

    # Intermediate
    elif positive >= 1:
        verdict = "MODERATELY ROBUST"
        grade = "YELLOW"

    # Weak
    else:
        verdict = "WEAK ROBUSTNESS"
        grade = "RED"

    return pd.Series({
        "Robustness_Flags_Audit": ";".join(flags),
        "Positive_Robustness_Domains": positive,
        "Negative_Robustness_Domains": negative,
        "Robustness_Verdict": verdict,
        "Robustness_Grade": grade,
    })


audit_flags = integrated.apply(classify_row, axis=1)

audit = pd.concat(
    [integrated.reset_index(drop=True),
     audit_flags.reset_index(drop=True)],
    axis=1
)

# ------------------------------------------------------------
# 6. ADD LOO RANGE EXPLICITLY
# ------------------------------------------------------------

loo_ranges = []

for target in TARGETS:

    sub = loo[
        loo["Target"].astype(str) == target
    ]

    vals = pd.to_numeric(
        sub["R2_Mean"], errors="coerce"
    ).dropna()

    if len(vals):
        loo_ranges.append({
            "Target": target,
            "LOO_R2_Range": float(vals.max() - vals.min()),
            "LOO_R2_Min": float(vals.min()),
            "LOO_R2_Max": float(vals.max()),
        })
    else:
        loo_ranges.append({
            "Target": target,
            "LOO_R2_Range": np.nan,
            "LOO_R2_Min": np.nan,
            "LOO_R2_Max": np.nan,
        })

loo_summary = pd.DataFrame(loo_ranges)

audit = audit.merge(
    loo_summary,
    on="Target",
    how="left",
    validate="one_to_one",
)

# ------------------------------------------------------------
# 7. HUMAN-READABLE INTERPRETATION
# ------------------------------------------------------------

def interpretation(row):

    statements = []

    if row["OuterFold_R2_CV"] <= 0.75:
        statements.append("outer-fold performance is reasonably stable")
    else:
        statements.append("outer-fold performance is heterogeneous")

    if pd.notna(row["LOO_R2_Range"]):
        if row["LOO_R2_Range"] <= 0.15:
            statements.append("the conclusion is stable to omission of any one outer fold")
        else:
            statements.append("the conclusion is sensitive to omission of some outer folds")

    if row["Permutation_Positive_Fraction"] >= 0.75:
        statements.append("permutation sensitivity provides consistent support for predictive dependence on the evaluated features")
    else:
        statements.append("permutation sensitivity is mixed across outer-fold/repeat evaluations")

    share = row["Strongest_Group_Importance_Share"]

    if pd.notna(share):
        if share >= 0.50:
            statements.append(
                f"importance is relatively concentrated in {row['Strongest_Feature_Group']}"
            )
        else:
            statements.append(
                "importance is distributed across multiple feature groups"
            )

    return "; ".join(statements) + "."

audit["Scientific_Interpretation"] = audit.apply(
    interpretation,
    axis=1
)

# ------------------------------------------------------------
# 8. POSITION CONTRIBUTION CROSS-CHECK
# ------------------------------------------------------------

position = position.copy()

if "Pooled_Delta_B_minus_A_R2" in position.columns:
    position["Pooled_Delta_B_minus_A_R2"] = pd.to_numeric(
        position["Pooled_Delta_B_minus_A_R2"],
        errors="coerce"
    )

position_cols = [
    "Target",
    "Selected_Architecture",
    "Model_A_Pooled_OOF_R2",
    "Model_B_Pooled_OOF_R2",
    "Pooled_Delta_B_minus_A_R2",
    "Mean_Fold_Delta_B_minus_A_R2",
    "N_Folds_Better",
    "N_Folds_Worse",
    "N_Folds_Tied",
]

position_available = [
    c for c in position_cols
    if c in position.columns
]

position_crosscheck = position[position_available].copy()

audit = audit.merge(
    position_crosscheck,
    on="Target",
    how="left",
    suffixes=("", "_Position"),
)

# ------------------------------------------------------------
# 9. FEATURE-GROUP SUMMARY
# ------------------------------------------------------------

group["Permutation_Importance_Absolute_Share"] = pd.to_numeric(
    group["Permutation_Importance_Absolute_Share"],
    errors="coerce"
)

group_top = (
    group.sort_values(
        ["Target", "Permutation_Importance_Absolute_Share"],
        ascending=[True, False]
    )
    .groupby("Target", as_index=False)
    .first()
)

group_top = group_top[
    [
        "Target",
        "Feature_Group",
        "Permutation_Importance_Absolute_Share",
        "Best_Permutation_Rank",
        "Median_Permutation_Rank",
    ]
].rename(
    columns={
        "Feature_Group": "Top_Feature_Group",
        "Permutation_Importance_Absolute_Share":
            "Top_Group_Importance_Share",
    }
)

# ------------------------------------------------------------
# 10. SAVE MASTER AUDIT
# ------------------------------------------------------------

audit = audit.sort_values(
    "Target",
    key=lambda s: pd.Categorical(
        s, categories=TARGETS, ordered=True
    )
)

audit.to_csv(
    OUTPUT / "VIM2_Step9_Professional_Target_Audit.csv",
    index=False
)

position_crosscheck.to_csv(
    OUTPUT / "VIM2_Step9_Position_Crosscheck.csv",
    index=False
)

group_top.to_csv(
    OUTPUT / "VIM2_Step9_Top_FeatureGroup_By_Target.csv",
    index=False
)

# ------------------------------------------------------------
# 11. PRINT TARGET-LEVEL VERDICT
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TARGET-LEVEL ROBUSTNESS VERDICT")
print("=" * 90)

display_cols = [
    "Target",
    "Final_Architecture",
    "OuterFold_R2_Mean",
    "OuterFold_R2_SD",
    "OuterFold_R2_CV",
    "LOO_R2_Range",
    "Permutation_Positive_Fraction",
    "Median_R2_Degradation",
    "Strongest_Feature_Group",
    "Strongest_Group_Importance_Share",
    "Robustness_Verdict",
]

print(
    audit[display_cols].to_string(index=False)
)

# ------------------------------------------------------------
# 12. PROJECT-LEVEL SUMMARY
# ------------------------------------------------------------

n_green = int(
    (audit["Robustness_Grade"] == "GREEN").sum()
)

n_yellow = int(
    (audit["Robustness_Grade"] == "YELLOW").sum()
)

n_red = int(
    (audit["Robustness_Grade"] == "RED").sum()
)

n_total = len(audit)

print("\n" + "=" * 90)
print("OVERALL PROJECT ROBUSTNESS")
print("=" * 90)

print(f"Targets evaluated       : {n_total}")
print(f"Strong/acceptable       : {n_green}")
print(f"Moderate                : {n_yellow}")
print(f"Weak                    : {n_red}")

if n_red == 0 and n_yellow <= 2:
    overall = "STRONG OVERALL ROBUSTNESS"
elif n_red == 0:
    overall = "ACCEPTABLE OVERALL ROBUSTNESS"
elif n_red <= 2:
    overall = "MIXED OVERALL ROBUSTNESS — INVESTIGATE FLAGGED TARGETS"
else:
    overall = "WEAK OVERALL ROBUSTNESS — INVESTIGATION REQUIRED"

print(f"\nFINAL VERDICT: {overall}")

# ------------------------------------------------------------
# 13. DOMAIN-LEVEL COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("ROBUSTNESS DOMAIN SUMMARY")
print("=" * 90)

domain_summary = pd.DataFrame({
    "Domain": [
        "Stable outer-fold R2",
        "Stable leave-one-fold-out",
        "Permutation-supported",
        "Distributed feature-group importance",
    ],
    "N_Targets": [
        int((audit["OuterFold_R2_CV"] <= 0.75).sum()),
        int((audit["LOO_R2_Range"] <= 0.15).sum()),
        int((audit["Permutation_Positive_Fraction"] >= 0.75).sum()),
        int((audit["Strongest_Group_Importance_Share"] < 0.50).sum()),
    ],
})

domain_summary["Fraction"] = (
    domain_summary["N_Targets"] / n_total
)

print(domain_summary.to_string(index=False))

domain_summary.to_csv(
    OUTPUT / "VIM2_Step9_Professional_Domain_Summary.csv",
    index=False
)

# ------------------------------------------------------------
# 14. FLAGGED TARGETS
# ------------------------------------------------------------

flagged = audit[
    audit["Robustness_Grade"] != "GREEN"
][[
    "Target",
    "Robustness_Verdict",
    "Robustness_Flags_Audit",
    "Scientific_Interpretation",
]]

print("\n" + "=" * 90)
print("TARGETS REQUIRING CAUTION")
print("=" * 90)

if flagged.empty:
    print("None. All targets passed the audit at the predefined descriptive thresholds.")
else:
    print(flagged.to_string(index=False))

flagged.to_csv(
    OUTPUT / "VIM2_Step9_Targets_Requiring_Caution.csv",
    index=False
)

# ------------------------------------------------------------
# 15. FINAL MESSAGE
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STEP 9 AUDIT COMPLETE")
print("=" * 90)

print("\nGenerated files:")
for p in sorted(OUTPUT.glob("*.csv")):
    print(" -", p.name)

print("\nIMPORTANT:")
print("This audit is descriptive robustness assessment.")
print("It does NOT change Step 6 architecture selection.")
print("It does NOT perform new HPO.")
print("It does NOT create new folds.")
print("It does NOT claim statistical significance from sensitivity scores.")

VIM-2 STEP 9 — PROFESSIONAL ROBUSTNESS RESULT AUDIT

All Step 9 output files found.

Integrated summary : (9, 13)
Fold robustness    : (27, 11)
LOO robustness     : (45, 8)
Feature groups     : (60, 8)
Model family       : (23, 10)
Permutation        : (59100, 18)
Position           : (9, 11)
Error robustness   : (36, 10)

TARGET-LEVEL ROBUSTNESS VERDICT
            Target    Final_Architecture  OuterFold_R2_Mean  OuterFold_R2_SD  OuterFold_R2_CV  LOO_R2_Range  Permutation_Positive_Fraction  Median_R2_Degradation Strongest_Feature_Group  Strongest_Group_Importance_Share Robustness_Verdict
0.031ug/mL_MEM_37C   Model_A_No_Position           0.504215         0.044589         0.088433      0.029554                       0.672769               0.000267              Structural                          0.647120  ACCEPTABLY ROBUST
  0.5ug/mL_CTX_37C Model_B_With_Position           0.450763         0.051799         0.114914      0.031428                       0.634242               0.000249    

In [ ]:
# @title

# ============================================================
# DOWNLOAD STEP 9 analysis OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step9_Professional_Result_Audit"

# Output ZIP archive
zip_base = "/content/VIM2_Step9_Professional_Result_Audit"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step9_Professional_Result_Audit"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("STEP 9 analysis DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

STEP 9 analysis DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step9_Professional_Result_Audit
ZIP archive      : /content/VIM2_Step9_Professional_Result_Audit.zip
Archive size     : 0.00 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
# ============================================================
# VIM-2 STEP 4 — FULL OOF PREDICTION AUDIT
# ============================================================
#
# Purpose
# -------
# Comprehensive quality-control and diagnostic analysis of
# Step 4 Position-Aware Out-of-Fold (OOF) predictions.
#
# This script DOES NOT:
#   - retrain models
#   - perform hyperparameter optimization
#   - change folds
#   - select architectures
#   - modify predictions
#
# It only audits the frozen OOF predictions generated by Step 4.
#
# Main questions addressed:
#   1. Are OOF predictions structurally complete?
#   2. Is aggregate R² genuinely supported by the predictions?
#   3. Does the model regress strongly toward the mean?
#   4. Are extreme fitness values poorly predicted?
#   5. Is performance driven by one particular fold?
#   6. Are Model A and Model B behaving differently?
#   7. Are predictions well calibrated?
#
# Input:
#   /content/VIM2_Step4_Final_OOF_Predictions.csv
#
# Output directory:
#   /content/VIM2_Step4_OOF_AUDIT
#
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_FILE = "/content/VIM2_Step4_Final_OOF_Predictions.csv"

OUTPUT_DIR = "/content/VIM2_Step4_OOF_AUDIT"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

print("=" * 80)
print("VIM-2 STEP 4 — FULL OOF PREDICTION AUDIT")
print("=" * 80)

print(f"\nInput file:")
print(INPUT_FILE)

print(f"\nOutput directory:")
print(OUTPUT_DIR)


# ============================================================
# 2. LOAD DATA SAFELY
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"\nInput file was not found:\n{INPUT_FILE}\n\n"
        "Please upload VIM2_Step4_Final_OOF_Predictions.csv "
        "to /content/ in Google Colab."
    )

print("\n[1/12] Loading OOF predictions...")

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print(f"Loaded shape: {df.shape}")

print("\nColumns:")
for c in df.columns:
    print("  -", c)


# ============================================================
# 3. COLUMN DETECTION
# ============================================================

print("\n[2/12] Detecting required columns...")

required_columns = [
    "Target",
    "Architecture",
    "Outer_Fold",
    "Observed",
    "Predicted"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        "\nMissing required columns:\n"
        + "\n".join(f"  - {x}" for x in missing)
    )

# Optional columns
optional_columns = [
    "Residual",
    "Best_Inner_R2",
    "Row_Index",
    "PositionAware_Fold",
    "Random_Fold",
    "Model"
]

print("\nRequired columns: PASS")

print("\nOptional columns present:")
for c in optional_columns:
    print(f"  {c}: {'YES' if c in df.columns else 'NO'}")


# ============================================================
# 4. BASIC DATA CLEANING FOR AUDIT ONLY
# ============================================================

print("\n[3/12] Preparing numerical fields...")

for col in ["Observed", "Predicted"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

if "Residual" in df.columns:
    df["Residual"] = pd.to_numeric(
        df["Residual"],
        errors="coerce"
    )
else:
    df["Residual"] = (
        df["Observed"] - df["Predicted"]
    )

# Keep original rows but create valid prediction subset
valid = df[
    df["Observed"].notna() &
    df["Predicted"].notna()
].copy()

valid["Residual_Calculated"] = (
    valid["Observed"] - valid["Predicted"]
)

valid["Absolute_Error"] = (
    valid["Residual_Calculated"].abs()
)

valid["Squared_Error"] = (
    valid["Residual_Calculated"] ** 2
)

print(f"Total rows: {len(df):,}")
print(f"Valid prediction rows: {len(valid):,}")
print(f"Rows excluded due to missing Observed/Predicted: "
      f"{len(df) - len(valid):,}")


# ============================================================
# 5. DATA INTEGRITY / QC
# ============================================================

print("\n[4/12] Running structural QC...")

qc_rows = []

qc_rows.append({
    "Metric": "Total rows",
    "Value": len(df)
})

qc_rows.append({
    "Metric": "Valid prediction rows",
    "Value": len(valid)
})

qc_rows.append({
    "Metric": "Missing Observed",
    "Value": df["Observed"].isna().sum()
})

qc_rows.append({
    "Metric": "Missing Predicted",
    "Value": df["Predicted"].isna().sum()
})

qc_rows.append({
    "Metric": "Unique Targets",
    "Value": df["Target"].nunique()
})

qc_rows.append({
    "Metric": "Unique Architectures",
    "Value": df["Architecture"].nunique()
})

qc_rows.append({
    "Metric": "Unique Outer Folds",
    "Value": df["Outer_Fold"].nunique()
})

qc_df = pd.DataFrame(qc_rows)

print(qc_df.to_string(index=False))

# ------------------------------------------------------------
# Duplicate detection
# ------------------------------------------------------------

duplicate_subset = [
    c for c in [
        "Target",
        "Architecture",
        "Outer_Fold",
        "Row_Index"
    ]
    if c in valid.columns
]

if len(duplicate_subset) >= 3:

    duplicate_mask = valid.duplicated(
        subset=duplicate_subset,
        keep=False
    )

    duplicate_count = duplicate_mask.sum()

else:

    duplicate_count = np.nan

print(
    f"\nPotential duplicate prediction rows: "
    f"{duplicate_count}"
)

# Save QC
qc_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "01_data_integrity_QC.csv"
    ),
    index=False
)


# ============================================================
# 6. OVERALL PERFORMANCE METRICS
# ============================================================

print("\n[5/12] Calculating global prediction metrics...")


def safe_pearson(y_true, y_pred):
    if len(y_true) < 3:
        return np.nan
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    return pearsonr(y_true, y_pred)[0]


def safe_spearman(y_true, y_pred):
    if len(y_true) < 3:
        return np.nan
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    return spearmanr(y_true, y_pred)[0]


def calculate_metrics(group):

    y = group["Observed"].to_numpy()
    p = group["Predicted"].to_numpy()

    residual = y - p

    # Calibration:
    # Observed ≈ intercept + slope * Predicted
    X = p.reshape(-1, 1)

    if len(group) >= 3 and np.std(p) > 0:

        reg = LinearRegression()
        reg.fit(X, y)

        calibration_slope = float(
            reg.coef_[0]
        )

        calibration_intercept = float(
            reg.intercept_
        )

    else:

        calibration_slope = np.nan
        calibration_intercept = np.nan

    return pd.Series({

        "N": len(group),

        "R2": r2_score(y, p)
        if len(np.unique(y)) > 1
        else np.nan,

        "RMSE": np.sqrt(
            mean_squared_error(y, p)
        ),

        "MAE": mean_absolute_error(
            y, p
        ),

        "Pearson_r": safe_pearson(
            y, p
        ),

        "Spearman_rho": safe_spearman(
            y, p
        ),

        "Observed_Mean": np.mean(y),

        "Observed_SD": np.std(
            y,
            ddof=1
        ),

        "Predicted_Mean": np.mean(p),

        "Predicted_SD": np.std(
            p,
            ddof=1
        ),

        "Mean_Bias_PredMinusObs": np.mean(
            p - y
        ),

        "Mean_Residual_ObsMinusPred": np.mean(
            residual
        ),

        "Prediction_SD_to_Observed_SD": (
            np.std(p, ddof=1) /
            np.std(y, ddof=1)
            if np.std(y, ddof=1) > 0
            else np.nan
        ),

        "Calibration_Slope": calibration_slope,

        "Calibration_Intercept": calibration_intercept,

        "Max_Absolute_Error": np.max(
            np.abs(residual)
        ),

        "Median_Absolute_Error": np.median(
            np.abs(residual)
        )
    })


overall = calculate_metrics(valid)

overall_df = pd.DataFrame(
    [overall]
)

print("\nOVERALL OOF PERFORMANCE")
print("=" * 80)
print(overall_df.T.to_string(
    header=False
))

overall_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "02_global_OOF_performance.csv"
    ),
    index=False
)


# ============================================================
# 7. TARGET × ARCHITECTURE PERFORMANCE
# ============================================================

print("\n[6/12] Calculating Target × Architecture performance...")

target_architecture = (
    valid
    .groupby(
        ["Target", "Architecture"],
        sort=True
    )
    .apply(calculate_metrics)
    .reset_index()
)

target_architecture.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "03_Target_Architecture_performance.csv"
    ),
    index=False
)

print(
    target_architecture[
        [
            "Target",
            "Architecture",
            "N",
            "R2",
            "RMSE",
            "MAE",
            "Pearson_r",
            "Spearman_rho",
            "Prediction_SD_to_Observed_SD",
            "Calibration_Slope"
        ]
    ].to_string(index=False)
)


# ============================================================
# 8. OUTER-FOLD PERFORMANCE
# ============================================================

print("\n[7/12] Calculating outer-fold diagnostics...")

fold_metrics = (
    valid
    .groupby(
        [
            "Target",
            "Architecture",
            "Outer_Fold"
        ],
        sort=True
    )
    .apply(calculate_metrics)
    .reset_index()
)

fold_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "04_outer_fold_performance.csv"
    ),
    index=False
)

print(
    "\nFold-level performance saved."
)


# ============================================================
# 9. FITNESS-RANGE ERROR ANALYSIS
# ============================================================

print("\n[8/12] Performing fitness-range error analysis...")

# Quantile-based bins ensure reasonable numbers of observations
# in each bin.

valid["Observed_Quantile_Bin"] = pd.qcut(
    valid["Observed"],
    q=10,
    duplicates="drop"
)

def bin_metrics(group):

    y = group["Observed"].to_numpy()
    p = group["Predicted"].to_numpy()

    return pd.Series({

        "N": len(group),

        "Observed_Mean": np.mean(y),

        "Predicted_Mean": np.mean(p),

        "Observed_SD": np.std(
            y,
            ddof=1
        ),

        "Predicted_SD": np.std(
            p,
            ddof=1
        ),

        "Bias_PredMinusObs": np.mean(
            p - y
        ),

        "MAE": mean_absolute_error(
            y, p
        ),

        "RMSE": np.sqrt(
            mean_squared_error(y, p)
        ),

        "Pearson_r": safe_pearson(
            y, p
        ),

        "Spearman_rho": safe_spearman(
            y, p
        )
    })


bin_summary = (
    valid
    .groupby(
        "Observed_Quantile_Bin",
        observed=True
    )
    .apply(bin_metrics)
    .reset_index()
)

bin_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "05_observed_fitness_range_error.csv"
    ),
    index=False
)

print(
    bin_summary.to_string(index=False)
)


# ============================================================
# 10. EXTREME FITNESS ANALYSIS
# ============================================================

print("\n[9/12] Evaluating extreme fitness predictions...")

# Lower 10% = most negative / lowest fitness
# Upper 10% = highest fitness

q10 = valid["Observed"].quantile(0.10)
q90 = valid["Observed"].quantile(0.90)

lower_tail = valid[
    valid["Observed"] <= q10
].copy()

upper_tail = valid[
    valid["Observed"] >= q90
].copy()


def tail_summary(group, label):

    if len(group) == 0:
        return {
            "Region": label,
            "N": 0
        }

    y = group["Observed"].to_numpy()
    p = group["Predicted"].to_numpy()

    return {
        "Region": label,
        "N": len(group),
        "Observed_Mean": np.mean(y),
        "Predicted_Mean": np.mean(p),
        "Bias_PredMinusObs": np.mean(p - y),
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(
            mean_squared_error(y, p)
        ),
        "Pearson_r": safe_pearson(y, p),
        "Spearman_rho": safe_spearman(y, p)
    }


tails_df = pd.DataFrame([
    tail_summary(
        lower_tail,
        "Lowest 10% observed fitness"
    ),
    tail_summary(
        upper_tail,
        "Highest 10% observed fitness"
    )
])

tails_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "06_extreme_fitness_analysis.csv"
    ),
    index=False
)

print(tails_df.to_string(index=False))


# ============================================================
# 11. WORST INDIVIDUAL PREDICTIONS
# ============================================================

print("\n[10/12] Identifying worst individual predictions...")

worst = valid.sort_values(
    "Absolute_Error",
    ascending=False
).copy()

worst_columns = [
    c for c in [
        "Target",
        "Architecture",
        "Outer_Fold",
        "Row_Index",
        "Observed",
        "Predicted",
        "Residual",
        "Residual_Calculated",
        "Absolute_Error",
        "Best_Inner_R2"
    ]
    if c in worst.columns
]

worst_100 = worst[
    worst_columns
].head(100)

worst_100.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "07_worst_100_predictions.csv"
    ),
    index=False
)

print("\nTOP 20 WORST PREDICTIONS")
print("=" * 80)

print(
    worst_100.head(20).to_string(
        index=False
    )
)


# ============================================================
# 12. CHECK FOR PREDICTION COMPRESSION
# ============================================================

print("\n[11/12] Diagnosing prediction compression...")

compression_rows = []

for (target, architecture), group in valid.groupby(
    ["Target", "Architecture"]
):

    y = group["Observed"].to_numpy()
    p = group["Predicted"].to_numpy()

    observed_sd = np.std(y, ddof=1)
    predicted_sd = np.std(p, ddof=1)

    ratio = (
        predicted_sd / observed_sd
        if observed_sd > 0
        else np.nan
    )

    # Linear calibration:
    # Observed = intercept + slope * Predicted

    if len(group) >= 3 and predicted_sd > 0:

        reg = LinearRegression()
        reg.fit(
            p.reshape(-1, 1),
            y
        )

        slope = float(
            reg.coef_[0]
        )

        intercept = float(
            reg.intercept_
        )

    else:

        slope = np.nan
        intercept = np.nan

    compression_rows.append({

        "Target": target,
        "Architecture": architecture,

        "Observed_SD": observed_sd,

        "Predicted_SD": predicted_sd,

        "Prediction_SD_Ratio": ratio,

        "Calibration_Slope": slope,

        "Calibration_Intercept": intercept
    })

compression_df = pd.DataFrame(
    compression_rows
)

compression_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "08_prediction_compression_calibration.csv"
    ),
    index=False
)

print(
    compression_df.to_string(index=False)
)


# ============================================================
# 13. PLOT 1 — OBSERVED VS PREDICTED
# ============================================================

print("\n[12/12] Generating diagnostic figures...")


def make_scatter_plot(
    data,
    title,
    filename
):

    x = data["Observed"].to_numpy()
    y = data["Predicted"].to_numpy()

    fig, ax = plt.subplots(
        figsize=(7.2, 6.4)
    )

    ax.scatter(
        x,
        y,
        s=10,
        alpha=0.30
    )

    # Identity line
    lo = min(
        np.min(x),
        np.min(y)
    )

    hi = max(
        np.max(x),
        np.max(y)
    )

    ax.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        linewidth=1.5
    )

    # Calibration line
    if np.std(x) > 0:

        reg = LinearRegression()
        reg.fit(
            x.reshape(-1, 1),
            y
        )

        xx = np.linspace(
            lo,
            hi,
            200
        )

        yy = reg.predict(
            xx.reshape(-1, 1)
        )

        ax.plot(
            xx,
            yy,
            linewidth=2
        )

    r2 = r2_score(x, y)

    ax.text(
        0.04,
        0.95,
        f"R² = {r2:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11
    )

    ax.set_xlabel(
        "Observed fitness"
    )

    ax.set_ylabel(
        "Predicted fitness"
    )

    ax.set_title(
        title
    )

    ax.grid(
        alpha=0.20
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            FIG_DIR,
            filename
        ),
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)


make_scatter_plot(
    valid,
    "VIM-2 Step 4 — Pooled OOF Observed vs Predicted",
    "01_pooled_OOF_observed_vs_predicted.png"
)


# ============================================================
# 14. PLOT 2 — RESIDUAL VS OBSERVED
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.2, 6.4)
)

ax.scatter(
    valid["Observed"],
    valid["Residual_Calculated"],
    s=10,
    alpha=0.25
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel(
    "Observed fitness"
)

ax.set_ylabel(
    "Residual (Observed − Predicted)"
)

ax.set_title(
    "VIM-2 Step 4 — Residual Structure"
)

ax.grid(
    alpha=0.20
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "02_residual_vs_observed.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# 15. PLOT 3 — OBSERVED VS PREDICTED SD
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.5, 5.8)
)

plot_df = compression_df.copy()

for architecture in sorted(
    plot_df["Architecture"].unique()
):

    sub = plot_df[
        plot_df["Architecture"] == architecture
    ]

    ax.scatter(
        sub["Observed_SD"],
        sub["Predicted_SD"],
        s=70,
        alpha=0.85,
        label=architecture
    )

max_sd = max(
    plot_df["Observed_SD"].max(),
    plot_df["Predicted_SD"].max()
)

ax.plot(
    [0, max_sd],
    [0, max_sd],
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel(
    "Observed fitness SD"
)

ax.set_ylabel(
    "Predicted fitness SD"
)

ax.set_title(
    "Prediction Dispersion / Compression"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.20
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "03_prediction_dispersion_compression.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# 16. PLOT 4 — MAE BY FITNESS QUANTILE
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.5, 5.8)
)

x = np.arange(
    len(bin_summary)
)

ax.plot(
    x,
    bin_summary["MAE"],
    marker="o",
    linewidth=2
)

ax.set_xticks(x)

ax.set_xticklabels(
    [
        f"Q{i+1}"
        for i in range(
            len(bin_summary)
        )
    ]
)

ax.set_xlabel(
    "Observed fitness quantile"
)

ax.set_ylabel(
    "MAE"
)

ax.set_title(
    "Prediction Error Across the Observed Fitness Distribution"
)

ax.grid(
    alpha=0.20
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "04_MAE_across_fitness_distribution.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# 17. PLOT 5 — FOLD PERFORMANCE
# ============================================================

fold_plot = (
    fold_metrics
    .groupby(
        [
            "Architecture",
            "Outer_Fold"
        ]
    )["R2"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(
    figsize=(8.5, 5.8)
)

for architecture in sorted(
    fold_plot["Architecture"].unique()
):

    sub = fold_plot[
        fold_plot["Architecture"] == architecture
    ]

    ax.plot(
        sub["Outer_Fold"],
        sub["R2"],
        marker="o",
        linewidth=2,
        label=architecture
    )

ax.set_xlabel(
    "Outer fold"
)

ax.set_ylabel(
    "Mean R² across targets"
)

ax.set_title(
    "Outer-Fold Stability"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.20
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "05_outer_fold_stability.png"
    ),
    dpi=400,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# 18. IDENTIFY EXTREME UNDER-PREDICTION
# ============================================================

print("\nIdentifying severe under-predictions...")

severe_under = valid[
    (
        valid["Observed"] -
        valid["Predicted"]
    ) < -2.0
].copy()

severe_under = severe_under.sort_values(
    "Residual_Calculated"
)

severe_under.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "09_severe_under_predictions.csv"
    ),
    index=False
)

print(
    f"Rows with Observed - Predicted < -2.0: "
    f"{len(severe_under):,}"
)


# ============================================================
# 19. IDENTIFY EXTREME OVER-PREDICTION
# ============================================================

severe_over = valid[
    (
        valid["Predicted"] -
        valid["Observed"]
    ) > 2.0
].copy()

severe_over = severe_over.sort_values(
    "Residual_Calculated",
    ascending=False
)

severe_over.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "10_severe_over_predictions.csv"
    ),
    index=False
)

print(
    f"Rows with Predicted - Observed > 2.0: "
    f"{len(severe_over):,}"
)


# ============================================================
# 20. TARGET-LEVEL BIAS
# ============================================================

target_bias = (
    valid
    .groupby(
        ["Target", "Architecture"]
    )
    .agg(
        N=("Observed", "size"),
        Observed_Mean=("Observed", "mean"),
        Predicted_Mean=("Predicted", "mean"),
        Observed_SD=("Observed", "std"),
        Predicted_SD=("Predicted", "std"),
        Mean_Bias=("Residual_Calculated", "mean"),
        MAE=("Absolute_Error", "mean")
    )
    .reset_index()
)

target_bias[
    "Prediction_SD_Ratio"
] = (
    target_bias["Predicted_SD"] /
    target_bias["Observed_SD"]
)

target_bias.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "11_target_level_bias_and_dispersion.csv"
    ),
    index=False
)


# ============================================================
# 21. FINAL AUTOMATED INTERPRETATION FLAGS
# ============================================================

print("\n" + "=" * 80)
print("AUTOMATED DIAGNOSTIC FLAGS")
print("=" * 80)

global_r2 = float(
    overall_df.loc[0, "R2"]
)

global_ratio = float(
    overall_df.loc[
        0,
        "Prediction_SD_to_Observed_SD"
    ]
)

global_slope = float(
    overall_df.loc[
        0,
        "Calibration_Slope"
    ]
)

print(
    f"\nGlobal pooled OOF R²: "
    f"{global_r2:.4f}"
)

print(
    f"Prediction SD / Observed SD: "
    f"{global_ratio:.4f}"
)

print(
    f"Calibration slope "
    f"(Observed ~ Predicted): "
    f"{global_slope:.4f}"
)


# ------------------------------------------------------------
# Compression flag
# ------------------------------------------------------------

if global_ratio < 0.70:

    print(
        "\n⚠️ STRONG PREDICTION COMPRESSION DETECTED"
    )

    print(
        "Predictions occupy substantially less "
        "variance than the observed fitness values."
    )

elif global_ratio < 0.85:

    print(
        "\n⚠️ MODERATE PREDICTION COMPRESSION"
    )

else:

    print(
        "\n✓ Prediction dispersion is reasonably preserved."
    )


# ------------------------------------------------------------
# Calibration flag
# ------------------------------------------------------------

if global_slope < 0.70:

    print(
        "\n⚠️ STRONG CALIBRATION COMPRESSION"
    )

    print(
        "Extreme observed values are likely being "
        "pulled toward the center."
    )

elif global_slope < 0.85:

    print(
        "\n⚠️ MODERATE CALIBRATION COMPRESSION"
    )

else:

    print(
        "\n✓ Calibration slope is relatively strong."
    )


# ------------------------------------------------------------
# Severe underprediction flag
# ------------------------------------------------------------

under_fraction = (
    len(severe_under) /
    len(valid)
)

print(
    f"\nSevere under-prediction fraction "
    f"(error > 2 fitness units): "
    f"{under_fraction:.3%}"
)

if under_fraction > 0.05:

    print(
        "⚠️ A substantial fraction of observations "
        "are severely under-predicted."
    )

else:

    print(
        "✓ Severe under-predictions are relatively uncommon."
    )


# ============================================================
# 22. WRITE HUMAN-READABLE REPORT
# ============================================================

report_path = os.path.join(
    OUTPUT_DIR,
    "VIM2_Step4_OOF_Audit_Report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "VIM-2 STEP 4 — OOF PREDICTION AUDIT REPORT\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        f"Input file: {INPUT_FILE}\n"
    )

    f.write(
        f"Total rows: {len(df):,}\n"
    )

    f.write(
        f"Valid prediction rows: {len(valid):,}\n\n"
    )

    f.write(
        "GLOBAL PERFORMANCE\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    for col in overall_df.columns:

        value = overall_df.loc[0, col]

        f.write(
            f"{col}: {value}\n"
        )

    f.write(
        "\nINTERPRETATION FLAGS\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        f"Prediction SD / Observed SD = "
        f"{global_ratio:.4f}\n"
    )

    f.write(
        f"Calibration slope = "
        f"{global_slope:.4f}\n"
    )

    f.write(
        f"Severe under-prediction fraction = "
        f"{under_fraction:.4%}\n"
    )

    f.write(
        "\nTarget × Architecture performance "
        "saved separately.\n"
    )

    f.write(
        "Outer-fold performance saved separately.\n"
    )

    f.write(
        "Worst predictions saved separately.\n"
    )


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

print(
    f"\nResults saved to:\n{OUTPUT_DIR}"
)

print("\nMain outputs:")

output_files = [
    "01_data_integrity_QC.csv",
    "02_global_OOF_performance.csv",
    "03_Target_Architecture_performance.csv",
    "04_outer_fold_performance.csv",
    "05_observed_fitness_range_error.csv",
    "06_extreme_fitness_analysis.csv",
    "07_worst_100_predictions.csv",
    "08_prediction_compression_calibration.csv",
    "09_severe_under_predictions.csv",
    "10_severe_over_predictions.csv",
    "11_target_level_bias_and_dispersion.csv",
    "VIM2_Step4_OOF_Audit_Report.txt"
]

for filename in output_files:

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(path):
        print("  ✓", filename)

print("\nFigures:")
for filename in sorted(
    os.listdir(FIG_DIR)
):
    print("  ✓", filename)

print(
    "\nIMPORTANT:"
)

print(
    "This audit does not retrain or modify any model. "
    "It only evaluates the frozen Step 4 OOF predictions."
)

print("=" * 80)

VIM-2 STEP 4 — FULL OOF PREDICTION AUDIT

Input file:
/content/VIM2_Step4_Final_OOF_Predictions.csv

Output directory:
/content/VIM2_Step4_OOF_AUDIT

[1/12] Loading OOF predictions...
Loaded shape: (90102, 10)

Columns:
  - Row_Index
  - Stable_Mutation_Key
  - Target
  - Architecture
  - Model
  - Outer_Fold
  - Observed
  - Predicted
  - Residual
  - Best_Inner_R2

[2/12] Detecting required columns...

Required columns: PASS

Optional columns present:
  Residual: YES
  Best_Inner_R2: YES
  Row_Index: YES
  PositionAware_Fold: NO
  Random_Fold: NO
  Model: YES

[3/12] Preparing numerical fields...
Total rows: 90,102
Valid prediction rows: 90,102
Rows excluded due to missing Observed/Predicted: 0

[4/12] Running structural QC...
               Metric  Value
           Total rows  90102
Valid prediction rows  90102
     Missing Observed      0
    Missing Predicted      0
       Unique Targets      9
 Unique Architectures      2
   Unique Outer Folds      5

Potential duplicate predicti

In [ ]:
# @title

# ============================================================
# DOWNLOAD /content/VIM2_Step4_OOF_AUDIT analysis OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step4_OOF_AUDIT"

# Output ZIP archive
zip_base = "/content/VIM2_Step4_OOF_AUDITt"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step4_OOF_AUDIT"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("/content/VIM2_Step4_OOF_AUDIT DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

/content/VIM2_Step4_OOF_AUDIT DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step4_OOF_AUDIT
ZIP archive      : /content/VIM2_Step4_OOF_AUDITt.zip
Archive size     : 4.65 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title
# =============================================================================
# VIM-2 STEP 4 — PROFESSIONAL OOF FORENSIC AUDIT
# =============================================================================
#
# Purpose
# -------
# This script performs a rigorous post-hoc audit of the frozen Step 4
# position-aware out-of-fold (OOF) predictions.
#
# IMPORTANT
# ---------
# - No model is retrained.
# - No hyperparameters are optimized.
# - No new folds are created.
# - No architecture is selected.
# - No information is added from the original ML dataset.
#
# The analysis is performed ONLY on:
#
#   /content/VIM2_Step4_Final_OOF_Predictions.csv
#
# The main objective is to determine whether the apparent prediction quality
# is driven by genuine mutation-level signal, target-specific structure,
# prediction compression, calibration artifacts, or a small number of extreme
# errors.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

warnings.filterwarnings("ignore")


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

INPUT_FILE = Path(
    "/content/VIM2_Step4_Final_OOF_Predictions.csv"
)

OUTPUT_DIR = Path(
    "/content/VIM2_Step4_OOF_FORENSIC_AUDIT"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_STATE = 42

# Thresholds used for diagnostic classification.
SEVERE_ERROR_THRESHOLD = 2.0
EXTREME_ERROR_THRESHOLD = 3.0

# Quantile bins used for distribution-aware diagnostics.
N_QUANTILE_BINS = 10

# Number of worst predictions to export.
N_WORST = 250


# =============================================================================
# 2. DISPLAY HELPERS
# =============================================================================

def print_section(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def safe_round(df, decimals=4):
    """
    Return a rounded copy without modifying the original dataframe.
    """
    out = df.copy()

    numeric_cols = out.select_dtypes(
        include=[np.number]
    ).columns

    out[numeric_cols] = out[numeric_cols].round(decimals)

    return out


# =============================================================================
# 3. LOAD DATA
# =============================================================================

print_section("1/15 — LOADING FROZEN STEP 4 OOF PREDICTIONS")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file was not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print(f"Input file : {INPUT_FILE}")
print(f"Loaded shape: {df.shape}")

print("\nColumns:")
for col in df.columns:
    print(f"  - {col}")


# =============================================================================
# 4. REQUIRED COLUMN VALIDATION
# =============================================================================

print_section("2/15 — VALIDATING REQUIRED COLUMNS")

REQUIRED_COLUMNS = [
    "Target",
    "Architecture",
    "Outer_Fold",
    "Observed",
    "Predicted"
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_columns)
    )

print("Required columns: PASS")

OPTIONAL_COLUMNS = [
    "Row_Index",
    "Stable_Mutation_Key",
    "Model",
    "Residual",
    "Best_Inner_R2"
]

print("\nOptional columns:")
for col in OPTIONAL_COLUMNS:
    print(
        f"  {col}: "
        f"{'YES' if col in df.columns else 'NO'}"
    )


# =============================================================================
# 5. NUMERIC PREPARATION
# =============================================================================

print_section("3/15 — PREPARING NUMERICAL FIELDS")

df["Observed"] = pd.to_numeric(
    df["Observed"],
    errors="coerce"
)

df["Predicted"] = pd.to_numeric(
    df["Predicted"],
    errors="coerce"
)

if "Residual" in df.columns:
    df["Residual"] = pd.to_numeric(
        df["Residual"],
        errors="coerce"
    )

if "Outer_Fold" in df.columns:
    df["Outer_Fold"] = pd.to_numeric(
        df["Outer_Fold"],
        errors="coerce"
    )

valid_mask = (
    df["Observed"].notna()
    & df["Predicted"].notna()
)

analysis_df = df.loc[
    valid_mask
].copy()

analysis_df["Error_ObsMinusPred"] = (
    analysis_df["Observed"]
    - analysis_df["Predicted"]
)

analysis_df["Error_PredMinusObs"] = (
    analysis_df["Predicted"]
    - analysis_df["Observed"]
)

analysis_df["Absolute_Error"] = (
    analysis_df["Observed"]
    - analysis_df["Predicted"]
).abs()

analysis_df["Squared_Error"] = (
    analysis_df["Observed"]
    - analysis_df["Predicted"]
) ** 2

print(f"Total rows              : {len(df):,}")
print(f"Valid prediction rows   : {len(analysis_df):,}")
print(
    f"Excluded rows            : "
    f"{len(df) - len(analysis_df):,}"
)


# =============================================================================
# 6. STRUCTURAL / DUPLICATE QC
# =============================================================================

print_section("4/15 — STRUCTURAL QUALITY CONTROL")

qc_records = []

qc_records.append({
    "Metric": "Total_rows",
    "Value": len(df)
})

qc_records.append({
    "Metric": "Valid_prediction_rows",
    "Value": len(analysis_df)
})

qc_records.append({
    "Metric": "Missing_Observed",
    "Value": df["Observed"].isna().sum()
})

qc_records.append({
    "Metric": "Missing_Predicted",
    "Value": df["Predicted"].isna().sum()
})

qc_records.append({
    "Metric": "Unique_Targets",
    "Value": analysis_df["Target"].nunique()
})

qc_records.append({
    "Metric": "Unique_Architectures",
    "Value": analysis_df["Architecture"].nunique()
})

qc_records.append({
    "Metric": "Unique_Outer_Folds",
    "Value": analysis_df["Outer_Fold"].nunique()
})


# Check duplicated mutation-level prediction records.
duplicate_keys = [
    c for c in [
        "Target",
        "Architecture",
        "Outer_Fold",
        "Row_Index"
    ]
    if c in analysis_df.columns
]

if duplicate_keys:
    duplicate_count = analysis_df.duplicated(
        subset=duplicate_keys,
        keep=False
    ).sum()
else:
    duplicate_count = 0

qc_records.append({
    "Metric": "Potential_duplicate_prediction_rows",
    "Value": duplicate_count
})

qc_df = pd.DataFrame(qc_records)

print(
    safe_round(
        qc_df
    ).to_string(index=False)
)

qc_df.to_csv(
    OUTPUT_DIR / "01_data_integrity_QC.csv",
    index=False
)


# =============================================================================
# 7. METRIC FUNCTION
# =============================================================================

def calculate_metrics(sub):
    """
    Calculate regression and correlation metrics for one prediction subset.
    """

    y_true = sub["Observed"].to_numpy(dtype=float)
    y_pred = sub["Predicted"].to_numpy(dtype=float)

    n = len(sub)

    if n < 3:
        return {
            "N": n,
            "R2": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan,
            "Pearson_r": np.nan,
            "Spearman_rho": np.nan,
            "Observed_Mean": np.nan,
            "Predicted_Mean": np.nan,
            "Observed_SD": np.nan,
            "Predicted_SD": np.nan,
            "Prediction_SD_to_Observed_SD": np.nan,
            "Mean_Bias_PredMinusObs": np.nan,
            "Calibration_Slope_ObsOnPred": np.nan,
            "Calibration_Intercept_ObsOnPred": np.nan,
            "Calibration_R2_ObsOnPred": np.nan
        }

    observed_sd = np.std(
        y_true,
        ddof=1
    )

    predicted_sd = np.std(
        y_pred,
        ddof=1
    )

    # Pearson correlation.
    try:
        pearson_r = pearsonr(
            y_true,
            y_pred
        )[0]
    except Exception:
        pearson_r = np.nan

    # Spearman rank correlation.
    try:
        spearman_rho = spearmanr(
            y_true,
            y_pred
        )[0]
    except Exception:
        spearman_rho = np.nan

    # Calibration:
    # Observed = slope * Predicted + intercept
    try:
        calibration = linregress(
            y_pred,
            y_true
        )

        calibration_slope = calibration.slope
        calibration_intercept = calibration.intercept
        calibration_r2 = calibration.rvalue ** 2

    except Exception:
        calibration_slope = np.nan
        calibration_intercept = np.nan
        calibration_r2 = np.nan

    return {
        "N": n,
        "R2": r2_score(
            y_true,
            y_pred
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "Pearson_r": pearson_r,
        "Spearman_rho": spearman_rho,
        "Observed_Mean": np.mean(y_true),
        "Predicted_Mean": np.mean(y_pred),
        "Observed_SD": observed_sd,
        "Predicted_SD": predicted_sd,
        "Prediction_SD_to_Observed_SD": (
            predicted_sd / observed_sd
            if observed_sd > 0
            else np.nan
        ),
        "Mean_Bias_PredMinusObs": np.mean(
            y_pred - y_true
        ),
        "Calibration_Slope_ObsOnPred": (
            calibration_slope
        ),
        "Calibration_Intercept_ObsOnPred": (
            calibration_intercept
        ),
        "Calibration_R2_ObsOnPred": (
            calibration_r2
        )
    }


# =============================================================================
# 8. GLOBAL METRICS
# =============================================================================

print_section("5/15 — GLOBAL OOF PERFORMANCE")

global_metrics = calculate_metrics(
    analysis_df
)

global_metrics_df = pd.DataFrame(
    [global_metrics]
)

print(
    safe_round(
        global_metrics_df
    ).T.to_string(
        header=False
    )
)

global_metrics_df.to_csv(
    OUTPUT_DIR / "02_global_OOF_performance.csv",
    index=False
)


# =============================================================================
# 9. TARGET × ARCHITECTURE ANALYSIS
# =============================================================================

print_section(
    "6/15 — TARGET × ARCHITECTURE PERFORMANCE"
)

target_arch_records = []

for (target, architecture), sub in analysis_df.groupby(
    ["Target", "Architecture"],
    sort=True
):

    metrics = calculate_metrics(sub)

    metrics.update({
        "Target": target,
        "Architecture": architecture
    })

    target_arch_records.append(metrics)

target_arch_df = pd.DataFrame(
    target_arch_records
)

target_arch_df = target_arch_df[
    [
        "Target",
        "Architecture",
        "N",
        "R2",
        "RMSE",
        "MAE",
        "Pearson_r",
        "Spearman_rho",
        "Observed_Mean",
        "Predicted_Mean",
        "Observed_SD",
        "Predicted_SD",
        "Prediction_SD_to_Observed_SD",
        "Mean_Bias_PredMinusObs",
        "Calibration_Slope_ObsOnPred",
        "Calibration_Intercept_ObsOnPred",
        "Calibration_R2_ObsOnPred"
    ]
]

print(
    safe_round(
        target_arch_df
    ).to_string(index=False)
)

target_arch_df.to_csv(
    OUTPUT_DIR /
    "03_Target_Architecture_performance.csv",
    index=False
)


# =============================================================================
# 10. OUTER-FOLD PERFORMANCE
# =============================================================================

print_section(
    "7/15 — OUTER-FOLD PERFORMANCE AND STABILITY"
)

fold_records = []

for (
    target,
    architecture,
    outer_fold
), sub in analysis_df.groupby(
    [
        "Target",
        "Architecture",
        "Outer_Fold"
    ],
    sort=True
):

    metrics = calculate_metrics(sub)

    metrics.update({
        "Target": target,
        "Architecture": architecture,
        "Outer_Fold": outer_fold
    })

    fold_records.append(metrics)

fold_df = pd.DataFrame(
    fold_records
)

fold_df = fold_df[
    [
        "Target",
        "Architecture",
        "Outer_Fold",
        "N",
        "R2",
        "RMSE",
        "MAE",
        "Pearson_r",
        "Spearman_rho",
        "Prediction_SD_to_Observed_SD",
        "Mean_Bias_PredMinusObs",
        "Calibration_Slope_ObsOnPred"
    ]
]

print(
    safe_round(
        fold_df
    ).to_string(index=False)
)

fold_df.to_csv(
    OUTPUT_DIR /
    "04_outer_fold_performance.csv",
    index=False
)


# =============================================================================
# 11. TARGET-WISE FITNESS QUANTILE ERROR
# =============================================================================

print_section(
    "8/15 — ERROR ACROSS THE OBSERVED FITNESS DISTRIBUTION"
)

quantile_records = []

for (
    target,
    architecture
), sub in analysis_df.groupby(
    ["Target", "Architecture"],
    sort=True
):

    sub = sub.copy()

    # Rank-based bins avoid problems caused by uneven distributions.
    try:
        sub["Fitness_Quantile"] = pd.qcut(
            sub["Observed"],
            q=N_QUANTILE_BINS,
            labels=False,
            duplicates="drop"
        )
    except Exception:
        continue

    for qbin, qsub in sub.groupby(
        "Fitness_Quantile",
        sort=True
    ):

        if len(qsub) == 0:
            continue

        quantile_records.append({
            "Target": target,
            "Architecture": architecture,
            "Fitness_Quantile": int(qbin) + 1,
            "N": len(qsub),
            "Observed_Mean": qsub["Observed"].mean(),
            "Predicted_Mean": qsub["Predicted"].mean(),
            "Mean_Error_PredMinusObs": (
                qsub["Predicted"]
                - qsub["Observed"]
            ).mean(),
            "MAE": qsub["Absolute_Error"].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    qsub["Squared_Error"]
                )
            ),
            "Median_Absolute_Error": (
                qsub["Absolute_Error"]
                .median()
            ),
            "UnderPrediction_Fraction": np.mean(
                qsub["Predicted"]
                < qsub["Observed"]
            ),
            "Severe_Error_Fraction": np.mean(
                qsub["Absolute_Error"]
                > SEVERE_ERROR_THRESHOLD
            )
        })

quantile_df = pd.DataFrame(
    quantile_records
)

print(
    safe_round(
        quantile_df
    ).to_string(index=False)
)

quantile_df.to_csv(
    OUTPUT_DIR /
    "05_observed_fitness_quantile_error.csv",
    index=False
)


# =============================================================================
# 12. EXTREME FITNESS / TAIL ANALYSIS
# =============================================================================

print_section(
    "9/15 — LOWER-TAIL AND UPPER-TAIL PERFORMANCE"
)

tail_records = []

for (
    target,
    architecture
), sub in analysis_df.groupby(
    ["Target", "Architecture"],
    sort=True
):

    sub = sub.copy()

    lower_cutoff = sub["Observed"].quantile(
        0.10
    )

    upper_cutoff = sub["Observed"].quantile(
        0.90
    )

    lower = sub[
        sub["Observed"] <= lower_cutoff
    ]

    upper = sub[
        sub["Observed"] >= upper_cutoff
    ]

    for tail_name, tail_sub in [
        ("Lower_10pct", lower),
        ("Upper_10pct", upper)
    ]:

        tail_records.append({
            "Target": target,
            "Architecture": architecture,
            "Tail": tail_name,
            "N": len(tail_sub),
            "Observed_Mean": (
                tail_sub["Observed"].mean()
            ),
            "Predicted_Mean": (
                tail_sub["Predicted"].mean()
            ),
            "Bias_PredMinusObs": (
                tail_sub["Predicted"]
                - tail_sub["Observed"]
            ).mean(),
            "MAE": (
                tail_sub["Absolute_Error"]
                .mean()
            ),
            "RMSE": np.sqrt(
                np.mean(
                    tail_sub["Squared_Error"]
                )
            ),
            "Median_Absolute_Error": (
                tail_sub["Absolute_Error"]
                .median()
            ),
            "Severe_Error_Fraction": np.mean(
                tail_sub["Absolute_Error"]
                > SEVERE_ERROR_THRESHOLD
            )
        })

tail_df = pd.DataFrame(
    tail_records
)

print(
    safe_round(
        tail_df
    ).to_string(index=False)
)

tail_df.to_csv(
    OUTPUT_DIR /
    "06_extreme_fitness_tail_analysis.csv",
    index=False
)


# =============================================================================
# 13. SEVERE ERROR ANALYSIS
# =============================================================================

print_section(
    "10/15 — SEVERE UNDER/OVER-PREDICTION ANALYSIS"
)

analysis_df["Severe_UnderPrediction"] = (
    analysis_df["Observed"]
    - analysis_df["Predicted"]
) > SEVERE_ERROR_THRESHOLD

analysis_df["Severe_OverPrediction"] = (
    analysis_df["Predicted"]
    - analysis_df["Observed"]
) > SEVERE_ERROR_THRESHOLD

analysis_df["Extreme_Error"] = (
    analysis_df["Absolute_Error"]
    > EXTREME_ERROR_THRESHOLD
)

severe_under = analysis_df[
    analysis_df["Severe_UnderPrediction"]
].copy()

severe_over = analysis_df[
    analysis_df["Severe_OverPrediction"]
].copy()

extreme_errors = analysis_df[
    analysis_df["Extreme_Error"]
].copy()

print(
    f"Severe under-predictions "
    f"(error > {SEVERE_ERROR_THRESHOLD}): "
    f"{len(severe_under):,} "
    f"({100 * len(severe_under) / len(analysis_df):.2f}%)"
)

print(
    f"Severe over-predictions "
    f"(error > {SEVERE_ERROR_THRESHOLD}): "
    f"{len(severe_over):,} "
    f"({100 * len(severe_over) / len(analysis_df):.2f}%)"
)

print(
    f"Extreme absolute errors "
    f"(>|{EXTREME_ERROR_THRESHOLD}|): "
    f"{len(extreme_errors):,} "
    f"({100 * len(extreme_errors) / len(analysis_df):.2f}%)"
)

severe_under.sort_values(
    "Absolute_Error",
    ascending=False
).to_csv(
    OUTPUT_DIR /
    "07_severe_under_predictions.csv",
    index=False
)

severe_over.sort_values(
    "Absolute_Error",
    ascending=False
).to_csv(
    OUTPUT_DIR /
    "08_severe_over_predictions.csv",
    index=False
)


# =============================================================================
# 14. WORST PREDICTIONS
# =============================================================================

print_section(
    "11/15 — WORST MUTATION-LEVEL PREDICTIONS"
)

worst_columns = [
    c for c in [
        "Row_Index",
        "Stable_Mutation_Key",
        "Target",
        "Architecture",
        "Model",
        "Outer_Fold",
        "Observed",
        "Predicted",
        "Residual",
        "Best_Inner_R2",
        "Absolute_Error",
        "Error_PredMinusObs"
    ]
    if c in analysis_df.columns
]

worst_df = (
    analysis_df
    .sort_values(
        "Absolute_Error",
        ascending=False
    )
    .head(N_WORST)
    [worst_columns]
)

print(
    safe_round(
        worst_df.head(25)
    ).to_string(index=False)
)

worst_df.to_csv(
    OUTPUT_DIR /
    "09_worst_250_predictions.csv",
    index=False
)


# =============================================================================
# 15. SPECIFIC AUDIT OF THE EXTREME EXAMPLES
# =============================================================================

print_section(
    "12/15 — DIRECT AUDIT OF EXTREME LOWER-TAIL PREDICTIONS"
)

# These are the observations most similar to the problematic examples:
#
# Observed approximately -3.5 to -4.0
# Predicted close to zero
#
# The exact thresholds are deliberately broad so that the analysis does not
# depend on any single manually selected row.

extreme_lower_examples = analysis_df[
    (
        analysis_df["Observed"] <= -3.0
    )
    &
    (
        analysis_df["Predicted"] >= -0.5
    )
].copy()

print(
    "Rows with Observed <= -3.0 and "
    "Predicted >= -0.5:"
)

print(
    f"  N = {len(extreme_lower_examples):,}"
)

if len(extreme_lower_examples) > 0:

    diagnostic_columns = [
        c for c in [
            "Row_Index",
            "Stable_Mutation_Key",
            "Target",
            "Architecture",
            "Model",
            "Outer_Fold",
            "Observed",
            "Predicted",
            "Residual",
            "Best_Inner_R2",
            "Absolute_Error"
        ]
        if c in extreme_lower_examples.columns
    ]

    print(
        safe_round(
            extreme_lower_examples
            .sort_values(
                "Absolute_Error",
                ascending=False
            )
            .head(50)
            [diagnostic_columns]
        ).to_string(index=False)
    )

    extreme_lower_examples[
        diagnostic_columns
    ].sort_values(
        "Absolute_Error",
        ascending=False
    ).to_csv(
        OUTPUT_DIR /
        "10_extreme_lower_tail_examples.csv",
        index=False
    )


# =============================================================================
# 16. MODEL A VS MODEL B DIRECT COMPARISON
# =============================================================================

print_section(
    "13/15 — MODEL A VS MODEL B DIRECT OOF COMPARISON"
)

architecture_summary_records = []

architectures = sorted(
    analysis_df["Architecture"]
    .dropna()
    .unique()
)

for target, target_sub in analysis_df.groupby(
    "Target",
    sort=True
):

    arch_results = {}

    for architecture in architectures:

        sub = target_sub[
            target_sub["Architecture"]
            == architecture
        ]

        if len(sub) == 0:
            continue

        arch_results[
            architecture
        ] = calculate_metrics(sub)

    record = {
        "Target": target
    }

    for architecture, metrics in arch_results.items():

        prefix = architecture.replace(
            "Model_",
            ""
        )

        record[
            f"{prefix}_R2"
        ] = metrics["R2"]

        record[
            f"{prefix}_RMSE"
        ] = metrics["RMSE"]

        record[
            f"{prefix}_MAE"
        ] = metrics["MAE"]

        record[
            f"{prefix}_Spearman"
        ] = metrics["Spearman_rho"]

        record[
            f"{prefix}_Prediction_SD_Ratio"
        ] = metrics[
            "Prediction_SD_to_Observed_SD"
        ]

    # Explicit A-B deltas if both architectures exist.
    if (
        "Model_A_No_Position" in arch_results
        and
        "Model_B_With_Position" in arch_results
    ):

        a = arch_results[
            "Model_A_No_Position"
        ]

        b = arch_results[
            "Model_B_With_Position"
        ]

        record["Delta_R2_B_minus_A"] = (
            b["R2"] - a["R2"]
        )

        record["Delta_RMSE_B_minus_A"] = (
            b["RMSE"] - a["RMSE"]
        )

        record["Delta_MAE_B_minus_A"] = (
            b["MAE"] - a["MAE"]
        )

        record["Delta_Spearman_B_minus_A"] = (
            b["Spearman_rho"]
            - a["Spearman_rho"]
        )

    architecture_summary_records.append(
        record
    )

architecture_summary_df = pd.DataFrame(
    architecture_summary_records
)

print(
    safe_round(
        architecture_summary_df
    ).to_string(index=False)
)

architecture_summary_df.to_csv(
    OUTPUT_DIR /
    "11_ModelA_vs_ModelB_direct_comparison.csv",
    index=False
)


# =============================================================================
# 17. TARGET-WISE BIAS AND DISPERSION
# =============================================================================

print_section(
    "14/15 — TARGET-WISE BIAS, DISPERSION, AND CALIBRATION"
)

target_dispersion_records = []

for target, sub in analysis_df.groupby(
    "Target",
    sort=True
):

    # Combine both architectures only for a descriptive target-level view.
    # Architecture-specific results remain in the main performance table.
    observed_sd = sub["Observed"].std(
        ddof=1
    )

    predicted_sd = sub["Predicted"].std(
        ddof=1
    )

    calibration = linregress(
        sub["Predicted"],
        sub["Observed"]
    )

    target_dispersion_records.append({
        "Target": target,
        "N": len(sub),
        "Observed_Mean": sub["Observed"].mean(),
        "Predicted_Mean": sub["Predicted"].mean(),
        "Bias_PredMinusObs": (
            sub["Predicted"]
            - sub["Observed"]
        ).mean(),
        "Observed_SD": observed_sd,
        "Predicted_SD": predicted_sd,
        "Prediction_SD_Ratio": (
            predicted_sd / observed_sd
            if observed_sd > 0
            else np.nan
        ),
        "Calibration_Slope": calibration.slope,
        "Calibration_Intercept": calibration.intercept,
        "Calibration_R2": calibration.rvalue ** 2
    })

target_dispersion_df = pd.DataFrame(
    target_dispersion_records
)

print(
    safe_round(
        target_dispersion_df
    ).to_string(index=False)
)

target_dispersion_df.to_csv(
    OUTPUT_DIR /
    "12_target_level_bias_dispersion_calibration.csv",
    index=False
)


# =============================================================================
# 18. MACRO-AVERAGED PERFORMANCE
# =============================================================================

print_section(
    "15/15 — MACRO-AVERAGED TARGET PERFORMANCE"
)

macro_records = []

for architecture in architectures:

    arch_df = target_arch_df[
        target_arch_df["Architecture"]
        == architecture
    ].copy()

    if len(arch_df) == 0:
        continue

    macro_records.append({
        "Architecture": architecture,
        "Number_of_Targets": len(arch_df),
        "MacroMean_R2": arch_df["R2"].mean(),
        "MacroMean_RMSE": arch_df["RMSE"].mean(),
        "MacroMean_MAE": arch_df["MAE"].mean(),
        "MacroMean_Pearson": arch_df["Pearson_r"].mean(),
        "MacroMean_Spearman": arch_df["Spearman_rho"].mean(),
        "MacroMean_SD_Ratio": (
            arch_df[
                "Prediction_SD_to_Observed_SD"
            ].mean()
        ),
        "MacroMean_Bias": (
            arch_df[
                "Mean_Bias_PredMinusObs"
            ].mean()
        ),
        "MacroMean_Calibration_Slope": (
            arch_df[
                "Calibration_Slope_ObsOnPred"
            ].mean()
        )
    })

macro_df = pd.DataFrame(
    macro_records
)

print(
    safe_round(
        macro_df
    ).to_string(index=False)
)

macro_df.to_csv(
    OUTPUT_DIR /
    "13_macro_averaged_performance.csv",
    index=False
)


# =============================================================================
# 19. FIGURE 1 — OBSERVED VS PREDICTED
# =============================================================================

print_section(
    "FIGURE 1 — OBSERVED VS PREDICTED"
)

fig, ax = plt.subplots(
    figsize=(8.5, 7.0)
)

# Plot only a random subset when necessary to keep rendering efficient.
plot_df = analysis_df

if len(plot_df) > 30000:

    plot_df = plot_df.sample(
        30000,
        random_state=RANDOM_STATE
    )

ax.scatter(
    plot_df["Observed"],
    plot_df["Predicted"],
    s=8,
    alpha=0.18,
    linewidths=0
)

min_value = min(
    analysis_df["Observed"].min(),
    analysis_df["Predicted"].min()
)

max_value = max(
    analysis_df["Observed"].max(),
    analysis_df["Predicted"].max()
)

ax.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    linewidth=1.5,
    label="Identity"
)

ax.set_xlabel(
    "Observed fitness"
)

ax.set_ylabel(
    "Predicted fitness"
)

ax.set_title(
    "VIM-2 Step 4 — Pooled Position-Aware OOF Predictions"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "01_observed_vs_predicted_pooled.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 20. FIGURE 2 — TARGET-SPECIFIC CALIBRATION
# =============================================================================

fig, ax = plt.subplots(
    figsize=(9.0, 7.0)
)

for target, sub in analysis_df.groupby(
    "Target",
    sort=True
):

    metrics = calculate_metrics(sub)

    x = np.linspace(
        sub["Predicted"].min(),
        sub["Predicted"].max(),
        100
    )

    y = (
        metrics[
            "Calibration_Slope_ObsOnPred"
        ] * x
        +
        metrics[
            "Calibration_Intercept_ObsOnPred"
        ]
    )

    ax.plot(
        x,
        y,
        linewidth=1.5,
        label=target
    )

ax.axline(
    (0, 0),
    slope=1,
    linestyle="--",
    linewidth=1.2,
    label="Ideal calibration"
)

ax.set_xlabel(
    "Predicted fitness"
)

ax.set_ylabel(
    "Observed fitness"
)

ax.set_title(
    "Target-Level Calibration"
)

ax.legend(
    fontsize=7,
    frameon=False,
    ncol=2
)

ax.grid(
    alpha=0.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "02_target_specific_calibration.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 21. FIGURE 3 — PREDICTION RANGE COMPRESSION
# =============================================================================

fig, ax = plt.subplots(
    figsize=(9.0, 6.5)
)

compression_plot = target_arch_df.copy()

labels = (
    compression_plot["Target"]
    + "\n"
    + compression_plot["Architecture"]
    .str.replace(
        "Model_",
        "",
        regex=False
    )
)

ax.bar(
    np.arange(len(compression_plot)),
    compression_plot[
        "Prediction_SD_to_Observed_SD"
    ]
)

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1.3,
    label="No compression"
)

ax.set_xticks(
    np.arange(len(compression_plot))
)

ax.set_xticklabels(
    labels,
    rotation=75,
    ha="right",
    fontsize=7
)

ax.set_ylabel(
    "Predicted SD / Observed SD"
)

ax.set_title(
    "Prediction Range Compression by Target and Architecture"
)

ax.legend(
    frameon=False
)

ax.grid(
    axis="y",
    alpha=0.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "03_prediction_range_compression.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 22. FIGURE 4 — ERROR ACROSS FITNESS QUANTILES
# =============================================================================

fig, ax = plt.subplots(
    figsize=(9.0, 6.5)
)

# Average across targets for each architecture and quantile.
quantile_summary = (
    quantile_df
    .groupby(
        [
            "Architecture",
            "Fitness_Quantile"
        ],
        as_index=False
    )
    .agg(
        MAE=("MAE", "mean"),
        Severe_Error_Fraction=(
            "Severe_Error_Fraction",
            "mean"
        )
    )
)

for architecture, sub in quantile_summary.groupby(
    "Architecture",
    sort=True
):

    ax.plot(
        sub["Fitness_Quantile"],
        sub["MAE"],
        marker="o",
        linewidth=1.8,
        label=architecture
    )

ax.set_xlabel(
    "Observed fitness decile"
)

ax.set_ylabel(
    "Mean absolute error"
)

ax.set_title(
    "Prediction Error Across the Observed Fitness Distribution"
)

ax.legend(
    frameon=False
)

ax.grid(
    alpha=0.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "04_MAE_across_fitness_quantiles.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 23. FIGURE 5 — OUTER-FOLD STABILITY
# =============================================================================

fig, ax = plt.subplots(
    figsize=(10.0, 6.5)
)

fold_plot = (
    fold_df
    .groupby(
        [
            "Target",
            "Architecture"
        ],
        as_index=False
    )
    .agg(
        Mean_R2=("R2", "mean"),
        SD_R2=("R2", "std")
    )
)

x = np.arange(
    len(fold_plot)
)

ax.errorbar(
    x,
    fold_plot["Mean_R2"],
    yerr=fold_plot["SD_R2"],
    fmt="o",
    capsize=4
)

ax.set_xticks(x)

ax.set_xticklabels(
    (
        fold_plot["Target"]
        + "\n"
        + fold_plot["Architecture"]
    ),
    rotation=75,
    ha="right",
    fontsize=7
)

ax.set_ylabel(
    "Outer-fold R²"
)

ax.set_title(
    "Position-Aware Outer-Fold Stability"
)

ax.grid(
    axis="y",
    alpha=0.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "05_outer_fold_R2_stability.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 24. FIGURE 6 — RESIDUAL VS OBSERVED
# =============================================================================

fig, ax = plt.subplots(
    figsize=(8.5, 6.5)
)

plot_df = analysis_df

if len(plot_df) > 30000:

    plot_df = plot_df.sample(
        30000,
        random_state=RANDOM_STATE
    )

ax.scatter(
    plot_df["Observed"],
    plot_df["Error_PredMinusObs"],
    s=8,
    alpha=0.18,
    linewidths=0
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.3
)

ax.axhline(
    SEVERE_ERROR_THRESHOLD,
    linestyle=":",
    linewidth=1.0
)

ax.axhline(
    -SEVERE_ERROR_THRESHOLD,
    linestyle=":",
    linewidth=1.0
)

ax.set_xlabel(
    "Observed fitness"
)

ax.set_ylabel(
    "Prediction error (Predicted − Observed)"
)

ax.set_title(
    "Residual Structure Across Observed Fitness"
)

ax.grid(
    alpha=0.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "06_residual_vs_observed.png",
    dpi=400,
    bbox_inches="tight"
)

plt.close()


# =============================================================================
# 25. AUTOMATED SCIENTIFIC DIAGNOSTIC FLAGS
# =============================================================================

print_section(
    "AUTOMATED SCIENTIFIC DIAGNOSTIC SUMMARY"
)

overall_r2 = global_metrics[
    "R2"
]

overall_sd_ratio = global_metrics[
    "Prediction_SD_to_Observed_SD"
]

overall_calibration_slope = global_metrics[
    "Calibration_Slope_ObsOnPred"
]

severe_fraction = (
    len(severe_under)
    /
    len(analysis_df)
)

diagnostic_flags = []

# -------------------------------------------------------------------------
# Signal
# -------------------------------------------------------------------------

if overall_r2 >= 0.50:
    diagnostic_flags.append(
        "STRONG_AGGREGATE_SIGNAL: "
        "pooled OOF R² is >= 0.50."
    )

elif overall_r2 >= 0.25:
    diagnostic_flags.append(
        "MODERATE_AGGREGATE_SIGNAL: "
        "pooled OOF R² is between 0.25 and 0.50."
    )

else:
    diagnostic_flags.append(
        "WEAK_AGGREGATE_SIGNAL: "
        "pooled OOF R² is < 0.25."
    )


# -------------------------------------------------------------------------
# Prediction compression
# -------------------------------------------------------------------------

if overall_sd_ratio < 0.80:

    diagnostic_flags.append(
        "PREDICTION_COMPRESSION: "
        "global predicted SD is substantially smaller "
        "than observed SD."
    )

elif overall_sd_ratio < 0.90:

    diagnostic_flags.append(
        "MILD_PREDICTION_COMPRESSION: "
        "predicted range is somewhat narrower "
        "than observed range."
    )

else:

    diagnostic_flags.append(
        "NO_MAJOR_GLOBAL_COMPRESSION: "
        "predicted and observed SDs are relatively similar."
    )


# -------------------------------------------------------------------------
# Calibration
# -------------------------------------------------------------------------

if (
    0.90
    <= overall_calibration_slope
    <= 1.10
):

    diagnostic_flags.append(
        "GLOBAL_CALIBRATION_NEAR_IDENTITY: "
        "global calibration slope is close to 1."
    )

else:

    diagnostic_flags.append(
        "GLOBAL_CALIBRATION_DEVIATION: "
        "global calibration slope differs materially from 1."
    )


# -------------------------------------------------------------------------
# Severe error fraction
# -------------------------------------------------------------------------

if severe_fraction >= 0.10:

    diagnostic_flags.append(
        "HIGH_SEVERE_ERROR_FRACTION: "
        ">=10% of predictions differ from observations "
        "by more than 2 fitness units."
    )

elif severe_fraction >= 0.05:

    diagnostic_flags.append(
        "MODERATE_SEVERE_ERROR_FRACTION: "
        "5–10% of predictions differ from observations "
        "by more than 2 fitness units."
    )

else:

    diagnostic_flags.append(
        "LOW_SEVERE_ERROR_FRACTION: "
        "<5% of predictions differ from observations "
        "by more than 2 fitness units."
    )


for flag in diagnostic_flags:
    print("•", flag)


# =============================================================================
# 26. FINAL HUMAN-READABLE REPORT
# =============================================================================

report_lines = []

report_lines.append(
    "VIM-2 STEP 4 — PROFESSIONAL OOF FORENSIC AUDIT"
)

report_lines.append(
    "=" * 80
)

report_lines.append(
    f"Input: {INPUT_FILE}"
)

report_lines.append(
    f"Valid OOF rows: {len(analysis_df):,}"
)

report_lines.append(
    ""
)

report_lines.append(
    "GLOBAL OOF PERFORMANCE"
)

report_lines.append(
    f"R²                         : "
    f"{overall_r2:.4f}"
)

report_lines.append(
    f"RMSE                       : "
    f"{global_metrics['RMSE']:.4f}"
)

report_lines.append(
    f"MAE                        : "
    f"{global_metrics['MAE']:.4f}"
)

report_lines.append(
    f"Pearson r                  : "
    f"{global_metrics['Pearson_r']:.4f}"
)

report_lines.append(
    f"Spearman rho               : "
    f"{global_metrics['Spearman_rho']:.4f}"
)

report_lines.append(
    f"Observed SD                : "
    f"{global_metrics['Observed_SD']:.4f}"
)

report_lines.append(
    f"Predicted SD               : "
    f"{global_metrics['Predicted_SD']:.4f}"
)

report_lines.append(
    f"Prediction SD ratio        : "
    f"{overall_sd_ratio:.4f}"
)

report_lines.append(
    f"Mean prediction bias       : "
    f"{global_metrics['Mean_Bias_PredMinusObs']:.4f}"
)

report_lines.append(
    f"Calibration slope          : "
    f"{overall_calibration_slope:.4f}"
)

report_lines.append(
    f"Calibration intercept      : "
    f"{global_metrics['Calibration_Intercept_ObsOnPred']:.4f}"
)

report_lines.append(
    ""
)

report_lines.append(
    "ERROR STRUCTURE"
)

report_lines.append(
    f"Severe under-predictions   : "
    f"{len(severe_under):,} "
    f"({100 * len(severe_under) / len(analysis_df):.2f}%)"
)

report_lines.append(
    f"Severe over-predictions    : "
    f"{len(severe_over):,} "
    f"({100 * len(severe_over) / len(analysis_df):.2f}%)"
)

report_lines.append(
    f"Extreme errors > 3 units   : "
    f"{len(extreme_errors):,} "
    f"({100 * len(extreme_errors) / len(analysis_df):.2f}%)"
)

report_lines.append(
    ""
)

report_lines.append(
    "EXTREME LOWER-TAIL DIAGNOSTIC"
)

report_lines.append(
    "Observed <= -3.0 and Predicted >= -0.5:"
)

report_lines.append(
    f"  N = {len(extreme_lower_examples):,}"
)

report_lines.append(
    ""
)

report_lines.append(
    "AUTOMATED INTERPRETATION FLAGS"
)

report_lines.extend(
    f"- {flag}"
    for flag in diagnostic_flags
)

report_lines.append(
    ""
)

report_lines.append(
    "IMPORTANT INTERPRETATION"
)

report_lines.append(
    "The pooled global metrics should not be interpreted as "
    "mutation-level accuracy for every variant."
)

report_lines.append(
    "Target-specific OOF performance, calibration, prediction "
    "dispersion, and tail-specific error are required to determine "
    "whether the model generalizes across individual phenotypes."
)

report_lines.append(
    "Best_Inner_R2 is an inner-model selection metric and must not "
    "be interpreted as the performance of an individual OOF prediction."
)

report_lines.append(
    ""
)

report_lines.append(
    f"Output directory: {OUTPUT_DIR}"
)

report_path = (
    OUTPUT_DIR /
    "VIM2_Step4_OOF_Forensic_Audit_Report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(report_lines)
    )


# =============================================================================
# 27. FINAL CONSOLE SUMMARY
# =============================================================================

print_section(
    "AUDIT COMPLETE"
)

print(
    f"Input file:"
    f"\n  {INPUT_FILE}"
)

print(
    f"\nOutput directory:"
    f"\n  {OUTPUT_DIR}"
)

print(
    "\nMain outputs:"
)

for filename in [
    "01_data_integrity_QC.csv",
    "02_global_OOF_performance.csv",
    "03_Target_Architecture_performance.csv",
    "04_outer_fold_performance.csv",
    "05_observed_fitness_quantile_error.csv",
    "06_extreme_fitness_tail_analysis.csv",
    "07_severe_under_predictions.csv",
    "08_severe_over_predictions.csv",
    "09_worst_250_predictions.csv",
    "10_extreme_lower_tail_examples.csv",
    "11_ModelA_vs_ModelB_direct_comparison.csv",
    "12_target_level_bias_dispersion_calibration.csv",
    "13_macro_averaged_performance.csv",
    "VIM2_Step4_OOF_Forensic_Audit_Report.txt"
]:

    print(f"  ✓ {filename}")

print(
    "\nFigures:"
)

for filename in [
    "01_observed_vs_predicted_pooled.png",
    "02_target_specific_calibration.png",
    "03_prediction_range_compression.png",
    "04_MAE_across_fitness_quantiles.png",
    "05_outer_fold_R2_stability.png",
    "06_residual_vs_observed.png"
]:

    print(f"  ✓ {filename}")

print(
    "\nIMPORTANT:"
)

print(
    "This audit only evaluates frozen Step 4 OOF predictions."
)

print(
    "It does NOT retrain, optimize, refit, or modify any model."
)

print(
    "\nNext step: inspect the target-specific tables and the "
    "extreme lower-tail examples before making manuscript-level "
    "claims about mutation-level predictive accuracy."
)


1/15 — LOADING FROZEN STEP 4 OOF PREDICTIONS
Input file : /content/VIM2_Step4_Final_OOF_Predictions.csv
Loaded shape: (90102, 10)

Columns:
  - Row_Index
  - Stable_Mutation_Key
  - Target
  - Architecture
  - Model
  - Outer_Fold
  - Observed
  - Predicted
  - Residual
  - Best_Inner_R2

2/15 — VALIDATING REQUIRED COLUMNS
Required columns: PASS

Optional columns:
  Row_Index: YES
  Stable_Mutation_Key: YES
  Model: YES
  Residual: YES
  Best_Inner_R2: YES

3/15 — PREPARING NUMERICAL FIELDS
Total rows              : 90,102
Valid prediction rows   : 90,102
Excluded rows            : 0

4/15 — STRUCTURAL QUALITY CONTROL
                             Metric  Value
                         Total_rows  90102
              Valid_prediction_rows  90102
                   Missing_Observed      0
                  Missing_Predicted      0
                     Unique_Targets      9
               Unique_Architectures      2
                 Unique_Outer_Folds      5
Potential_duplicate_predictio

In [ ]:
# @title

# ============================================================
# DOWNLOAD /content/VIM2_Step4_OOF_FORENSIC_AUDIT analysis OUTPUT DIRECTORY AS A ZIP ARCHIVE
# ============================================================

import os
import shutil
from google.colab import files

# Source directory
source_dir = "/content/VIM2_Step4_OOF_FORENSIC_AUDIT"

# Output ZIP archive
zip_base = "/content/VIM2_Step4_OOF_FORENSIC_AUDIT"
zip_file = f"{zip_base}.zip"

# Validate source directory
if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"Source directory not found: {source_dir}"
    )

# Remove previous ZIP if it exists
if os.path.exists(zip_file):
    os.remove(zip_file)

# Create ZIP archive
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir="/content",
    base_dir="VIM2_Step4_OOF_FORENSIC_AUDIT"
)

# Validate ZIP creation
if not os.path.isfile(zip_file):
    raise RuntimeError("ZIP archive was not created successfully.")

# Report archive size
size_mb = os.path.getsize(zip_file) / (1024 ** 2)

print("=" * 70)
print("/content/VIM2_Step4_OOF_FORENSIC_AUDIT DOWNLOAD PACKAGE")
print("=" * 70)
print(f"Source directory : {source_dir}")
print(f"ZIP archive      : {zip_file}")
print(f"Archive size     : {size_mb:.2f} MB")
print("=" * 70)
print("ZIP archive created successfully.")
print("Starting download...")

# Download ZIP
files.download(zip_file)

/content/VIM2_Step4_OOF_FORENSIC_AUDIT DOWNLOAD PACKAGE
Source directory : /content/VIM2_Step4_OOF_FORENSIC_AUDIT
ZIP archive      : /content/VIM2_Step4_OOF_FORENSIC_AUDIT.zip
Archive size     : 4.32 MB
ZIP archive created successfully.
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>